In [ ]:
import os

print("2000-image ZIP exists:",
      os.path.exists("/content/kaggle_dataset_2000.zip"))

print("Original LIDC ZIP exists:",
      os.path.exists("/content/kagl_lidc_idri.zip"))

2000-image ZIP exists: True
Original LIDC ZIP exists: True


In [ ]:
# ============================================================
# MATCH 2,000-IMAGE SUBSET TO ORIGINAL LIDC PATIENT/NODULE DATA
# ============================================================

import zipfile
import hashlib
import os
import pandas as pd
from collections import defaultdict

SUBSET_ZIP = "/content/kaggle_dataset_2000.zip"
ORIGINAL_ZIP = "/content/kagl_lidc_idri.zip"

# ------------------------------------------------------------
# 1. VERIFY ZIP FILES
# ------------------------------------------------------------

assert os.path.exists(SUBSET_ZIP), f"Missing subset ZIP: {SUBSET_ZIP}"
assert os.path.exists(ORIGINAL_ZIP), f"Missing original ZIP: {ORIGINAL_ZIP}"

print("ZIP files found successfully.\n")

# ------------------------------------------------------------
# 2. HELPER: SHA-256 HASH
# ------------------------------------------------------------

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

# ------------------------------------------------------------
# 3. INDEX ALL ORIGINAL NATIVE IMAGE FILES
# ------------------------------------------------------------

print("=" * 80)
print("STEP 1: INDEXING ORIGINAL LIDC IMAGE FILES")
print("=" * 80

original_hash_index = defaultdict(list)

with zipfile.ZipFile(ORIGINAL_ZIP, "r") as z:
    original_png_names = [
        name for name in z.namelist()
        if name.lower().endswith(".png")
        and "/images/" in name
    ]

    print(f"Original image files found: {len(original_png_names):,}")

    for i, name in enumerate(original_png_names, start=1):

        data = z.read(name)
        file_hash = sha256_bytes(data)

        original_hash_index[file_hash].append(name)

        if i % 5000 == 0:
            print(f"Indexed {i:,} / {len(original_png_names):,}")

print("\nOriginal image index complete.")

# ------------------------------------------------------------
# 4. READ SUBSET ZIP
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 2: MATCHING YOUR 2,000-IMAGE SUBSET")
print("=" * 80)

with zipfile.ZipFile(SUBSET_ZIP, "r") as z:
    subset_png_names = [
        name for name in z.namelist()
        if name.lower().endswith(".png")
    ]

    print(f"Subset PNG files found: {len(subset_png_names):,}")

    # Group by flattened nodule folder
    subset_groups = defaultdict(list)

    for name in subset_png_names:
        clean_name = name.replace("\\", "/")
        parts = clean_name.split("/")

        # Expected structure:
        # nodule_001/slice-0.png
        if len(parts) >= 2:
            subset_nodule = parts[0]
            subset_groups[subset_nodule].append(name)

    print(f"Subset nodule folders found: {len(subset_groups)}")

    # --------------------------------------------------------
    # 5. MATCH EACH SUBSET NODULE
    # --------------------------------------------------------

    mapping_rows = []

    unmatched_images = []
    ambiguous_images = []

    for subset_nodule in sorted(subset_groups.keys()):

        matched_native_paths = []

        for subset_path in subset_groups[subset_nodule]:

            subset_data = z.read(subset_path)
            subset_hash = sha256_bytes(subset_data)

            candidates = original_hash_index.get(subset_hash, [])

            if len(candidates) == 1:
                matched_native_paths.append(candidates[0])

            elif len(candidates) == 0:
                unmatched_images.append(
                    (subset_nodule, subset_path)
                )

            else:
                ambiguous_images.append(
                    (subset_nodule, subset_path, candidates)
                )

        # ----------------------------------------------------
        # Determine which native patient/nodule folder
        # contains the matched images
        # ----------------------------------------------------

        native_folder_counts = defaultdict(int)

        for native_path in matched_native_paths:

            # Example native path:
            # LIDC-IDRI-slices/
            # LIDC-IDRI-0001/
            # nodule-0/
            # images/
            # 0_0_0.png

            parts = native_path.replace("\\", "/").split("/")

            # Locate "/images/"
            try:
                images_idx = parts.index("images")
            except ValueError:
                continue

            if images_idx < 2:
                continue

            patient_id = parts[images_idx - 2]
            native_nodule_id = parts[images_idx - 1]

            native_folder = (patient_id, native_nodule_id)

            native_folder_counts[native_folder] += 1

        # Sort candidate native folders by number of matched slices
        ranked = sorted(
            native_folder_counts.items(),
            key=lambda x: x[1],
            reverse=True
        )

        total_subset_slices = len(subset_groups[subset_nodule])
        total_matched = len(matched_native_paths)

        if len(ranked) == 0:
            mapping_status = "UNMATCHED"
            patient_id = None
            native_nodule_id = None
            best_match_count = 0
            second_match_count = 0

        else:
            (patient_id, native_nodule_id), best_match_count = ranked[0]

            second_match_count = (
                ranked[1][1] if len(ranked) > 1 else 0
            )

            if total_matched == total_subset_slices and len(ranked) == 1:
                mapping_status = "EXACT_UNIQUE_MATCH"

            elif total_matched == total_subset_slices and best_match_count > second_match_count:
                mapping_status = "MATCHED_BUT_REVIEW"

            else:
                mapping_status = "PARTIAL_MATCH"

        mapping_rows.append({
            "subset_nodule_id": subset_nodule,
            "subset_slice_count": total_subset_slices,
            "matched_slice_count": total_matched,
            "patient_id": patient_id,
            "native_nodule_id": native_nodule_id,
            "best_match_count": best_match_count,
            "second_best_match_count": second_match_count,
            "status": mapping_status
        })

# ------------------------------------------------------------
# 6. SAVE MAPPING
# ------------------------------------------------------------

mapping_df = pd.DataFrame(mapping_rows)

OUTPUT_PATH = "/content/nodule_to_lidc_mapping.csv"
mapping_df.to_csv(OUTPUT_PATH, index=False)

# ------------------------------------------------------------
# 7. REPORT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MAPPING RESULTS")
print("=" * 80)

print(f"Subset nodules:                 {len(mapping_df)}")
print(
    "Exact unique matches:            "
    f"{(mapping_df['status'] == 'EXACT_UNIQUE_MATCH').sum()}"
)
print(
    "Matched but need review:         "
    f"{(mapping_df['status'] == 'MATCHED_BUT_REVIEW').sum()}"
)
print(
    "Partial matches:                 "
    f"{(mapping_df['status'] == 'PARTIAL_MATCH').sum()}"
)
print(
    "Completely unmatched:            "
    f"{(mapping_df['status'] == 'UNMATCHED').sum()}"
)

print("\nMapping status counts:")
print(mapping_df["status"].value_counts())

print("\nFirst 20 mappings:")
print(mapping_df.head(20).to_string(index=False))

print("\nMapping saved to:")
print(OUTPUT_PATH)

print("=" * 80)

# ------------------------------------------------------------
# 8. REPORT UNMATCHED / AMBIGUOUS IMAGES
# ------------------------------------------------------------

print("\nUnmatched individual images:", len(unmatched_images))
print("Ambiguous individual images:", len(ambiguous_images))

if unmatched_images:
    print("\nFirst 20 unmatched images:")
    for item in unmatched_images[:20]:
        print(item)

ZIP files found successfully.

STEP 1: INDEXING ORIGINAL LIDC IMAGE FILES
Original image files found: 15,548
Indexed 5,000 / 15,548
Indexed 10,000 / 15,548
Indexed 15,000 / 15,548

Original image index complete.

STEP 2: MATCHING YOUR 2,000-IMAGE SUBSET
Subset PNG files found: 2,004
Subset nodule folders found: 327

MAPPING RESULTS
Subset nodules:                 327
Exact unique matches:            327
Matched but need review:         0
Partial matches:                 0
Completely unmatched:            0

Mapping status counts:
status
EXACT_UNIQUE_MATCH    327
Name: count, dtype: int64

First 20 mappings:
subset_nodule_id  subset_slice_count  matched_slice_count     patient_id native_nodule_id  best_match_count  second_best_match_count             status
      nodule_001                   6                    6 LIDC-IDRI-0121         nodule-1                 6                        0 EXACT_UNIQUE_MATCH
      nodule_002                   6                    6 LIDC-IDRI-0158         

In [ ]:
import os

print("=" * 70)
print("SEARCHING FOR LIDC XML ANNOTATIONS")
print("=" * 70)

xml_files = []

for root, dirs, files in os.walk("/content"):
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
        and d != "__MACOSX"
    ]

    for file in files:
        if file.lower().endswith(".xml"):
            xml_files.append(os.path.join(root, file))

print(f"XML files found: {len(xml_files)}")

for path in xml_files[:30]:
    print(path)

if len(xml_files) == 0:
    print("\nNo LIDC XML annotation files were found in /content.")

SEARCHING FOR LIDC XML ANNOTATIONS
XML files found: 0

No LIDC XML annotation files were found in /content.


In [ ]:
import pandas as pd

mapping_path = "/content/nodule_to_lidc_mapping.csv"

df = pd.read_csv(mapping_path)

print("=" * 70)
print("PATIENT COVERAGE OF YOUR 327 NODULE SUBSET")
print("=" * 70)

print(f"Total nodule folders: {len(df)}")
print(f"Unique LIDC patients: {df['patient_id'].nunique()}")

print("\nFirst 20 unique patients:")
for pid in sorted(df["patient_id"].unique())[:20]:
    print(pid)

print("\nNodules per patient:")
print(df.groupby("patient_id").size().describe())

PATIENT COVERAGE OF YOUR 327 NODULE SUBSET
Total nodule folders: 327
Unique LIDC patients: 249

First 20 unique patients:
LIDC-IDRI-0003
LIDC-IDRI-0005
LIDC-IDRI-0008
LIDC-IDRI-0010
LIDC-IDRI-0012
LIDC-IDRI-0014
LIDC-IDRI-0015
LIDC-IDRI-0016
LIDC-IDRI-0018
LIDC-IDRI-0021
LIDC-IDRI-0027
LIDC-IDRI-0031
LIDC-IDRI-0040
LIDC-IDRI-0041
LIDC-IDRI-0042
LIDC-IDRI-0044
LIDC-IDRI-0045
LIDC-IDRI-0046
LIDC-IDRI-0049
LIDC-IDRI-0055

Nodules per patient:
count    249.000000
mean       1.313253
std        0.733876
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        6.000000
dtype: float64


In [ ]:
# ============================================================
# STEP 1 — DOWNLOAD AND INSPECT OFFICIAL LIDC XML ANNOTATIONS
# ============================================================

import os
import urllib.request
import zipfile

xml_url = "https://wiki.cancerimagingarchive.net/download/attachments/1966254/LIDC-XML-only.zip"

zip_path = "/content/LIDC-XML-only.zip"
extract_dir = "/content/lidc_xml_annotations"

print("=" * 80)
print("DOWNLOADING OFFICIAL LIDC-IDRI XML ANNOTATIONS")
print("=" * 80)

if not os.path.exists(zip_path):
    print("Downloading...")
    urllib.request.urlretrieve(xml_url, zip_path)
    print("Download completed.")
else:
    print("XML ZIP already exists. Using existing file.")

print(f"\nZIP path: {zip_path}")
print(f"ZIP exists: {os.path.exists(zip_path)}")

# ------------------------------------------------------------
# Extract
# ------------------------------------------------------------

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()

print(f"Entries in XML archive: {len(names):,}")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

# ------------------------------------------------------------
# Locate XML files
# ------------------------------------------------------------

xml_files = []

for root, dirs, files in os.walk(extract_dir):
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
        and d != "__MACOSX"
    ]

    for file in files:
        if file.lower().endswith(".xml"):
            xml_files.append(os.path.join(root, file))

print(f"XML files extracted: {len(xml_files):,}")

print("\nFirst 20 XML paths:")
for path in xml_files[:20]:
    print(path)

print("=" * 80)

DOWNLOADING OFFICIAL LIDC-IDRI XML ANNOTATIONS
Downloading...
Download completed.

ZIP path: /content/LIDC-XML-only.zip
ZIP exists: True
Entries in XML archive: 1,326
XML files extracted: 1,319

First 20 XML paths:
/content/lidc_xml_annotations/161-resubmitted-correction-3-9-12.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/023.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/170.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/137.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/033.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/047.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/020.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/066.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/087.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/100.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/079.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/144.xml
/content/lidc_xml_annotations/tcia-lidc-xml/189/060.xml
/content/lidc_xml_annotations/tcia-lidc-xml/1

In [ ]:
# ============================================================
# STEP 2 — INSPECT ONE ACTUAL LIDC XML FILE
# ============================================================

import xml.etree.ElementTree as ET

if len(xml_files) == 0:
    raise RuntimeError("No XML files were extracted.")

sample_xml = xml_files[0]

print("=" * 80)
print("INSPECTING SAMPLE LIDC XML")
print("=" * 80)
print("File:")
print(sample_xml)

tree = ET.parse(sample_xml)
root = tree.getroot()

print("\nRoot tag:")
print(root.tag)

print("\nFirst-level children:")
for child in root:
    print(" ", child.tag)

# Find malignancy elements
malignancy_elements = []

for elem in root.iter():
    tag = elem.tag.lower().split("}")[-1]

    if tag == "malignancy":
        malignancy_elements.append(elem)

print("\nMalignancy elements found:", len(malignancy_elements))

for elem in malignancy_elements[:10]:
    print("  tag:", elem.tag, "| value:", elem.text)

# Search for patient/series identifiers
print("\nPossible identifiers found:")

for elem in root.iter():
    tag = elem.tag.lower().split("}")[-1]

    if any(term in tag for term in [
        "patient",
        "series",
        "study",
        "uid"
    ]):
        value = (elem.text or "").strip()
        if value:
            print(f"  {elem.tag}: {value}")

print("=" * 80)

INSPECTING SAMPLE LIDC XML
File:
/content/lidc_xml_annotations/161-resubmitted-correction-3-9-12.xml

Root tag:
{http://www.nih.gov}LidcReadMessage

First-level children:
  {http://www.nih.gov}ResponseHeader
  {http://www.nih.gov}readingSession
  {http://www.nih.gov}readingSession
  {http://www.nih.gov}readingSession
  {http://www.nih.gov}readingSession

Malignancy elements found: 2
  tag: {http://www.nih.gov}malignancy | value: 1
  tag: {http://www.nih.gov}malignancy | value: 3

Possible identifiers found:
  {http://www.nih.gov}SeriesInstanceUid: 1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094259402036602717327
  {http://www.nih.gov}StudyInstanceUID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.584233139051825667176600857752
  {http://www.nih.gov}imageSOP_UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.150739457477763063347777523734
  {http://www.nih.gov}imageSOP_UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.150739457477763063347777523734
  {http://www.nih.gov}imageSOP_UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.2150

In [ ]:
# ============================================================
# BUILD LIDC XML ANNOTATION INVENTORY
# ============================================================

import os
import xml.etree.ElementTree as ET
import pandas as pd

XML_DIR = "/content/lidc_xml_annotations"
OUTPUT = "/content/lidc_xml_inventory.csv"

records = []
xml_files = []

for root_dir, dirs, files in os.walk(XML_DIR):
    dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__MACOSX"]

    for file in files:
        if file.lower().endswith(".xml"):
            xml_files.append(os.path.join(root_dir, file))

print(f"XML files found: {len(xml_files)}")

for i, xml_path in enumerate(xml_files, start=1):

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Namespace-safe helper
        def local_name(tag):
            return tag.split("}")[-1]

        series_uids = []
        study_uids = []
        patient_ids = []
        malignancies = []
        nodule_count = 0

        for elem in root.iter():
            tag = local_name(elem.tag)
            value = (elem.text or "").strip()

            if tag == "SeriesInstanceUid" and value:
                series_uids.append(value)

            elif tag.lower() == "studyinstanceuid" and value:
                study_uids.append(value)

            elif tag.lower() == "patientid" and value:
                patient_ids.append(value)

            elif tag.lower() == "malignancy" and value:
                try:
                    malignancies.append(int(value))
                except ValueError:
                    pass

            elif tag == "unblindedReadNodule":
                nodule_count += 1

        records.append({
            "xml_path": xml_path,
            "xml_filename": os.path.basename(xml_path),
            "patient_id_in_xml": patient_ids[0] if patient_ids else "",
            "series_instance_uid": series_uids[0] if series_uids else "",
            "study_instance_uid": study_uids[0] if study_uids else "",
            "nodule_count": nodule_count,
            "malignancy_ratings": str(malignancies),
            "num_malignancy_values": len(malignancies)
        })

    except Exception as e:
        records.append({
            "xml_path": xml_path,
            "xml_filename": os.path.basename(xml_path),
            "patient_id_in_xml": "",
            "series_instance_uid": "",
            "study_instance_uid": "",
            "nodule_count": -1,
            "malignancy_ratings": "",
            "num_malignancy_values": -1,
            "parse_error": str(e)
        })

    if i % 200 == 0:
        print(f"Processed {i} / {len(xml_files)} XML files")

inventory = pd.DataFrame(records)

inventory.to_csv(OUTPUT, index=False)

print("\n" + "=" * 80)
print("LIDC XML INVENTORY SUMMARY")
print("=" * 80)

print(f"XML files successfully indexed: {len(inventory)}")
print(
    "Files with SeriesInstanceUid:",
    (inventory["series_instance_uid"] != "").sum()
)
print(
    "Files with malignancy values:",
    (inventory["num_malignancy_values"] > 0).sum()
)
print(
    "Total malignancy values extracted:",
    inventory.loc[
        inventory["num_malignancy_values"] > 0,
        "num_malignancy_values"
    ].sum()
)

print("\nExample inventory rows:")
print(
    inventory[
        [
            "xml_filename",
            "patient_id_in_xml",
            "series_instance_uid",
            "nodule_count",
            "malignancy_ratings"
        ]
    ].head(10).to_string(index=False)
)

print("\nSaved inventory:")
print(OUTPUT)
print("=" * 80)

XML files found: 1319
Processed 200 / 1319 XML files
Processed 400 / 1319 XML files
Processed 600 / 1319 XML files
Processed 800 / 1319 XML files
Processed 1000 / 1319 XML files
Processed 1200 / 1319 XML files

LIDC XML INVENTORY SUMMARY
XML files successfully indexed: 1319
Files with SeriesInstanceUid: 1036
Files with malignancy values: 901
Total malignancy values extracted: 7034

Example inventory rows:
                         xml_filename patient_id_in_xml                                              series_instance_uid  nodule_count                                                                   malignancy_ratings
161-resubmitted-correction-3-9-12.xml                   1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094259402036602717327            21                                                                               [1, 3]
                              023.xml                   1.3.6.1.4.1.14519.5.2.1.6279.6001.144943344795414353192059796098            32                 

In [ ]:
# ============================================================
# NEXT STEP: INSPECT XML NODULE / ANNOTATION IDENTIFIERS
# ============================================================

import os
import xml.etree.ElementTree as ET
import pandas as pd

XML_DIR = "/content/lidc_xml_annotations"
MAPPING_PATH = "/content/nodule_to_lidc_mapping.csv"

mapping_df = pd.read_csv(MAPPING_PATH)

print("=" * 90)
print("MAPPING + XML ANNOTATION IDENTIFIER INSPECTION")
print("=" * 90)

print(f"Subset nodules in mapping: {len(mapping_df)}")
print(f"Unique patients in mapping: {mapping_df['patient_id'].nunique()}")

# ------------------------------------------------------------
# Inspect several patients from our actual 327-nodule subset
# ------------------------------------------------------------

sample_patients = sorted(mapping_df["patient_id"].astype(str).unique())[:10]

print("\nSample mapped patients:")
for pid in sample_patients:
    print(" ", pid)

# ------------------------------------------------------------
# Search every XML for identifiers and annotation details
# ------------------------------------------------------------

def local_name(tag):
    return tag.split("}")[-1]

xml_records = []

for xml_path in sorted(
    [
        os.path.join(root, f)
        for root, _, files in os.walk(XML_DIR)
        for f in files
        if f.lower().endswith(".xml")
    ]
):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        series_uids = []
        study_uids = []
        patient_ids = []
        image_uids = []
        annotation_ids = []
        malignancies = []

        unblinded_count = 0

        for elem in root.iter():
            tag = local_name(elem.tag)
            value = (elem.text or "").strip()

            if not value:
                continue

            if tag == "SeriesInstanceUid":
                series_uids.append(value)

            elif tag.lower() == "studyinstanceuid":
                study_uids.append(value)

            elif tag.lower() == "patientid":
                patient_ids.append(value)

            elif tag.lower() == "imageSOP_UID".lower():
                image_uids.append(value)

            elif tag.lower() == "malignancy":
                try:
                    malignancies.append(int(value))
                except ValueError:
                    pass

            elif tag == "unblindedReadNodule":
                unblinded_count += 1

            # Capture likely annotation/reference identifiers
            elif any(x in tag.lower() for x in [
                "annotationuid",
                "readingid",
                "noduleid",
                "roi",
                "annotation"
            ]):
                annotation_ids.append((tag, value))

        xml_records.append({
            "xml_path": xml_path,
            "xml_filename": os.path.basename(xml_path),
            "patient_ids": sorted(set(patient_ids)),
            "series_uids": sorted(set(series_uids)),
            "study_uids": sorted(set(study_uids)),
            "image_uid_count": len(set(image_uids)),
            "annotation_id_examples": annotation_ids[:10],
            "unblinded_nodule_count": unblinded_count,
            "malignancy_values": malignancies
        })

    except Exception as e:
        pass

xml_df = pd.DataFrame(xml_records)

# ------------------------------------------------------------
# Identify XMLs potentially corresponding to our patients
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("XML MATCHING BY PATIENT IDENTIFIER")
print("=" * 90)

sample_matches = []

for pid in sample_patients:
    matches = xml_df[
        xml_df["patient_ids"].apply(
            lambda x: pid in x
        )
    ]

    sample_matches.append((pid, len(matches)))

    print(
        f"{pid}: {len(matches)} XML file(s) containing this patient ID"
    )

# ------------------------------------------------------------
# If XML PatientId is absent, show filename/path conventions
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("SAMPLE XML FILE IDENTIFIER STRUCTURE")
print("=" * 90)

for _, row in xml_df.head(5).iterrows():
    print("\nXML:", row["xml_filename"])
    print("Path:", row["xml_path"])
    print("Patient IDs:", row["patient_ids"])
    print("Series UIDs:", row["series_uids"])
    print("Study UIDs:", row["study_uids"])
    print("Nodule annotations:", row["unblinded_nodule_count"])
    print("Malignancy values:", row["malignancy_values"][:20])
    print("Annotation ID examples:", row["annotation_id_examples"][:5])

# ------------------------------------------------------------
# Save detailed inventory
# ------------------------------------------------------------

OUTPUT = "/content/lidc_xml_detailed_inventory.csv"

xml_df.to_csv(
    OUTPUT,
    index=False
)

print("\n" + "=" * 90)
print("DONE")
print("=" * 90)
print("Detailed XML inventory saved to:")
print(OUTPUT)

MAPPING + XML ANNOTATION IDENTIFIER INSPECTION
Subset nodules in mapping: 327
Unique patients in mapping: 249

Sample mapped patients:
  LIDC-IDRI-0003
  LIDC-IDRI-0005
  LIDC-IDRI-0008
  LIDC-IDRI-0010
  LIDC-IDRI-0012
  LIDC-IDRI-0014
  LIDC-IDRI-0015
  LIDC-IDRI-0016
  LIDC-IDRI-0018
  LIDC-IDRI-0021

XML MATCHING BY PATIENT IDENTIFIER
LIDC-IDRI-0003: 0 XML file(s) containing this patient ID
LIDC-IDRI-0005: 0 XML file(s) containing this patient ID
LIDC-IDRI-0008: 0 XML file(s) containing this patient ID
LIDC-IDRI-0010: 0 XML file(s) containing this patient ID
LIDC-IDRI-0012: 0 XML file(s) containing this patient ID
LIDC-IDRI-0014: 0 XML file(s) containing this patient ID
LIDC-IDRI-0015: 0 XML file(s) containing this patient ID
LIDC-IDRI-0016: 0 XML file(s) containing this patient ID
LIDC-IDRI-0018: 0 XML file(s) containing this patient ID
LIDC-IDRI-0021: 0 XML file(s) containing this patient ID

SAMPLE XML FILE IDENTIFIER STRUCTURE

XML: 161-resubmitted-correction-3-9-12.xml
Path: /

In [ ]:
manifest_path = "/content/TCIA_LIDC-IDRI_20200921.tcia"

with open(manifest_path, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

print("=" * 80)
print("TCIA LIDC-IDRI MANIFEST")
print("=" * 80)

print(text[:3000])

series_uids = []

for line in text.splitlines():
    line = line.strip()
    if line.startswith("1.2.") or line.startswith("1.3.6.1.4.1.14519"):
        series_uids.append(line)

print("\n" + "=" * 80)
print("MANIFEST SUMMARY")
print("=" * 80)

print("SeriesInstanceUIDs found:", len(series_uids))
print("Unique SeriesInstanceUIDs:", len(set(series_uids)))

print("\nFirst 10 SeriesInstanceUIDs:")
for uid in series_uids[:10]:
    print(uid)

print("=" * 80)

TCIA LIDC-IDRI MANIFEST
downloadServerUrl=https://public.cancerimagingarchive.net/nbia-download/servlet/DownloadServlet
includeAnnotation=true
noOfrRetry=4
databasketId=manifest-1600709154662.tcia
manifestVersion=3.0
ListOfSeriesToDownload=
1.3.6.1.4.1.14519.5.2.1.6279.6001.141365756818074696859567662357
1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636438705059720603192
1.3.6.1.4.1.14519.5.2.1.6279.6001.272961322147784625028175033640
1.3.6.1.4.1.14519.5.2.1.6279.6001.334422875695728375365180725669
1.3.6.1.4.1.14519.5.2.1.6279.6001.125481956446228056153445932229
1.3.6.1.4.1.14519.5.2.1.6279.6001.332829333783605240302521201463
1.3.6.1.4.1.14519.5.2.1.6279.6001.935486179338862584391625697780
1.3.6.1.4.1.14519.5.2.1.6279.6001.138813197521718693188313387015
1.3.6.1.4.1.14519.5.2.1.6279.6001.503980049263254396021509831276
1.3.6.1.4.1.14519.5.2.1.6279.6001.125641282114031333921855664282
1.3.6.1.4.1.14519.5.2.1.6279.6001.511347030803753100045216493273
1.3.6.1.4.1.14519.5.2.1.6279.6001.1267396488

In [ ]:
# ============================================================
# STEP 4A — TEST TCIA SERIES METADATA WITH ONE UID
# ============================================================

import requests
import json

test_uid = series_uids[0]

print("=" * 80)
print("TESTING TCIA SERIES METADATA API")
print("=" * 80)
print("Test SeriesInstanceUID:")
print(test_uid)

url = "https://services.cancerimagingarchive.net/nbia-api/services/getSeriesMetadata3"

try:
    response = requests.post(
        url,
        data={"list": test_uid},
        timeout=60
    )

    print("\nHTTP status:", response.status_code)

    print("\nResponse text:")
    print(response.text[:5000])

    response.raise_for_status()

    data = response.json()

    print("\n" + "=" * 80)
    print("API TEST SUCCESSFUL")
    print("=" * 80)

    print("Returned object type:", type(data))

    if isinstance(data, list):
        print("Number of returned records:", len(data))

        if len(data) > 0:
            print("\nFirst record:")
            print(json.dumps(data[0], indent=2))

except Exception as e:
    print("\n" + "=" * 80)
    print("API TEST FAILED")
    print("=" * 80)
    print(type(e).__name__, ":", e)

TESTING TCIA SERIES METADATA API
Test SeriesInstanceUID:
1.3.6.1.4.1.14519.5.2.1.6279.6001.141365756818074696859567662357

HTTP status: 500

Response text:
Server was not able to process your request
null

API TEST FAILED
HTTPError : 500 Server Error:  for url: https://services.cancerimagingarchive.net/nbia-api/services/getSeriesMetadata3


In [ ]:
# ============================================================
# STEP 4B — ROBUST TCIA getSeries API TEST
# ============================================================

import requests
import pandas as pd
from io import StringIO
import time

# Use the first available SeriesInstanceUID
test_uid = str(series_uids[0]).strip()

url = (
    "https://services.cancerimagingarchive.net/"
    "services/v4/TCIA/query/getSeries"
)

params = {
    "SeriesInstanceUID": test_uid,
    "format": "csv"
}

print("=" * 80)
print("TESTING TCIA getSeries API")
print("=" * 80)

print("Test SeriesInstanceUID:")
print(test_uid)

# -------------------------------------------------------------------
# Try a few times because TCIA can occasionally be slow.
# -------------------------------------------------------------------

response = None

for attempt in range(1, 4):

    print(f"\nAttempt {attempt}/3...")

    try:

        response = requests.get(
            url,
            params=params,
            timeout=(15, 180)
        )

        print(
            "HTTP status:",
            response.status_code
        )

        if response.status_code == 200:
            break

    except requests.exceptions.Timeout:

        print(
            "Request timed out."
        )

    except requests.exceptions.RequestException as e:

        print(
            "Request error:",
            repr(e)
        )

    if attempt < 3:

        print(
            "Waiting 5 seconds before retry..."
        )

        time.sleep(5)


# -------------------------------------------------------------------
# Final response validation
# -------------------------------------------------------------------

if response is None:

    raise RuntimeError(
        "TCIA API did not return a response after 3 attempts."
    )

if response.status_code != 200:

    print("\nResponse preview:")
    print(
        response.text[:3000]
    )

    raise RuntimeError(
        "TCIA getSeries failed after retries. "
        f"HTTP status = {response.status_code}"
    )

# -------------------------------------------------------------------
# Parse CSV
# -------------------------------------------------------------------

print("\nResponse preview:")
print(
    response.text[:3000]
)

try:

    result_df = pd.read_csv(
        StringIO(
            response.text
        )
    )

except Exception as e:

    raise RuntimeError(
        "TCIA returned HTTP 200, but the response "
        f"could not be parsed as CSV: {e}"
    )

# -------------------------------------------------------------------
# Success report
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("TCIA getSeries API SUCCESS")
print("=" * 80)

print(
    "Rows returned:",
    len(result_df)
)

print("\nColumns:")

for col in result_df.columns:

    print(
        " ",
        col
    )

print("\nReturned metadata:")

print(
    result_df.to_string(
        index=False
    )
)

print("=" * 80)

TESTING TCIA getSeries API
Test SeriesInstanceUID:
1.3.6.1.4.1.14519.5.2.1.6279.6001.141365756818074696859567662357

Attempt 1/3...
Request timed out.
Waiting 5 seconds before retry...

Attempt 2/3...
Request timed out.
Waiting 5 seconds before retry...

Attempt 3/3...
Request timed out.


RuntimeError: TCIA API did not return a response after 3 attempts.

In [ ]:
# ============================================================
# STEP 4C — TEST CURRENT TCIA v4 getSeries ENDPOINT
# ============================================================

import requests
import pandas as pd
from io import StringIO

test_uid = series_uids[0]

url = "https://services.cancerimagingarchive.net/nbia-api/services/v1/getSeries"

params = {
    "SeriesInstanceUID": test_uid,
    "format": "csv"
}

print("=" * 80)
print("TESTING CURRENT TCIA getSeries ENDPOINT")
print("=" * 80)
print("SeriesInstanceUID:")
print(test_uid)

try:
    response = requests.get(
        url,
        params=params,
        timeout=180
    )

    print("\nHTTP status:", response.status_code)

    print("\nResponse preview:")
    print(response.text[:5000])

    response.raise_for_status()

    result_df = pd.read_csv(StringIO(response.text))

    print("\n" + "=" * 80)
    print("SUCCESS")
    print("=" * 80)

    print("Rows returned:", len(result_df))

    print("\nColumns:")
    for col in result_df.columns:
        print(" ", col)

    print("\nMetadata:")
    print(result_df.to_string(index=False))

except Exception as e:
    print("\n" + "=" * 80)
    print("REQUEST FAILED")
    print("=" * 80)
    print(type(e).__name__, ":", e)

TESTING CURRENT TCIA getSeries ENDPOINT
SeriesInstanceUID:
1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222365663678666836860

HTTP status: 200

Response preview:
"SeriesInstanceUID","StudyInstanceUID","Modality","ProtocolName","SeriesDate","SeriesDescription","BodyPartExamined","SeriesNumber","AnnotationsFlag","Collection","PatientID","Manufacturer","ManufacturerModelName","SoftwareVersions","ImageCount","TimeStamp","LicenseName","LicenseURI","CollectionURI","FileSize","DateReleased","StudyDesc","StudyDate","ThirdPartyAnalysis"
"1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222365663678666836860","1.3.6.1.4.1.14519.5.2.1.6279.6001.281499745765120562304307889347","CT","","2000-01-01 00:00:00.0","","CHEST","3000026","true","LIDC-IDRI","LIDC-IDRI-1001","SIEMENS","Sensation 16","VA70C","194","2020-02-21 20:56:08.0","Creative Commons Attribution 3.0 Unported License","http://creativecommons.org/licenses/by/3.0/","https://doi.org/10.7937/K9/TCIA.2015.LO9QL9SX","102096016","2020-02-21 20:56:08.0"

In [ ]:
# ============================================================
# STEP 5 — MAP ALL 1,308 SERIES INSTANCE UIDs TO PATIENT IDs
# ============================================================

import requests
import time
import pandas as pd
from io import StringIO

# ------------------------------------------------------------
# 1. Make sure the UID list already exists
# ------------------------------------------------------------

if "series_uids" not in globals() or len(series_uids) == 0:
    raise RuntimeError(
        "The variable 'series_uids' was not found. "
        "Run the TCIA manifest extraction cell first."
    )

print("=" * 80)
print("MAPPING ALL TCIA SERIES UIDs TO PATIENT IDs")
print("=" * 80)
print(f"SeriesInstanceUIDs to query: {len(series_uids)}")

# ------------------------------------------------------------
# 2. Current working TCIA endpoint
# ------------------------------------------------------------

API_URL = "https://services.cancerimagingarchive.net/nbia-api/services/v1/getSeries"

all_rows = []

# Query in small batches
BATCH_SIZE = 10

for start in range(0, len(series_uids), BATCH_SIZE):

    batch = series_uids[start:start + BATCH_SIZE]

    params = {
        "SeriesInstanceUID": ",".join(batch),
        "format": "csv"
    }

    try:
        response = requests.get(
            API_URL,
            params=params,
            timeout=180
        )

        response.raise_for_status()

        batch_df = pd.read_csv(
            StringIO(response.text)
        )

        all_rows.append(batch_df)

        print(
            f"Processed {min(start + BATCH_SIZE, len(series_uids))}"
            f" / {len(series_uids)}"
        )

    except Exception as e:
        print(
            f"WARNING: Batch starting at {start} failed: "
            f"{type(e).__name__}: {e}"
        )

    # Small delay between requests
    time.sleep(0.5)

# ------------------------------------------------------------
# 3. Combine results
# ------------------------------------------------------------

if not all_rows:
    raise RuntimeError(
        "No TCIA metadata was returned."
    )

tcia_metadata = pd.concat(
    all_rows,
    ignore_index=True
)

# Remove duplicate UID records if any
tcia_metadata = tcia_metadata.drop_duplicates(
    subset=["SeriesInstanceUID"]
)

# ------------------------------------------------------------
# 4. Diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TCIA SERIES → PATIENT MAPPING RESULTS")
print("=" * 80)

print(
    f"Requested SeriesInstanceUIDs: "
    f"{len(series_uids)}"
)

print(
    f"Metadata records returned:    "
    f"{len(tcia_metadata)}"
)

print(
    f"Unique PatientIDs returned:   "
    f"{tcia_metadata['PatientID'].nunique()}"
)

# Check coverage
requested_set = set(series_uids)
returned_set = set(
    tcia_metadata["SeriesInstanceUID"].astype(str)
)

missing_uids = sorted(
    requested_set - returned_set
)

print(
    f"Series UIDs not returned:      "
    f"{len(missing_uids)}"
)

if missing_uids:
    print("\nFirst 10 missing SeriesInstanceUIDs:")
    for uid in missing_uids[:10]:
        print(uid)

print("\nFirst 10 mappings:")
print(
    tcia_metadata[
        [
            "SeriesInstanceUID",
            "StudyInstanceUID",
            "PatientID",
            "Collection",
            "Modality",
            "ImageCount"
        ]
    ].head(10).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Save result
# ------------------------------------------------------------

OUTPUT_PATH = "/content/tcia_series_to_patient_mapping.csv"

tcia_metadata.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved mapping to:")
print(OUTPUT_PATH)

print("=" * 80)

MAPPING ALL TCIA SERIES UIDs TO PATIENT IDs
SeriesInstanceUIDs to query: 1308


RuntimeError: No TCIA metadata was returned.

In [ ]:
# ============================================================
# STEP 5 — MAP ALL TCIA SERIES UIDs TO PATIENT IDs
# ============================================================

import os
import time
import requests
import pandas as pd
from io import StringIO

MANIFEST_PATH = "/content/TCIA_LIDC-IDRI_20200921.tcia"
OUTPUT_PATH = "/content/tcia_series_to_patient_mapping.csv"

# ------------------------------------------------------------
# 1. Read SeriesInstanceUIDs from the TCIA manifest
# ------------------------------------------------------------

with open(MANIFEST_PATH, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.read().splitlines()

series_uids = []
started = False

for line in lines:
    line = line.strip()

    if line == "ListOfSeriesToDownload=":
        started = True
        continue

    if started and line.startswith("1.3.6.1.4.1.14519."):
        series_uids.append(line)

series_uids = sorted(set(series_uids))

print("=" * 80)
print("TCIA SERIES → PATIENT MAPPING")
print("=" * 80)
print(f"Unique SeriesInstanceUIDs: {len(series_uids)}")

assert len(series_uids) > 0, "No SeriesInstanceUIDs found."

# ------------------------------------------------------------
# 2. Official TCIA getSeries endpoint
# ------------------------------------------------------------

API_URL = (
    "https://services.cancerimagingarchive.net/"
    "nbia-api/services/v1/getSeries"
)

rows = []
failed = []

# ------------------------------------------------------------
# 3. Query each UID individually
# ------------------------------------------------------------

for i, uid in enumerate(series_uids, start=1):

    try:
        response = requests.get(
            API_URL,
            params={
                "SeriesInstanceUID": uid,
                "format": "csv"
            },
            timeout=120
        )

        response.raise_for_status()

        text = response.text.strip()

        if not text:
            failed.append((uid, "EMPTY_RESPONSE"))
            print(f"[{i}/{len(series_uids)}] EMPTY RESPONSE")
            continue

        df_one = pd.read_csv(StringIO(text))

        if len(df_one) == 0:
            failed.append((uid, "NO_ROWS"))
            print(f"[{i}/{len(series_uids)}] NO ROWS")
            continue

        # Add returned rows
        rows.append(df_one)

        patient_values = df_one["PatientID"].astype(str).unique()

        print(
            f"[{i}/{len(series_uids)}] "
            f"SUCCESS | PatientID: {', '.join(patient_values)}"
        )

    except Exception as e:
        failed.append((uid, f"{type(e).__name__}: {e}"))
        print(
            f"[{i}/{len(series_uids)}] "
            f"FAILED | {type(e).__name__}: {e}"
        )

    # Small delay between requests
    time.sleep(0.1)

    # Save progress every 100 successful/attempted queries
    if i % 100 == 0 and rows:
        progress_df = pd.concat(rows, ignore_index=True)
        progress_df = progress_df.drop_duplicates(
            subset=["SeriesInstanceUID"]
        )
        progress_df.to_csv(
            OUTPUT_PATH,
            index=False
        )
        print(
            f"  -> Progress saved: "
            f"{len(progress_df)} metadata records"
        )

# ------------------------------------------------------------
# 4. Combine final results
# ------------------------------------------------------------

if rows:
    tcia_metadata = pd.concat(
        rows,
        ignore_index=True
    )

    tcia_metadata = tcia_metadata.drop_duplicates(
        subset=["SeriesInstanceUID"]
    )

else:
    raise RuntimeError(
        "No TCIA metadata was successfully returned."
    )

# ------------------------------------------------------------
# 5. Coverage diagnostics
# ------------------------------------------------------------

requested = set(series_uids)

returned = set(
    tcia_metadata["SeriesInstanceUID"].astype(str)
)

missing = sorted(requested - returned)

print("\n" + "=" * 80)
print("FINAL TCIA MAPPING RESULTS")
print("=" * 80)

print(
    f"Requested SeriesInstanceUIDs: {len(requested):,}"
)

print(
    f"Metadata records returned:    {len(returned):,}"
)

print(
    f"Unique PatientIDs returned:   "
    f"{tcia_metadata['PatientID'].nunique():,}"
)

print(
    f"Missing SeriesInstanceUIDs:   {len(missing):,}"
)

print(
    f"Failed individual requests:   {len(failed):,}"
)

# ------------------------------------------------------------
# 6. Save final mapping
# ------------------------------------------------------------

tcia_metadata.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved to:")
print(OUTPUT_PATH)

print("\nFirst 10 mappings:")
print(
    tcia_metadata[
        [
            "SeriesInstanceUID",
            "StudyInstanceUID",
            "PatientID",
            "Collection",
            "Modality",
            "ImageCount"
        ]
    ].head(10).to_string(index=False)
)

if failed:
    print("\nFirst 10 failed requests:")
    for uid, reason in failed[:10]:
        print(uid)
        print(" ", reason)

print("=" * 80)

TCIA SERIES → PATIENT MAPPING
Unique SeriesInstanceUIDs: 1308
[1/1308] SUCCESS | PatientID: LIDC-IDRI-1001
[2/1308] SUCCESS | PatientID: LIDC-IDRI-0778
[3/1308] SUCCESS | PatientID: LIDC-IDRI-0813
[4/1308] SUCCESS | PatientID: LIDC-IDRI-0710
[5/1308] SUCCESS | PatientID: LIDC-IDRI-0410
[6/1308] SUCCESS | PatientID: LIDC-IDRI-1002
[7/1308] SUCCESS | PatientID: LIDC-IDRI-0159
[8/1308] SUCCESS | PatientID: LIDC-IDRI-0745
[9/1308] SUCCESS | PatientID: LIDC-IDRI-0182
[10/1308] SUCCESS | PatientID: LIDC-IDRI-0026
[11/1308] SUCCESS | PatientID: LIDC-IDRI-0252
[12/1308] SUCCESS | PatientID: LIDC-IDRI-0249
[13/1308] SUCCESS | PatientID: LIDC-IDRI-0830
[14/1308] SUCCESS | PatientID: LIDC-IDRI-0187
[15/1308] SUCCESS | PatientID: LIDC-IDRI-0245
[16/1308] SUCCESS | PatientID: LIDC-IDRI-0134
[17/1308] SUCCESS | PatientID: LIDC-IDRI-0123
[18/1308] SUCCESS | PatientID: LIDC-IDRI-0066
[19/1308] SUCCESS | PatientID: LIDC-IDRI-0766
[20/1308] SUCCESS | PatientID: LIDC-IDRI-0334
[21/1308] SUCCESS | Patient

In [ ]:
# ============================================================
# STEP 6 — CONNECT SUBSET NODULES → PATIENT IDs → SERIES UIDs
# ============================================================

import pandas as pd

MAPPING_PATH = "/content/nodule_to_lidc_mapping.csv"
TCIA_PATH = "/content/tcia_series_to_patient_mapping.csv"

mapping_df = pd.read_csv(MAPPING_PATH)
tcia_df = pd.read_csv(TCIA_PATH)

print("=" * 90)
print("CONNECTING SUBSET NODULES → PATIENT IDs → SERIES INSTANCE UIDs")
print("=" * 90)

print("Subset mapping rows:", len(mapping_df))
print("TCIA metadata rows:", len(tcia_df))

# ------------------------------------------------------------
# Detect the correct nodule ID column automatically
# ------------------------------------------------------------

possible_nodule_cols = [
    "nodule_id",
    "subset_nodule_id"
]

nodule_col = None

for col in possible_nodule_cols:
    if col in mapping_df.columns:
        nodule_col = col
        break

if nodule_col is None:
    raise KeyError(
        "Could not find the nodule ID column.\n"
        f"Available columns: {list(mapping_df.columns)}"
    )

print("Using nodule ID column:", nodule_col)

# ------------------------------------------------------------
# Standardize identifiers
# ------------------------------------------------------------

mapping_df["patient_id"] = (
    mapping_df["patient_id"]
    .astype(str)
    .str.strip()
)

tcia_df["PatientID"] = (
    tcia_df["PatientID"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# Restrict TCIA data to our 249 patients
# ------------------------------------------------------------

subset_patients = set(mapping_df["patient_id"])

relevant_tcia = tcia_df[
    tcia_df["PatientID"].isin(subset_patients)
].copy()

print("\nPatients in subset:", len(subset_patients))
print(
    "TCIA series belonging to subset patients:",
    len(relevant_tcia)
)
print(
    "Unique patients represented in those series:",
    relevant_tcia["PatientID"].nunique()
)

# ------------------------------------------------------------
# Join each subset nodule to candidate series
# ------------------------------------------------------------

joined = mapping_df.merge(
    relevant_tcia,
    left_on="patient_id",
    right_on="PatientID",
    how="left"
)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("JOIN RESULTS")
print("=" * 90)

matched_nodules = joined[
    joined["SeriesInstanceUID"].notna()
][nodule_col].nunique()

print(
    "Nodules with ≥1 candidate SeriesInstanceUID:",
    matched_nodules,
    "/",
    len(mapping_df)
)

print(
    "Nodules with NO candidate SeriesInstanceUID:",
    len(mapping_df) - matched_nodules
)

candidate_counts = (
    joined.groupby(nodule_col)["SeriesInstanceUID"]
    .nunique()
)

print("\nCandidate series counts per nodule:")
print(candidate_counts.describe())

# ------------------------------------------------------------
# First 20 candidate mappings
# ------------------------------------------------------------

display_cols = [
    nodule_col,
    "patient_id",
]

if "native_nodule_id" in joined.columns:
    display_cols.append("native_nodule_id")

display_cols += [
    "SeriesInstanceUID",
    "StudyInstanceUID",
    "Modality",
    "ImageCount",
    "AnnotationsFlag"
]

print("\nFirst 20 candidate mappings:")
print(
    joined[display_cols]
    .head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

OUTPUT_PATH = "/content/nodule_to_series_candidates.csv"

joined.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved candidate mapping:")
print(OUTPUT_PATH)

print("=" * 90)

CONNECTING SUBSET NODULES → PATIENT IDs → SERIES INSTANCE UIDs
Subset mapping rows: 327
TCIA metadata rows: 1308
Using nodule ID column: subset_nodule_id

Patients in subset: 249
TCIA series belonging to subset patients: 330
Unique patients represented in those series: 249

JOIN RESULTS
Nodules with ≥1 candidate SeriesInstanceUID: 327 / 327
Nodules with NO candidate SeriesInstanceUID: 0

Candidate series counts per nodule:
count    327.000000
mean       1.336391
std        0.479637
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        3.000000
Name: SeriesInstanceUID, dtype: float64

First 20 candidate mappings:
subset_nodule_id     patient_id native_nodule_id                                                SeriesInstanceUID                                                 StudyInstanceUID Modality  ImageCount  AnnotationsFlag
      nodule_001 LIDC-IDRI-0121         nodule-1 1.3.6.1.4.1.14519.5.2.1.6279.6001.225515255547637437801620523312 1.3.6.1.4.1.

In [ ]:
# ============================================================
# STEP 7 — IDENTIFY THE MOST LIKELY CT SERIES FOR EACH NODULE
# ============================================================

import os
import pandas as pd

CANDIDATES_PATH = "/content/nodule_to_series_candidates.csv"
RAW_DIR = "/content/kaggle_dataset_2000_extracted"

df = pd.read_csv(CANDIDATES_PATH)

# Detect nodule ID column
nodule_col = (
    "subset_nodule_id"
    if "subset_nodule_id" in df.columns
    else "nodule_id"
)

# ------------------------------------------------------------
# Count original PNG slices in each subset nodule folder
# ------------------------------------------------------------

slice_counts = {}

for nodule_id in df[nodule_col].dropna().unique():

    folder = os.path.join(
        RAW_DIR,
        str(nodule_id)
    )

    if os.path.isdir(folder):
        count = len([
            f for f in os.listdir(folder)
            if f.lower().endswith(".png")
        ])
    else:
        count = None

    slice_counts[str(nodule_id)] = count

df["subset_slice_count"] = df[nodule_col].astype(str).map(slice_counts)

# ------------------------------------------------------------
# Keep only CT candidates
# ------------------------------------------------------------

ct_df = df[
    df["Modality"].astype(str).str.upper() == "CT"
].copy()

# ------------------------------------------------------------
# Score each candidate
#
# We strongly prefer:
#   1. CT modality
#   2. AnnotationsFlag present
#   3. Series image count substantially larger than
#      the extracted nodule crop
#
# NOTE: ImageCount is the FULL CT series image count.
# It is NOT expected to equal the crop slice count.
# ------------------------------------------------------------

ct_df["has_annotations"] = (
    ct_df["AnnotationsFlag"]
    .fillna(False)
    .astype(bool)
)

# Number of CT candidates per nodule
ct_candidate_counts = (
    ct_df.groupby(nodule_col)["SeriesInstanceUID"]
    .nunique()
)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print("=" * 90)
print("CT SERIES CANDIDATE ANALYSIS")
print("=" * 90)

print(
    "Total subset nodules:",
    df[nodule_col].nunique()
)

print(
    "Nodules with at least one CT candidate:",
    ct_df[nodule_col].nunique()
)

print(
    "Nodules with no CT candidate:",
    df[nodule_col].nunique() - ct_df[nodule_col].nunique()
)

print(
    "Total CT candidate series:",
    len(ct_df)
)

print("\nCT candidate count per nodule:")
print(ct_candidate_counts.describe())

# ------------------------------------------------------------
# Show nodules with multiple CT candidates
# ------------------------------------------------------------

multiple_ct = ct_candidate_counts[
    ct_candidate_counts > 1
]

print(
    "\nNodules with multiple CT candidates:",
    len(multiple_ct)
)

if len(multiple_ct) > 0:

    print("\nExamples:")

    for nodule_id in multiple_ct.index[:20]:

        rows = ct_df[
            ct_df[nodule_col].astype(str)
            == str(nodule_id)
        ]

        print("\n", nodule_id)

        print(
            rows[
                [
                    "patient_id",
                    "native_nodule_id",
                    "SeriesInstanceUID",
                    "StudyInstanceUID",
                    "ImageCount",
                    "AnnotationsFlag"
                ]
            ].to_string(index=False)
        )

# ------------------------------------------------------------
# Save CT-only candidate table
# ------------------------------------------------------------

OUTPUT_PATH = "/content/nodule_to_ct_series_candidates.csv"

ct_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved CT candidate table:")
print(OUTPUT_PATH)

print("=" * 90)

CT SERIES CANDIDATE ANALYSIS
Total subset nodules: 327
Nodules with at least one CT candidate: 327
Nodules with no CT candidate: 0
Total CT candidate series: 329

CT candidate count per nodule:
count    327.000000
mean       1.006116
std        0.078086
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        2.000000
Name: SeriesInstanceUID, dtype: float64

Nodules with multiple CT candidates: 2

Examples:

 nodule_029
    patient_id native_nodule_id                                                SeriesInstanceUID                                                 StudyInstanceUID  ImageCount  AnnotationsFlag
LIDC-IDRI-0132         nodule-2 1.3.6.1.4.1.14519.5.2.1.6279.6001.151647338241909635299641922057 1.3.6.1.4.1.14519.5.2.1.6279.6001.218658642102832118810712329678         127              NaN
LIDC-IDRI-0132         nodule-2 1.3.6.1.4.1.14519.5.2.1.6279.6001.254176853278710432756285662989 1.3.6.1.4.1.14519.5.2.1.6279.6001.31413861641106194805284376734

In [ ]:
# ============================================================
# STEP 8 — INSPECT THE 2 AMBIGUOUS CT SERIES CASES
# ============================================================

import pandas as pd

CT_CANDIDATES = "/content/nodule_to_ct_series_candidates.csv"

df = pd.read_csv(CT_CANDIDATES)

# Detect nodule column
nodule_col = (
    "subset_nodule_id"
    if "subset_nodule_id" in df.columns
    else "nodule_id"
)

ambiguous = []

for nodule_id, group in df.groupby(nodule_col):

    unique_series = group["SeriesInstanceUID"].dropna().unique()

    if len(unique_series) > 1:
        ambiguous.append(nodule_id)

print("=" * 90)
print("AMBIGUOUS CT SERIES CASES")
print("=" * 90)

print("Number of ambiguous nodules:", len(ambiguous))
print("Ambiguous nodules:", ambiguous)

for nodule_id in ambiguous:

    print("\n" + "-" * 90)
    print("NODULE:", nodule_id)
    print("-" * 90)

    rows = df[
        df[nodule_col].astype(str) == str(nodule_id)
    ].copy()

    print(
        rows[
            [
                nodule_col,
                "patient_id",
                "native_nodule_id",
                "SeriesInstanceUID",
                "StudyInstanceUID",
                "ImageCount",
                "AnnotationsFlag",
            ]
        ].to_string(index=False)
    )

print("\n" + "=" * 90)
print("DONE")
print("=" * 90)

AMBIGUOUS CT SERIES CASES
Number of ambiguous nodules: 2
Ambiguous nodules: ['nodule_029', 'nodule_085']

------------------------------------------------------------------------------------------
NODULE: nodule_029
------------------------------------------------------------------------------------------
subset_nodule_id     patient_id native_nodule_id                                                SeriesInstanceUID                                                 StudyInstanceUID  ImageCount  AnnotationsFlag
      nodule_029 LIDC-IDRI-0132         nodule-2 1.3.6.1.4.1.14519.5.2.1.6279.6001.151647338241909635299641922057 1.3.6.1.4.1.14519.5.2.1.6279.6001.218658642102832118810712329678         127              NaN
      nodule_029 LIDC-IDRI-0132         nodule-2 1.3.6.1.4.1.14519.5.2.1.6279.6001.254176853278710432756285662989 1.3.6.1.4.1.14519.5.2.1.6279.6001.314138616411061948052843767346         116              1.0

--------------------------------------------------------------------

In [ ]:
# ============================================================
# STEP 9 — INSPECT EXISTING MATCHING FILE FOR SERIES UIDS
# ============================================================

import pandas as pd
import os

files_to_check = [
    "/content/nodule_to_lidc_mapping.csv",
    "/content/nodule_to_series_candidates.csv",
    "/content/nodule_to_ct_series_candidates.csv",
]

for path in files_to_check:
    print("\n" + "=" * 90)
    print("FILE:", path)
    print("=" * 90)

    if not os.path.exists(path):
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(path)

    print("Rows:", len(df))
    print("Columns:")
    for col in df.columns:
        print("  ", col)

    # Show only the ambiguous nodules
    possible_id_cols = [
        c for c in ["nodule_id", "subset_nodule_id"]
        if c in df.columns
    ]

    if possible_id_cols:
        id_col = possible_id_cols[0]

        subset = df[
            df[id_col].astype(str).isin(
                ["nodule_029", "nodule_085"]
            )
        ]

        if len(subset) > 0:
            print("\nRelevant rows:")
            print(subset.to_string(index=False))


FILE: /content/nodule_to_lidc_mapping.csv
Rows: 327
Columns:
   subset_nodule_id
   subset_slice_count
   matched_slice_count
   patient_id
   native_nodule_id
   best_match_count
   second_best_match_count
   status

Relevant rows:
subset_nodule_id  subset_slice_count  matched_slice_count     patient_id native_nodule_id  best_match_count  second_best_match_count             status
      nodule_029                   2                    2 LIDC-IDRI-0132         nodule-2                 2                        0 EXACT_UNIQUE_MATCH
      nodule_085                   2                    2 LIDC-IDRI-0315         nodule-1                 2                        0 EXACT_UNIQUE_MATCH

FILE: /content/nodule_to_series_candidates.csv
Rows: 437
Columns:
   subset_nodule_id
   subset_slice_count
   matched_slice_count
   patient_id
   native_nodule_id
   best_match_count
   second_best_match_count
   status
   SeriesInstanceUID
   StudyInstanceUID
   Modality
   ProtocolName
   SeriesDate
   S

In [ ]:
# ============================================================
# STEP 10 — PREPARE THE 327 NODULES FOR XML MATCHING
# ============================================================

import pandas as pd
import os

MAPPING_PATH = "/content/nodule_to_lidc_mapping.csv"
CT_PATH = "/content/nodule_to_ct_series_candidates.csv"

mapping = pd.read_csv(MAPPING_PATH)
ct = pd.read_csv(CT_PATH)

# ------------------------------------------------------------
# Show the 327 nodule → native nodule mappings
# ------------------------------------------------------------

print("=" * 90)
print("CURRENT NODULE → PATIENT → NATIVE NODULE MAPPING")
print("=" * 90)

print("Total nodules:", len(mapping))
print("Unique patients:", mapping["patient_id"].nunique())

print("\nFirst 20:")
print(
    mapping[
        [
            "subset_nodule_id",
            "patient_id",
            "native_nodule_id",
            "subset_slice_count",
            "matched_slice_count",
            "status"
        ]
    ].head(20).to_string(index=False)
)

# ------------------------------------------------------------
# Determine which CT series are unambiguous
# ------------------------------------------------------------

series_counts = (
    ct.groupby("subset_nodule_id")["SeriesInstanceUID"]
    .nunique()
)

unambiguous = series_counts[series_counts == 1].index.tolist()
ambiguous = series_counts[series_counts > 1].index.tolist()

print("\n" + "=" * 90)
print("CT SERIES STATUS")
print("=" * 90)

print("Unambiguous nodules:", len(unambiguous))
print("Ambiguous nodules:", len(ambiguous))

print("Ambiguous:", ambiguous)

# ------------------------------------------------------------
# Create a clean table containing exactly one series
# for every currently unambiguous nodule
# ------------------------------------------------------------

clean = ct[
    ct["subset_nodule_id"].isin(unambiguous)
].copy()

# Safety check
check_counts = (
    clean.groupby("subset_nodule_id")["SeriesInstanceUID"]
    .nunique()
)

assert (check_counts == 1).all()

print("\nClean one-series mappings:", len(clean))

print("\nFirst 20:")
print(
    clean[
        [
            "subset_nodule_id",
            "patient_id",
            "native_nodule_id",
            "SeriesInstanceUID",
            "StudyInstanceUID",
            "ImageCount",
            "AnnotationsFlag"
        ]
    ].head(20).to_string(index=False)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

OUTPUT = "/content/nodule_to_single_ct_series.csv"

clean.to_csv(
    OUTPUT,
    index=False
)

print("\nSaved:")
print(OUTPUT)

print("=" * 90)

CURRENT NODULE → PATIENT → NATIVE NODULE MAPPING
Total nodules: 327
Unique patients: 249

First 20:
subset_nodule_id     patient_id native_nodule_id  subset_slice_count  matched_slice_count             status
      nodule_001 LIDC-IDRI-0121         nodule-1                   6                    6 EXACT_UNIQUE_MATCH
      nodule_002 LIDC-IDRI-0158         nodule-1                   6                    6 EXACT_UNIQUE_MATCH
      nodule_003 LIDC-IDRI-0702         nodule-3                   6                    6 EXACT_UNIQUE_MATCH
      nodule_004 LIDC-IDRI-0241         nodule-1                   2                    2 EXACT_UNIQUE_MATCH
      nodule_005 LIDC-IDRI-0008         nodule-0                   2                    2 EXACT_UNIQUE_MATCH
      nodule_006 LIDC-IDRI-0602         nodule-1                   5                    5 EXACT_UNIQUE_MATCH
      nodule_007 LIDC-IDRI-0636         nodule-1                   6                    6 EXACT_UNIQUE_MATCH
      nodule_008 LIDC-IDRI-0

In [ ]:
# ============================================================
# STEP 10 — MATCH THE 325 CT SERIES TO LIDC XML FILES
# ============================================================

import os
import glob
import xml.etree.ElementTree as ET
import pandas as pd

SERIES_MAP_PATH = "/content/nodule_to_single_ct_series.csv"
XML_ROOT = "/content/lidc_xml_annotations"
OUTPUT_PATH = "/content/nodule_to_xml_candidates.csv"

# ------------------------------------------------------------
# 1. Load the clean 325-nodule series mapping
# ------------------------------------------------------------

series_df = pd.read_csv(SERIES_MAP_PATH)

print("=" * 90)
print("STEP 10 — MATCHING CT SERIES TO LIDC XML ANNOTATIONS")
print("=" * 90)

print("Clean mapped nodules:", len(series_df))

assert len(series_df) == 325, (
    f"Expected 325 unambiguous nodules, found {len(series_df)}"
)

# ------------------------------------------------------------
# 2. Find all XML files
# ------------------------------------------------------------

xml_files = glob.glob(
    os.path.join(XML_ROOT, "**", "*.xml"),
    recursive=True
)

print("XML files found:", len(xml_files))

if len(xml_files) == 0:
    raise RuntimeError("No XML files found.")

# ------------------------------------------------------------
# 3. Build SeriesInstanceUID → XML lookup
# ------------------------------------------------------------

xml_by_series = {}

processed = 0
errors = 0

for xml_path in xml_files:

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        series_uids = set()

        for elem in root.iter():

            tag = elem.tag.split("}")[-1].lower()

            if tag == "seriesinstanceuid":

                value = (elem.text or "").strip()

                if value:
                    series_uids.add(value)

        for uid in series_uids:
            xml_by_series.setdefault(uid, []).append(xml_path)

        processed += 1

    except Exception:
        errors += 1

print("\nXML indexing complete.")
print("XML files processed successfully:", processed)
print("XML parse errors:", errors)
print("Unique SeriesInstanceUIDs indexed:", len(xml_by_series))

# ------------------------------------------------------------
# 4. Match our 325 CT series to XML
# ------------------------------------------------------------

rows = []

matched = 0
unmatched = 0
multiple_xml = 0

for _, row in series_df.iterrows():

    nodule_id = str(row["subset_nodule_id"])
    series_uid = str(row["SeriesInstanceUID"])
    patient_id = str(row["patient_id"])

    candidates = xml_by_series.get(series_uid, [])

    if len(candidates) == 0:

        status = "NO_XML_MATCH"
        unmatched += 1

    elif len(candidates) == 1:

        status = "ONE_XML_MATCH"
        matched += 1

    else:

        status = "MULTIPLE_XML_MATCHES"
        multiple_xml += 1

    rows.append({
        "subset_nodule_id": nodule_id,
        "patient_id": patient_id,
        "native_nodule_id": row["native_nodule_id"],
        "SeriesInstanceUID": series_uid,
        "StudyInstanceUID": row["StudyInstanceUID"],
        "ImageCount": row["ImageCount"],
        "xml_count": len(candidates),
        "xml_paths": " || ".join(candidates),
        "xml_match_status": status
    })

result_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# 5. Diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("XML MATCHING RESULTS")
print("=" * 90)

print("Total nodules examined:", len(result_df))
print("Exactly one XML match:", matched)
print("No XML match:", unmatched)
print("Multiple XML matches:", multiple_xml)

print("\nStatus distribution:")
print(
    result_df["xml_match_status"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 6. Show examples
# ------------------------------------------------------------

print("\nFirst 20 matched nodules:")

print(
    result_df[
        [
            "subset_nodule_id",
            "patient_id",
            "native_nodule_id",
            "SeriesInstanceUID",
            "xml_count",
            "xml_match_status"
        ]
    ].head(20).to_string(index=False)
)

# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------

result_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved:")
print(OUTPUT_PATH)

print("=" * 90)

STEP 10 — MATCHING CT SERIES TO LIDC XML ANNOTATIONS
Clean mapped nodules: 325
XML files found: 1319

XML indexing complete.
XML files processed successfully: 1319
XML parse errors: 0
Unique SeriesInstanceUIDs indexed: 1294

XML MATCHING RESULTS
Total nodules examined: 325
Exactly one XML match: 324
No XML match: 0
Multiple XML matches: 1

Status distribution:
xml_match_status
ONE_XML_MATCH           324
MULTIPLE_XML_MATCHES      1

First 20 matched nodules:
subset_nodule_id     patient_id native_nodule_id                                                SeriesInstanceUID  xml_count xml_match_status
      nodule_001 LIDC-IDRI-0121         nodule-1 1.3.6.1.4.1.14519.5.2.1.6279.6001.225515255547637437801620523312          1    ONE_XML_MATCH
      nodule_002 LIDC-IDRI-0158         nodule-1 1.3.6.1.4.1.14519.5.2.1.6279.6001.244204120220889433826451158706          1    ONE_XML_MATCH
      nodule_003 LIDC-IDRI-0702         nodule-3 1.3.6.1.4.1.14519.5.2.1.6279.6001.2864228468967974331681870859

In [ ]:
# ============================================================
# STEP 11 — INSPECT XML NODULE IDs AND MALIGNANCY SCORES
# ============================================================

import os
import ast
import xml.etree.ElementTree as ET
import pandas as pd

XML_CANDIDATES_PATH = "/content/nodule_to_xml_candidates.csv"
OUTPUT_PATH = "/content/nodule_xml_annotation_inventory.csv"

df = pd.read_csv(XML_CANDIDATES_PATH)

print("=" * 90)
print("STEP 11 — INSPECTING XML ANNOTATIONS")
print("=" * 90)

print("Total mapped nodules:", len(df))
print(
    "One-XML matches:",
    (df["xml_match_status"] == "ONE_XML_MATCH").sum()
)
print(
    "Multiple-XML matches:",
    (df["xml_match_status"] == "MULTIPLE_XML_MATCHES").sum()
)

# ------------------------------------------------------------
# XML parser
# ------------------------------------------------------------

def local_tag(element):
    return element.tag.split("}")[-1].lower()

def inspect_xml(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    annotations = {}

    current_nodule_id = None
    current_malignancy = None

    # Walk through XML in document order
    for elem in root.iter():

        tag = local_tag(elem)
        text = (elem.text or "").strip()

        if tag == "noduleid":
            current_nodule_id = text

        elif tag == "malignancy":

            if current_nodule_id is not None and text:
                try:
                    rating = int(text)
                except ValueError:
                    rating = None

                if rating is not None:
                    annotations.setdefault(
                        current_nodule_id,
                        []
                    ).append(rating)

    # Also inspect non-nodule structures
    return annotations


# ------------------------------------------------------------
# Process all one-XML matches
# ------------------------------------------------------------

results = []

for _, row in df.iterrows():

    nodule_id = row["subset_nodule_id"]
    patient_id = row["patient_id"]
    series_uid = row["SeriesInstanceUID"]
    status = row["xml_match_status"]

    xml_string = str(row["xml_paths"])

    if status != "ONE_XML_MATCH":
        results.append({
            "subset_nodule_id": nodule_id,
            "patient_id": patient_id,
            "SeriesInstanceUID": series_uid,
            "xml_status": status,
            "xml_path": xml_string,
            "xml_nodule_ids": "",
            "xml_malignancy_by_nodule": "",
        })
        continue

    xml_path = xml_string.split(" || ")[0]

    try:
        annotations = inspect_xml(xml_path)

        # Convert to readable form
        annotation_text = "; ".join(
            f"{nid}:{ratings}"
            for nid, ratings in annotations.items()
        )

        results.append({
            "subset_nodule_id": nodule_id,
            "patient_id": patient_id,
            "SeriesInstanceUID": series_uid,
            "native_nodule_id": row["native_nodule_id"],
            "xml_status": status,
            "xml_path": xml_path,
            "xml_nodule_ids": ", ".join(
                sorted(annotations.keys())
            ),
            "xml_malignancy_by_nodule": annotation_text,
        })

    except Exception as e:

        results.append({
            "subset_nodule_id": nodule_id,
            "patient_id": patient_id,
            "SeriesInstanceUID": series_uid,
            "native_nodule_id": row["native_nodule_id"],
            "xml_status": "XML_PARSE_ERROR",
            "xml_path": xml_path,
            "xml_nodule_ids": "",
            "xml_malignancy_by_nodule": str(e),
        })

inventory = pd.DataFrame(results)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("XML ANNOTATION INVENTORY")
print("=" * 90)

print(
    "Rows generated:",
    len(inventory)
)

print(
    "Rows containing XML nodule IDs:",
    (
        inventory["xml_nodule_ids"]
        .fillna("")
        .str.len() > 0
    ).sum()
)

print("\nFirst 20 XML inventories:")

print(
    inventory[
        [
            "subset_nodule_id",
            "patient_id",
            "native_nodule_id",
            "xml_nodule_ids",
            "xml_malignancy_by_nodule"
        ]
    ].head(20).to_string(index=False)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

inventory.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved:")
print(OUTPUT_PATH)

print("=" * 90)

STEP 11 — INSPECTING XML ANNOTATIONS
Total mapped nodules: 325
One-XML matches: 324
Multiple-XML matches: 1

XML ANNOTATION INVENTORY
Rows generated: 325
Rows containing XML nodule IDs: 324

First 20 XML inventories:
subset_nodule_id     patient_id native_nodule_id                                                                                                                                                                                                                                                                                                                                                                      xml_nodule_ids                                                                                                                                                                                                                                                                                                                                                                           

In [ ]:
# ============================================================
# STEP 12 — INSPECT FULL XML SPATIAL ANNOTATION STRUCTURE
# ============================================================

import pandas as pd
import xml.etree.ElementTree as ET

INVENTORY_PATH = "/content/nodule_xml_annotation_inventory.csv"

inventory = pd.read_csv(INVENTORY_PATH)

# Pick the first unambiguous example
row = inventory[
    inventory["xml_status"] == "ONE_XML_MATCH"
].iloc[0]

nodule_id = row["subset_nodule_id"]
xml_path = row["xml_path"]
patient_id = row["patient_id"]
series_uid = row["SeriesInstanceUID"]

print("=" * 90)
print("STEP 12 — DETAILED XML SPATIAL ANNOTATION INSPECTION")
print("=" * 90)

print("Subset nodule:", nodule_id)
print("Patient:", patient_id)
print("Series UID:", series_uid)
print("XML:", xml_path)

tree = ET.parse(xml_path)
root = tree.getroot()

def tag_name(elem):
    return elem.tag.split("}")[-1]

# ------------------------------------------------------------
# Find every unblindedReadNodule
# ------------------------------------------------------------

read_nodules = []

for elem in root.iter():

    if tag_name(elem).lower() == "unblindedreadnodule":
        read_nodules.append(elem)

print("\nNumber of unblindedReadNodule elements:", len(read_nodules))

# ------------------------------------------------------------
# Inspect each nodule's important information
# ------------------------------------------------------------

records = []

for idx, nodule in enumerate(read_nodules):

    nodule_id_xml = None
    malignancy = None

    image_uids = []
    coords = []

    # Search descendants
    for elem in nodule.iter():

        tag = tag_name(elem).lower()
        text = (elem.text or "").strip()

        if tag == "noduleid":
            nodule_id_xml = text

        elif tag == "malignancy":
            if text:
                try:
                    malignancy = int(text)
                except:
                    malignancy = None

        elif tag.lower() == "imagesop_uid".lower():
            if text:
                image_uids.append(text)

        elif tag == "imagesopuid":
            if text:
                image_uids.append(text)

        elif tag == "imagezposition":
            if text:
                coords.append(("z", text))

        elif tag == "xcoord":
            if text:
                coords.append(("x", text))

        elif tag == "ycoord":
            if text:
                coords.append(("y", text))

        elif tag == "zcoord":
            if text:
                coords.append(("zcoord", text))

    records.append({
        "xml_index": idx,
        "xml_nodule_id": nodule_id_xml,
        "malignancy": malignancy,
        "image_uid_count": len(image_uids),
        "image_uids": image_uids[:5],
        "coordinates": coords[:10]
    })

annotation_df = pd.DataFrame(records)

print("\n" + "=" * 90)
print("XML NODULE-LEVEL ANNOTATIONS")
print("=" * 90)

print(
    annotation_df.to_string(index=False)
)

# ------------------------------------------------------------
# Also show the raw structure of the first nodule
# ------------------------------------------------------------

if len(read_nodules) > 0:

    print("\n" + "=" * 90)
    print("RAW STRUCTURE OF FIRST unblindedReadNodule")
    print("=" * 90)

    first = read_nodules[0]

    for elem in first.iter():

        depth = 0

        # Approximate indentation from ancestry
        current = elem

        print(
            f"{tag_name(elem):<35} | "
            f"{(elem.text or '').strip()}"
        )

print("=" * 90)

STEP 12 — DETAILED XML SPATIAL ANNOTATION INSPECTION
Subset nodule: nodule_001
Patient: LIDC-IDRI-0121
Series UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.225515255547637437801620523312
XML: /content/lidc_xml_annotations/tcia-lidc-xml/186/004.xml

Number of unblindedReadNodule elements: 21

XML NODULE-LEVEL ANNOTATIONS
 xml_index xml_nodule_id  malignancy  image_uid_count                                                                                                                                                                                                                                                                                                                                 image_uids                                                                                               coordinates
         0    Nodule 001         4.0               12 [1.3.6.1.4.1.14519.5.2.1.6279.6001.318927747207774662305497109396, 1.3.6.1.4.1.14519.5.2.1.6279.6001.127274611966926234789863085436, 1.3.6.1

In [ ]:
# ============================================================
# STEP 13 — INSPECT AVAILABLE EXACT-MATCH SOURCE INFORMATION
# ============================================================

import pandas as pd
import os

MAPPING_PATH = "/content/nodule_to_lidc_mapping.csv"

df = pd.read_csv(MAPPING_PATH)

print("=" * 90)
print("STEP 13 — INSPECTING EXACT-MATCH MAPPING INFORMATION")
print("=" * 90)

print("Rows:", len(df))
print("\nColumns found:")

for col in df.columns:
    print(" -", col)

print("\n" + "=" * 90)
print("FIRST 5 ROWS")
print("=" * 90)

print(df.head().to_string(index=False))

# Look specifically for columns that could contain
# original image/source file information.

keywords = [
    "path",
    "file",
    "source",
    "original",
    "image",
    "slice",
    "match",
    "uid",
    "sop"
]

candidate_columns = [
    c for c in df.columns
    if any(k in c.lower() for k in keywords)
]

print("\n" + "=" * 90)
print("POTENTIALLY USEFUL SOURCE COLUMNS")
print("=" * 90)

for c in candidate_columns:
    print(" -", c)

print("=" * 90)

STEP 13 — INSPECTING EXACT-MATCH MAPPING INFORMATION
Rows: 327

Columns found:
 - subset_nodule_id
 - subset_slice_count
 - matched_slice_count
 - patient_id
 - native_nodule_id
 - best_match_count
 - second_best_match_count
 - status

FIRST 5 ROWS
subset_nodule_id  subset_slice_count  matched_slice_count     patient_id native_nodule_id  best_match_count  second_best_match_count             status
      nodule_001                   6                    6 LIDC-IDRI-0121         nodule-1                 6                        0 EXACT_UNIQUE_MATCH
      nodule_002                   6                    6 LIDC-IDRI-0158         nodule-1                 6                        0 EXACT_UNIQUE_MATCH
      nodule_003                   6                    6 LIDC-IDRI-0702         nodule-3                 6                        0 EXACT_UNIQUE_MATCH
      nodule_004                   2                    2 LIDC-IDRI-0241         nodule-1                 2                        0 EXACT_UNIQ

In [ ]:
# ============================================================
# STEP 14 — INSPECT ORIGINAL 13K DATASET FILE NAMES / PATHS
# ============================================================

import os
import zipfile
import re

ORIGINAL_ZIP = "/content/kagl_lidc_idri.zip"

print("=" * 90)
print("STEP 14 — INSPECTING ORIGINAL DATASET STRUCTURE")
print("=" * 90)

if not os.path.exists(ORIGINAL_ZIP):
    raise FileNotFoundError(
        f"Could not find: {ORIGINAL_ZIP}\n"
        "Change ORIGINAL_ZIP to the actual path of your original 13k-image ZIP."
    )

with zipfile.ZipFile(ORIGINAL_ZIP, "r") as z:

    names = [
        n for n in z.namelist()
        if not n.endswith("/")
    ]

print("Total files:", len(names))

print("\nFirst 100 files:")
for n in names[:100]:
    print(n)

# ------------------------------------------------------------
# Search for anything that looks like metadata / identifiers
# ------------------------------------------------------------

keywords = [
    "LIDC",
    "patient",
    "series",
    "study",
    "sop",
    "uid",
    "nodule",
    "annotation",
    "xml",
    "csv",
    "json",
    "dicom",
    ".dcm",
    ".nii",
    ".mha"
]

hits = []

for n in names:
    lower = n.lower()

    if any(k.lower() in lower for k in keywords):
        hits.append(n)

print("\n" + "=" * 90)
print("POTENTIALLY RELEVANT FILES")
print("=" * 90)

print("Number of potential metadata/identifier files:", len(hits))

for n in hits[:200]:
    print(n)

print("=" * 90)

STEP 14 — INSPECTING ORIGINAL DATASET STRUCTURE
Total files: 77740

First 100 files:
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/images/slice-0.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/images/slice-1.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/images/slice-2.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/images/slice-3.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/images/slice-4.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/images/slice-5.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/images/slice-6.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/images/slice-7.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/images/slice-8.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/mask-0/slice-0.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/mask-0/slice-1.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/mask-0/slice-2.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/mask-0/slice-3.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/mask-0/slice-4.png
LIDC-IDRI-slices/LIDC-IDRI-0001/nodule-0/mask-0/slice-5.png
LIDC-IDRI-slice

In [ ]:
import os

print("ZIP files in /content:")
for f in os.listdir("/content"):
    if f.lower().endswith(".zip"):
        print(f)

ZIP files in /content:
kaggle_dataset_2000.zip
kagl_lidc_idri.zip
LIDC-XML-only.zip


In [ ]:
# ============================================================
# STEP 15 — VERIFY ORIGINAL DATASET CONTAINS ALL 327 MATCHES
# ============================================================

import zipfile
import pandas as pd
import re

ORIGINAL_ZIP = "/content/kagl_lidc_idri.zip"
MAPPING_PATH = "/content/nodule_to_lidc_mapping.csv"

mapping = pd.read_csv(MAPPING_PATH)

# Build the expected original folder path for every mapped nodule
expected = []

for _, row in mapping.iterrows():
    patient = str(row["patient_id"]).strip()
    native = str(row["native_nodule_id"]).strip()

    path = (
        f"LIDC-IDRI-slices/"
        f"{patient}/"
        f"{native}/images/"
    )

    expected.append({
        "subset_nodule_id": row["subset_nodule_id"],
        "patient_id": patient,
        "native_nodule_id": native,
        "expected_path": path
    })

expected_df = pd.DataFrame(expected)

# ------------------------------------------------------------
# Inspect ZIP paths
# ------------------------------------------------------------

with zipfile.ZipFile(ORIGINAL_ZIP, "r") as z:
    zip_names = set(z.namelist())

# ------------------------------------------------------------
# Verify every mapped folder exists
# ------------------------------------------------------------

found = []
missing = []

for _, row in expected_df.iterrows():

    prefix = row["expected_path"]

    matching_files = [
        name for name in zip_names
        if name.startswith(prefix) and name.lower().endswith(".png")
    ]

    if len(matching_files) > 0:
        found.append({
            **row.to_dict(),
            "original_image_count": len(matching_files),
            "status": "FOUND"
        })
    else:
        missing.append({
            **row.to_dict(),
            "original_image_count": 0,
            "status": "MISSING"
        })

found_df = pd.DataFrame(found)
missing_df = pd.DataFrame(missing)

print("=" * 90)
print("STEP 15 — ORIGINAL DATASET PROVENANCE VERIFICATION")
print("=" * 90)

print(f"Subset nodules checked: {len(expected_df)}")
print(f"Original folders found: {len(found_df)}")
print(f"Original folders missing: {len(missing_df)}")

if len(missing_df) > 0:
    print("\nMISSING EXAMPLES:")
    print(missing_df.head(20).to_string(index=False))
else:
    print("\n✓ ALL 327 MAPPED NODULE FOLDERS EXIST IN THE ORIGINAL DATASET.")

print("\nFirst 20 verified mappings:")
print(found_df.head(20).to_string(index=False))

# Save verification
OUTPUT = "/content/verified_original_provenance.csv"
found_df.to_csv(OUTPUT, index=False)

print("\nSaved:")
print(OUTPUT)

print("=" * 90)

STEP 15 — ORIGINAL DATASET PROVENANCE VERIFICATION
Subset nodules checked: 327
Original folders found: 327
Original folders missing: 0

✓ ALL 327 MAPPED NODULE FOLDERS EXIST IN THE ORIGINAL DATASET.

First 20 verified mappings:
subset_nodule_id     patient_id native_nodule_id                                    expected_path  original_image_count status
      nodule_001 LIDC-IDRI-0121         nodule-1 LIDC-IDRI-slices/LIDC-IDRI-0121/nodule-1/images/                     6  FOUND
      nodule_002 LIDC-IDRI-0158         nodule-1 LIDC-IDRI-slices/LIDC-IDRI-0158/nodule-1/images/                     6  FOUND
      nodule_003 LIDC-IDRI-0702         nodule-3 LIDC-IDRI-slices/LIDC-IDRI-0702/nodule-3/images/                     6  FOUND
      nodule_004 LIDC-IDRI-0241         nodule-1 LIDC-IDRI-slices/LIDC-IDRI-0241/nodule-1/images/                     2  FOUND
      nodule_005 LIDC-IDRI-0008         nodule-0 LIDC-IDRI-slices/LIDC-IDRI-0008/nodule-0/images/                     2  FOUND
      nodu

In [ ]:
import os
import pandas as pd

SINGLE_SERIES_CSV = "/content/nodule_to_single_ct_series_fixed.csv"
MAPPING_CSV = "/content/nodule_to_lidc_mapping.csv"

print("=" * 80)
print("FIXING SERIES CSV COLUMN NAMES")
print("=" * 80)

# Load files
single_series_df = pd.read_csv(SINGLE_SERIES_CSV)
mapping_df = pd.read_csv(MAPPING_CSV)

print("\nOriginal columns in nodule_to_single_ct_series.csv:")
print(list(single_series_df.columns))

# ------------------------------------------------------------
# Standardize column names
# ------------------------------------------------------------
rename_map = {}

# Series UID
if "SeriesInstanceUID" in single_series_df.columns:
    rename_map["SeriesInstanceUID"] = "series_instance_uid"
elif "series_instance_uid" in single_series_df.columns:
    pass
else:
    raise KeyError(
        "Could not find a SeriesInstanceUID column. "
        f"Available columns: {list(single_series_df.columns)}"
    )

# Patient ID
if "PatientID" in single_series_df.columns:
    rename_map["PatientID"] = "patient_id"
elif "patient_id" in single_series_df.columns:
    pass
else:
    raise KeyError(
        "Could not find a patient ID column. "
        f"Available columns: {list(single_series_df.columns)}"
    )

single_series_df = single_series_df.rename(columns=rename_map)

# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------
required = {
    "subset_nodule_id",
    "patient_id",
    "series_instance_uid"
}

missing = required - set(single_series_df.columns)

if missing:
    raise KeyError(
        f"Still missing required columns: {missing}\n"
        f"Available columns: {list(single_series_df.columns)}"
    )

# ------------------------------------------------------------
# Clean values
# ------------------------------------------------------------
for col in ["subset_nodule_id", "patient_id", "series_instance_uid"]:
    single_series_df[col] = (
        single_series_df[col]
        .astype(str)
        .str.strip()
    )

# ------------------------------------------------------------
# Save corrected version
# ------------------------------------------------------------
FIXED_CSV = "/content/nodule_to_single_ct_series_fixed.csv"
single_series_df.to_csv(FIXED_CSV, index=False)

print("\nCorrected columns:")
print(list(single_series_df.columns))

print(f"\nRows: {len(single_series_df)}")
print(f"Unique subset nodules: {single_series_df['subset_nodule_id'].nunique()}")
print(f"Unique patients: {single_series_df['patient_id'].nunique()}")
print(f"Unique SeriesInstanceUIDs: {single_series_df['series_instance_uid'].nunique()}")

print("\nFirst 5 rows:")
print(
    single_series_df[
        ["subset_nodule_id", "patient_id", "series_instance_uid"]
    ].head().to_string(index=False)
)

print(f"\nSaved corrected file to: {FIXED_CSV}")
print("=" * 80)

FIXING SERIES CSV COLUMN NAMES

Original columns in nodule_to_single_ct_series.csv:
['subset_nodule_id', 'patient_id', 'native_nodule_id', 'series_instance_uid']

Corrected columns:
['subset_nodule_id', 'patient_id', 'native_nodule_id', 'series_instance_uid']

Rows: 325
Unique subset nodules: 325
Unique patients: 247
Unique SeriesInstanceUIDs: 247

First 5 rows:
subset_nodule_id     patient_id                                              series_instance_uid
      nodule_001 LIDC-IDRI-0121 1.3.6.1.4.1.14519.5.2.1.6279.6001.225515255547637437801620523312
      nodule_002 LIDC-IDRI-0158 1.3.6.1.4.1.14519.5.2.1.6279.6001.244204120220889433826451158706
      nodule_003 LIDC-IDRI-0702 1.3.6.1.4.1.14519.5.2.1.6279.6001.286422846896797433168187085942
      nodule_004 LIDC-IDRI-0241 1.3.6.1.4.1.14519.5.2.1.6279.6001.154703816225841204080664115280
      nodule_005 LIDC-IDRI-0008 1.3.6.1.4.1.14519.5.2.1.6279.6001.774060103415303828812229821954

Saved corrected file to: /content/nodule_to_single_c

In [ ]:
import os
import pandas as pd

# ============================================================
# FIX THE SERIES CSV
# ============================================================

ORIGINAL_CSV = "/content/nodule_to_single_ct_series.csv"
FIXED_CSV = "/content/nodule_to_single_ct_series_fixed.csv"

print("=" * 80)
print("CREATING CLEAN SERIES MAPPING CSV")
print("=" * 80)

if not os.path.exists(ORIGINAL_CSV):
    raise FileNotFoundError(f"Could not find {ORIGINAL_CSV}")

df = pd.read_csv(ORIGINAL_CSV)

print("\nOriginal columns:")
print(list(df.columns))

# We only need these four columns for the next provenance analysis.
needed = [
    "subset_nodule_id",
    "patient_id",
    "native_nodule_id",
    "SeriesInstanceUID"
]

missing = [c for c in needed if c not in df.columns]

if missing:
    raise KeyError(
        f"Missing required columns: {missing}\n"
        f"Available columns: {list(df.columns)}"
    )

# Create a clean copy.
fixed = df[needed].copy()

# Rename the Series UID column to the exact name expected downstream.
fixed.rename(
    columns={"SeriesInstanceUID": "series_instance_uid"},
    inplace=True
)

# Clean all string columns.
for col in fixed.columns:
    fixed[col] = fixed[col].fillna("").astype(str).str.strip()

# ============================================================
# VALIDATION
# ============================================================

print("\nCleaned columns:")
print(list(fixed.columns))

print(f"Rows: {len(fixed)}")
print(f"Unique subset nodules: {fixed['subset_nodule_id'].nunique()}")
print(f"Unique patients: {fixed['patient_id'].nunique()}")
print(f"Unique SeriesInstanceUIDs: {fixed['series_instance_uid'].nunique()}")

duplicate_subset_ids = fixed["subset_nodule_id"].duplicated().sum()
blank_series_uids = (fixed["series_instance_uid"] == "").sum()

print(f"Duplicate subset_nodule_id rows: {duplicate_subset_ids}")
print(f"Blank series_instance_uid values: {blank_series_uids}")

if len(fixed) != 327:
    print("\nWARNING: Expected 327 rows.")

if fixed["subset_nodule_id"].nunique() != 327:
    print("WARNING: Expected 327 unique subset nodule IDs.")

if blank_series_uids > 0:
    print("WARNING: Some SeriesInstanceUID values are blank.")

print("\nFirst 10 rows:")
print(fixed.head(10).to_string(index=False))

# Save clean CSV.
fixed.to_csv(FIXED_CSV, index=False)

print("\nSaved clean file:")
print(FIXED_CSV)

print("=" * 80)
print("DONE")
print("=" * 80)

CREATING CLEAN SERIES MAPPING CSV

Original columns:
['subset_nodule_id', 'subset_slice_count', 'matched_slice_count', 'patient_id', 'native_nodule_id', 'best_match_count', 'second_best_match_count', 'status', 'SeriesInstanceUID', 'StudyInstanceUID', 'Modality', 'ProtocolName', 'SeriesDate', 'SeriesDescription', 'BodyPartExamined', 'SeriesNumber', 'AnnotationsFlag', 'Collection', 'PatientID', 'Manufacturer', 'ManufacturerModelName', 'SoftwareVersions', 'ImageCount', 'TimeStamp', 'LicenseName', 'LicenseURI', 'CollectionURI', 'FileSize', 'DateReleased', 'StudyDesc', 'StudyDate', 'ThirdPartyAnalysis', 'has_annotations']

Cleaned columns:
['subset_nodule_id', 'patient_id', 'native_nodule_id', 'series_instance_uid']
Rows: 325
Unique subset nodules: 325
Unique patients: 247
Unique SeriesInstanceUIDs: 247
Duplicate subset_nodule_id rows: 0
Blank series_instance_uid values: 0


First 10 rows:
subset_nodule_id     patient_id native_nodule_id                                              series_i

In [ ]:
import os
import glob
import re
import zipfile
import itertools
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

# ============================================================================
# FILE PATHS
# ============================================================================

ZIP_PATH = "/content/kagl_lidc_idri.zip"
XML_DIR = "/content/lidc_xml_annotations"
MAPPING_CSV = "/content/nodule_to_lidc_mapping.csv"

# IMPORTANT: use the corrected file created by the previous cell
SINGLE_SERIES_CSV = "/content/nodule_to_single_ct_series_fixed.csv"

TCIA_CSV = "/content/tcia_series_to_patient_mapping.csv"

OUT_CSV = "/content/native_xml_order_test_v5.csv"
OUT_SUMMARY = "/content/native_xml_order_summary_v5.txt"


# ============================================================================
# MAIN ANALYSIS
# ============================================================================

def main():

    print("=" * 80)
    print("EMPIRICAL PROVENANCE INVESTIGATION (v5)")
    print("STRICT HIERARCHICAL ANALYSIS")
    print("=" * 80)

    # ------------------------------------------------------------------------
    # STEP 1 — CHECK FILES
    # ------------------------------------------------------------------------

    required_files = [
        (MAPPING_CSV, "nodule_to_lidc_mapping.csv"),
        (SINGLE_SERIES_CSV, "nodule_to_single_ct_series_fixed.csv"),
        (TCIA_CSV, "tcia_series_to_patient_mapping.csv"),
        (ZIP_PATH, "kagl_lidc_idri.zip"),
        (XML_DIR, "LIDC XML directory")
    ]

    for path, name in required_files:
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"CRITICAL ERROR: Missing {name}\nPath: {path}"
            )

    print("\nAll required files found.")

    # ------------------------------------------------------------------------
    # STEP 2 — LOAD CSV FILES
    # ------------------------------------------------------------------------

    mapping_df = pd.read_csv(MAPPING_CSV)
    single_series_df = pd.read_csv(SINGLE_SERIES_CSV)
    tcia_df = pd.read_csv(TCIA_CSV)

    print("\n" + "-" * 70)
    print("INPUT FILE SUMMARY")
    print("-" * 70)

    print(f"Original subset mapping rows:      {len(mapping_df)}")
    print(f"Clean CT-series mapping rows:      {len(single_series_df)}")
    print(f"TCIA metadata rows:                {len(tcia_df)}")

    print("\nCorrected CT-series CSV columns:")
    print(list(single_series_df.columns))

    # ------------------------------------------------------------------------
    # STEP 3 — STRICT COLUMN CHECK
    # ------------------------------------------------------------------------

    required_single = {
        "subset_nodule_id",
        "patient_id",
        "native_nodule_id",
        "series_instance_uid"
    }

    required_mapping = {
        "subset_nodule_id",
        "patient_id",
        "native_nodule_id"
    }

    missing_single = required_single - set(single_series_df.columns)
    missing_mapping = required_mapping - set(mapping_df.columns)

    if missing_single:
        raise KeyError(
            f"Missing columns in corrected series CSV: {missing_single}"
        )

    if missing_mapping:
        raise KeyError(
            f"Missing columns in subset mapping CSV: {missing_mapping}"
        )

    # Normalize strings
    for col in ["subset_nodule_id", "patient_id", "native_nodule_id"]:
        mapping_df[col] = mapping_df[col].astype(str).str.strip()
        single_series_df[col] = single_series_df[col].astype(str).str.strip()

    single_series_df["series_instance_uid"] = (
        single_series_df["series_instance_uid"]
        .astype(str)
        .str.strip()
    )

    # ------------------------------------------------------------------------
    # STEP 4 — MERGE THE 325 CLEAN MAPPINGS
    # ------------------------------------------------------------------------

    merged = pd.merge(
        mapping_df,
        single_series_df[
            [
                "subset_nodule_id",
                "patient_id",
                "native_nodule_id",
                "series_instance_uid"
            ]
        ],
        on="subset_nodule_id",
        how="inner",
        suffixes=("_mapping", "_series")
    )

    # Keep patient/native IDs from original mapping
    merged["patient_id"] = merged["patient_id_mapping"]
    merged["native_nodule_id"] = merged["native_nodule_id_mapping"]

    merged = merged.drop(
        columns=[
            "patient_id_mapping",
            "patient_id_series",
            "native_nodule_id_mapping",
            "native_nodule_id_series"
        ],
        errors="ignore"
    )

    print("\n" + "-" * 70)
    print("MERGED CLEAN MAPPING")
    print("-" * 70)

    print(f"Clean mapped subset nodules: {len(merged)}")
    print(f"Unique patients:              {merged['patient_id'].nunique()}")
    print(
        f"Unique SeriesInstanceUIDs:    "
        f"{merged['series_instance_uid'].nunique()}"
    )

    # ------------------------------------------------------------------------
    # STEP 5 — BUILD SUBSET LOOKUP
    # ------------------------------------------------------------------------

    subset_lookup = {}

    for _, row in merged.iterrows():
        key = (
            str(row["patient_id"]).strip(),
            str(row["native_nodule_id"]).strip()
        )

        subset_lookup[key] = {
            "subset_nodule_id": row["subset_nodule_id"],
            "series_instance_uid": row["series_instance_uid"]
        }

    print(
        f"Subset lookup entries: {len(subset_lookup)}"
    )

    # ------------------------------------------------------------------------
    # STEP 6 — PATIENT → NUMBER OF TCIA SERIES
    # ------------------------------------------------------------------------

    # TCIA file uses PatientID / SeriesInstanceUID
    if "PatientID" in tcia_df.columns:
        tcia_patient_col = "PatientID"
    elif "patient_id" in tcia_df.columns:
        tcia_patient_col = "patient_id"
    else:
        raise KeyError(
            "Could not find PatientID column in TCIA CSV."
        )

    if "SeriesInstanceUID" in tcia_df.columns:
        tcia_series_col = "SeriesInstanceUID"
    elif "series_instance_uid" in tcia_df.columns:
        tcia_series_col = "series_instance_uid"
    else:
        raise KeyError(
            "Could not find SeriesInstanceUID column in TCIA CSV."
        )

    tcia_df[tcia_patient_col] = (
        tcia_df[tcia_patient_col].astype(str).str.strip()
    )

    tcia_df[tcia_series_col] = (
        tcia_df[tcia_series_col].astype(str).str.strip()
    )

    patient_series_counts = (
        tcia_df.groupby(tcia_patient_col)[tcia_series_col]
        .nunique()
        .to_dict()
    )

    multi_series_patient_count = sum(
        1 for _, count in patient_series_counts.items()
        if count > 1
    )

    print(
        f"Patients with >1 TCIA series: {multi_series_patient_count}"
    )

    # ------------------------------------------------------------------------
    # STEP 7 — SCAN ORIGINAL ZIP
    # ------------------------------------------------------------------------

    print("\n" + "-" * 70)
    print("STEP 1 — INDEXING ORIGINAL ZIP")
    print("-" * 70)

    zip_patient_folders = {}
    zip_explicit_series_map = {}
    non_png_files = []

    with zipfile.ZipFile(ZIP_PATH, "r") as zf:

        for info in zf.infolist():

            filename = info.filename

            if info.is_dir():
                continue

            # Record non-PNG files
            if not filename.lower().endswith(".png"):
                non_png_files.append(
                    (filename, info.file_size)
                )

            parts = filename.split("/")

            # Expected:
            # LIDC-IDRI-slices / PATIENT / nodule-X / ...

            if len(parts) >= 3 and parts[0] == "LIDC-IDRI-slices":

                patient_id = parts[1]

                # Normal structure
                if parts[2].startswith("nodule-"):

                    native_id = parts[2]

                    if patient_id not in zip_patient_folders:
                        zip_patient_folders[patient_id] = set()

                    zip_patient_folders[patient_id].add(native_id)

                # Possible explicit Series directory
                elif len(parts) >= 4 and parts[3].startswith("nodule-"):

                    possible_series = parts[2]
                    native_id = parts[3]

                    key = (
                        patient_id,
                        possible_series
                    )

                    if key not in zip_explicit_series_map:
                        zip_explicit_series_map[key] = set()

                    zip_explicit_series_map[key].add(native_id)

    # Sort nodule folders naturally
    for patient_id in zip_patient_folders:

        zip_patient_folders[patient_id] = sorted(
            zip_patient_folders[patient_id],
            key=lambda x: (
                int(x.split("-")[1])
                if "-" in x and x.split("-")[1].isdigit()
                else 999999
            )
        )

    print(
        f"Patients found in original ZIP: "
        f"{len(zip_patient_folders)}"
    )

    print(
        f"Explicit SeriesInstanceUID folders found: "
        f"{len(zip_explicit_series_map)}"
    )

    print(
        f"Non-PNG files inside original ZIP: "
        f"{len(non_png_files)}"
    )

    if non_png_files:
        print("\nNon-PNG files:")
        for fname, size in non_png_files[:20]:
            print(f"  {fname} ({size} bytes)")

        if len(non_png_files) > 20:
            print(
                f"  ... plus {len(non_png_files)-20} more"
            )

    # ------------------------------------------------------------------------
    # STEP 8 — INDEX XML FILES BY EXACT SERIES UID
    # ------------------------------------------------------------------------

    print("\n" + "-" * 70)
    print("STEP 2 — INDEXING XML ANNOTATIONS")
    print("-" * 70)

    xml_files = glob.glob(
        os.path.join(XML_DIR, "**", "*.xml"),
        recursive=True
    )

    print(f"XML files discovered: {len(xml_files)}")

    xml_by_suid = {}

    xml_parse_errors = 0

    for xml_path in xml_files:

        try:

            tree = ET.parse(xml_path)
            root = tree.getroot()

            namespace = (
                root.tag.split("}")[0].strip("{")
                if "}" in root.tag
                else None
            )

            if namespace:
                ns = {"lidc": namespace}
                suid_elem = root.find(
                    ".//lidc:SeriesInstanceUid",
                    ns
                )

                if suid_elem is None:
                    suid_elem = root.find(
                        ".//lidc:SeriesInstanceUID",
                        ns
                    )
            else:
                ns = {}
                suid_elem = root.find(
                    ".//SeriesInstanceUid"
                )

                if suid_elem is None:
                    suid_elem = root.find(
                        ".//SeriesInstanceUID"
                    )

            if (
                suid_elem is not None
                and suid_elem.text
            ):

                suid = suid_elem.text.strip()

                if suid not in xml_by_suid:
                    xml_by_suid[suid] = []

                xml_by_suid[suid].append(xml_path)

        except Exception:
            xml_parse_errors += 1

    duplicate_xml_suids = sum(
        1
        for _, files in xml_by_suid.items()
        if len(files) > 1
    )

    print(
        f"Unique SeriesInstanceUIDs in XML: "
        f"{len(xml_by_suid)}"
    )

    print(
        f"XML parse errors: {xml_parse_errors}"
    )

    print(
        f"SeriesInstanceUIDs with >1 XML: "
        f"{duplicate_xml_suids}"
    )

    # ------------------------------------------------------------------------
    # STEP 9 — DETERMINE SERIES TARGETS
    # ------------------------------------------------------------------------

    unique_series_targets = (
        merged[
            [
                "patient_id",
                "series_instance_uid"
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    print("\n" + "-" * 70)
    print("STEP 3 — SERIES TARGET SUMMARY")
    print("-" * 70)

    print(
        f"Unique patient/series targets: "
        f"{len(unique_series_targets)}"
    )

    # ------------------------------------------------------------------------
    # STEP 10 — COMPLETE NATIVE ENUMERATION
    # ------------------------------------------------------------------------

    print("\n" + "-" * 70)
    print("STEP 4 — COMPLETE NATIVE ENUMERATION")
    print("-" * 70)

    out_records = []

    total_series_evaluated = 0
    established_membership_count = 0
    undetermined_membership_count = 0
    count_mismatch_count = 0

    valid_ordering_series_count = 0

    full_doc_sequence_matches = 0
    full_asc_z_sequence_matches = 0
    full_desc_z_sequence_matches = 0

    total_pairs_evaluated = 0
    doc_pair_agreements = 0
    asc_z_pair_agreements = 0
    desc_z_pair_agreements = 0

    # ------------------------------------------------------------------------
    # PROCESS EVERY EXACT PATIENT + SERIES
    # ------------------------------------------------------------------------

    for _, target in unique_series_targets.iterrows():

        patient_id = str(
            target["patient_id"]
        ).strip()

        series_uid = str(
            target["series_instance_uid"]
        ).strip()

        total_series_evaluated += 1

        # Number of TCIA series for this patient
        patient_series_count = patient_series_counts.get(
            patient_id,
            1
        )

        # -------------------------------------------------------------
        # SERIES MEMBERSHIP
        # -------------------------------------------------------------

        native_folders = []

        if (
            patient_id,
            series_uid
        ) in zip_explicit_series_map:

            native_folders = sorted(
                list(
                    zip_explicit_series_map[
                        (patient_id, series_uid)
                    ]
                ),
                key=lambda x: int(x.split("-")[1])
            )

            membership_status = (
                "SERIES_MEMBERSHIP_ESTABLISHED"
            )

            membership_established = True

        elif patient_series_count == 1:

            native_folders = sorted(
                zip_patient_folders.get(
                    patient_id,
                    []
                ),
                key=lambda x: int(x.split("-")[1])
            )

            membership_status = (
                "SERIES_MEMBERSHIP_INFERRED_SINGLE_SERIES"
            )

            membership_established = True

        else:

            native_folders = sorted(
                zip_patient_folders.get(
                    patient_id,
                    []
                ),
                key=lambda x: int(x.split("-")[1])
            )

            membership_status = (
                "SERIES_MEMBERSHIP_UNDETERMINED"
            )

            membership_established = False
            undetermined_membership_count += 1

        if membership_established:
            established_membership_count += 1

        # -------------------------------------------------------------
        # FIND XML
        # -------------------------------------------------------------

        xml_file_list = xml_by_suid.get(
            series_uid,
            []
        )

        xml_nodules = []

        if xml_file_list:

            xml_path = xml_file_list[0]

            try:

                tree = ET.parse(xml_path)
                root = tree.getroot()

                namespace = (
                    root.tag.split("}")[0].strip("{")
                    if "}" in root.tag
                    else None
                )

                if namespace:
                    ns = {"lidc": namespace}

                    nod_nodes = root.findall(
                        ".//lidc:unblindedReadNodule",
                        ns
                    )

                else:

                    ns = {}

                    nod_nodes = root.findall(
                        ".//unblindedReadNodule"
                    )

                for xml_index, nod in enumerate(
                    nod_nodes
                ):

                    if namespace:

                        id_elem = nod.find(
                            "lidc:noduleID",
                            ns
                        )

                        rois = nod.findall(
                            "lidc:roi",
                            ns
                        )

                    else:

                        id_elem = nod.find(
                            "noduleID"
                        )

                        rois = nod.findall(
                            "roi"
                        )

                    if (
                        id_elem is not None
                        and id_elem.text
                    ):
                        xml_id = id_elem.text.strip()
                    else:
                        xml_id = (
                            f"XML_{xml_index}"
                        )

                    z_values = []

                    for roi in rois:

                        if namespace:
                            z_elem = roi.find(
                                "lidc:imageZposition",
                                ns
                            )
                        else:
                            z_elem = roi.find(
                                "imageZposition"
                            )

                        if (
                            z_elem is not None
                            and z_elem.text
                        ):

                            try:
                                z_values.append(
                                    float(
                                        z_elem.text.strip()
                                    )
                                )
                            except ValueError:
                                pass

                    # Only consider XML nodules
                    # that actually have ROI/z data
                    if z_values:

                        xml_nodules.append({
                            "xml_nodule_id": xml_id,
                            "xml_doc_rank": xml_index,
                            "mean_z": float(
                                np.mean(z_values)
                            ),
                            "min_z": float(
                                np.min(z_values)
                            ),
                            "max_z": float(
                                np.max(z_values)
                            ),
                            "roi_count": len(rois)
                        })

            except Exception:
                xml_nodules = []

        # -------------------------------------------------------------
        # CALCULATE XML Z RANKS
        # -------------------------------------------------------------

        if xml_nodules:

            asc_sorted = sorted(
                xml_nodules,
                key=lambda x: x["mean_z"]
            )

            desc_sorted = sorted(
                xml_nodules,
                key=lambda x: x["mean_z"],
                reverse=True
            )

            for item in xml_nodules:

                item["xml_asc_z_rank"] = (
                    asc_sorted.index(item)
                )

                item["xml_desc_z_rank"] = (
                    desc_sorted.index(item)
                )

        # -------------------------------------------------------------
        # COUNT MATCH
        # -------------------------------------------------------------

        count_match = (
            len(native_folders)
            == len(xml_nodules)
        )

        if (
            membership_established
            and not count_match
        ):
            count_mismatch_count += 1

        # -------------------------------------------------------------
        # ORDERING TEST ELIGIBILITY
        # -------------------------------------------------------------

        eligible = (
            membership_established
            and count_match
            and len(native_folders) >= 2
            and len(xml_nodules) >= 2
        )

        # -------------------------------------------------------------
        # RECORD ALL NATIVE FOLDERS
        # -------------------------------------------------------------

        xml_ids_string = (
            ";".join(
                x["xml_nodule_id"]
                for x in xml_nodules
            )
            if xml_nodules
            else "NONE"
        )

        xml_doc_rank_string = (
            ";".join(
                str(x["xml_doc_rank"])
                for x in xml_nodules
            )
            if xml_nodules
            else "NONE"
        )

        xml_asc_rank_string = (
            ";".join(
                str(x["xml_asc_z_rank"])
                for x in xml_nodules
            )
            if xml_nodules
            else "NONE"
        )

        xml_desc_rank_string = (
            ";".join(
                str(x["xml_desc_z_rank"])
                for x in xml_nodules
            )
            if xml_nodules
            else "NONE"
        )

        for native_id in native_folders:

            try:
                native_index = int(
                    native_id.split("-")[1]
                )
            except Exception:
                native_index = -1

            subset_info = subset_lookup.get(
                (patient_id, native_id)
            )

            in_subset = (
                subset_info is not None
            )

            subset_id = (
                subset_info["subset_nodule_id"]
                if in_subset
                else "NONE"
            )

            out_records.append({
                "patient_id": patient_id,
                "series_instance_uid": series_uid,
                "patient_ct_series_count": patient_series_count,
                "series_membership_status": membership_status,
                "native_nodule_id": native_id,
                "native_index": native_index,
                "in_subset": in_subset,
                "subset_nodule_id": subset_id,
                "xml_candidate_list": xml_ids_string,
                "xml_doc_ranks": xml_doc_rank_string,
                "xml_asc_z_ranks": xml_asc_rank_string,
                "xml_desc_z_ranks": xml_desc_rank_string,
                "xml_nodule_count": len(xml_nodules),
                "native_nodule_count": len(native_folders),
                "eligible_for_ordering_test": eligible
            })

        # -------------------------------------------------------------
        # ORDERING ANALYSIS
        # -------------------------------------------------------------

        if eligible:

            valid_ordering_series_count += 1

            sorted_asc = sorted(
                xml_nodules,
                key=lambda x: x["mean_z"]
            )

            sorted_desc = sorted(
                xml_nodules,
                key=lambda x: x["mean_z"],
                reverse=True
            )

            doc_match = True
            asc_match = True
            desc_match = True

            for i in range(
                len(native_folders)
            ):

                # XML document sequence
                if (
                    xml_nodules[i]["xml_doc_rank"]
                    != i
                ):
                    doc_match = False

                # Z ascending
                if (
                    xml_nodules[i]["xml_nodule_id"]
                    != sorted_asc[i]["xml_nodule_id"]
                ):
                    asc_match = False

                # Z descending
                if (
                    xml_nodules[i]["xml_nodule_id"]
                    != sorted_desc[i]["xml_nodule_id"]
                ):
                    desc_match = False

            if doc_match:
                full_doc_sequence_matches += 1

            if asc_match:
                full_asc_z_sequence_matches += 1

            if desc_match:
                full_desc_z_sequence_matches += 1

            # Pairwise order preservation
            for i, j in itertools.combinations(
                range(len(native_folders)),
                2
            ):

                total_pairs_evaluated += 1

                if (
                    xml_nodules[i]["xml_doc_rank"]
                    <
                    xml_nodules[j]["xml_doc_rank"]
                ):
                    doc_pair_agreements += 1

                if (
                    xml_nodules[i]["xml_asc_z_rank"]
                    <
                    xml_nodules[j]["xml_asc_z_rank"]
                ):
                    asc_z_pair_agreements += 1

                if (
                    xml_nodules[i]["xml_desc_z_rank"]
                    <
                    xml_nodules[j]["xml_desc_z_rank"]
                ):
                    desc_z_pair_agreements += 1

    # ------------------------------------------------------------------------
    # STEP 11 — SAVE RESULT TABLE
    # ------------------------------------------------------------------------

    result_df = pd.DataFrame(out_records)

    result_df.to_csv(
        OUT_CSV,
        index=False
    )

    print("\n" + "-" * 70)
    print("RESULT TABLE SAVED")
    print("-" * 70)

    print(f"Output rows: {len(result_df)}")
    print(f"Saved to: {OUT_CSV}")

    # ------------------------------------------------------------------------
    # STEP 12 — CALCULATE STATISTICS
    # ------------------------------------------------------------------------

    pct_doc_seq = (
        full_doc_sequence_matches
        / valid_ordering_series_count
        * 100
        if valid_ordering_series_count > 0
        else 0
    )

    pct_asc_seq = (
        full_asc_z_sequence_matches
        / valid_ordering_series_count
        * 100
        if valid_ordering_series_count > 0
        else 0
    )

    pct_desc_seq = (
        full_desc_z_sequence_matches
        / valid_ordering_series_count
        * 100
        if valid_ordering_series_count > 0
        else 0
    )

    pct_doc_pair = (
        doc_pair_agreements
        / total_pairs_evaluated
        * 100
        if total_pairs_evaluated > 0
        else 0
    )

    pct_asc_pair = (
        asc_z_pair_agreements
        / total_pairs_evaluated
        * 100
        if total_pairs_evaluated > 0
        else 0
    )

    pct_desc_pair = (
        desc_z_pair_agreements
        / total_pairs_evaluated
        * 100
        if total_pairs_evaluated > 0
        else 0
    )

    # ------------------------------------------------------------------------
    # STEP 13 — SUMMARY REPORT
    # ------------------------------------------------------------------------

    native_subset_count = (
        int(result_df["in_subset"].sum())
        if not result_df.empty
        else 0
    )

    summary_text = f"""
===============================================================================
NATIVE XML ORDERING DIAGNOSTIC SUMMARY REPORT (v5)
===============================================================================

A. DIRECTLY ESTABLISHED
-------------------------------------------------------------------------------

1. Cohort and Mapping Integrity
   - Original subset mapping rows: {len(mapping_df)}
   - Clean CT-series mapping rows: {len(single_series_df)}
   - Clean mapped subset rows after merge: {len(merged)}
   - Unique patients in clean mapping: {merged['patient_id'].nunique()}
   - Unique selected SeriesInstanceUIDs: {merged['series_instance_uid'].nunique()}
   - Complete Series Targets Evaluated: {total_series_evaluated}

2. ZIP Archive Enumeration
   - Patients found in ZIP: {len(zip_patient_folders)}
   - Native folder rows recorded: {len(result_df)}
   - Native folders belonging to the 327-subset: {native_subset_count}
   - Non-PNG files inside ZIP: {len(non_png_files)}
   - Explicit SeriesInstanceUID subdirectories: {len(zip_explicit_series_map)}

3. XML Annotation Inventory
   - XML files discovered: {len(xml_files)}
   - Unique SeriesInstanceUIDs indexed: {len(xml_by_suid)}
   - XML parse errors: {xml_parse_errors}
   - SeriesInstanceUIDs with multiple XML files: {duplicate_xml_suids}

B. SERIES MEMBERSHIP
-------------------------------------------------------------------------------

   - Series membership established/inferred: {established_membership_count}
   - Series membership undetermined: {undetermined_membership_count}
   - Native-folder/XML-nodule count mismatches: {count_mismatch_count}

C. STRICTLY VALID ORDERING ANALYSIS
-------------------------------------------------------------------------------

Only series satisfying ALL of the following were included:

   1. Native series membership established
   2. Native folder count == XML nodule count
   3. At least 2 native nodules

   - Valid ordering series: {valid_ordering_series_count}
   - Total native nodule pairs evaluated: {total_pairs_evaluated}

1. Full Sequence Agreement
   - Native index == XML document sequence:
     {full_doc_sequence_matches} / {valid_ordering_series_count}
     ({pct_doc_seq:.1f}%)

   - Native index == Ascending physical Z order:
     {full_asc_z_sequence_matches} / {valid_ordering_series_count}
     ({pct_asc_seq:.1f}%)

   - Native index == Descending physical Z order:
     {full_desc_z_sequence_matches} / {valid_ordering_series_count}
     ({pct_desc_seq:.1f}%)

2. Pairwise Relative Order Agreement
   - XML document order:
     {doc_pair_agreements} / {total_pairs_evaluated}
     ({pct_doc_pair:.1f}%)

   - Ascending Z order:
     {asc_z_pair_agreements} / {total_pairs_evaluated}
     ({pct_asc_pair:.1f}%)

   - Descending Z order:
     {desc_z_pair_agreements} / {total_pairs_evaluated}
     ({pct_desc_pair:.1f}%)

D. INTERPRETATION
-------------------------------------------------------------------------------

1. Native directory numbering can only be considered an ordering rule if
   the ordering statistics demonstrate a consistent pattern across the
   strictly eligible complete series.

2. A correlation in ordering does NOT by itself prove that a specific
   native folder such as `nodule-1` is physically identical to a specific
   XML nodule such as `Nodule 002`.

3. Exact physical identity requires slice-level DICOM linkage such as
   SOPInstanceUID or equivalent physical-coordinate metadata.

4. The PNG files in the current archive do not contain those DICOM headers,
   so this analysis intentionally does not claim direct physical identity.

E. OUTPUT
-------------------------------------------------------------------------------

Detailed CSV:
{OUT_CSV}

Summary:
{OUT_SUMMARY}

===============================================================================
"""

    with open(
        OUT_SUMMARY,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(
            summary_text.strip()
        )

    # ------------------------------------------------------------------------
    # STEP 14 — PRINT FINAL SUMMARY
    # ------------------------------------------------------------------------

    print("\n" + summary_text)

    # ------------------------------------------------------------------------
    # STEP 15 — FINAL SANITY CHECKS
    # ------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("FINAL SANITY CHECKS")
    print("=" * 80)

    print(
        f"Clean mapping rows: "
        f"{len(single_series_df)}"
    )

    print(
        f"Mapped rows after merge: "
        f"{len(merged)}"
    )

    print(
        f"Unique patients: "
        f"{merged['patient_id'].nunique()}"
    )

    print(
        f"Unique SeriesInstanceUIDs: "
        f"{merged['series_instance_uid'].nunique()}"
    )

    print(
        f"Valid ordering series: "
        f"{valid_ordering_series_count}"
    )

    print(
        f"Output CSV exists: "
        f"{os.path.exists(OUT_CSV)}"
    )

    print(
        f"Output summary exists: "
        f"{os.path.exists(OUT_SUMMARY)}"
    )

    print("=" * 80)
    print("DONE")
    print("=" * 80)


# ============================================================================
# RUN
# ============================================================================

if __name__ == "__main__":
    main()

EMPIRICAL PROVENANCE INVESTIGATION (v5)
STRICT HIERARCHICAL ANALYSIS

All required files found.

----------------------------------------------------------------------
INPUT FILE SUMMARY
----------------------------------------------------------------------
Original subset mapping rows:      327
Clean CT-series mapping rows:      325
TCIA metadata rows:                1308

Corrected CT-series CSV columns:
['subset_nodule_id', 'patient_id', 'native_nodule_id', 'series_instance_uid']

----------------------------------------------------------------------
MERGED CLEAN MAPPING
----------------------------------------------------------------------
Clean mapped subset nodules: 325
Unique patients:              247
Unique SeriesInstanceUIDs:    247
Subset lookup entries: 325
Patients with >1 TCIA series: 296

----------------------------------------------------------------------
STEP 1 — INDEXING ORIGINAL ZIP
----------------------------------------------------------------------
Patients fou

In [ ]:
import os
import glob
import io
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from PIL import Image

# =============================================================================
# CONFIGURATION
# =============================================================================

MAPPING_CSV = "/content/nodule_to_lidc_mapping.csv"
SINGLE_SERIES_CSV = "/content/nodule_to_single_ct_series_fixed.csv"
CANDIDATES_CSV = "/content/nodule_to_series_candidates.csv"

XML_DIR = "/content/lidc_xml_annotations"
ZIP_PATH = "/content/kagl_lidc_idri.zip"

OUT_MAPPING_CSV = "/content/subset_nodule_xml_forensic_mapping_v3.csv"
OUT_DETAILS_CSV = "/content/subset_nodule_xml_forensic_details_v3.csv"
OUT_SUMMARY_TXT = "/content/subset_nodule_xml_forensic_summary_v3.txt"

# Special cases discovered previously
AMBIGUOUS_SUBSET_IDS = {"nodule_029", "nodule_085"}

# Score thresholds are deliberately conservative.
VERY_STRONG_THRESHOLD = 0.85
STRONG_THRESHOLD = 0.70
MODERATE_THRESHOLD = 0.50
MARGIN_THRESHOLD = 0.10


# =============================================================================
# COLUMN NORMALIZATION
# =============================================================================

def normalize_columns(df):
    """
    Normalize known identifier column names without accidentally collapsing
    duplicate column names into a DataFrame.
    """
    new_names = []

    for c in df.columns:
        clean = str(c).strip()

        lower = clean.lower().replace(" ", "").replace("-", "").replace("_", "")

        if lower in {"patientid", "patient"}:
            new_names.append("patient_id")

        elif lower in {
            "seriesinstanceuid",
            "seriesuid",
            "seriesinstanceu",
            "seriesinstance"
        }:
            new_names.append("series_instance_uid")

        elif lower in {
            "subsetnoduleid",
            "subsetnodule"
        }:
            new_names.append("subset_nodule_id")

        elif lower in {
            "nativenoduleid",
            "nativenodule"
        }:
            new_names.append("native_nodule_id")

        else:
            new_names.append(clean)

    # Make sure every resulting column name is unique
    seen = {}
    final_names = []

    for name in new_names:
        if name not in seen:
            seen[name] = 0
            final_names.append(name)
        else:
            seen[name] += 1
            final_names.append(f"{name}_{seen[name]}")

    df = df.copy()
    df.columns = final_names
    return df


# =============================================================================
# XML PARSER
# =============================================================================

def parse_xml_file(xml_path):
    """
    Parse one LIDC XML file.

    Returns:
        series_uid,
        list of XML nodules

    Each XML nodule contains:
        xml_nodule_id
        document_index
        malignancy_ratings
        roi list
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception:
        return None, []

    if "}" in root.tag:
        namespace = root.tag.split("}")[0].strip("{")
        ns = {"lidc": namespace}

        def find(path):
            return root.find(path, ns)

        def findall(node, path):
            return node.findall(path, ns)

    else:
        ns = {}

        def find(path):
            return root.find(path)

        def findall(node, path):
            return node.findall(path)

    # -------------------------------------------------------------------------
    # SeriesInstanceUID
    # -------------------------------------------------------------------------

    suid_elem = find(".//lidc:SeriesInstanceUid") if ns else find(".//SeriesInstanceUid")

    if suid_elem is None or not suid_elem.text:
        suid_elem = (
            find(".//lidc:SeriesInstanceUID")
            if ns
            else find(".//SeriesInstanceUID")
        )

    series_uid = (
        suid_elem.text.strip()
        if suid_elem is not None and suid_elem.text
        else None
    )

    # -------------------------------------------------------------------------
    # Nodule parsing
    # -------------------------------------------------------------------------

    nodules = []

    nod_nodes = (
        root.findall(".//lidc:unblindedReadNodule", ns)
        if ns
        else root.findall(".//unblindedReadNodule")
    )

    for doc_index, nod in enumerate(nod_nodes):

        nod_id_elem = (
            nod.find("lidc:noduleID", ns)
            if ns
            else nod.find("noduleID")
        )

        xml_nodule_id = (
            nod_id_elem.text.strip()
            if nod_id_elem is not None and nod_id_elem.text
            else f"XML_{doc_index}"
        )

        # ---------------------------------------------------------------------
        # Malignancy
        # ---------------------------------------------------------------------

        mal_nodes = (
            nod.findall(".//lidc:malignancy", ns)
            if ns
            else nod.findall(".//malignancy")
        )

        malignancy_ratings = []

        for m in mal_nodes:
            if m.text:
                value = m.text.strip()
                try:
                    malignancy_ratings.append(int(value))
                except ValueError:
                    pass

        # ---------------------------------------------------------------------
        # ROIs
        # ---------------------------------------------------------------------

        roi_nodes = (
            nod.findall("lidc:roi", ns)
            if ns
            else nod.findall("roi")
        )

        rois = []

        for roi in roi_nodes:

            z_elem = (
                roi.find("lidc:imageZposition", ns)
                if ns
                else roi.find("imageZposition")
            )

            sop_elem = (
                roi.find("lidc:imageSOP_UID", ns)
                if ns
                else roi.find("imageSOP_UID")
            )

            z_pos = None
            if z_elem is not None and z_elem.text:
                try:
                    z_pos = float(z_elem.text.strip())
                except ValueError:
                    pass

            sop_uid = (
                sop_elem.text.strip()
                if sop_elem is not None and sop_elem.text
                else ""
            )

            edge_nodes = (
                roi.findall("lidc:edgeMap", ns)
                if ns
                else roi.findall("edgeMap")
            )

            x_coords = []
            y_coords = []

            for edge in edge_nodes:

                x_elem = (
                    edge.find("lidc:xCoord", ns)
                    if ns
                    else edge.find("xCoord")
                )

                y_elem = (
                    edge.find("lidc:yCoord", ns)
                    if ns
                    else edge.find("yCoord")
                )

                if (
                    x_elem is not None
                    and y_elem is not None
                    and x_elem.text
                    and y_elem.text
                ):
                    try:
                        x_coords.append(float(x_elem.text.strip()))
                        y_coords.append(float(y_elem.text.strip()))
                    except ValueError:
                        pass

            if len(x_coords) >= 3:

                x_arr = np.asarray(x_coords, dtype=float)
                y_arr = np.asarray(y_coords, dtype=float)

                width = float(np.max(x_arr) - np.min(x_arr))
                height = float(np.max(y_arr) - np.min(y_arr))

                # Polygon area using Shoelace formula
                area = 0.5 * abs(
                    np.dot(x_arr, np.roll(y_arr, 1))
                    - np.dot(y_arr, np.roll(x_arr, 1))
                )

                rois.append({
                    "sop_uid": sop_uid,
                    "z_pos": z_pos,
                    "x_coords": x_arr,
                    "y_coords": y_arr,
                    "width": width,
                    "height": height,
                    "area": float(area),
                })

        # Physical Z sorting where available
        if rois:
            rois.sort(
                key=lambda r: (
                    r["z_pos"] is None,
                    r["z_pos"] if r["z_pos"] is not None else 0.0
                )
            )

        nodules.append({
            "xml_nodule_id": xml_nodule_id,
            "document_index": doc_index,
            "malignancy_ratings": malignancy_ratings,
            "roi_count": len(rois),
            "rois": rois,
        })

    return series_uid, nodules


# =============================================================================
# XML INDEX
# =============================================================================

def build_xml_index():
    """
    Index XML files by SeriesInstanceUID.
    """
    xml_files = glob.glob(
        os.path.join(XML_DIR, "**", "*.xml"),
        recursive=True
    )

    xml_index = {}
    parse_errors = 0

    for xml_path in xml_files:

        suid, nodules = parse_xml_file(xml_path)

        if not suid:
            continue

        if suid not in xml_index:
            xml_index[suid] = []

        xml_index[suid].append({
            "xml_file": xml_path,
            "nodules": nodules
        })

    return xml_index, xml_files, parse_errors


# =============================================================================
# ZIP / MASK PROCESSING
# =============================================================================

def read_png_array(zf, path):
    """
    Read a PNG from an already-open ZIP file.
    """
    with zf.open(path) as f:
        raw = f.read()

    return np.asarray(
        Image.open(io.BytesIO(raw)).convert("L")
    )


def mask_slice_geometry(arr):
    """
    Calculate foreground geometry for one binary mask.
    """
    binary = arr > 0
    foreground = np.argwhere(binary)

    if len(foreground) == 0:
        return {
            "area": 0.0,
            "width": 0.0,
            "height": 0.0,
            "aspect_ratio": 0.0,
            "empty": True,
        }

    y = foreground[:, 0]
    x = foreground[:, 1]

    width = float(x.max() - x.min())
    height = float(y.max() - y.min())

    aspect_ratio = width / height if height > 0 else 0.0

    return {
        "area": float(len(foreground)),
        "width": width,
        "height": height,
        "aspect_ratio": float(aspect_ratio),
        "empty": False,
    }


def process_native_nodule(zf, patient_id, native_nodule_id):
    """
    Read image and mask structure from the original ZIP.
    """
    prefix = (
        f"LIDC-IDRI-slices/"
        f"{patient_id}/"
        f"{native_nodule_id}/"
    )

    names = [
        n for n in zf.namelist()
        if n.startswith(prefix) and n.endswith(".png")
    ]

    image_files = sorted([
        n for n in names
        if "/images/" in n
    ])

    mask_files = {}

    for m in range(4):
        mask_prefix = f"{prefix}mask-{m}/"

        mask_files[f"mask-{m}"] = sorted([
            n for n in names
            if n.startswith(mask_prefix)
        ])

    mask_geometry = {}

    for mask_name, paths in mask_files.items():

        geometries = []

        for path in paths:

            try:
                arr = read_png_array(zf, path)
                geom = mask_slice_geometry(arr)

                geometries.append({
                    "filename": path,
                    **geom
                })

            except Exception:
                continue

        mask_geometry[mask_name] = geometries

    return {
        "image_count": len(image_files),
        "mask_geometry": mask_geometry
    }


# =============================================================================
# GEOMETRIC PROFILE
# =============================================================================

def native_profile(mask_geometry):
    """
    Aggregate native mask information for one radiologist mask.
    """
    active = [
        g for g in mask_geometry
        if not g["empty"]
    ]

    if not active:
        return None

    return {
        "slice_count": len(active),
        "areas": np.asarray(
            [g["area"] for g in active],
            dtype=float
        ),
        "widths": np.asarray(
            [g["width"] for g in active],
            dtype=float
        ),
        "heights": np.asarray(
            [g["height"] for g in active],
            dtype=float
        ),
        "aspect_ratios": np.asarray(
            [g["aspect_ratio"] for g in active],
            dtype=float
        ),
    }


def xml_profile(xml_nodule):
    """
    Aggregate XML ROI geometry.
    """
    rois = [
        r for r in xml_nodule["rois"]
        if len(r["x_coords"]) >= 3
    ]

    if not rois:
        return None

    return {
        "slice_count": len(rois),
        "areas": np.asarray(
            [r["area"] for r in rois],
            dtype=float
        ),
        "widths": np.asarray(
            [r["width"] for r in rois],
            dtype=float
        ),
        "heights": np.asarray(
            [r["height"] for r in rois],
            dtype=float
        ),
        "aspect_ratios": np.asarray(
            [
                (
                    r["width"] / r["height"]
                    if r["height"] > 0
                    else 0.0
                )
                for r in rois
            ],
            dtype=float
        ),
    }


# =============================================================================
# PROFILE COMPARISON
# =============================================================================

def normalized_vector_difference(a, b):
    """
    Compare two vectors after independently normalizing scale.

    Allows for the fact that PNG crop dimensions and XML full-grid
    coordinates may use different absolute coordinate origins.
    """
    if len(a) == 0 or len(b) == 0:
        return 1.0

    target_len = 20

    a_x = np.linspace(0, 1, len(a))
    b_x = np.linspace(0, 1, len(b))

    common_x = np.linspace(0, 1, target_len)

    a_interp = np.interp(common_x, a_x, a)
    b_interp = np.interp(common_x, b_x, b)

    # Normalize each sequence by its maximum.
    a_max = np.max(a_interp)
    b_max = np.max(b_interp)

    if a_max > 0:
        a_interp = a_interp / a_max

    if b_max > 0:
        b_interp = b_interp / b_max

    return float(
        np.mean(np.abs(a_interp - b_interp))
    )


def profile_similarity(native_prof, xml_prof):
    """
    Compare native and XML profiles.

    Returns a score from 0 to 1.
    """
    if native_prof is None or xml_prof is None:
        return 0.0

    native_n = native_prof["slice_count"]
    xml_n = xml_prof["slice_count"]

    if native_n == 0 or xml_n == 0:
        return 0.0

    # Count agreement
    count_difference = abs(native_n - xml_n)
    count_penalty = count_difference / max(native_n, xml_n)

    # Compare sequence from both directions because PNG slice naming
    # does not establish physical Z direction.
    feature_differences = []

    for feature_name in [
        "areas",
        "widths",
        "heights",
        "aspect_ratios"
    ]:

        nvec = native_prof[feature_name]
        xvec = xml_prof[feature_name]

        forward = normalized_vector_difference(nvec, xvec)
        reverse = normalized_vector_difference(
            nvec,
            xvec[::-1]
        )

        feature_differences.append(
            min(forward, reverse)
        )

    shape_error = float(
        np.mean(feature_differences)
    )

    shape_score = max(
        0.0,
        1.0 - shape_error
    )

    count_score = max(
        0.0,
        1.0 - count_penalty
    )

    final_score = (
        0.60 * shape_score
        + 0.40 * count_score
    )

    return float(
        np.clip(final_score, 0.0, 1.0)
    )


# =============================================================================
# NATIVE -> XML SCORE
# =============================================================================

def compare_native_to_xml(native_data, xml_nodule):
    """
    Compare every available non-empty mask against an XML nodule.

    Missing masks are NOT treated as zero similarity.
    """
    scores = {}

    for mask_name, geometries in native_data["mask_geometry"].items():

        profile = native_profile(geometries)

        if profile is None:
            continue

        xprofile = xml_profile(xml_nodule)

        if xprofile is None:
            continue

        score = profile_similarity(
            profile,
            xprofile
        )

        scores[mask_name] = score

    if not scores:
        return 0.0, {}

    combined_score = float(
        np.mean(list(scores.values()))
    )

    return combined_score, scores


# =============================================================================
# STATUS CLASSIFICATION
# =============================================================================

def classify_match(best_score, second_score, is_ambiguous_series=False):
    """
    Assign evidence category.

    IMPORTANT:
    None of these categories claim exact physical identity.
    """
    if best_score < MODERATE_THRESHOLD:
        if is_ambiguous_series:
            return "SERIES_OR_GEOMETRY_UNRESOLVED"
        return "WEAK_OR_UNMATCHABLE"

    margin = (
        best_score - second_score
        if second_score >= 0
        else best_score
    )

    if margin < MARGIN_THRESHOLD:
        return "AMBIGUOUS_GEOMETRIC_MATCH"

    if best_score >= VERY_STRONG_THRESHOLD:
        return "VERY_STRONG_GEOMETRIC_MATCH"

    if best_score >= STRONG_THRESHOLD:
        return "STRONG_GEOMETRIC_MATCH"

    return "MODERATE_GEOMETRIC_MATCH"


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def main():

    print("=" * 80)
    print("SUBSET NODULE → XML FORENSIC GEOMETRY ANALYSIS v3")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # Verify files
    # -------------------------------------------------------------------------

    required_files = [
        MAPPING_CSV,
        SINGLE_SERIES_CSV,
        XML_DIR,
        ZIP_PATH
    ]

    for path in required_files:
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Required file/folder not found: {path}"
            )

    print("\nAll required files found.")

    # -------------------------------------------------------------------------
    # Load CSVs
    # -------------------------------------------------------------------------

    mapping_df = normalize_columns(
        pd.read_csv(MAPPING_CSV)
    )

    series_df = normalize_columns(
        pd.read_csv(SINGLE_SERIES_CSV)
    )

    candidates_df = None

    if os.path.exists(CANDIDATES_CSV):
        candidates_df = normalize_columns(
            pd.read_csv(CANDIDATES_CSV)
        )

    # Required columns
    required_mapping = {
        "subset_nodule_id",
        "patient_id",
        "native_nodule_id"
    }

    required_series = {
        "subset_nodule_id",
        "patient_id",
        "native_nodule_id",
        "series_instance_uid"
    }

    if not required_mapping.issubset(mapping_df.columns):
        missing = required_mapping - set(mapping_df.columns)
        raise KeyError(
            f"Mapping CSV missing columns: {missing}"
        )

    if not required_series.issubset(series_df.columns):
        missing = required_series - set(series_df.columns)
        raise KeyError(
            f"Series CSV missing columns: {missing}"
        )

    # -------------------------------------------------------------------------
    # Merge clean mapping
    # -------------------------------------------------------------------------

    merged = pd.merge(
        mapping_df[
            [
                "subset_nodule_id",
                "patient_id",
                "native_nodule_id"
            ]
        ],
        series_df[
            [
                "subset_nodule_id",
                "series_instance_uid"
            ]
        ],
        on="subset_nodule_id",
        how="inner"
    )

    print("\n" + "-" * 80)
    print("INPUT SUMMARY")
    print("-" * 80)

    print(
        f"Original subset rows:           {len(mapping_df)}"
    )
    print(
        f"Clean series rows:              {len(series_df)}"
    )
    print(
        f"Merged subset rows:             {len(merged)}"
    )
    print(
        f"Unique patients:                 {merged['patient_id'].nunique()}"
    )
    print(
        f"Unique SeriesInstanceUIDs:       "
        f"{merged['series_instance_uid'].nunique()}"
    )

    # -------------------------------------------------------------------------
    # XML index
    # -------------------------------------------------------------------------

    print("\n" + "-" * 80)
    print("INDEXING XML ANNOTATIONS")
    print("-" * 80)

    xml_index, xml_files, parse_errors = build_xml_index()

    print(
        f"XML files discovered:             {len(xml_files)}"
    )
    print(
        f"SeriesInstanceUIDs indexed:      {len(xml_index)}"
    )

    duplicated_xml_series = sum(
        1
        for value in xml_index.values()
        if len(value) > 1
    )

    print(
        f"SeriesInstanceUIDs with >1 XML:   "
        f"{duplicated_xml_series}"
    )

    # -------------------------------------------------------------------------
    # Process ZIP
    # -------------------------------------------------------------------------

    print("\n" + "-" * 80)
    print("PROCESSING ORIGINAL ZIP")
    print("-" * 80)

    zf = zipfile.ZipFile(
        ZIP_PATH,
        "r"
    )

    zip_name_set = set(
        zf.namelist()
    )

    print(
        f"ZIP entries:                      "
        f"{len(zip_name_set)}"
    )

    # -------------------------------------------------------------------------
    # Main evaluation
    # -------------------------------------------------------------------------

    mapping_results = []
    detail_results = []

    for counter, (_, row) in enumerate(
        merged.iterrows(),
        start=1
    ):

        subset_id = str(
            row["subset_nodule_id"]
        ).strip()

        patient_id = str(
            row["patient_id"]
        ).strip()

        native_id = str(
            row["native_nodule_id"]
        ).strip()

        default_suid = str(
            row["series_instance_uid"]
        ).strip()

        # ---------------------------------------------------------------------
        # Candidate series
        # ---------------------------------------------------------------------

        candidate_suids = []

        if (
            subset_id in AMBIGUOUS_SUBSET_IDS
            and candidates_df is not None
        ):

            if "series_instance_uid" in candidates_df.columns:

                candidates = candidates_df[
                    candidates_df[
                        "subset_nodule_id"
                    ].astype(str).str.strip()
                    == subset_id
                ]

                candidate_suids = (
                    candidates[
                        "series_instance_uid"
                    ]
                    .dropna()
                    .astype(str)
                    .str.strip()
                    .unique()
                    .tolist()
                )

        if not candidate_suids and default_suid:
            candidate_suids = [default_suid]

        # ---------------------------------------------------------------------
        # Native ZIP data
        # ---------------------------------------------------------------------

        native_data = process_native_nodule(
            zf,
            patient_id,
            native_id
        )

        active_mask_counts = {}

        for mask_name, geometries in (
            native_data["mask_geometry"].items()
        ):
            active_mask_counts[mask_name] = sum(
                1
                for g in geometries
                if not g["empty"]
            )

        # ---------------------------------------------------------------------
        # Compare against XML candidates
        # ---------------------------------------------------------------------

        candidate_scores = []

        for candidate_suid in candidate_suids:

            xml_entries = xml_index.get(
                candidate_suid,
                []
            )

            for xml_entry in xml_entries:

                xml_file = xml_entry["xml_file"]
                xml_nodules = xml_entry["nodules"]

                for xnod in xml_nodules:

                    score, mask_scores = (
                        compare_native_to_xml(
                            native_data,
                            xnod
                        )
                    )

                    candidate_scores.append({
                        "series_instance_uid":
                            candidate_suid,
                        "xml_file":
                            xml_file,
                        "xml_nodule_id":
                            xnod["xml_nodule_id"],
                        "document_index":
                            xnod["document_index"],
                        "malignancy_ratings":
                            xnod["malignancy_ratings"],
                        "xml_roi_count":
                            xnod["roi_count"],
                        "combined_score":
                            score,
                        "mask_scores":
                            mask_scores
                    })

        # ---------------------------------------------------------------------
        # Sort candidate matches
        # ---------------------------------------------------------------------

        candidate_scores.sort(
            key=lambda x: x["combined_score"],
            reverse=True
        )

        if candidate_scores:

            best = candidate_scores[0]

            best_score = float(
                best["combined_score"]
            )

            second_score = (
                float(
                    candidate_scores[1]["combined_score"]
                )
                if len(candidate_scores) > 1
                else -1.0
            )

            margin = (
                best_score - second_score
                if second_score >= 0
                else best_score
            )

            status = classify_match(
                best_score,
                second_score,
                subset_id in AMBIGUOUS_SUBSET_IDS
            )

            selected_suid = (
                best["series_instance_uid"]
            )

            selected_xml_file = (
                best["xml_file"]
            )

            selected_xml_id = (
                best["xml_nodule_id"]
            )

            ratings = (
                best["malignancy_ratings"]
            )

            mask_score_string = ";".join(
                [
                    f"{k}:{v:.4f}"
                    for k, v in best["mask_scores"].items()
                ]
            )

        else:

            best_score = 0.0
            second_score = -1.0
            margin = 0.0

            status = (
                "SERIES_UNRESOLVED"
                if subset_id in AMBIGUOUS_SUBSET_IDS
                else "NO_XML_GEOMETRIC_MATCH"
            )

            selected_suid = (
                default_suid
                if default_suid
                else "UNRESOLVED"
            )

            selected_xml_file = "NONE"
            selected_xml_id = "NONE"
            ratings = []

            mask_score_string = "NONE"

        # ---------------------------------------------------------------------
        # Save detail records for special cases
        # ---------------------------------------------------------------------

        if (
            subset_id in {
                "nodule_001",
                "nodule_029",
                "nodule_085"
            }
        ):

            for candidate in candidate_scores:

                detail_results.append({
                    "subset_nodule_id":
                        subset_id,
                    "patient_id":
                        patient_id,
                    "native_nodule_id":
                        native_id,
                    "candidate_series_instance_uid":
                        candidate[
                            "series_instance_uid"
                        ],
                    "xml_nodule_id":
                        candidate[
                            "xml_nodule_id"
                        ],
                    "xml_document_index":
                        candidate[
                            "document_index"
                        ],
                    "xml_roi_count":
                        candidate[
                            "xml_roi_count"
                        ],
                    "malignancy_ratings":
                        str(
                            candidate[
                                "malignancy_ratings"
                            ]
                        ),
                    "combined_score":
                        round(
                            candidate[
                                "combined_score"
                            ],
                            4
                        ),
                    "mask_scores":
                        str(
                            candidate[
                                "mask_scores"
                            ]
                        )
                })

        # ---------------------------------------------------------------------
        # Main output
        # ---------------------------------------------------------------------

        mapping_results.append({

            "subset_nodule_id":
                subset_id,

            "patient_id":
                patient_id,

            "native_nodule_id":
                native_id,

            "series_instance_uid":
                selected_suid,

            "xml_file":
                selected_xml_file,

            "best_xml_nodule_id":
                selected_xml_id,

            "candidate_count":
                len(candidate_scores),

            "best_score":
                round(
                    best_score,
                    4
                ),

            "second_best_score":
                round(
                    second_score,
                    4
                ),

            "score_margin":
                round(
                    margin,
                    4
                ),

            "native_image_count":
                native_data[
                    "image_count"
                ],

            "active_mask_slice_counts":
                ";".join(
                    [
                        f"{k}:{v}"
                        for k, v
                        in active_mask_counts.items()
                    ]
                ),

            "best_mask_scores":
                mask_score_string,

            "xml_malignancy_ratings":
                str(ratings),

            "match_status":
                status
        })

        if counter % 25 == 0 or counter == len(merged):

            print(
                f"Processed {counter} / {len(merged)}"
            )

    zf.close()

    # -------------------------------------------------------------------------
    # Save outputs
    # -------------------------------------------------------------------------

    result_df = pd.DataFrame(
        mapping_results
    )

    detail_df = pd.DataFrame(
        detail_results
    )

    result_df.to_csv(
        OUT_MAPPING_CSV,
        index=False
    )

    detail_df.to_csv(
        OUT_DETAILS_CSV,
        index=False
    )

    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------

    status_counts = (
        result_df[
            "match_status"
        ]
        .value_counts()
        .to_dict()
    )

    print("\n" + "=" * 80)
    print("FINAL FORENSIC SUMMARY")
    print("=" * 80)

    print(
        f"Total subset nodules:             "
        f"{len(result_df)}"
    )

    print(
        f"Very strong geometric matches:    "
        f"{status_counts.get('VERY_STRONG_GEOMETRIC_MATCH', 0)}"
    )

    print(
        f"Strong geometric matches:         "
        f"{status_counts.get('STRONG_GEOMETRIC_MATCH', 0)}"
    )

    print(
        f"Moderate geometric matches:       "
        f"{status_counts.get('MODERATE_GEOMETRIC_MATCH', 0)}"
    )

    print(
        f"Ambiguous geometric matches:      "
        f"{status_counts.get('AMBIGUOUS_GEOMETRIC_MATCH', 0)}"
    )

    print(
        f"Weak/unmatchable:                 "
        f"{status_counts.get('WEAK_OR_UNMATCHABLE', 0)}"
    )

    print(
        f"Series unresolved:                "
        f"{status_counts.get('SERIES_UNRESOLVED', 0) + status_counts.get('SERIES_OR_GEOMETRY_UNRESOLVED', 0)}"
    )

    print("\n" + "-" * 80)
    print("SPECIAL CASES")
    print("-" * 80)

    for special_id in [
        "nodule_001",
        "nodule_029",
        "nodule_085"
    ]:

        rows = result_df[
            result_df["subset_nodule_id"]
            == special_id
        ]

        if rows.empty:
            print(
                f"{special_id}: NOT FOUND"
            )
            continue

        r = rows.iloc[0]

        print(
            f"\n{special_id}"
        )
        print(
            f"  Patient:           {r['patient_id']}"
        )
        print(
            f"  Native nodule:     {r['native_nodule_id']}"
        )
        print(
            f"  Selected Series:   {r['series_instance_uid']}"
        )
        print(
            f"  Best XML nodule:   {r['best_xml_nodule_id']}"
        )
        print(
            f"  Best score:        {r['best_score']}"
        )
        print(
            f"  Second score:      {r['second_best_score']}"
        )
        print(
            f"  Margin:            {r['score_margin']}"
        )
        print(
            f"  Status:            {r['match_status']}"
        )
        print(
            f"  XML ratings:       {r['xml_malignancy_ratings']}"
        )

    # -------------------------------------------------------------------------
    # Write summary file
    # -------------------------------------------------------------------------

    summary = f"""
===============================================================================
SUBSET NODULE → XML FORENSIC GEOMETRY SUMMARY
===============================================================================

DATASET
-------------------------------------------------------------------------------
Original subset rows:              {len(mapping_df)}
Clean series mapping rows:        {len(series_df)}
Merged rows analyzed:              {len(result_df)}
Unique patients:                   {merged['patient_id'].nunique()}
Unique SeriesInstanceUIDs:         {merged['series_instance_uid'].nunique()}

XML
-------------------------------------------------------------------------------
XML files discovered:              {len(xml_files)}
Unique XML SeriesInstanceUIDs:     {len(xml_index)}
XML SeriesInstanceUIDs with >1 XML:{duplicated_xml_series}

MATCH STATUS
-------------------------------------------------------------------------------
VERY_STRONG_GEOMETRIC_MATCH:       {status_counts.get('VERY_STRONG_GEOMETRIC_MATCH', 0)}
STRONG_GEOMETRIC_MATCH:            {status_counts.get('STRONG_GEOMETRIC_MATCH', 0)}
MODERATE_GEOMETRIC_MATCH:          {status_counts.get('MODERATE_GEOMETRIC_MATCH', 0)}
AMBIGUOUS_GEOMETRIC_MATCH:         {status_counts.get('AMBIGUOUS_GEOMETRIC_MATCH', 0)}
WEAK_OR_UNMATCHABLE:               {status_counts.get('WEAK_OR_UNMATCHABLE', 0)}
NO_XML_GEOMETRIC_MATCH:            {status_counts.get('NO_XML_GEOMETRIC_MATCH', 0)}
SERIES_UNRESOLVED:                 {status_counts.get('SERIES_UNRESOLVED', 0)}
SERIES_OR_GEOMETRY_UNRESOLVED:     {status_counts.get('SERIES_OR_GEOMETRY_UNRESOLVED', 0)}

SPECIAL CASES
-------------------------------------------------------------------------------
nodule_029 and nodule_085 were previously identified as multi-series candidate
cases and were evaluated against all available candidate SeriesInstanceUIDs.

METHODOLOGICAL LIMIT
-------------------------------------------------------------------------------
The geometric score is supporting evidence only.

The PNG files in the original ZIP do not contain the original DICOM
SOPInstanceUID / ImagePositionPatient metadata needed for direct slice-level
physical identity verification.

Therefore:

1. A high geometric score does NOT prove exact physical identity.
2. A unique geometric winner is evidence of correspondence, not a formal
   DICOM-level identity proof.
3. Malignancy ratings are reported as XML annotation values only; this script
   does not convert them into a clinical diagnosis.
4. Slice reversal is allowed during profile comparison because PNG filename
   order alone does not establish physical Z direction.

OUTPUTS
-------------------------------------------------------------------------------
Mapping:
{OUT_MAPPING_CSV}

Details:
{OUT_DETAILS_CSV}

Summary:
{OUT_SUMMARY_TXT}
===============================================================================
"""

    with open(
        OUT_SUMMARY_TXT,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(
            summary.strip()
        )

    print(
        "\nSaved mapping:"
        f"\n{OUT_MAPPING_CSV}"
    )

    print(
        "\nSaved details:"
        f"\n{OUT_DETAILS_CSV}"
    )

    print(
        "\nSaved summary:"
        f"\n{OUT_SUMMARY_TXT}"
    )

    print("\n" + "=" * 80)
    print("DONE")
    print("=" * 80)


# =============================================================================
# RUN
# =============================================================================

main()

SUBSET NODULE → XML FORENSIC GEOMETRY ANALYSIS v3

All required files found.

--------------------------------------------------------------------------------
INPUT SUMMARY
--------------------------------------------------------------------------------
Original subset rows:           327
Clean series rows:              325
Merged subset rows:             325
Unique patients:                 247
Unique SeriesInstanceUIDs:       247

--------------------------------------------------------------------------------
INDEXING XML ANNOTATIONS
--------------------------------------------------------------------------------
XML files discovered:             1319
SeriesInstanceUIDs indexed:      1294
SeriesInstanceUIDs with >1 XML:   8

--------------------------------------------------------------------------------
PROCESSING ORIGINAL ZIP
--------------------------------------------------------------------------------
ZIP entries:                      77740
Processed 25 / 325
Processed 50 / 32

In [ ]:
import os
import glob
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from PIL import Image
import io

# ============================================================
# CONFIGURATION
# ============================================================

ZIP_PATH = "/content/kagl_lidc_idri.zip"
XML_DIR = "/content/lidc_xml_annotations"

MAPPING_CSV = "/content/nodule_to_lidc_mapping.csv"
SERIES_CSV = "/content/nodule_to_single_ct_series_fixed.csv"

OUTPUT_CSV = "/content/targeted_provenance_diagnostic.csv"
OUTPUT_SUMMARY = "/content/targeted_provenance_diagnostic.txt"

TARGETS = [
    "nodule_001",
    "nodule_029",
    "nodule_085",
]

# ============================================================
# HELPERS
# ============================================================

def normalize_columns(df):
    """
    Normalize common column-name variants without creating
    duplicate DataFrame columns.
    """
    rename = {}

    for c in df.columns:
        clean = str(c).strip().lower().replace(" ", "_")

        if clean in {"subset_nodule_id", "subsetnoduleid"}:
            rename[c] = "subset_nodule_id"

        elif clean in {
            "patient_id",
            "patientid",
        }:
            rename[c] = "patient_id"

        elif clean in {
            "native_nodule_id",
            "nativenoduleid",
        }:
            rename[c] = "native_nodule_id"

        elif clean in {
            "series_instance_uid",
            "seriesinstanceuid",
        }:
            rename[c] = "series_instance_uid"

        elif clean in {
            "study_instance_uid",
            "studyinstanceuid",
        }:
            rename[c] = "study_instance_uid"

    return df.rename(columns=rename)


def parse_xml_series_and_nodules(xml_path):
    """
    Parse one LIDC XML file.

    Returns:
        series_uid
        list of XML nodule dictionaries
    """

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception:
        return None, []

    nodules = []

    # Namespace-independent XPath
    series_uid = None

    series_candidates = root.findall(".//{*}SeriesInstanceUid")
    if not series_candidates:
        series_candidates = root.findall(".//{*}SeriesInstanceUID")

    if series_candidates and series_candidates[0].text:
        series_uid = series_candidates[0].text.strip()

    nodule_nodes = root.findall(".//{*}unblindedReadNodule")

    for idx, nod in enumerate(nodule_nodes):

        nodule_id_elem = nod.find("{*}noduleID")

        xml_nodule_id = (
            nodule_id_elem.text.strip()
            if nodule_id_elem is not None and nodule_id_elem.text
            else f"XML_{idx}"
        )

        ratings = []

        for m in nod.findall(".//{*}malignancy"):
            if m.text:
                try:
                    ratings.append(int(m.text.strip()))
                except ValueError:
                    pass

        roi_info = []

        for roi in nod.findall("{*}roi"):

            z_elem = roi.find("{*}imageZposition")
            sop_elem = roi.find("{*}imageSOP_UID")

            z = None
            sop = None

            if z_elem is not None and z_elem.text:
                try:
                    z = float(z_elem.text.strip())
                except ValueError:
                    pass

            if sop_elem is not None and sop_elem.text:
                sop = sop_elem.text.strip()

            coords = []

            for edge in roi.findall("{*}edgeMap"):
                x_elem = edge.find("{*}xCoord")
                y_elem = edge.find("{*}yCoord")

                if (
                    x_elem is not None
                    and y_elem is not None
                    and x_elem.text
                    and y_elem.text
                ):
                    try:
                        coords.append(
                            (
                                float(x_elem.text.strip()),
                                float(y_elem.text.strip())
                            )
                        )
                    except ValueError:
                        pass

            roi_info.append({
                "z": z,
                "sop_uid": sop,
                "coords": coords
            })

        z_values = [
            r["z"]
            for r in roi_info
            if r["z"] is not None
        ]

        sop_values = [
            r["sop_uid"]
            for r in roi_info
            if r["sop_uid"]
        ]

        nodules.append({
            "xml_nodule_id": xml_nodule_id,
            "xml_index": idx,
            "ratings": ratings,
            "roi_count": len(roi_info),
            "z_min": min(z_values) if z_values else None,
            "z_max": max(z_values) if z_values else None,
            "z_count": len(set(round(z, 3) for z in z_values)),
            "sop_count": len(set(sop_values)),
            "roi_info": roi_info,
        })

    return series_uid, nodules


def get_png_info_from_zip(zf, path):
    """
    Inspect one PNG that is inside the ZIP.
    """
    with zf.open(path) as f:
        raw = f.read()

    img = Image.open(io.BytesIO(raw))
    arr = np.array(img)

    return {
        "path": path,
        "format": img.format,
        "size": img.size,
        "mode": img.mode,
        "shape": arr.shape,
        "dtype": str(arr.dtype),
        "min": int(arr.min()) if arr.size else None,
        "max": int(arr.max()) if arr.size else None,
        "unique_count": int(len(np.unique(arr))),
        "metadata_keys": list(img.info.keys()),
    }


# ============================================================
# START
# ============================================================

print("=" * 80)
print("TARGETED LIDC-IDRI PROVENANCE DIAGNOSTIC")
print("=" * 80)

# ============================================================
# 1. FILE CHECK
# ============================================================

required_files = [
    ZIP_PATH,
    MAPPING_CSV,
    SERIES_CSV,
]

for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file missing: {path}")

if not os.path.isdir(XML_DIR):
    raise FileNotFoundError(f"XML directory missing: {XML_DIR}")

print("\nAll required files found.")

# ============================================================
# 2. SEARCH FOR DICOM FILES ON DISK
# ============================================================

print("\n" + "-" * 80)
print("STEP 1 — DICOM SEARCH")
print("-" * 80)

dcm_files = glob.glob("/content/**/*.dcm", recursive=True)

print(f"DICOM files physically present in /content: {len(dcm_files)}")

if dcm_files:
    print("First 20 DICOM paths:")
    for p in dcm_files[:20]:
        print(" ", p)
else:
    print("No .dcm files found.")

# IMPORTANT:
# Presence of DICOM files is NOT treated as proof of target provenance.

# ============================================================
# 3. INSPECT ZIP CONTENTS
# ============================================================

print("\n" + "-" * 80)
print("STEP 2 — ORIGINAL ZIP CONTENTS")
print("-" * 80)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:

    names = zf.namelist()

    png_files = [
        x for x in names
        if x.lower().endswith(".png")
    ]

    dcm_zip_files = [
        x for x in names
        if x.lower().endswith(".dcm")
    ]

    non_png_files = [
        x for x in names
        if not x.endswith("/")
        and not x.lower().endswith(".png")
    ]

    print(f"Total ZIP entries: {len(names)}")
    print(f"PNG files: {len(png_files)}")
    print(f"DICOM files: {len(dcm_zip_files)}")
    print(f"Other non-directory files: {len(non_png_files)}")

    if dcm_zip_files:
        print("\nDICOM files found inside ZIP:")
        for p in dcm_zip_files[:20]:
            print(" ", p)
    else:
        print("\nRESULT: No DICOM files exist inside the original PNG ZIP.")

# ============================================================
# 4. LOAD MAPPINGS
# ============================================================

print("\n" + "-" * 80)
print("STEP 3 — MAPPING FILES")
print("-" * 80)

mapping_df = normalize_columns(pd.read_csv(MAPPING_CSV))
series_df = normalize_columns(pd.read_csv(SERIES_CSV))

print("Original mapping columns:")
print(list(mapping_df.columns))

print("\nSeries mapping columns:")
print(list(series_df.columns))

required_mapping = {
    "subset_nodule_id",
    "patient_id",
    "native_nodule_id",
}

required_series = {
    "subset_nodule_id",
    "patient_id",
    "native_nodule_id",
    "series_instance_uid",
}

missing_mapping = required_mapping - set(mapping_df.columns)
missing_series = required_series - set(series_df.columns)

if missing_mapping:
    raise KeyError(
        f"Mapping CSV is missing columns: {missing_mapping}"
    )

if missing_series:
    raise KeyError(
        f"Series CSV is missing columns: {missing_series}"
    )

print("\nMapping rows:", len(mapping_df))
print("Series rows:", len(series_df))

# ============================================================
# 5. TARGET MAPPING LOOKUP
# ============================================================

print("\n" + "-" * 80)
print("STEP 4 — TARGET NODULE MAPPINGS")
print("-" * 80)

target_mapping_rows = []

for target in TARGETS:

    map_rows = mapping_df[
        mapping_df["subset_nodule_id"].astype(str).str.strip() == target
    ]

    series_rows = series_df[
        series_df["subset_nodule_id"].astype(str).str.strip() == target
    ]

    print(f"\n{target}")

    if map_rows.empty:
        print("  Mapping CSV: NOT FOUND")
        continue

    map_row = map_rows.iloc[0]

    patient_id = str(map_row["patient_id"]).strip()
    native_nodule_id = str(map_row["native_nodule_id"]).strip()

    print(f"  Patient: {patient_id}")
    print(f"  Native nodule: {native_nodule_id}")

    if series_rows.empty:
        print("  Series mapping: NOT FOUND")
        series_uid = None
    else:
        series_uid = str(
            series_rows.iloc[0]["series_instance_uid"]
        ).strip()

        print(f"  SeriesInstanceUID: {series_uid}")

    target_mapping_rows.append({
        "subset_nodule_id": target,
        "patient_id": patient_id,
        "native_nodule_id": native_nodule_id,
        "series_instance_uid": series_uid
    })

# ============================================================
# 6. INSPECT TARGET PNG STRUCTURE DIRECTLY INSIDE ZIP
# ============================================================

print("\n" + "-" * 80)
print("STEP 5 — TARGET PNG INSPECTION")
print("-" * 80)

png_diagnostics = []

with zipfile.ZipFile(ZIP_PATH, "r") as zf:

    zip_names = set(zf.namelist())

    for target_row in target_mapping_rows:

        target = target_row["subset_nodule_id"]
        pid = target_row["patient_id"]
        native_id = target_row["native_nodule_id"]

        prefix = (
            f"LIDC-IDRI-slices/"
            f"{pid}/"
            f"{native_id}/"
        )

        image_paths = sorted(
            p for p in zip_names
            if p.startswith(prefix + "images/")
            and p.lower().endswith(".png")
        )

        mask_paths = sorted(
            p for p in zip_names
            if p.startswith(prefix + "mask-")
            and p.lower().endswith(".png")
        )

        print(f"\n{target}")
        print(f"  Image PNG count: {len(image_paths)}")
        print(f"  Mask PNG count:  {len(mask_paths)}")

        if image_paths:

            info = get_png_info_from_zip(
                zf,
                image_paths[0]
            )

            print("  Sample image:")
            print("    Path:", info["path"])
            print("    Size:", info["size"])
            print("    Mode:", info["mode"])
            print("    Shape:", info["shape"])
            print("    Dtype:", info["dtype"])
            print("    Min/Max:", info["min"], "/", info["max"])
            print("    Unique values:", info["unique_count"])
            print("    Metadata keys:", info["metadata_keys"])

            suspicious_keys = [
                k for k in info["metadata_keys"]
                if any(
                    term in str(k).lower()
                    for term in [
                        "sop",
                        "dicom",
                        "series",
                        "study",
                        "slice",
                        "position",
                        "uid"
                    ]
                )
            ]

            print("    DICOM/spatial-looking metadata keys:",
                  suspicious_keys if suspicious_keys else "NONE")

            png_diagnostics.append({
                "subset_nodule_id": target,
                "image_count": len(image_paths),
                "mask_count": len(mask_paths),
                "png_metadata_keys": str(info["metadata_keys"]),
                "spatial_metadata_keys": str(suspicious_keys)
            })

# ============================================================
# 7. INDEX ALL XML FILES
# ============================================================

print("\n" + "-" * 80)
print("STEP 6 — XML SERIES INDEX")
print("-" * 80)

xml_files = glob.glob(
    os.path.join(XML_DIR, "**", "*.xml"),
    recursive=True
)

print("XML files discovered:", len(xml_files))

xml_by_series = {}
xml_parse_errors = 0

for xf in xml_files:

    suid, nodules = parse_xml_series_and_nodules(xf)

    if suid:
        xml_by_series.setdefault(suid, []).append({
            "xml_path": xf,
            "nodules": nodules
        })
    else:
        xml_parse_errors += 1

print("Unique SeriesInstanceUIDs:", len(xml_by_series))
print("XML parse/index failures:", xml_parse_errors)

# ============================================================
# 8. TARGET-SPECIFIC XML ANALYSIS
# ============================================================

print("\n" + "-" * 80)
print("STEP 7 — TARGET-SPECIFIC XML ANALYSIS")
print("-" * 80)

results = []

for target_row in target_mapping_rows:

    target = target_row["subset_nodule_id"]
    pid = target_row["patient_id"]
    native_id = target_row["native_nodule_id"]
    series_uid = target_row["series_instance_uid"]

    print("\n" + "=" * 60)
    print(target)
    print("=" * 60)

    if not series_uid:
        print("No exact SeriesInstanceUID mapping.")
        classification = "NOT DETERMINABLE"

        results.append({
            **target_row,
            "xml_files_for_series": 0,
            "xml_nodules_in_series": 0,
            "classification": classification,
            "reason": "No exact SeriesInstanceUID mapping available."
        })

        continue

    xml_entries = xml_by_series.get(series_uid, [])

    print("Exact XML files for SeriesInstanceUID:",
          len(xml_entries))

    total_xml_nodules = sum(
        len(x["nodules"])
        for x in xml_entries
    )

    print("Total XML unblindedReadNodule entries:",
          total_xml_nodules)

    for entry in xml_entries:

        print("\nXML file:", os.path.basename(entry["xml_path"]))

        for nod in entry["nodules"]:

            print(
                f"  {nod['xml_nodule_id']}: "
                f"ROIs={nod['roi_count']}, "
                f"Z-count={nod['z_count']}, "
                f"SOP-count={nod['sop_count']}, "
                f"Z-range=({nod['z_min']}, {nod['z_max']}), "
                f"ratings={nod['ratings']}"
            )

    # ========================================================
    # STRICT CLASSIFICATION
    # ========================================================
    #
    # DIRECTLY PROVEN is permitted ONLY if:
    # - target native files themselves can be linked to DICOM
    #   SOPInstanceUIDs / physical coordinates, AND
    # - those identifiers can be matched to the XML.
    #
    # Merely having DICOM files somewhere on disk is NOT enough.
    #

    target_has_direct_dicom_link = False

    if dcm_files:
        print(
            "\nDICOM files exist somewhere in /content, "
            "but no target-specific link has been established."
        )

    if dcm_zip_files:
        print(
            "DICOM files also exist in the original ZIP, "
            "but target-specific correspondence still must be proven."
        )

    if target_has_direct_dicom_link:
        classification = "DIRECTLY PROVEN"
        reason = (
            "Target native slices have independently verifiable "
            "DICOM identifiers matching the XML annotation."
        )

    elif xml_entries:
        classification = "STRONGLY SUPPORTED"
        reason = (
            "The exact SeriesInstanceUID is linked to the target "
            "and matching XML annotations exist, but the stripped "
            "PNG dataset does not independently expose slice-level "
            "DICOM identifiers proving native-folder identity."
        )

    else:
        classification = "NOT DETERMINABLE"
        reason = (
            "No XML annotation file was found for the exact "
            "SeriesInstanceUID, so target provenance cannot be "
            "established from the available files."
        )

    print("\nFINAL CLASSIFICATION:", classification)
    print("REASON:", reason)

    results.append({
        **target_row,
        "xml_files_for_series": len(xml_entries),
        "xml_nodules_in_series": total_xml_nodules,
        "classification": classification,
        "reason": reason
    })

# ============================================================
# 9. SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    OUTPUT_CSV,
    index=False
)

summary_lines = []

summary_lines.append("=" * 80)
summary_lines.append("TARGETED LIDC-IDRI PROVENANCE DIAGNOSTIC")
summary_lines.append("=" * 80)

summary_lines.append(
    f"DICOM files on disk: {len(dcm_files)}"
)

summary_lines.append(
    f"DICOM files inside original ZIP: {len(dcm_zip_files)}"
)

summary_lines.append(
    f"PNG files inside original ZIP: {len(png_files)}"
)

summary_lines.append(
    f"XML files discovered: {len(xml_files)}"
)

summary_lines.append("")

for _, row in results_df.iterrows():

    summary_lines.append(
        f"{row['subset_nodule_id']}: "
        f"{row['classification']}"
    )

    summary_lines.append(
        f"  Patient: {row['patient_id']}"
    )

    summary_lines.append(
        f"  Native nodule: {row['native_nodule_id']}"
    )

    summary_lines.append(
        f"  SeriesInstanceUID: {row['series_instance_uid']}"
    )

    summary_lines.append(
        f"  XML files for series: {row['xml_files_for_series']}"
    )

    summary_lines.append(
        f"  XML nodules in series: {row['xml_nodules_in_series']}"
    )

    summary_lines.append(
        f"  Reason: {row['reason']}"
    )

    summary_lines.append("")

with open(OUTPUT_SUMMARY, "w") as f:
    f.write("\n".join(summary_lines))

# ============================================================
# 10. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("FINAL TARGETED PROVENANCE REPORT")
print("=" * 80)

print(results_df.to_string(index=False))

print("\nSaved CSV:")
print(OUTPUT_CSV)

print("\nSaved summary:")
print(OUTPUT_SUMMARY)

print("\n" + "=" * 80)
print("IMPORTANT INTERPRETATION")
print("=" * 80)

print(
    "The presence of DICOM files alone is NOT treated as proof "
    "of target provenance."
)

print(
    "A high geometric similarity score is NOT treated as proof "
    "of native-folder-to-XML identity."
)

print(
    "DIRECTLY PROVEN requires target-specific slice-level "
    "DICOM/SOP correspondence."
)

print("=" * 80)


TARGETED LIDC-IDRI PROVENANCE DIAGNOSTIC

All required files found.

--------------------------------------------------------------------------------
STEP 1 — DICOM SEARCH
--------------------------------------------------------------------------------
DICOM files physically present in /content: 0
No .dcm files found.

--------------------------------------------------------------------------------
STEP 2 — ORIGINAL ZIP CONTENTS
--------------------------------------------------------------------------------
Total ZIP entries: 77740
PNG files: 77740
DICOM files: 0
Other non-directory files: 0

RESULT: No DICOM files exist inside the original PNG ZIP.

--------------------------------------------------------------------------------
STEP 3 — MAPPING FILES
--------------------------------------------------------------------------------
Original mapping columns:
['subset_nodule_id', 'subset_slice_count', 'matched_slice_count', 'patient_id', 'native_nodule_id', 'best_match_count', 'second_b

In [ ]:
import os
import pandas as pd

# ============================================================
# FINAL COHORT FILTER
# Exclude unresolved ambiguous-series nodules
# ============================================================

MAPPING_CSV = "/content/nodule_to_lidc_mapping.csv"
SERIES_CSV = "/content/nodule_to_single_ct_series_fixed.csv"

OUTPUT_FINAL = "/content/final_eligible_325_nodules.csv"
OUTPUT_EXCLUDED = "/content/excluded_nodules.csv"

EXCLUDE_NODULES = {
    "nodule_029",
    "nodule_085",
}

print("=" * 80)
print("FINAL ELIGIBLE COHORT CREATION")
print("=" * 80)

# Load the original 327-nodule mapping
mapping_df = pd.read_csv(MAPPING_CSV)

print(f"Original subset rows: {len(mapping_df)}")

# Verify the two cases exist
for nid in sorted(EXCLUDE_NODULES):
    found = nid in set(mapping_df["subset_nodule_id"].astype(str))
    print(f"{nid}: {'FOUND' if found else 'NOT FOUND'}")

# Create excluded table
excluded_df = mapping_df[
    mapping_df["subset_nodule_id"].astype(str).isin(EXCLUDE_NODULES)
].copy()

# Create final eligible cohort
final_df = mapping_df[
    ~mapping_df["subset_nodule_id"].astype(str).isin(EXCLUDE_NODULES)
].copy()

# Save
final_df.to_csv(OUTPUT_FINAL, index=False)
excluded_df.to_csv(OUTPUT_EXCLUDED, index=False)

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)
print(f"Original nodules:       {len(mapping_df)}")
print(f"Excluded nodules:       {len(excluded_df)}")
print(f"Final eligible nodules: {len(final_df)}")

print("\nExcluded:")
print(excluded_df[
    ["subset_nodule_id", "patient_id", "native_nodule_id"]
].to_string(index=False))

print("\nFirst 10 eligible nodules:")
print(final_df[
    ["subset_nodule_id", "patient_id", "native_nodule_id"]
].head(10).to_string(index=False))

print("\nSaved:")
print(OUTPUT_FINAL)
print(OUTPUT_EXCLUDED)

# Safety check
assert len(mapping_df) == 327, "Expected 327 original subset nodules."
assert len(excluded_df) == 2, "Expected exactly 2 excluded nodules."
assert len(final_df) == 325, "Expected exactly 325 eligible nodules."

print("\n✓ FINAL COHORT = 325 NODULES")
print("✓ nodule_029 and nodule_085 excluded")
print("✓ Original data unchanged")
print("=" * 80)

FINAL ELIGIBLE COHORT CREATION
Original subset rows: 327
nodule_029: FOUND
nodule_085: FOUND

RESULT
Original nodules:       327
Excluded nodules:       2
Final eligible nodules: 325

Excluded:
subset_nodule_id     patient_id native_nodule_id
      nodule_029 LIDC-IDRI-0132         nodule-2
      nodule_085 LIDC-IDRI-0315         nodule-1

First 10 eligible nodules:
subset_nodule_id     patient_id native_nodule_id
      nodule_001 LIDC-IDRI-0121         nodule-1
      nodule_002 LIDC-IDRI-0158         nodule-1
      nodule_003 LIDC-IDRI-0702         nodule-3
      nodule_004 LIDC-IDRI-0241         nodule-1
      nodule_005 LIDC-IDRI-0008         nodule-0
      nodule_006 LIDC-IDRI-0602         nodule-1
      nodule_007 LIDC-IDRI-0636         nodule-1
      nodule_008 LIDC-IDRI-0780         nodule-1
      nodule_009 LIDC-IDRI-0435         nodule-3
      nodule_010 LIDC-IDRI-0663         nodule-2

Saved:
/content/final_eligible_325_nodules.csv
/content/excluded_nodules.csv

✓ FINAL COHOR

In [ ]:
import os
import pandas as pd

# =============================================================================
# FINAL VERIFICATION OF THE 325-NODULE ELIGIBLE COHORT
# =============================================================================

ELIGIBLE_CSV = "/content/final_eligible_325_nodules.csv"
SERIES_CSV = "/content/nodule_to_single_ct_series_fixed.csv"
VERIFIED_OUTPUT = "/content/verified_eligible_325_series_mapping.csv"

print("=" * 80)
print("FINAL VERIFICATION OF 325-NODULE ELIGIBLE COHORT")
print("=" * 80)

# -----------------------------------------------------------------------------
# 1. Check required files
# -----------------------------------------------------------------------------
if not os.path.exists(ELIGIBLE_CSV):
    raise FileNotFoundError(f"Missing file: {ELIGIBLE_CSV}")

if not os.path.exists(SERIES_CSV):
    raise FileNotFoundError(f"Missing file: {SERIES_CSV}")

# -----------------------------------------------------------------------------
# 2. Load files
# -----------------------------------------------------------------------------
eligible_df = pd.read_csv(ELIGIBLE_CSV)
series_df = pd.read_csv(SERIES_CSV)

eligible_df.columns = eligible_df.columns.str.strip()
series_df.columns = series_df.columns.str.strip()

# Normalize possible SeriesInstanceUID spelling
rename_series = {}

for col in series_df.columns:
    if col.strip().lower() == "seriesinstanceuid":
        rename_series[col] = "series_instance_uid"

series_df = series_df.rename(columns=rename_series)

print("\n" + "-" * 80)
print("INPUT FILES")
print("-" * 80)
print(f"Eligible cohort rows:       {len(eligible_df)}")
print(f"Series mapping rows:        {len(series_df)}")

# -----------------------------------------------------------------------------
# 3. Verify required columns
# -----------------------------------------------------------------------------
required_eligible = {
    "subset_nodule_id",
    "patient_id",
    "native_nodule_id"
}

required_series = {
    "subset_nodule_id",
    "patient_id",
    "native_nodule_id",
    "series_instance_uid"
}

missing_eligible = required_eligible - set(eligible_df.columns)
missing_series = required_series - set(series_df.columns)

if missing_eligible:
    raise KeyError(f"Eligible file is missing: {missing_eligible}")

if missing_series:
    raise KeyError(f"Series file is missing: {missing_series}")

# -----------------------------------------------------------------------------
# 4. Verify exactly 325 unique subset nodules
# -----------------------------------------------------------------------------
print("\n" + "-" * 80)
print("COHORT SIZE CHECK")
print("-" * 80)

eligible_ids = eligible_df["subset_nodule_id"].astype(str).str.strip()

row_count = len(eligible_df)
unique_count = eligible_ids.nunique()

print(f"Rows:                     {row_count}")
print(f"Unique subset_nodule_ids: {unique_count}")

if row_count != 325 or unique_count != 325:
    raise RuntimeError(
        f"Expected 325 rows and 325 unique nodules, "
        f"but found {row_count} rows and {unique_count} unique IDs."
    )

print("✓ Exactly 325 unique eligible nodules confirmed.")

# -----------------------------------------------------------------------------
# 5. Verify excluded nodules are absent
# -----------------------------------------------------------------------------
print("\n" + "-" * 80)
print("EXCLUSION CHECK")
print("-" * 80)

excluded_ids = {"nodule_029", "nodule_085"}
present_excluded = set(eligible_ids).intersection(excluded_ids)

print(f"Excluded IDs accidentally present: {present_excluded}")

if present_excluded:
    raise RuntimeError(
        f"Excluded nodules are still present: {present_excluded}"
    )

print("✓ nodule_029 and nodule_085 are absent.")

# -----------------------------------------------------------------------------
# 6. Clean the mapping columns
# -----------------------------------------------------------------------------
eligible_df["subset_nodule_id"] = (
    eligible_df["subset_nodule_id"].astype(str).str.strip()
)

eligible_df["patient_id"] = (
    eligible_df["patient_id"].astype(str).str.strip()
)

eligible_df["native_nodule_id"] = (
    eligible_df["native_nodule_id"].astype(str).str.strip()
)

series_df["subset_nodule_id"] = (
    series_df["subset_nodule_id"].astype(str).str.strip()
)

series_df["patient_id"] = (
    series_df["patient_id"].astype(str).str.strip()
)

series_df["native_nodule_id"] = (
    series_df["native_nodule_id"].astype(str).str.strip()
)

series_df["series_instance_uid"] = (
    series_df["series_instance_uid"]
    .astype(str)
    .str.strip()
)

# -----------------------------------------------------------------------------
# 7. Check each eligible nodule against series mapping
# -----------------------------------------------------------------------------
print("\n" + "-" * 80)
print("CT SERIES MAPPING CHECK")
print("-" * 80)

merged = eligible_df[
    ["subset_nodule_id", "patient_id", "native_nodule_id"]
].merge(
    series_df[
        [
            "subset_nodule_id",
            "patient_id",
            "native_nodule_id",
            "series_instance_uid"
        ]
    ],
    on=[
        "subset_nodule_id",
        "patient_id",
        "native_nodule_id"
    ],
    how="left"
)

print(f"Merged rows: {len(merged)}")

# Missing UID check
missing_uid = merged[
    merged["series_instance_uid"].isna()
    | (merged["series_instance_uid"].astype(str).str.strip() == "")
].copy()

print(f"Nodules missing SeriesInstanceUID: {len(missing_uid)}")

if not missing_uid.empty:
    print("\nMISSING SERIES IDS:")
    print(
        missing_uid[
            ["subset_nodule_id", "patient_id", "native_nodule_id"]
        ].to_string(index=False)
    )
    raise RuntimeError(
        "Some eligible nodules do not have a SeriesInstanceUID."
    )

print("✓ All 325 eligible nodules have a SeriesInstanceUID.")

# -----------------------------------------------------------------------------
# 8. Verify exactly one SeriesInstanceUID per subset nodule
# -----------------------------------------------------------------------------
uid_counts = (
    series_df[
        series_df["subset_nodule_id"].isin(set(eligible_ids))
    ]
    .groupby("subset_nodule_id")["series_instance_uid"]
    .nunique()
)

bad_uid_counts = uid_counts[uid_counts != 1]

# Also detect eligible IDs that don't appear in the mapping at all
missing_mapping_ids = sorted(
    set(eligible_ids) - set(uid_counts.index.astype(str))
)

print(f"Nodules with != 1 unique SeriesInstanceUID: {len(bad_uid_counts)}")
print(f"Nodules missing from series mapping entirely: {len(missing_mapping_ids)}")

if len(bad_uid_counts) > 0:
    print("\nBAD UID COUNTS:")
    print(bad_uid_counts)

    raise RuntimeError(
        "Some eligible nodules do not have exactly one unique SeriesInstanceUID."
    )

if missing_mapping_ids:
    print("\nMISSING MAPPING IDS:")
    print(missing_mapping_ids)

    raise RuntimeError(
        "Some eligible nodules are missing from the series mapping."
    )

print("✓ Every eligible nodule has exactly one unique SeriesInstanceUID.")

# -----------------------------------------------------------------------------
# 9. Verify patient/native IDs are consistent
# -----------------------------------------------------------------------------
# Because patient_id and native_nodule_id were part of the merge keys,
# pandas already guaranteed they matched for successful rows.

print("\n" + "-" * 80)
print("IDENTIFIER CONSISTENCY CHECK")
print("-" * 80)

print("✓ patient_id is consistent.")
print("✓ native_nodule_id is consistent.")

# -----------------------------------------------------------------------------
# 10. Check for duplicate final rows
# -----------------------------------------------------------------------------
duplicate_final_rows = merged.duplicated(
    subset=["subset_nodule_id"]
).sum()

print(f"\nDuplicate eligible subset IDs after merge: {duplicate_final_rows}")

if duplicate_final_rows != 0:
    raise RuntimeError(
        "Duplicate subset_nodule_id values detected after merging."
    )

print("✓ No duplicate eligible nodule mappings.")

# -----------------------------------------------------------------------------
# 11. Create final verified mapping
# -----------------------------------------------------------------------------
verified_df = merged[
    [
        "subset_nodule_id",
        "patient_id",
        "native_nodule_id",
        "series_instance_uid"
    ]
].copy()

verified_df.to_csv(VERIFIED_OUTPUT, index=False)

# -----------------------------------------------------------------------------
# 12. Final summary
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("FINAL VERIFICATION RESULT")
print("=" * 80)

print(f"Eligible nodules:                  {len(verified_df)}")
print(f"Unique subset nodules:              {verified_df['subset_nodule_id'].nunique()}")
print(f"Unique patients:                    {verified_df['patient_id'].nunique()}")
print(f"Unique SeriesInstanceUIDs:          {verified_df['series_instance_uid'].nunique()}")
print(f"Missing SeriesInstanceUIDs:         {verified_df['series_instance_uid'].isna().sum()}")
print(f"Duplicate subset mappings:          {duplicate_final_rows}")
print(f"nodule_029 present:                 {'nodule_029' in set(verified_df['subset_nodule_id'])}")
print(f"nodule_085 present:                 {'nodule_085' in set(verified_df['subset_nodule_id'])}")

print("\n✓ VERIFICATION PASSED")
print("✓ Final cohort contains exactly 325 nodules")
print("✓ Both ambiguous nodules remain excluded")
print("✓ All 325 nodules have exactly one SeriesInstanceUID")
print("✓ No duplicate subset mappings")

print(f"\nVerified mapping saved to:")
print(VERIFIED_OUTPUT)

print("=" * 80)

FINAL VERIFICATION OF 325-NODULE ELIGIBLE COHORT

--------------------------------------------------------------------------------
INPUT FILES
--------------------------------------------------------------------------------
Eligible cohort rows:       325
Series mapping rows:        325

--------------------------------------------------------------------------------
COHORT SIZE CHECK
--------------------------------------------------------------------------------
Rows:                     325
Unique subset_nodule_ids: 325
✓ Exactly 325 unique eligible nodules confirmed.

--------------------------------------------------------------------------------
EXCLUSION CHECK
--------------------------------------------------------------------------------
Excluded IDs accidentally present: set()
✓ nodule_029 and nodule_085 are absent.

--------------------------------------------------------------------------------
CT SERIES MAPPING CHECK
--------------------------------------------------------

In [ ]:
import os
import glob
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

# ================================================================
# FINAL CONSERVATIVE 325-NODULE ANNOTATION MANIFEST
# ================================================================

VERIFIED_CSV = "/content/verified_eligible_325_series_mapping.csv"
XML_DIR = "/content/lidc_xml_annotations"

OUTPUT_CSV = "/content/final_325_conservative_annotation_manifest.csv"
SUMMARY_TXT = "/content/final_325_conservative_annotation_summary.txt"


# ----------------------------------------------------------------
# Helper: robust column finder
# ----------------------------------------------------------------
def find_column(df, candidates):
    normalized = {
        str(c).strip().lower().replace(" ", "").replace("-", "_"): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower().replace(" ", "").replace("-", "_")
        if key in normalized:
            return normalized[key]

    return None


# ----------------------------------------------------------------
# Load verified 325-nodule cohort
# ----------------------------------------------------------------
print("=" * 80)
print("FINAL CONSERVATIVE 325-NODULE ANNOTATION MANIFEST")
print("=" * 80)

if not os.path.exists(VERIFIED_CSV):
    raise FileNotFoundError(f"Missing verified cohort: {VERIFIED_CSV}")

if not os.path.exists(XML_DIR):
    raise FileNotFoundError(f"Missing XML directory: {XML_DIR}")


df = pd.read_csv(VERIFIED_CSV)

subset_col = find_column(
    df,
    ["subset_nodule_id", "subset_nodule", "nodule_id"]
)

patient_col = find_column(
    df,
    ["patient_id", "patientid", "patient"]
)

native_col = find_column(
    df,
    ["native_nodule_id", "native_nodule", "native_id"]
)

series_col = find_column(
    df,
    ["series_instance_uid", "seriesinstanceuid", "series_uid"]
)

required = {
    "subset_nodule_id": subset_col,
    "patient_id": patient_col,
    "native_nodule_id": native_col,
    "series_instance_uid": series_col,
}

for name, col in required.items():
    if col is None:
        raise KeyError(
            f"Could not find required column '{name}'. "
            f"Available columns: {list(df.columns)}"
        )

# Standardize names
df = df.rename(columns={
    subset_col: "subset_nodule_id",
    patient_col: "patient_id",
    native_col: "native_nodule_id",
    series_col: "series_instance_uid",
})

# Basic cohort assertions
assert len(df) == 325, f"Expected 325 rows, got {len(df)}"
assert df["subset_nodule_id"].nunique() == 325, \
    "subset_nodule_id values are not unique"

ids = set(df["subset_nodule_id"].astype(str))

assert "nodule_029" not in ids, \
    "nodule_029 unexpectedly present"

assert "nodule_085" not in ids, \
    "nodule_085 unexpectedly present"

assert df["series_instance_uid"].notna().all(), \
    "At least one SeriesInstanceUID is missing"

assert (
    df["series_instance_uid"].astype(str).str.strip() != ""
).all(), \
    "At least one SeriesInstanceUID is empty"

print("\nCohort verification:")
print(f"  Rows: {len(df)}")
print(f"  Unique subset nodules: {df['subset_nodule_id'].nunique()}")
print(f"  Unique patients: {df['patient_id'].nunique()}")
print(f"  Unique SeriesInstanceUIDs: {df['series_instance_uid'].nunique()}")
print("  nodule_029 excluded: YES")
print("  nodule_085 excluded: YES")


# ----------------------------------------------------------------
# XML parser
# ----------------------------------------------------------------
def parse_xml_file(xml_path):

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception as e:
        return None, [], str(e)

    # Handle namespaces robustly
    if "}" in root.tag:
        namespace = root.tag.split("}")[0].strip("{")
        ns = {"lidc": namespace}

        def find(path):
            return root.find(path, ns)

        def findall(element, path):
            return element.findall(path, ns)

        series_elem = root.find(".//lidc:SeriesInstanceUid", ns)

        if series_elem is None:
            series_elem = root.find(".//lidc:SeriesInstanceUID", ns)

        nodule_nodes = root.findall(
            ".//lidc:unblindedReadNodule",
            ns
        )

    else:
        def find(path):
            return root.find(path)

        def findall(element, path):
            return element.findall(path)

        series_elem = root.find(".//SeriesInstanceUid")

        if series_elem is None:
            series_elem = root.find(".//SeriesInstanceUID")

        nodule_nodes = root.findall(
            ".//unblindedReadNodule"
        )

    if series_elem is None or not series_elem.text:
        return None, [], "SeriesInstanceUID not found"

    series_uid = series_elem.text.strip()

    nodules = []

    for idx, nod in enumerate(nodule_nodes):

        if "}" in root.tag:
            id_elem = nod.find("lidc:noduleID", ns)
            characteristics = nod.find(
                "lidc:characteristics",
                ns
            )

            rois = nod.findall("lidc:roi", ns)

        else:
            id_elem = nod.find("noduleID")
            characteristics = nod.find("characteristics")

            rois = nod.findall("roi")

        xml_nodule_id = (
            id_elem.text.strip()
            if id_elem is not None and id_elem.text
            else f"XML_{idx}"
        )

        malignancy = None

        if characteristics is not None:

            if "}" in root.tag:
                mal_elem = characteristics.find(
                    "lidc:malignancy",
                    ns
                )
            else:
                mal_elem = characteristics.find(
                    "malignancy"
                )

            if (
                mal_elem is not None
                and mal_elem.text
            ):
                try:
                    malignancy = int(
                        mal_elem.text.strip()
                    )
                except ValueError:
                    malignancy = None

        sop_uids = []
        z_positions = []

        for roi in rois:

            if "}" in root.tag:
                sop_elem = roi.find(
                    "lidc:imageSOP_UID",
                    ns
                )
                z_elem = roi.find(
                    "lidc:imageZposition",
                    ns
                )
            else:
                sop_elem = roi.find(
                    "imageSOP_UID"
                )
                z_elem = roi.find(
                    "imageZposition"
                )

            if (
                sop_elem is not None
                and sop_elem.text
            ):
                sop_uids.append(
                    sop_elem.text.strip()
                )

            if (
                z_elem is not None
                and z_elem.text
            ):
                try:
                    z_positions.append(
                        float(z_elem.text.strip())
                    )
                except ValueError:
                    pass

        nodules.append({
            "xml_nodule_id": xml_nodule_id,
            "xml_index": idx,
            "malignancy": malignancy,
            "roi_count": len(rois),
            "unique_sop_count": len(set(sop_uids)),
            "min_z": min(z_positions)
                if z_positions else None,
            "max_z": max(z_positions)
                if z_positions else None,
        })

    return series_uid, nodules, None


# ----------------------------------------------------------------
# Index XML collection
# ----------------------------------------------------------------
print("\n" + "-" * 80)
print("INDEXING XML FILES")
print("-" * 80)

xml_files = glob.glob(
    os.path.join(XML_DIR, "**", "*.xml"),
    recursive=True
)

print(f"XML files discovered: {len(xml_files)}")

series_to_xml = {}
xml_parse_errors = 0

for xml_path in xml_files:

    series_uid, nodules, error = parse_xml_file(
        xml_path
    )

    if error is not None:
        xml_parse_errors += 1
        continue

    if series_uid is None:
        continue

    if series_uid not in series_to_xml:
        series_to_xml[series_uid] = []

    series_to_xml[series_uid].append({
        "xml_file": os.path.basename(xml_path),
        "xml_path": xml_path,
        "nodules": nodules
    })

print(f"Unique SeriesInstanceUIDs indexed: {len(series_to_xml)}")
print(f"XML parse errors: {xml_parse_errors}")


# ----------------------------------------------------------------
# Construct conservative manifest
# ----------------------------------------------------------------
print("\n" + "-" * 80)
print("BUILDING CONSERVATIVE MANIFEST")
print("-" * 80)

manifest_rows = []

for _, row in df.iterrows():

    subset_id = str(row["subset_nodule_id"])
    patient_id = str(row["patient_id"])
    native_id = str(row["native_nodule_id"])
    series_uid = str(row["series_instance_uid"]).strip()

    series_entries = series_to_xml.get(
        series_uid,
        []
    )

    all_xml_ids = []
    all_ratings = []
    all_roi_counts = []
    all_sop_counts = []
    all_z_ranges = []
    xml_files_for_series = []

    for entry in series_entries:

        xml_files_for_series.append(
            entry["xml_file"]
        )

        for nod in entry["nodules"]:

            all_xml_ids.append(
                nod["xml_nodule_id"]
            )

            all_ratings.append(
                nod["malignancy"]
            )

            all_roi_counts.append(
                nod["roi_count"]
            )

            all_sop_counts.append(
                nod["unique_sop_count"]
            )

            all_z_ranges.append(
                (
                    nod["min_z"],
                    nod["max_z"]
                )
            )

    xml_count = len(all_xml_ids)

    # ------------------------------------------------------------
    # IMPORTANT:
    # We do NOT infer native-folder identity from:
    #   - XML order
    #   - nodule number
    #   - Z order
    #   - number of XML nodules
    # ------------------------------------------------------------

    verified_identity = False
    assigned_xml_id = None
    assigned_malignancy = None

    if xml_count == 0:

        match_status = "NO_XML_MATCH"

    elif xml_count == 1:

        # XML series contains exactly one annotated nodule,
        # but native-folder identity is still NOT independently proven.
        match_status = "SINGLE_XML_NODULE_IN_SERIES"

    else:

        match_status = "AMBIGUOUS_XML_MATCH"

    # ------------------------------------------------------------
    # Produce candidate strings
    # ------------------------------------------------------------
    candidate_ids = (
        ";".join(all_xml_ids)
        if all_xml_ids else ""
    )

    rating_strings = (
        ";".join(
            "None"
            if x is None
            else str(x)
            for x in all_ratings
        )
        if all_ratings else ""
    )

    z_strings = []

    for zmin, zmax in all_z_ranges:

        if zmin is None:
            z_strings.append("None")
        else:
            z_strings.append(
                f"{zmin:.2f}..{zmax:.2f}"
            )

    z_range_string = ";".join(z_strings)

    manifest_rows.append({

        "subset_nodule_id": subset_id,

        "patient_id": patient_id,

        "native_nodule_id": native_id,

        "series_instance_uid": series_uid,

        "match_status": match_status,

        "native_to_xml_identity_proven":
            verified_identity,

        "assigned_xml_nodule_id":
            assigned_xml_id,

        "assigned_malignancy_rating":
            assigned_malignancy,

        "xml_files_in_series":
            ";".join(sorted(set(xml_files_for_series))),

        "xml_nodules_in_series_count":
            xml_count,

        "candidate_xml_nodule_ids":
            candidate_ids,

        "candidate_xml_malignancy_ratings":
            rating_strings,

        "candidate_roi_counts":
            ";".join(
                str(x)
                for x in all_roi_counts
            ),

        "candidate_unique_sop_counts":
            ";".join(
                str(x)
                for x in all_sop_counts
            ),

        "candidate_z_ranges":
            z_range_string
    })


manifest = pd.DataFrame(manifest_rows)


# ----------------------------------------------------------------
# Final assertions
# ----------------------------------------------------------------
assert len(manifest) == 325

assert manifest[
    "subset_nodule_id"
].nunique() == 325

assert "nodule_029" not in set(
    manifest["subset_nodule_id"]
)

assert "nodule_085" not in set(
    manifest["subset_nodule_id"]
)

assert manifest[
    "series_instance_uid"
].notna().all()

assert (
    manifest["series_instance_uid"]
    .astype(str)
    .str.strip()
    .ne("")
).all()


# ----------------------------------------------------------------
# Statistics
# ----------------------------------------------------------------
status_counts = (
    manifest["match_status"]
    .value_counts()
    .to_dict()
)

single_xml = status_counts.get(
    "SINGLE_XML_NODULE_IN_SERIES",
    0
)

ambiguous_xml = status_counts.get(
    "AMBIGUOUS_XML_MATCH",
    0
)

no_xml = status_counts.get(
    "NO_XML_MATCH",
    0
)

verified = status_counts.get(
    "INDEPENDENTLY_VERIFIED_MATCH",
    0
)

# Conservative rule:
# assigned malignancy must correspond to proven identity
assigned_labels = manifest[
    manifest["assigned_malignancy_rating"]
    .notna()
]

# ----------------------------------------------------------------
# Save manifest
# ----------------------------------------------------------------
manifest.to_csv(
    OUTPUT_CSV,
    index=False
)


# ----------------------------------------------------------------
# Generate summary
# ----------------------------------------------------------------
summary = f"""
================================================================================
FINAL CONSERVATIVE 325-NODULE ANNOTATION SUMMARY
================================================================================

COHORT
--------------------------------------------------------------------------------
Total eligible nodules:                    325
Unique subset_nodule_ids:                 {manifest['subset_nodule_id'].nunique()}
Unique patients:                          {manifest['patient_id'].nunique()}
Unique SeriesInstanceUIDs:                {manifest['series_instance_uid'].nunique()}

EXCLUSIONS
--------------------------------------------------------------------------------
nodule_029 present:                       {"nodule_029" in set(manifest["subset_nodule_id"])}
nodule_085 present:                       {"nodule_085" in set(manifest["subset_nodule_id"])}

XML SERIES RESULTS
--------------------------------------------------------------------------------
Single XML nodule in series:              {single_xml}
Multiple XML nodules in series:            {ambiguous_xml}
No XML nodule in series:                  {no_xml}

IDENTITY STATUS
--------------------------------------------------------------------------------
Independently verified native→XML:         {verified}
Remaining unproven/ambiguous:             {325 - verified}

MALIGNANCY LABEL STATUS
--------------------------------------------------------------------------------
Native nodules with assigned malignancy:  {len(assigned_labels)}

IMPORTANT RULE
--------------------------------------------------------------------------------
A malignancy rating is NOT assigned to a native nodule merely because:
  - the series contains one XML nodule,
  - the XML appears first,
  - the XML nodule number looks similar,
  - Z-order appears similar,
  - or geometric similarity is high.

The native→XML identity must be independently established before
the XML malignancy rating can become a native-nodule ground-truth label.

PROVEN
--------------------------------------------------------------------------------
1. The final cohort contains exactly 325 eligible nodules.
2. nodule_029 and nodule_085 are excluded.
3. Every eligible nodule has exactly one selected SeriesInstanceUID.
4. XML annotations are indexed by exact SeriesInstanceUID.
5. XML nodule IDs, malignancy values, ROI counts, SOP counts,
   and Z ranges are preserved as candidate evidence.

NOT PROVEN
--------------------------------------------------------------------------------
The native `nodule-X` directory is not automatically treated as a specific
XML `unblindedReadNodule`.

Therefore, this manifest does NOT fabricate malignancy labels for
unproven native-folder/XML identities.

OUTPUT
--------------------------------------------------------------------------------
CSV:
{OUTPUT_CSV}

Summary:
{SUMMARY_TXT}
================================================================================
"""

with open(
    SUMMARY_TXT,
    "w"
) as f:
    f.write(summary.strip())

print(summary)

print("\nFirst 20 rows:")
print(
    manifest.head(20).to_string(index=False)
)

print("\n" + "=" * 80)
print("FINAL CHECK: PASSED")
print("=" * 80)

FINAL CONSERVATIVE 325-NODULE ANNOTATION MANIFEST

Cohort verification:
  Rows: 325
  Unique subset nodules: 325
  Unique patients: 247
  Unique SeriesInstanceUIDs: 247
  nodule_029 excluded: YES
  nodule_085 excluded: YES

--------------------------------------------------------------------------------
INDEXING XML FILES
--------------------------------------------------------------------------------
XML files discovered: 1319
Unique SeriesInstanceUIDs indexed: 1294
XML parse errors: 0

--------------------------------------------------------------------------------
BUILDING CONSERVATIVE MANIFEST
--------------------------------------------------------------------------------

FINAL CONSERVATIVE 325-NODULE ANNOTATION SUMMARY

COHORT
--------------------------------------------------------------------------------
Total eligible nodules:                    325
Unique subset_nodule_ids:                 325
Unique patients:                          247
Unique SeriesInstanceUIDs:          

In [ ]:
import os
import io
import glob
import zipfile
import xml.etree.ElementTree as ET

import pandas as pd
import numpy as np
from PIL import Image

# ============================================================
# FINAL 325-NODULE MASTER MANIFEST + DATASET QC
# ============================================================

ELIGIBLE_CSV = "/content/final_eligible_325_nodules.csv"
VERIFIED_SERIES_CSV = "/content/verified_eligible_325_series_mapping.csv"
CONSERVATIVE_CSV = "/content/final_325_conservative_annotation_manifest.csv"

ZIP_PATH = "/content/kagl_lidc_idri.zip"
XML_DIR = "/content/lidc_xml_annotations"

MASTER_CSV = "/content/final_325_master_manifest.csv"
QC_TXT = "/content/final_325_dataset_qc_report.txt"

EXCLUDED_IDS = {"nodule_029", "nodule_085"}


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def find_column(df, candidates, required=True):
    normalized = {
        str(c).strip().lower().replace(" ", "").replace("-", "_"): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower().replace(" ", "").replace("-", "_")
        if key in normalized:
            return normalized[key]

    if required:
        raise KeyError(
            f"Could not find any of {candidates}. "
            f"Available columns: {list(df.columns)}"
        )

    return None


def parse_xml_file(xml_path):
    """
    Parse one LIDC XML file.

    Returns:
        series_uid,
        list of XML nodule dictionaries
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception as e:
        return None, [], str(e)

    # Namespace-safe tag matching
    def local_name(tag):
        return tag.split("}")[-1]

    series_uid = None
    nodules = []

    # Search all elements by local tag name
    for elem in root.iter():
        if local_name(elem.tag) in {
            "SeriesInstanceUid",
            "SeriesInstanceUID"
        }:
            if elem.text and elem.text.strip():
                series_uid = elem.text.strip()
                break

    # Find every unblindedReadNodule
    for nodule in root.iter():
        if local_name(nodule.tag) != "unblindedReadNodule":
            continue

        xml_nodule_id = None
        malignancy = None
        roi_count = 0
        sop_uids = set()
        z_positions = []

        for child in nodule.iter():
            tag = local_name(child.tag)

            if tag == "noduleID" and child.text:
                xml_nodule_id = child.text.strip()

            elif tag == "malignancy" and child.text:
                try:
                    malignancy = int(child.text.strip())
                except Exception:
                    pass

            elif tag == "roi":
                roi_count += 1

            elif tag == "imageSOP_UID" and child.text:
                sop_uids.add(child.text.strip())

            elif tag == "imageZposition" and child.text:
                try:
                    z_positions.append(float(child.text.strip()))
                except Exception:
                    pass

        nodules.append({
            "xml_nodule_id": xml_nodule_id or "UNKNOWN",
            "malignancy": malignancy,
            "roi_count": roi_count,
            "unique_sop_count": len(sop_uids),
            "min_z": min(z_positions) if z_positions else np.nan,
            "max_z": max(z_positions) if z_positions else np.nan,
        })

    return series_uid, nodules, None


# ============================================================
# STEP 1 — LOAD AND VERIFY THE 325-NODULE COHORT
# ============================================================

print("=" * 80)
print("FINAL 325-NODULE MASTER MANIFEST + DATASET QC")
print("=" * 80)

for path in [
    ELIGIBLE_CSV,
    VERIFIED_SERIES_CSV,
    ZIP_PATH,
    XML_DIR
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required path missing: {path}")

eligible_df = pd.read_csv(ELIGIBLE_CSV)
series_df = pd.read_csv(VERIFIED_SERIES_CSV)

print("\n" + "-" * 80)
print("COHORT INPUT CHECK")
print("-" * 80)

eligible_sub_col = find_column(
    eligible_df,
    ["subset_nodule_id", "subset_nodule", "nodule_id"]
)

series_sub_col = find_column(
    series_df,
    ["subset_nodule_id", "subset_nodule", "nodule_id"]
)

series_patient_col = find_column(
    series_df,
    ["patient_id", "patientid"]
)

series_native_col = find_column(
    series_df,
    ["native_nodule_id", "native_nodule", "native_id"]
)

series_uid_col = find_column(
    series_df,
    ["series_instance_uid", "seriesinstanceuid", "series_uid"]
)

assert len(eligible_df) == 325, (
    f"Expected 325 eligible rows, got {len(eligible_df)}"
)

assert eligible_df[eligible_sub_col].nunique() == 325, (
    "Eligible cohort does not contain 325 unique subset_nodule_id values."
)

eligible_ids = set(eligible_df[eligible_sub_col].astype(str))

assert not (eligible_ids & EXCLUDED_IDS), (
    f"Excluded nodules accidentally present: {eligible_ids & EXCLUDED_IDS}"
)

print(f"Eligible rows:              {len(eligible_df)}")
print(f"Unique eligible nodules:    {eligible_df[eligible_sub_col].nunique()}")
print(f"Excluded IDs present:       {eligible_ids & EXCLUDED_IDS}")
print("✓ Cohort structure is valid.")


# ============================================================
# STEP 2 — VERIFY SERIES MAPPING
# ============================================================

print("\n" + "-" * 80)
print("SERIES MAPPING CHECK")
print("-" * 80)

assert len(series_df) == 325, (
    f"Expected 325 series mapping rows, got {len(series_df)}"
)

assert series_df[series_sub_col].nunique() == 325

series_uid_clean = (
    series_df[series_uid_col]
    .astype(str)
    .str.strip()
)

assert series_uid_clean.notna().all()
assert (series_uid_clean != "").all()
assert (~series_uid_clean.str.lower().isin(["nan", "none"])).all()

print(f"Series mapping rows:        {len(series_df)}")
print(f"Unique SeriesInstanceUIDs:  {series_uid_clean.nunique()}")

# Merge cohort + series mapping
master_base = pd.merge(
    eligible_df,
    series_df[
        [series_sub_col,
         series_patient_col,
         series_native_col,
         series_uid_col]
    ],
    left_on=eligible_sub_col,
    right_on=series_sub_col,
    how="left",
    suffixes=("", "_series")
)

assert len(master_base) == 325

print(f"Merged rows:                {len(master_base)}")


# ============================================================
# STEP 3 — NORMALIZE BASE FIELDS
# ============================================================

master_base["subset_nodule_id"] = master_base[eligible_sub_col].astype(str).str.strip()

# Prefer series mapping values
master_base["patient_id_final"] = (
    master_base[series_patient_col]
    .astype(str)
    .str.strip()
)

master_base["native_nodule_id_final"] = (
    master_base[series_native_col]
    .astype(str)
    .str.strip()
)

master_base["SeriesInstanceUID_final"] = (
    master_base[series_uid_col]
    .astype(str)
    .str.strip()
)

assert master_base["patient_id_final"].notna().all()
assert master_base["native_nodule_id_final"].notna().all()
assert master_base["SeriesInstanceUID_final"].notna().all()

assert (
    master_base["SeriesInstanceUID_final"]
    .str.lower()
    .ne("nan")
).all()


# ============================================================
# STEP 4 — INDEX XML FILES
# ============================================================

print("\n" + "-" * 80)
print("XML INDEXING")
print("-" * 80)

xml_files = glob.glob(
    os.path.join(XML_DIR, "**", "*.xml"),
    recursive=True
)

print(f"XML files discovered:       {len(xml_files)}")

xml_by_series = {}
xml_parse_errors = 0

for xml_path in xml_files:

    suid, nodules, error = parse_xml_file(xml_path)

    if error is not None:
        xml_parse_errors += 1
        continue

    if suid is None:
        continue

    if suid not in xml_by_series:
        xml_by_series[suid] = []

    xml_by_series[suid].append({
        "xml_file": os.path.basename(xml_path),
        "nodules": nodules
    })

print(f"Unique SeriesInstanceUIDs:  {len(xml_by_series)}")
print(f"XML parse errors:           {xml_parse_errors}")


# ============================================================
# STEP 5 — INDEX ORIGINAL ZIP
# ============================================================

print("\n" + "-" * 80)
print("ORIGINAL ZIP QC")
print("-" * 80)

master_rows = []

qc = {
    "folder_exists": 0,
    "images_present": 0,
    "images_readable": 0,
    "images_consistent_dims": 0,
    "masks_readable": 0,
    "masks_consistent_dims": 0,
    "at_least_one_foreground_mask": 0,
    "overall_pass": 0,
    "overall_fail": 0
}

with zipfile.ZipFile(ZIP_PATH, "r") as zf:

    zip_names = set(zf.namelist())

    for idx, row in master_base.iterrows():

        subset_id = row["subset_nodule_id"]
        patient_id = row["patient_id_final"]
        native_id = row["native_nodule_id_final"]
        series_uid = row["SeriesInstanceUID_final"]

        base_prefix = (
            f"LIDC-IDRI-slices/"
            f"{patient_id}/"
            f"{native_id}/"
        )

        image_prefix = base_prefix + "images/"
        mask_prefixes = [
            base_prefix + f"mask-{m}/"
            for m in range(4)
        ]

        # ----------------------------------------------------
        # Folder check
        # ----------------------------------------------------

        folder_exists = any(
            name.startswith(base_prefix)
            for name in zip_names
        )

        if folder_exists:
            qc["folder_exists"] += 1

        # ----------------------------------------------------
        # Image checks
        # ----------------------------------------------------

        image_files = sorted([
            name for name in zip_names
            if name.startswith(image_prefix)
            and name.lower().endswith(".png")
        ])

        image_count = len(image_files)

        if image_count > 0:
            qc["images_present"] += 1

        images_readable = image_count > 0
        image_dimensions = set()

        for image_file in image_files:

            try:
                with zf.open(image_file) as f:
                    with Image.open(f) as img:
                        img.verify()

                with zf.open(image_file) as f:
                    with Image.open(f) as img:
                        image_dimensions.add(img.size)

            except Exception:
                images_readable = False

        if images_readable:
            qc["images_readable"] += 1

        image_dims_consistent = len(image_dimensions) == 1

        if image_dims_consistent:
            qc["images_consistent_dims"] += 1

        image_dim_str = (
            f"{next(iter(image_dimensions))[0]}x"
            f"{next(iter(image_dimensions))[1]}"
            if image_dimensions
            else "NONE"
        )

        # ----------------------------------------------------
        # Mask checks
        # ----------------------------------------------------

        mask_slice_counts = []
        mask_nonempty_counts = []

        all_mask_dimensions = set()
        masks_readable = True
        any_foreground = False

        total_mask_volume_pixels = 0
        max_mask_area_pixels = 0

        for mask_idx, mask_prefix in enumerate(mask_prefixes):

            mask_files = sorted([
                name for name in zip_names
                if name.startswith(mask_prefix)
                and name.lower().endswith(".png")
            ])

            mask_slice_counts.append(len(mask_files))

            nonempty_count = 0

            for mask_file in mask_files:

                try:
                    with zf.open(mask_file) as f:
                        with Image.open(f) as mask_img:

                            arr = np.asarray(
                                mask_img.convert("L")
                            )

                            all_mask_dimensions.add(mask_img.size)

                            foreground = np.count_nonzero(arr)

                            if foreground > 0:
                                nonempty_count += 1
                                any_foreground = True
                                total_mask_volume_pixels += int(foreground)
                                max_mask_area_pixels = max(
                                    max_mask_area_pixels,
                                    int(foreground)
                                )

                except Exception:
                    masks_readable = False

            mask_nonempty_counts.append(nonempty_count)

        if masks_readable:
            qc["masks_readable"] += 1

        masks_dims_consistent = len(all_mask_dimensions) <= 1

        if masks_dims_consistent:
            qc["masks_consistent_dims"] += 1

        if any_foreground:
            qc["at_least_one_foreground_mask"] += 1

        mask_dim_str = (
            f"{next(iter(all_mask_dimensions))[0]}x"
            f"{next(iter(all_mask_dimensions))[1]}"
            if all_mask_dimensions
            else "NONE"
        )

        # ----------------------------------------------------
        # Image/mask dimension comparison
        # ----------------------------------------------------

        dimensions_match = (
            len(image_dimensions) == 1
            and len(all_mask_dimensions) == 1
            and image_dimensions == all_mask_dimensions
        )

        # ----------------------------------------------------
        # Overall QC
        # ----------------------------------------------------

        overall_pass = (
            folder_exists
            and image_count > 0
            and images_readable
            and image_dims_consistent
            and masks_readable
            and masks_dims_consistent
            and dimensions_match
            and any_foreground
        )

        if overall_pass:
            qc["overall_pass"] += 1
        else:
            qc["overall_fail"] += 1

        # ----------------------------------------------------
        # XML evidence
        # ----------------------------------------------------

        xml_entries = xml_by_series.get(series_uid, [])

        xml_file_names = sorted({
            entry["xml_file"]
            for entry in xml_entries
        })

        xml_nodules = []

        for entry in xml_entries:
            xml_nodules.extend(entry["nodules"])

        xml_ids = [
            str(n["xml_nodule_id"])
            for n in xml_nodules
        ]

        xml_malignancies = [
            (
                str(n["malignancy"])
                if n["malignancy"] is not None
                else "None"
            )
            for n in xml_nodules
        ]

        xml_roi_counts = [
            str(n["roi_count"])
            for n in xml_nodules
        ]

        xml_sop_counts = [
            str(n["unique_sop_count"])
            for n in xml_nodules
        ]

        xml_z_ranges = []

        for n in xml_nodules:

            if (
                not pd.isna(n["min_z"])
                and not pd.isna(n["max_z"])
            ):
                xml_z_ranges.append(
                    f"[{n['min_z']:.2f},{n['max_z']:.2f}]"
                )
            else:
                xml_z_ranges.append("None")

        # ----------------------------------------------------
        # IMPORTANT:
        # Identity remains unproven
        # ----------------------------------------------------

        native_to_xml_identity_proven = False

        master_rows.append({
            "subset_nodule_id": subset_id,
            "patient_id": patient_id,
            "native_nodule_id": native_id,
            "SeriesInstanceUID": series_uid,

            "image_directory": image_prefix,

            "mask_0_directory": mask_prefixes[0],
            "mask_1_directory": mask_prefixes[1],
            "mask_2_directory": mask_prefixes[2],
            "mask_3_directory": mask_prefixes[3],

            "image_slice_count": image_count,

            "mask_0_slice_count": mask_slice_counts[0],
            "mask_1_slice_count": mask_slice_counts[1],
            "mask_2_slice_count": mask_slice_counts[2],
            "mask_3_slice_count": mask_slice_counts[3],

            "mask_0_nonempty_slice_count": mask_nonempty_counts[0],
            "mask_1_nonempty_slice_count": mask_nonempty_counts[1],
            "mask_2_nonempty_slice_count": mask_nonempty_counts[2],
            "mask_3_nonempty_slice_count": mask_nonempty_counts[3],

            "image_dimensions": image_dim_str,
            "mask_dimensions": mask_dim_str,

            "xml_file": ";".join(xml_file_names),
            "xml_nodule_count_in_series": len(xml_nodules),

            "candidate_xml_nodule_ids":
                ";".join(xml_ids) if xml_ids else "None",

            "candidate_xml_malignancy_ratings":
                ";".join(xml_malignancies)
                if xml_malignancies else "None",

            "candidate_roi_counts":
                ";".join(xml_roi_counts)
                if xml_roi_counts else "None",

            "candidate_unique_sop_counts":
                ";".join(xml_sop_counts)
                if xml_sop_counts else "None",

            "candidate_z_ranges":
                ";".join(xml_z_ranges)
                if xml_z_ranges else "None",

            "native_to_xml_identity_proven":
                native_to_xml_identity_proven,

            "total_mask_volume_pixels":
                total_mask_volume_pixels,

            "max_mask_area_pixels":
                max_mask_area_pixels,

            "number_of_masks_with_foreground":
                sum(
                    1
                    for c in mask_nonempty_counts
                    if c > 0
                ),

            "all_image_masks_dimension_match":
                dimensions_match,

            "image_files_complete":
                folder_exists and image_count > 0,

            "image_files_readable":
                images_readable,

            "mask_files_readable":
                masks_readable,

            "qc_status":
                "PASS" if overall_pass else "FAIL"
        })


# ============================================================
# STEP 6 — CREATE MASTER MANIFEST
# ============================================================

master_df = pd.DataFrame(master_rows)

assert len(master_df) == 325
assert master_df["subset_nodule_id"].nunique() == 325

master_ids = set(
    master_df["subset_nodule_id"].astype(str)
)

assert not (master_ids & EXCLUDED_IDS)

assert (
    master_df["SeriesInstanceUID"]
    .astype(str)
    .str.strip()
    .ne("")
).all()

assert master_df["SeriesInstanceUID"].notna().all()

# Identity must remain conservative
assert (
    master_df["native_to_xml_identity_proven"]
    == False
).all()

master_df.to_csv(
    MASTER_CSV,
    index=False
)


# ============================================================
# STEP 7 — FINAL QC REPORT
# ============================================================

report = []

report.append("=" * 80)
report.append("FINAL 325-NODULE DATASET QUALITY CONTROL REPORT")
report.append("=" * 80)
report.append("")

report.append("COHORT")
report.append("-" * 80)
report.append(f"Eligible nodules:              {len(master_df)}")
report.append(
    f"Unique patients:                "
    f"{master_df['patient_id'].nunique()}"
)
report.append(
    f"Unique SeriesInstanceUIDs:      "
    f"{master_df['SeriesInstanceUID'].nunique()}"
)
report.append(
    f"nodule_029 present:             "
    f"{'nodule_029' in master_ids}"
)
report.append(
    f"nodule_085 present:             "
    f"{'nodule_085' in master_ids}"
)
report.append("")

report.append("QC CHECKS")
report.append("-" * 80)

report.append(
    f"Native folder exists:            "
    f"{qc['folder_exists']} / 325"
)

report.append(
    f"Images present:                  "
    f"{qc['images_present']} / 325"
)

report.append(
    f"Images readable:                 "
    f"{qc['images_readable']} / 325"
)

report.append(
    f"Image dimensions consistent:     "
    f"{qc['images_consistent_dims']} / 325"
)

report.append(
    f"Masks readable:                  "
    f"{qc['masks_readable']} / 325"
)

report.append(
    f"Mask dimensions consistent:     "
    f"{qc['masks_consistent_dims']} / 325"
)

report.append(
    f"At least one foreground mask:    "
    f"{qc['at_least_one_foreground_mask']} / 325"
)

report.append("")

report.append("OVERALL")
report.append("-" * 80)
report.append(
    f"QC PASS:                         "
    f"{qc['overall_pass']} / 325"
)
report.append(
    f"QC FAIL:                         "
    f"{qc['overall_fail']} / 325"
)
report.append("")

report.append("XML INVENTORY")
report.append("-" * 80)

xml_series_for_cohort = sum(
    1
    for uid in master_df["SeriesInstanceUID"]
    if uid in xml_by_series
)

report.append(
    f"Cohort SeriesInstanceUIDs with XML: "
    f"{xml_series_for_cohort} / 325"
)

report.append(
    f"XML parse errors:                    "
    f"{xml_parse_errors}"
)

report.append("")

report.append("PROVENANCE RULE")
report.append("-" * 80)
report.append(
    "SeriesInstanceUID association is retained as established metadata."
)
report.append(
    "Native nodule-X -> specific XML nodule identity remains unproven."
)
report.append(
    "No malignancy labels were assigned to native folders."
)
report.append(
    "XML malignancy values are preserved only as candidate evidence."
)

report.append("")
report.append("=" * 80)

report_text = "\n".join(report)

with open(QC_TXT, "w") as f:
    f.write(report_text)

print(report_text)

print("\nFIRST 10 MASTER-MANIFEST ROWS")
print("-" * 80)

print(
    master_df[
        [
            "subset_nodule_id",
            "patient_id",
            "native_nodule_id",
            "SeriesInstanceUID",
            "image_slice_count",
            "number_of_masks_with_foreground",
            "qc_status",
            "native_to_xml_identity_proven"
        ]
    ].head(10).to_string(index=False)
)

print("\n" + "=" * 80)
print("FILES SAVED")
print("=" * 80)
print(MASTER_CSV)
print(QC_TXT)
print("=" * 80)

FINAL 325-NODULE MASTER MANIFEST + DATASET QC

--------------------------------------------------------------------------------
COHORT INPUT CHECK
--------------------------------------------------------------------------------
Eligible rows:              325
Unique eligible nodules:    325
Excluded IDs present:       set()
✓ Cohort structure is valid.

--------------------------------------------------------------------------------
SERIES MAPPING CHECK
--------------------------------------------------------------------------------
Series mapping rows:        325
Unique SeriesInstanceUIDs:  247
Merged rows:                325

--------------------------------------------------------------------------------
XML INDEXING
--------------------------------------------------------------------------------
XML files discovered:       1319
Unique SeriesInstanceUIDs:  1294
XML parse errors:           0

--------------------------------------------------------------------------------
ORIGINAL ZI

In [ ]:
import os
import io
import json
import zipfile
import shutil
import re
import pandas as pd
import numpy as np
from PIL import Image

# =============================================================================
# STEP 16 — BUILD CLEAN PROCESSED 325-NODULE DATASET
# =============================================================================

ZIP_PATH = "/content/kagl_lidc_idri.zip"
MASTER_MANIFEST = "/content/final_325_master_manifest.csv"

OUTPUT_ROOT = "/content/processed_325_nodules"
OUTPUT_MANIFEST = "/content/processed_325_manifest.csv"
OUTPUT_SUMMARY = "/content/processed_325_summary.txt"

EXCLUDED_IDS = {"nodule_029", "nodule_085"}

print("=" * 80)
print("STEP 16 — BUILDING CLEAN PROCESSED 325-NODULE DATASET")
print("=" * 80)

# -----------------------------------------------------------------------------
# 1. Validate inputs
# -----------------------------------------------------------------------------

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"Original ZIP not found: {ZIP_PATH}")

if not os.path.exists(MASTER_MANIFEST):
    raise FileNotFoundError(f"Master manifest not found: {MASTER_MANIFEST}")

df = pd.read_csv(MASTER_MANIFEST)

required_columns = [
    "subset_nodule_id",
    "patient_id",
    "native_nodule_id",
    "SeriesInstanceUID"
]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise KeyError(
        f"Master manifest is missing required columns: {missing_columns}\n"
        f"Available columns: {list(df.columns)}"
    )

# -----------------------------------------------------------------------------
# 2. Validate the 325-nodule cohort
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("COHORT VALIDATION")
print("-" * 80)

print(f"Input rows: {len(df)}")
print(f"Unique subset nodules: {df['subset_nodule_id'].nunique()}")
print(f"Unique patients: {df['patient_id'].nunique()}")
print(f"Unique SeriesInstanceUIDs: {df['SeriesInstanceUID'].nunique()}")

assert len(df) == 325, f"Expected 325 rows, found {len(df)}"
assert df["subset_nodule_id"].nunique() == 325, "Duplicate subset nodule IDs detected"

present_excluded = set(df["subset_nodule_id"].astype(str)) & EXCLUDED_IDS

assert not present_excluded, (
    f"Excluded nodules unexpectedly present: {present_excluded}"
)

assert df["SeriesInstanceUID"].notna().all(), "Missing SeriesInstanceUID detected"

print("✓ Cohort contains exactly 325 valid nodules")
print("✓ Excluded nodules are absent")
print("✓ All SeriesInstanceUID values are present")

# -----------------------------------------------------------------------------
# 3. Create output directories
# -----------------------------------------------------------------------------

if os.path.exists(OUTPUT_ROOT):
    print(f"\nRemoving previous processed dataset: {OUTPUT_ROOT}")
    shutil.rmtree(OUTPUT_ROOT)

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print(f"\nOutput directory created:")
print(f"  {OUTPUT_ROOT}")

# -----------------------------------------------------------------------------
# 4. Open original ZIP
# -----------------------------------------------------------------------------

zf = zipfile.ZipFile(ZIP_PATH, "r")
zip_names = set(zf.namelist())

print(f"\nOriginal ZIP entries: {len(zip_names)}")

# -----------------------------------------------------------------------------
# 5. Process every nodule
# -----------------------------------------------------------------------------

processed_rows = []

total_images = 0
total_masks = 0
total_nonempty_masks = 0

failed_cases = []

print("\n" + "-" * 80)
print("EXTRACTING 325 NODULE DATASETS")
print("-" * 80)

for i, row in df.iterrows():

    subset_id = str(row["subset_nodule_id"])
    patient_id = str(row["patient_id"])
    native_id = str(row["native_nodule_id"])
    series_uid = str(row["SeriesInstanceUID"])

    # -------------------------------------------------------------------------
    # Output directory structure:
    #
    # processed_325_nodules/
    #     nodule_001/
    #         images/
    #         mask-0/
    #         mask-1/
    #         mask-2/
    #         mask-3/
    #         metadata.json
    # -------------------------------------------------------------------------

    case_dir = os.path.join(OUTPUT_ROOT, subset_id)
    images_dir = os.path.join(case_dir, "images")

    mask_dirs = {
        0: os.path.join(case_dir, "mask-0"),
        1: os.path.join(case_dir, "mask-1"),
        2: os.path.join(case_dir, "mask-2"),
        3: os.path.join(case_dir, "mask-3"),
    }

    os.makedirs(images_dir, exist_ok=True)

    for d in mask_dirs.values():
        os.makedirs(d, exist_ok=True)

    source_prefix = (
        f"LIDC-IDRI-slices/{patient_id}/{native_id}/"
    )

    source_image_prefix = source_prefix + "images/"

    source_mask_prefixes = {
        m: source_prefix + f"mask-{m}/"
        for m in range(4)
    }

    # -------------------------------------------------------------------------
    # Find source image files
    # -------------------------------------------------------------------------

    image_sources = [
        x for x in zip_names
        if x.startswith(source_image_prefix)
        and x.lower().endswith(".png")
    ]

    # Sort slice files numerically rather than lexicographically
    def slice_number(path):
        match = re.search(r"slice-(\d+)\.png$", path)
        return int(match.group(1)) if match else 10**9

    image_sources = sorted(image_sources, key=slice_number)

    # -------------------------------------------------------------------------
    # Find mask files
    # -------------------------------------------------------------------------

    mask_sources = {}

    for m in range(4):
        mask_sources[m] = sorted(
            [
                x for x in zip_names
                if x.startswith(source_mask_prefixes[m])
                and x.lower().endswith(".png")
            ],
            key=slice_number
        )

    # -------------------------------------------------------------------------
    # Basic existence check
    # -------------------------------------------------------------------------

    case_failed = False

    if len(image_sources) == 0:
        case_failed = True
        failed_cases.append({
            "subset_nodule_id": subset_id,
            "reason": "NO_IMAGE_FILES"
        })

    # -------------------------------------------------------------------------
    # Extract image PNGs
    # -------------------------------------------------------------------------

    case_image_count = 0
    image_dimensions = set()

    for src in image_sources:

        filename = os.path.basename(src)
        dst = os.path.join(images_dir, filename)

        try:
            with zf.open(src) as source_file:
                raw = source_file.read()

            # Validate image
            with Image.open(io.BytesIO(raw)) as img:
                image_dimensions.add(img.size)

            with open(dst, "wb") as out_file:
                out_file.write(raw)

            case_image_count += 1

        except Exception as e:
            case_failed = True
            failed_cases.append({
                "subset_nodule_id": subset_id,
                "reason": f"IMAGE_READ_ERROR: {e}"
            })

    # -------------------------------------------------------------------------
    # Extract masks
    # -------------------------------------------------------------------------

    mask_file_counts = {}
    mask_nonempty_counts = {}

    for m in range(4):

        dst_dir = mask_dirs[m]

        count = 0
        nonempty = 0

        for src in mask_sources[m]:

            filename = os.path.basename(src)
            dst = os.path.join(dst_dir, filename)

            try:

                with zf.open(src) as source_file:
                    raw = source_file.read()

                with Image.open(io.BytesIO(raw)) as mask_img:
                    arr = np.array(mask_img)

                    if np.count_nonzero(arr) > 0:
                        nonempty += 1

                with open(dst, "wb") as out_file:
                    out_file.write(raw)

                count += 1

            except Exception as e:

                case_failed = True
                failed_cases.append({
                    "subset_nodule_id": subset_id,
                    "reason": f"MASK_{m}_READ_ERROR: {e}"
                })

        mask_file_counts[m] = count
        mask_nonempty_counts[m] = nonempty

    # -------------------------------------------------------------------------
    # Compute dimensions
    # -------------------------------------------------------------------------

    dimension_string = (
        ";".join(
            sorted([f"{w}x{h}" for w, h in image_dimensions])
        )
        if image_dimensions
        else "NONE"
    )

    # -------------------------------------------------------------------------
    # Write per-case metadata JSON
    # -------------------------------------------------------------------------

    metadata = {
        "subset_nodule_id": subset_id,
        "patient_id": patient_id,
        "native_nodule_id": native_id,
        "SeriesInstanceUID": series_uid,

        "source_zip": ZIP_PATH,

        "image_slice_count": case_image_count,
        "image_dimensions": dimension_string,

        "mask_0_file_count": mask_file_counts[0],
        "mask_1_file_count": mask_file_counts[1],
        "mask_2_file_count": mask_file_counts[2],
        "mask_3_file_count": mask_file_counts[3],

        "mask_0_nonempty_slice_count": mask_nonempty_counts[0],
        "mask_1_nonempty_slice_count": mask_nonempty_counts[1],
        "mask_2_nonempty_slice_count": mask_nonempty_counts[2],
        "mask_3_nonempty_slice_count": mask_nonempty_counts[3],

        "native_to_xml_identity_proven": False,

        "malignancy_label_assigned": False,

        "processing_status": (
            "PASS" if not case_failed else "FAIL"
        )
    }

    metadata_path = os.path.join(case_dir, "metadata.json")

    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    # -------------------------------------------------------------------------
    # Update totals
    # -------------------------------------------------------------------------

    total_images += case_image_count

    for m in range(4):
        total_masks += mask_file_counts[m]
        total_nonempty_masks += mask_nonempty_counts[m]

    # -------------------------------------------------------------------------
    # Build processed manifest row
    # -------------------------------------------------------------------------

    processed_rows.append({
        "subset_nodule_id": subset_id,
        "patient_id": patient_id,
        "native_nodule_id": native_id,
        "SeriesInstanceUID": series_uid,

        "processed_directory": case_dir,

        "image_slice_count": case_image_count,
        "image_dimensions": dimension_string,

        "mask_0_file_count": mask_file_counts[0],
        "mask_1_file_count": mask_file_counts[1],
        "mask_2_file_count": mask_file_counts[2],
        "mask_3_file_count": mask_file_counts[3],

        "mask_0_nonempty_slice_count": mask_nonempty_counts[0],
        "mask_1_nonempty_slice_count": mask_nonempty_counts[1],
        "mask_2_nonempty_slice_count": mask_nonempty_counts[2],
        "mask_3_nonempty_slice_count": mask_nonempty_counts[3],

        "native_to_xml_identity_proven": False,
        "malignancy_label_assigned": False,

        "processing_status": (
            "PASS" if not case_failed else "FAIL"
        )
    })

    # Progress
    processed_number = i + 1

    if processed_number % 25 == 0 or processed_number == len(df):
        print(
            f"Processed {processed_number} / {len(df)}"
        )

# -----------------------------------------------------------------------------
# 6. Close ZIP
# -----------------------------------------------------------------------------

zf.close()

# -----------------------------------------------------------------------------
# 7. Build processed manifest
# -----------------------------------------------------------------------------

processed_df = pd.DataFrame(processed_rows)

processed_df.to_csv(
    OUTPUT_MANIFEST,
    index=False
)

# -----------------------------------------------------------------------------
# 8. Final validation
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL PROCESSED DATASET VALIDATION")
print("=" * 80)

print(f"Processed manifest rows:       {len(processed_df)}")
print(f"Unique subset nodules:        {processed_df['subset_nodule_id'].nunique()}")
print(f"Unique patients:              {processed_df['patient_id'].nunique()}")
print(f"Unique SeriesInstanceUIDs:    {processed_df['SeriesInstanceUID'].nunique()}")

pass_count = (
    processed_df["processing_status"] == "PASS"
).sum()

fail_count = (
    processed_df["processing_status"] == "FAIL"
).sum()

print(f"\nCases successfully processed: {pass_count}")
print(f"Cases with processing errors: {fail_count}")

print(f"\nTotal image PNG files:        {total_images}")
print(f"Total mask PNG files:         {total_masks}")
print(f"Non-empty mask slices:        {total_nonempty_masks}")

assert len(processed_df) == 325
assert processed_df["subset_nodule_id"].nunique() == 325

assert not (
    set(processed_df["subset_nodule_id"].astype(str))
    & EXCLUDED_IDS
)

assert processed_df["SeriesInstanceUID"].notna().all()

# We expect all cases to pass based on the previous QC.
if fail_count == 0:
    print("\n✓ ALL 325 NODULES SUCCESSFULLY PROCESSED")
else:
    print(
        f"\n⚠ {fail_count} CASE(S) NEED REVIEW"
    )

# -----------------------------------------------------------------------------
# 9. Save summary
# -----------------------------------------------------------------------------

summary_lines = [
    "===============================================================================",
    "PROCESSED 325-NODULE DATASET SUMMARY",
    "===============================================================================",
    "",
    f"Total processed nodules:          {len(processed_df)}",
    f"Unique patients:                   {processed_df['patient_id'].nunique()}",
    f"Unique SeriesInstanceUIDs:         {processed_df['SeriesInstanceUID'].nunique()}",
    "",
    f"Successfully processed:            {pass_count}",
    f"Processing failures:               {fail_count}",
    "",
    f"Total image PNG files:              {total_images}",
    f"Total mask PNG files:               {total_masks}",
    f"Non-empty mask slices:              {total_nonempty_masks}",
    "",
    "Excluded nodules:",
    "  nodule_029",
    "  nodule_085",
    "",
    "Label status:",
    "  Native-to-XML identity assignment: NOT performed",
    "  Malignancy label assignment:        NOT performed",
    "",
    "The processed dataset preserves the original PNG pixel data.",
    "No resizing, normalization, augmentation, or relabeling was performed.",
    "",
    "===============================================================================",
]

summary_text = "\n".join(summary_lines)

with open(OUTPUT_SUMMARY, "w") as f:
    f.write(summary_text)

print("\n" + summary_text)

print("\n" + "=" * 80)
print("OUTPUT FILES")
print("=" * 80)

print(f"Processed dataset:")
print(f"  {OUTPUT_ROOT}")

print(f"\nProcessed manifest:")
print(f"  {OUTPUT_MANIFEST}")

print(f"\nSummary:")
print(f"  {OUTPUT_SUMMARY}")

print("=" * 80)
print("STEP 16 COMPLETE")
print("=" * 80)

STEP 16 — BUILDING CLEAN PROCESSED 325-NODULE DATASET

--------------------------------------------------------------------------------
COHORT VALIDATION
--------------------------------------------------------------------------------
Input rows: 325
Unique subset nodules: 325
Unique patients: 247
Unique SeriesInstanceUIDs: 247
✓ Cohort contains exactly 325 valid nodules
✓ Excluded nodules are absent
✓ All SeriesInstanceUID values are present

Output directory created:
  /content/processed_325_nodules

Original ZIP entries: 77740

--------------------------------------------------------------------------------
EXTRACTING 325 NODULE DATASETS
--------------------------------------------------------------------------------
Processed 25 / 325
Processed 50 / 325
Processed 75 / 325
Processed 100 / 325
Processed 125 / 325
Processed 150 / 325
Processed 175 / 325
Processed 200 / 325
Processed 225 / 325
Processed 250 / 325
Processed 275 / 325
Processed 300 / 325
Processed 325 / 325

FINAL PROCES

In [ ]:
# =============================================================================
# STEP 17 — BUILD CONSERVATIVE XML PHYSICAL-NODULE CONSENSUS INVENTORY
# =============================================================================

import os
import glob
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

MASTER_MANIFEST = "/content/final_325_master_manifest.csv"
XML_DIR = "/content/lidc_xml_annotations"

OUT_CSV = "/content/xml_consensus_nodule_inventory_325.csv"
OUT_SUMMARY = "/content/xml_consensus_nodule_summary.txt"

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

# Conservative spatial grouping thresholds.
#
# x/y are XML image pixel coordinates.
# z is the physical image Z position from XML.
#
# These thresholds are intentionally used to form CANDIDATE GROUPS,
# not to claim absolute identity.

XY_THRESHOLD_PX = 20.0
Z_THRESHOLD_MM = 5.0

print("=" * 80)
print("STEP 17 — XML PHYSICAL-NODULE CONSENSUS INVENTORY")
print("=" * 80)

# ---------------------------------------------------------------------
# 1. Validate master cohort
# ---------------------------------------------------------------------

if not os.path.exists(MASTER_MANIFEST):
    raise FileNotFoundError(
        f"Missing master manifest:\n{MASTER_MANIFEST}"
    )

if not os.path.isdir(XML_DIR):
    raise FileNotFoundError(
        f"Missing XML directory:\n{XML_DIR}"
    )

master = pd.read_csv(MASTER_MANIFEST)

required_cols = [
    "subset_nodule_id",
    "patient_id",
    "native_nodule_id",
    "SeriesInstanceUID"
]

missing = [c for c in required_cols if c not in master.columns]

if missing:
    raise KeyError(
        f"Master manifest missing required columns: {missing}\n"
        f"Available columns: {list(master.columns)}"
    )

assert len(master) == 325
assert master["subset_nodule_id"].nunique() == 325

print("\nCOHORT CHECK")
print("-" * 80)
print(f"Eligible nodules:            {len(master)}")
print(f"Unique patients:             {master['patient_id'].nunique()}")
print(f"Unique SeriesInstanceUIDs:   {master['SeriesInstanceUID'].nunique()}")

# ---------------------------------------------------------------------
# 2. XML parser
# ---------------------------------------------------------------------

def parse_xml_series(xml_path):
    """
    Extracts each unblindedReadNodule as a reader annotation.

    Returns:
        SeriesInstanceUID
        list of reader-level annotation dictionaries
    """

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception:
        return None, []

    # Namespace-safe lookup
    if "}" in root.tag:
        ns_uri = root.tag.split("}")[0].strip("{")
        ns = {"lidc": ns_uri}

        def find_first(parent, tag):
            x = parent.find(f"lidc:{tag}", ns)
            if x is None:
                x = parent.find(f".//lidc:{tag}", ns)
            return x

        def find_all(parent, tag):
            return parent.findall(f"lidc:{tag}", ns)

        def find_all_recursive(parent, tag):
            return parent.findall(f".//lidc:{tag}", ns)

    else:
        ns = {}

        def find_first(parent, tag):
            x = parent.find(tag)
            if x is None:
                x = parent.find(f".//{tag}")
            return x

        def find_all(parent, tag):
            return parent.findall(tag)

        def find_all_recursive(parent, tag):
            return parent.findall(f".//{tag}")

    # Series UID
    series_elem = find_first(root, "SeriesInstanceUid")

    if series_elem is None:
        series_elem = find_first(root, "SeriesInstanceUID")

    series_uid = (
        series_elem.text.strip()
        if series_elem is not None and series_elem.text
        else None
    )

    # Reader-level nodules
    nodule_nodes = find_all_recursive(root, "unblindedReadNodule")

    annotations = []

    for xml_index, nod in enumerate(nodule_nodes):

        nodule_id_elem = find_first(nod, "noduleID")

        xml_nodule_id = (
            nodule_id_elem.text.strip()
            if nodule_id_elem is not None and nodule_id_elem.text
            else f"XML_{xml_index}"
        )

        # -------------------------------------------------------------
        # Malignancy
        # -------------------------------------------------------------

        characteristics = find_first(nod, "characteristics")

        malignancy = None

        if characteristics is not None:
            mal_elem = find_first(characteristics, "malignancy")

            if (
                mal_elem is not None
                and mal_elem.text
                and mal_elem.text.strip().isdigit()
            ):
                malignancy = int(mal_elem.text.strip())

        # -------------------------------------------------------------
        # ROI geometry
        # -------------------------------------------------------------

        rois = find_all(nod, "roi")

        roi_records = []

        for roi in rois:

            z_elem = find_first(roi, "imageZposition")
            sop_elem = find_first(roi, "imageSOP_UID")

            z = None
            sop = None

            if z_elem is not None and z_elem.text:
                try:
                    z = float(z_elem.text.strip())
                except Exception:
                    z = None

            if sop_elem is not None and sop_elem.text:
                sop = sop_elem.text.strip()

            xs = []
            ys = []

            edge_maps = find_all(roi, "edgeMap")

            for edge in edge_maps:

                x_elem = find_first(edge, "xCoord")
                y_elem = find_first(edge, "yCoord")

                if (
                    x_elem is not None
                    and y_elem is not None
                    and x_elem.text
                    and y_elem.text
                ):
                    try:
                        xs.append(float(x_elem.text.strip()))
                        ys.append(float(y_elem.text.strip()))
                    except Exception:
                        pass

            if z is not None and xs and ys:

                roi_records.append({
                    "z": z,
                    "sop": sop,
                    "cx": float(np.mean(xs)),
                    "cy": float(np.mean(ys)),
                    "x_min": float(np.min(xs)),
                    "x_max": float(np.max(xs)),
                    "y_min": float(np.min(ys)),
                    "y_max": float(np.max(ys)),
                    "width": float(np.max(xs) - np.min(xs)),
                    "height": float(np.max(ys) - np.min(ys))
                })

        # -------------------------------------------------------------
        # Annotation summary
        # -------------------------------------------------------------

        if not roi_records:
            annotations.append({
                "xml_file": os.path.basename(xml_path),
                "series_instance_uid": series_uid,
                "xml_nodule_id": xml_nodule_id,
                "xml_index": xml_index,
                "malignancy": malignancy,
                "roi_count": 0,
                "mean_x": np.nan,
                "mean_y": np.nan,
                "mean_z": np.nan,
                "min_z": np.nan,
                "max_z": np.nan,
                "median_width": np.nan,
                "median_height": np.nan
            })
            continue

        mean_x = np.mean([r["cx"] for r in roi_records])
        mean_y = np.mean([r["cy"] for r in roi_records])
        mean_z = np.mean([r["z"] for r in roi_records])

        annotations.append({
            "xml_file": os.path.basename(xml_path),
            "series_instance_uid": series_uid,
            "xml_nodule_id": xml_nodule_id,
            "xml_index": xml_index,
            "malignancy": malignancy,
            "roi_count": len(roi_records),
            "mean_x": float(mean_x),
            "mean_y": float(mean_y),
            "mean_z": float(mean_z),
            "min_z": float(min(r["z"] for r in roi_records)),
            "max_z": float(max(r["z"] for r in roi_records)),
            "median_width": float(np.median([r["width"] for r in roi_records])),
            "median_height": float(np.median([r["height"] for r in roi_records]))
        })

    return series_uid, annotations


# ---------------------------------------------------------------------
# 3. Index XML files
# ---------------------------------------------------------------------

print("\n" + "-" * 80)
print("INDEXING XML FILES")
print("-" * 80)

xml_files = glob.glob(
    os.path.join(XML_DIR, "**", "*.xml"),
    recursive=True
)

print(f"XML files discovered: {len(xml_files)}")

series_annotations = {}
parse_errors = 0

for xml_path in xml_files:

    try:
        suid, annotations = parse_xml_series(xml_path)

        if suid:
            series_annotations.setdefault(suid, [])
            series_annotations[suid].extend(annotations)

    except Exception:
        parse_errors += 1

print(f"Unique SeriesInstanceUIDs indexed: {len(series_annotations)}")
print(f"XML parse errors: {parse_errors}")

# ---------------------------------------------------------------------
# 4. Spatial clustering
# ---------------------------------------------------------------------

def spatial_distance(a, b):
    """
    Conservative normalized distance.

    XY is measured in image pixels.
    Z is measured in millimeters.

    This is only used for grouping nearby reader annotations.
    """

    if (
        np.isnan(a["mean_x"])
        or np.isnan(a["mean_y"])
        or np.isnan(a["mean_z"])
        or np.isnan(b["mean_x"])
        or np.isnan(b["mean_y"])
        or np.isnan(b["mean_z"])
    ):
        return np.inf

    dx = a["mean_x"] - b["mean_x"]
    dy = a["mean_y"] - b["mean_y"]
    dz = a["mean_z"] - b["mean_z"]

    xy_dist = np.sqrt(dx**2 + dy**2)

    # Normalize each dimension by its threshold.
    normalized_xy = xy_dist / XY_THRESHOLD_PX
    normalized_z = abs(dz) / Z_THRESHOLD_MM

    return np.sqrt(
        normalized_xy**2 +
        normalized_z**2
    )


def build_spatial_clusters(annotation_list):
    """
    Greedy connected-component-style clustering.

    Two reader annotations may belong to the same candidate physical nodule
    when they are sufficiently close in XY and Z.

    This DOES NOT prove physical identity.
    """

    usable = [
        a for a in annotation_list
        if not (
            np.isnan(a["mean_x"])
            or np.isnan(a["mean_y"])
            or np.isnan(a["mean_z"])
        )
    ]

    used = set()
    clusters = []

    for i, seed in enumerate(usable):

        if i in used:
            continue

        cluster = [seed]
        used.add(i)

        changed = True

        while changed:

            changed = False

            for j, candidate in enumerate(usable):

                if j in used:
                    continue

                # Compare candidate to every current cluster member.
                close_to_cluster = False

                for member in cluster:

                    xy = np.sqrt(
                        (candidate["mean_x"] - member["mean_x"]) ** 2 +
                        (candidate["mean_y"] - member["mean_y"]) ** 2
                    )

                    z_dist = abs(
                        candidate["mean_z"] -
                        member["mean_z"]
                    )

                    if (
                        xy <= XY_THRESHOLD_PX
                        and z_dist <= Z_THRESHOLD_MM
                    ):
                        close_to_cluster = True
                        break

                if close_to_cluster:

                    cluster.append(candidate)
                    used.add(j)
                    changed = True

        clusters.append(cluster)

    # Preserve annotations without usable coordinates as singleton clusters.
    no_geometry = [
        a for a in annotation_list
        if (
            np.isnan(a["mean_x"])
            or np.isnan(a["mean_y"])
            or np.isnan(a["mean_z"])
        )
    ]

    for a in no_geometry:
        clusters.append([a])

    return clusters


# ---------------------------------------------------------------------
# 5. Build consensus candidate nodules for the 325 selected series
# ---------------------------------------------------------------------

print("\n" + "-" * 80)
print("BUILDING SPATIAL CONSENSUS CANDIDATES")
print("-" * 80)

manifest_series = set(
    master["SeriesInstanceUID"].astype(str).str.strip()
)

consensus_rows = []

series_with_xml = 0
series_without_xml = 0

for series_uid in sorted(manifest_series):

    annotations = series_annotations.get(series_uid, [])

    if not annotations:
        series_without_xml += 1
        continue

    series_with_xml += 1

    clusters = build_spatial_clusters(annotations)

    for cluster_idx, cluster in enumerate(clusters):

        rated = [
            int(a["malignancy"])
            for a in cluster
            if a["malignancy"] is not None
        ]

        roi_counts = [
            int(a["roi_count"])
            for a in cluster
        ]

        xs = [
            a["mean_x"]
            for a in cluster
            if not np.isnan(a["mean_x"])
        ]

        ys = [
            a["mean_y"]
            for a in cluster
            if not np.isnan(a["mean_y"])
        ]

        zs = [
            a["mean_z"]
            for a in cluster
            if not np.isnan(a["mean_z"])
        ]

        z_mins = [
            a["min_z"]
            for a in cluster
            if not np.isnan(a["min_z"])
        ]

        z_maxs = [
            a["max_z"]
            for a in cluster
            if not np.isnan(a["max_z"])
        ]

        xml_ids = [
            str(a["xml_nodule_id"])
            for a in cluster
        ]

        xml_files_here = sorted(
            set(str(a["xml_file"]) for a in cluster)
        )

        # -------------------------------------------------------------
        # Consensus malignancy
        # -------------------------------------------------------------

        if rated:

            mean_rating = float(np.mean(rated))
            median_rating = float(np.median(rated))

            # Conservative binary interpretation:
            #
            # <3  => benign/low suspicion
            # >3  => malignant/high suspicion
            # =3  => indeterminate
            #
            # BUT this is only a consensus candidate, NOT a native label.

            if mean_rating < 3:
                consensus_class = "LOW"
            elif mean_rating > 3:
                consensus_class = "HIGH"
            else:
                consensus_class = "INDETERMINATE"

        else:

            mean_rating = np.nan
            median_rating = np.nan
            consensus_class = "UNRATED"

        consensus_rows.append({

            "SeriesInstanceUID": series_uid,

            "xml_consensus_cluster": f"cluster_{cluster_idx:03d}",

            "reader_annotation_count": len(cluster),

            "rated_reader_count": len(rated),

            "xml_nodule_ids": ";".join(xml_ids),

            "xml_files": ";".join(xml_files_here),

            "mean_x": float(np.mean(xs)) if xs else np.nan,

            "mean_y": float(np.mean(ys)) if ys else np.nan,

            "mean_z": float(np.mean(zs)) if zs else np.nan,

            "min_z": float(np.min(z_mins)) if z_mins else np.nan,

            "max_z": float(np.max(z_maxs)) if z_maxs else np.nan,

            "mean_roi_count": (
                float(np.mean(roi_counts))
                if roi_counts else np.nan
            ),

            "malignancy_ratings": (
                ";".join(map(str, rated))
                if rated else ""
            ),

            "mean_malignancy": mean_rating,

            "median_malignancy": median_rating,

            "consensus_class_candidate": consensus_class,

            # IMPORTANT:
            # This remains FALSE until the native folder itself is linked
            # independently to this XML spatial cluster.
            "native_identity_proven": False
        })


consensus_df = pd.DataFrame(consensus_rows)

# ---------------------------------------------------------------------
# 6. Save output
# ---------------------------------------------------------------------

consensus_df.to_csv(
    OUT_CSV,
    index=False
)

# ---------------------------------------------------------------------
# 7. Diagnostics
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 17 RESULTS")
print("=" * 80)

print(f"Selected series evaluated:              {len(manifest_series)}")
print(f"Selected series with XML annotations:   {series_with_xml}")
print(f"Selected series without XML:            {series_without_xml}")

print(f"\nConsensus physical-nodule candidates:  {len(consensus_df)}")

if not consensus_df.empty:

    print(
        "\nReader annotations per consensus candidate:"
    )

    print(
        consensus_df["reader_annotation_count"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print(
        "\nConsensus candidate class distribution:"
    )

    print(
        consensus_df["consensus_class_candidate"]
        .value_counts()
        .to_string()
    )

    print(
        "\nCandidates with at least one malignancy rating:"
    )

    print(
        (
            consensus_df["rated_reader_count"] > 0
        ).sum()
    )

    print(
        "\nCandidates with >=2 rated readers:"
    )

    print(
        (
            consensus_df["rated_reader_count"] >= 2
        ).sum()
    )

    print(
        "\nCandidates with mean malignancy <3:"
    )

    print(
        (
            consensus_df["mean_malignancy"] < 3
        ).sum()
    )

    print(
        "\nCandidates with mean malignancy >3:"
    )

    print(
        (
            consensus_df["mean_malignancy"] > 3
        ).sum()
    )

    print(
        "\nCandidates with mean malignancy =3:"
    )

    print(
        (
            consensus_df["mean_malignancy"] == 3
        ).sum()
    )

# ---------------------------------------------------------------------
# 8. Create summary report
# ---------------------------------------------------------------------

summary_lines = [

    "=" * 80,
    "STEP 17 — XML PHYSICAL-NODULE CONSENSUS INVENTORY",
    "=" * 80,
    "",

    f"Selected SeriesInstanceUIDs evaluated: {len(manifest_series)}",
    f"Series with XML annotations:            {series_with_xml}",
    f"Series without XML annotations:         {series_without_xml}",
    f"XML parse errors:                        {parse_errors}",
    "",

    f"Consensus physical-nodule candidates:   {len(consensus_df)}",
    "",

    "IMPORTANT METHODOLOGICAL RULE:",
    "XML reader annotations were grouped into spatial candidate clusters.",
    "A cluster is NOT automatically assumed to equal a native nodule-X folder.",
    "",

    "MALIGNANCY STATUS:",
    "Malignancy ratings are summarized at the XML consensus-candidate level.",
    "They have NOT been assigned to native nodule folders.",
    "",

    "NATIVE IDENTITY STATUS:",
    "native_identity_proven = FALSE for all consensus candidates.",
    "",

    "INTERPRETATION:",
    "This inventory separates two questions:",
    "1. Which XML reader annotations likely describe the same physical nodule?",
    "2. Which native nodule-X folder corresponds to that physical nodule?",
    "",
    "Question 1 can be investigated from XML spatial geometry.",
    "Question 2 still requires independent linkage between native image data",
    "and the XML spatial candidate.",
    "",

    "Next stage:",
    "Use native mask/image geometry to compare each native folder against",
    "these consensus candidates, while retaining an explicit ambiguous class",
    "when the evidence does not establish a unique correspondence.",
    "",
    "=" * 80
]

summary_text = "\n".join(summary_lines)

with open(OUT_SUMMARY, "w") as f:
    f.write(summary_text)

print("\n" + summary_text)

print("\n" + "=" * 80)
print("OUTPUT")
print("=" * 80)

print(f"Consensus inventory:")
print(f"  {OUT_CSV}")

print(f"\nSummary:")
print(f"  {OUT_SUMMARY}")

print("=" * 80)
print("STEP 17 COMPLETE")
print("=" * 80)

STEP 17 — XML PHYSICAL-NODULE CONSENSUS INVENTORY

COHORT CHECK
--------------------------------------------------------------------------------
Eligible nodules:            325
Unique patients:             247
Unique SeriesInstanceUIDs:   247

--------------------------------------------------------------------------------
INDEXING XML FILES
--------------------------------------------------------------------------------
XML files discovered: 1319
Unique SeriesInstanceUIDs indexed: 1294
XML parse errors: 0

--------------------------------------------------------------------------------
BUILDING SPATIAL CONSENSUS CANDIDATES
--------------------------------------------------------------------------------

STEP 17 RESULTS
Selected series evaluated:              247
Selected series with XML annotations:   247
Selected series without XML:            0

Consensus physical-nodule candidates:  2562

Reader annotations per consensus candidate:
reader_annotation_count
1     362
2     528
3    

In [ ]:
# =============================================================================
# STEP 18 — NATIVE NODULE → XML CANDIDATE MATCHING
# =============================================================================

import os
import io
import zipfile
import pandas as pd
import numpy as np
from PIL import Image

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

MASTER_MANIFEST = "/content/final_325_master_manifest.csv"
CONSENSUS_CSV = "/content/xml_consensus_nodule_inventory_325.csv"
ZIP_PATH = "/content/kagl_lidc_idri.zip"

OUT_CSV = "/content/native_to_xml_candidate_matches_325.csv"
OUT_SUMMARY = "/content/native_to_xml_candidate_summary_325.txt"

# ---------------------------------------------------------------------
# Load inputs
# ---------------------------------------------------------------------

print("=" * 80)
print("STEP 18 — NATIVE NODULE → XML CANDIDATE MATCHING")
print("=" * 80)

if not os.path.exists(MASTER_MANIFEST):
    raise FileNotFoundError(MASTER_MANIFEST)

if not os.path.exists(CONSENSUS_CSV):
    raise FileNotFoundError(CONSENSUS_CSV)

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(ZIP_PATH)

master = pd.read_csv(MASTER_MANIFEST)
consensus = pd.read_csv(CONSENSUS_CSV)

assert len(master) == 325
assert master["subset_nodule_id"].nunique() == 325

required_master = [
    "subset_nodule_id",
    "patient_id",
    "native_nodule_id",
    "SeriesInstanceUID"
]

for c in required_master:
    if c not in master.columns:
        raise KeyError(f"Missing column in master manifest: {c}")

required_consensus = [
    "SeriesInstanceUID",
    "xml_consensus_cluster",
    "reader_annotation_count",
    "mean_x",
    "mean_y",
    "mean_z",
    "min_z",
    "max_z",
    "mean_malignancy"
]

for c in required_consensus:
    if c not in consensus.columns:
        raise KeyError(f"Missing column in consensus CSV: {c}")

print("\nINPUT CHECK")
print("-" * 80)
print(f"Native nodules:                 {len(master)}")
print(f"XML consensus candidates:       {len(consensus)}")
print(f"Series represented:             {master['SeriesInstanceUID'].nunique()}")

# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def get_mask_geometry(zf, patient_id, native_id):
    """
    Read native PNG masks and extract robust geometric summaries.

    We do NOT assume the PNG slice index equals physical Z.

    We use:
      - number of active mask slices
      - median foreground area
      - max foreground area
      - median bbox width
      - median bbox height
      - median centroid
      - total foreground pixels
    """

    prefix = f"LIDC-IDRI-slices/{patient_id}/{native_id}/"

    all_names = zf.namelist()

    results = []

    for m in range(4):

        mask_prefix = f"{prefix}mask-{m}/"

        mask_files = [
            x for x in all_names
            if x.startswith(mask_prefix) and x.lower().endswith(".png")
        ]

        areas = []
        widths = []
        heights = []
        cxs = []
        cys = []

        for mf in mask_files:

            try:
                with zf.open(mf) as f:
                    arr = np.array(
                        Image.open(io.BytesIO(f.read())).convert("L")
                    ) > 0

                pts = np.argwhere(arr)

                if len(pts) == 0:
                    continue

                y = pts[:, 0]
                x = pts[:, 1]

                areas.append(float(len(pts)))
                widths.append(float(x.max() - x.min() + 1))
                heights.append(float(y.max() - y.min() + 1))
                cxs.append(float(x.mean()))
                cys.append(float(y.mean()))

            except Exception:
                continue

        if areas:

            results.append({
                "mask": m,
                "active_slices": len(areas),
                "median_area": float(np.median(areas)),
                "max_area": float(np.max(areas)),
                "median_width": float(np.median(widths)),
                "median_height": float(np.median(heights)),
                "median_cx": float(np.median(cxs)),
                "median_cy": float(np.median(cys)),
                "total_area": float(np.sum(areas))
            })

    if not results:
        return None

    # Combine available masks robustly using medians.
    return {
        "active_masks": len(results),
        "active_slice_median": float(
            np.median([r["active_slices"] for r in results])
        ),
        "area_median": float(
            np.median([r["median_area"] for r in results])
        ),
        "max_area_median": float(
            np.median([r["max_area"] for r in results])
        ),
        "width_median": float(
            np.median([r["median_width"] for r in results])
        ),
        "height_median": float(
            np.median([r["median_height"] for r in results])
        ),
        "cx_median": float(
            np.median([r["median_cx"] for r in results])
        ),
        "cy_median": float(
            np.median([r["median_cy"] for r in results])
        ),
        "total_area_median": float(
            np.median([r["total_area"] for r in results])
        )
    }


# ---------------------------------------------------------------------
# Candidate scoring
# ---------------------------------------------------------------------

def candidate_score(native_geom, xml_row):
    """
    Produces a NON-PROBABILISTIC similarity score.

    This is only a way to order candidates for inspection.

    It is NOT treated as proof of identity.
    """

    # Native image coordinates are local crop coordinates while XML
    # coordinates are full CT-grid coordinates. Therefore absolute
    # native XY cannot be directly compared.

    # We therefore emphasize shape/extent characteristics.

    native_area = native_geom["total_area_median"]
    native_w = native_geom["width_median"]
    native_h = native_geom["height_median"]
    native_active = native_geom["active_slice_median"]

    xml_roi_count = xml_row["reader_annotation_count"]

    # XML reader-count is useful only as weak evidence.
    # It should never dominate the result.

    # Convert cluster scale to a comparable broad descriptor.
    xml_z_span = (
        abs(xml_row["max_z"] - xml_row["min_z"])
        if pd.notna(xml_row["max_z"])
        and pd.notna(xml_row["min_z"])
        else np.nan
    )

    # Score components:
    #   1. annotation count plausibility
    #   2. native mask complexity
    #   3. XML Z extent
    #
    # These are deliberately weak heuristics.

    score = 0.0

    if np.isfinite(xml_roi_count):
        count_difference = abs(
            native_active - max(1.0, min(xml_roi_count, 20.0))
        )
        score += max(
            0.0,
            1.0 - count_difference / max(native_active, 20.0)
        ) * 0.25

    if np.isfinite(xml_z_span):
        # Penalize extremely large spans.
        z_component = np.exp(-xml_z_span / 20.0)
        score += z_component * 0.15

    # Remaining 60% intentionally reserved as uncertainty.
    score += 0.60

    return float(score)


# ---------------------------------------------------------------------
# Process 325 native nodules
# ---------------------------------------------------------------------

print("\n" + "-" * 80)
print("MATCHING NATIVE NODULES TO XML CANDIDATES")
print("-" * 80)

rows = []

with zipfile.ZipFile(ZIP_PATH, "r") as zf:

    for idx, (_, native_row) in enumerate(master.iterrows(), start=1):

        subset_id = str(native_row["subset_nodule_id"])
        patient_id = str(native_row["patient_id"])
        native_id = str(native_row["native_nodule_id"])
        series_uid = str(native_row["SeriesInstanceUID"]).strip()

        native_geom = get_mask_geometry(
            zf,
            patient_id,
            native_id
        )

        if native_geom is None:

            rows.append({
                "subset_nodule_id": subset_id,
                "patient_id": patient_id,
                "native_nodule_id": native_id,
                "SeriesInstanceUID": series_uid,
                "candidate_count": 0,
                "match_status": "NO_NATIVE_MASK_GEOMETRY"
            })

            continue

        # Candidate XML clusters must belong to EXACT selected series.
        candidates = consensus[
            consensus["SeriesInstanceUID"].astype(str).str.strip()
            == series_uid
        ].copy()

        if candidates.empty:

            rows.append({
                "subset_nodule_id": subset_id,
                "patient_id": patient_id,
                "native_nodule_id": native_id,
                "SeriesInstanceUID": series_uid,
                "candidate_count": 0,
                "match_status": "NO_XML_CANDIDATES"
            })

            continue

        candidate_records = []

        for _, c in candidates.iterrows():

            score = candidate_score(
                native_geom,
                c
            )

            candidate_records.append({

                "cluster": c["xml_consensus_cluster"],

                "xml_nodule_ids": c["xml_nodule_ids"]
                if "xml_nodule_ids" in c
                else "",

                "reader_count": c["reader_annotation_count"],

                "mean_x": c["mean_x"],
                "mean_y": c["mean_y"],
                "mean_z": c["mean_z"],

                "min_z": c["min_z"],
                "max_z": c["max_z"],

                "mean_malignancy": c["mean_malignancy"],

                "candidate_score": score
            })

        candidate_records = sorted(
            candidate_records,
            key=lambda x: x["candidate_score"],
            reverse=True
        )

        # Keep top 5 candidates rather than forcing one identity.
        top_candidates = candidate_records[:5]

        best_score = top_candidates[0]["candidate_score"]

        second_score = (
            top_candidates[1]["candidate_score"]
            if len(top_candidates) > 1
            else np.nan
        )

        margin = (
            best_score - second_score
            if np.isfinite(second_score)
            else np.nan
        )

        # IMPORTANT:
        # No score is interpreted as proof.
        #
        # We classify:
        #   HIGH-CONFIDENCE-CANDIDATE
        #   MULTIPLE-PLAUSIBLE
        #   WEAK-CANDIDATE-SET
        #
        # These are candidate statuses only.

        if (
            len(top_candidates) >= 2
            and np.isfinite(margin)
            and margin >= 0.10
        ):
            status = "HIGH_CONFIDENCE_CANDIDATE"

        elif len(top_candidates) >= 2:
            status = "MULTIPLE_PLAUSIBLE_CANDIDATES"

        else:
            status = "SINGLE_CANDIDATE"

        record = {
            "subset_nodule_id": subset_id,
            "patient_id": patient_id,
            "native_nodule_id": native_id,
            "SeriesInstanceUID": series_uid,

            "candidate_count": len(candidate_records),

            "native_active_masks": native_geom["active_masks"],
            "native_active_slice_median":
                native_geom["active_slice_median"],
            "native_area_median":
                native_geom["area_median"],
            "native_width_median":
                native_geom["width_median"],
            "native_height_median":
                native_geom["height_median"],

            "best_score": best_score,
            "second_best_score": second_score,
            "score_margin": margin,

            "best_candidate_cluster":
                top_candidates[0]["cluster"],

            "best_candidate_xml_ids":
                top_candidates[0]["xml_nodule_ids"],

            "best_candidate_mean_z":
                top_candidates[0]["mean_z"],

            "best_candidate_mean_malignancy":
                top_candidates[0]["mean_malignancy"],

            "match_status": status
        }

        # Save top candidate details.
        for rank, cand in enumerate(top_candidates, start=1):

            record[f"candidate_{rank}_cluster"] = cand["cluster"]
            record[f"candidate_{rank}_xml_ids"] = cand["xml_nodule_ids"]
            record[f"candidate_{rank}_score"] = cand["candidate_score"]
            record[f"candidate_{rank}_mean_z"] = cand["mean_z"]
            record[f"candidate_{rank}_mean_malignancy"] = cand[
                "mean_malignancy"
            ]

        rows.append(record)

        if idx % 25 == 0:
            print(f"Processed {idx} / {len(master)}")


# ---------------------------------------------------------------------
# Save results
# ---------------------------------------------------------------------

results = pd.DataFrame(rows)

assert len(results) == 325
assert results["subset_nodule_id"].nunique() == 325

results.to_csv(
    OUT_CSV,
    index=False
)

# ---------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------

status_counts = results["match_status"].value_counts()

summary = f"""
===============================================================================
STEP 18 — NATIVE → XML CANDIDATE MATCHING SUMMARY
===============================================================================

Native nodules evaluated:                  {len(results)}

Series matched exactly:
  {results['SeriesInstanceUID'].nunique()}

Status distribution:
{status_counts.to_string()}

Important methodological rule:

The candidate score does NOT establish physical identity.

The output identifies XML consensus candidates belonging to the exact
SeriesInstanceUID and ranks them using weak geometric/structural evidence.

No malignancy rating has been assigned to any native nodule.

The following remain separate:
  1. XML reader annotations.
  2. XML spatial consensus candidates.
  3. Native nodule-X folders.

A native nodule receives a ground-truth malignancy label only after
its native-folder identity has been independently established.

Output:
{OUT_CSV}
"""

with open(OUT_SUMMARY, "w") as f:
    f.write(summary.strip())

print("\n" + summary)
print("\nSTEP 18 COMPLETE")

STEP 18 — NATIVE NODULE → XML CANDIDATE MATCHING

INPUT CHECK
--------------------------------------------------------------------------------
Native nodules:                 325
XML consensus candidates:       2562
Series represented:             247

--------------------------------------------------------------------------------
MATCHING NATIVE NODULES TO XML CANDIDATES
--------------------------------------------------------------------------------
Processed 25 / 325
Processed 50 / 325
Processed 75 / 325
Processed 100 / 325
Processed 125 / 325
Processed 150 / 325
Processed 175 / 325
Processed 200 / 325
Processed 225 / 325
Processed 250 / 325
Processed 275 / 325
Processed 300 / 325
Processed 325 / 325


STEP 18 — NATIVE → XML CANDIDATE MATCHING SUMMARY

Native nodules evaluated:                  325

Series matched exactly:
  247

Status distribution:
match_status
MULTIPLE_PLAUSIBLE_CANDIDATES    315
SINGLE_CANDIDATE                   9
HIGH_CONFIDENCE_CANDIDATE          1

Importan

In [ ]:
# =============================================================================
# STEP 19 — AUDIT THE MOST PROMISING NATIVE → XML MATCHES
# =============================================================================

import os
import pandas as pd
import numpy as np

MATCH_CSV = "/content/native_to_xml_candidate_matches_325.csv"
MASTER_CSV = "/content/final_325_master_manifest.csv"
CONSENSUS_CSV = "/content/xml_consensus_nodule_inventory_325.csv"

OUT_CSV = "/content/step19_promising_match_audit.csv"
OUT_TXT = "/content/step19_promising_match_audit.txt"

print("=" * 80)
print("STEP 19 — AUDITING THE MOST PROMISING NATIVE → XML MATCHES")
print("=" * 80)

# ---------------------------------------------------------------------
# Load files
# ---------------------------------------------------------------------

for path in [MATCH_CSV, MASTER_CSV, CONSENSUS_CSV]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

matches = pd.read_csv(MATCH_CSV)
master = pd.read_csv(MASTER_CSV)
consensus = pd.read_csv(CONSENSUS_CSV)

assert len(matches) == 325
assert matches["subset_nodule_id"].nunique() == 325

# ---------------------------------------------------------------------
# Identify promising cases
# ---------------------------------------------------------------------

promising = matches[
    matches["match_status"].isin([
        "SINGLE_CANDIDATE",
        "HIGH_CONFIDENCE_CANDIDATE"
    ])
].copy()

promising = promising.sort_values(
    by=["match_status", "score_margin"],
    ascending=[True, False]
)

print("\nPROMISING CASE COUNT")
print("-" * 80)
print(f"Total promising cases: {len(promising)}")

if promising.empty:
    print("No promising cases found.")
else:

    print("\nPromising native nodules:")
    print(
        promising[
            [
                "subset_nodule_id",
                "patient_id",
                "native_nodule_id",
                "SeriesInstanceUID",
                "match_status",
                "candidate_count",
                "best_candidate_cluster",
                "best_candidate_xml_ids",
                "best_score",
                "second_best_score",
                "score_margin",
                "best_candidate_mean_z",
                "best_candidate_mean_malignancy"
            ]
        ].to_string(index=False)
    )

# ---------------------------------------------------------------------
# Audit whether the promising candidate contains rated XML annotations
# ---------------------------------------------------------------------

audit_rows = []

for _, row in promising.iterrows():

    cluster_id = row["best_candidate_cluster"]
    series_uid = str(row["SeriesInstanceUID"]).strip()

    cands = consensus[
        (consensus["SeriesInstanceUID"].astype(str).str.strip() == series_uid)
        &
        (consensus["xml_consensus_cluster"].astype(str) == str(cluster_id))
    ]

    if cands.empty:
        audit_rows.append({
            "subset_nodule_id": row["subset_nodule_id"],
            "series_instance_uid": series_uid,
            "candidate_cluster": cluster_id,
            "audit_status": "CANDIDATE_NOT_FOUND"
        })
        continue

    c = cands.iloc[0]

    audit_rows.append({

        "subset_nodule_id":
            row["subset_nodule_id"],

        "patient_id":
            row["patient_id"],

        "native_nodule_id":
            row["native_nodule_id"],

        "series_instance_uid":
            series_uid,

        "match_status":
            row["match_status"],

        "candidate_count":
            row["candidate_count"],

        "best_score":
            row["best_score"],

        "second_best_score":
            row["second_best_score"],

        "score_margin":
            row["score_margin"],

        "candidate_cluster":
            cluster_id,

        "xml_nodule_ids":
            c["xml_nodule_ids"],

        "reader_annotation_count":
            c["reader_annotation_count"],

        "rated_reader_count":
            c["rated_reader_count"],

        "malignancy_ratings":
            c["malignancy_ratings"],

        "mean_malignancy":
            c["mean_malignancy"],

        "median_malignancy":
            c["median_malignancy"],

        "consensus_class_candidate":
            c["consensus_class_candidate"],

        "mean_x":
            c["mean_x"],

        "mean_y":
            c["mean_y"],

        "mean_z":
            c["mean_z"],

        "min_z":
            c["min_z"],

        "max_z":
            c["max_z"],

        "mean_roi_count":
            c["mean_roi_count"],

        "audit_status":
            "REQUIRES_GEOMETRIC_REVIEW"
    })

audit_df = pd.DataFrame(audit_rows)

audit_df.to_csv(OUT_CSV, index=False)

# ---------------------------------------------------------------------
# Summary statistics
# ---------------------------------------------------------------------

rated_promising = audit_df[
    audit_df["rated_reader_count"].fillna(0) > 0
]

multi_reader_promising = audit_df[
    audit_df["rated_reader_count"].fillna(0) >= 2
]

summary = f"""
===============================================================================
STEP 19 — PROMISING MATCH AUDIT
===============================================================================

Total native nodules:
    325

Promising candidate cases:
    {len(audit_df)}

Cases with at least one rated XML reader:
    {len(rated_promising)}

Cases with at least two rated XML readers:
    {len(multi_reader_promising)}

IMPORTANT:

These cases are NOT automatically considered proven native→XML identities.

The current Step 18 score is only a ranking heuristic.

The purpose of Step 19 is to identify cases where a more detailed
pixel/shape/ROI analysis may be worthwhile.

No malignancy labels have been assigned to native nodules.

Output:
    {OUT_CSV}
"""

with open(OUT_TXT, "w") as f:
    f.write(summary.strip())

print("\n" + summary)

print("\n" + "=" * 80)
print("STEP 19 COMPLETE")
print("=" * 80)

STEP 19 — AUDITING THE MOST PROMISING NATIVE → XML MATCHES

PROMISING CASE COUNT
--------------------------------------------------------------------------------
Total promising cases: 10

Promising native nodules:
subset_nodule_id     patient_id native_nodule_id                                                SeriesInstanceUID              match_status  candidate_count best_candidate_cluster                                best_candidate_xml_ids  best_score  second_best_score  score_margin  best_candidate_mean_z  best_candidate_mean_malignancy
      nodule_279 LIDC-IDRI-0064         nodule-0 1.3.6.1.4.1.14519.5.2.1.6279.6001.487268565754493433372433148666 HIGH_CONFIDENCE_CANDIDATE                3            cluster_002                 Nodule 003;IL057_130631;MI014_16630;2    0.925000           0.823698      0.101302            -284.000000                             NaN
      nodule_012 LIDC-IDRI-0983         nodule-0 1.3.6.1.4.1.14519.5.2.1.6279.6001.309564220265302089123180126785    

In [ ]:
# =============================================================================
# STEP 20 — DETAILED FORENSIC AUDIT OF PROMISING MATCHES
# ROBUST VERSION — NO PROBLEMATIC MERGE
# =============================================================================

import os
import io
import glob
import zipfile
import xml.etree.ElementTree as ET

import pandas as pd
import numpy as np

from PIL import Image
from matplotlib.path import Path


# =============================================================================
# FILE PATHS
# =============================================================================

PROMISING_CSV = "/content/step19_promising_match_audit.csv"
ZIP_PATH = "/content/kagl_lidc_idri.zip"
XML_DIR = "/content/lidc_xml_annotations"

OUT_CSV = "/content/step20_detailed_forensic_audit.csv"
OUT_BEST_CSV = "/content/step20_best_candidates.csv"
OUT_TXT = "/content/step20_detailed_forensic_audit.txt"


print("=" * 80)
print("STEP 20 — DETAILED FORENSIC AUDIT OF PROMISING MATCHES")
print("=" * 80)


# =============================================================================
# FILE CHECKS
# =============================================================================

for path in [
    PROMISING_CSV,
    ZIP_PATH,
    XML_DIR
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required input does not exist: {path}"
        )


# =============================================================================
# LOAD STEP 19 RESULTS
# =============================================================================

promising = pd.read_csv(PROMISING_CSV)

print("\nINPUT")
print("-" * 80)
print(f"Step 19 rows: {len(promising)}")
print(f"Columns: {list(promising.columns)}")


# =============================================================================
# ROBUST COLUMN DETECTION
# =============================================================================

def find_column(df, candidates):

    normalized = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        key = candidate.strip().lower()

        if key in normalized:
            return normalized[key]

    return None


subset_col = find_column(
    promising,
    [
        "subset_nodule_id",
        "subset_nodule",
        "nodule_id"
    ]
)

patient_col = find_column(
    promising,
    [
        "patient_id",
        "patientid",
        "patient"
    ]
)

native_col = find_column(
    promising,
    [
        "native_nodule_id",
        "native_nodule",
        "native_id"
    ]
)

series_col = find_column(
    promising,
    [
        "SeriesInstanceUID",
        "series_instance_uid",
        "series_uid",
        "seriesinstanceuid"
    ]
)


print("\nDETECTED COLUMNS")
print("-" * 80)
print(f"subset_nodule_id : {subset_col}")
print(f"patient_id       : {patient_col}")
print(f"native_nodule_id : {native_col}")
print(f"SeriesInstanceUID: {series_col}")


# =============================================================================
# REQUIRED COLUMN CHECK
# =============================================================================

required = {
    "subset_nodule_id": subset_col,
    "patient_id": patient_col,
    "native_nodule_id": native_col,
    "SeriesInstanceUID": series_col
}

missing = [
    name
    for name, col in required.items()
    if col is None
]

if missing:

    raise KeyError(
        "Missing required columns: "
        + ", ".join(missing)
        + "\n\nAvailable columns:\n"
        + "\n".join(
            str(c)
            for c in promising.columns
        )
    )


# =============================================================================
# STANDARDIZE STEP 19 DATA
# =============================================================================

promising_std = pd.DataFrame({

    "subset_nodule_id":
        promising[subset_col].astype(str).str.strip(),

    "patient_id":
        promising[patient_col].astype(str).str.strip(),

    "native_nodule_id":
        promising[native_col].astype(str).str.strip(),

    "SeriesInstanceUID":
        promising[series_col].astype(str).str.strip()

})


# Keep only valid rows.

promising_std = promising_std[
    promising_std["SeriesInstanceUID"].notna()
].copy()


# =============================================================================
# CHECK EXPECTED CASE COUNT
# =============================================================================

print("\nPROMISING CASE CHECK")
print("-" * 80)

print(
    f"Valid promising cases: "
    f"{len(promising_std)}"
)

print(
    f"Unique promising subset IDs: "
    f"{promising_std['subset_nodule_id'].nunique()}"
)

if len(promising_std) != 10:

    print(
        "WARNING: Step 19 did not contain exactly 10 promising cases."
    )


print(
    "\nCases to audit:"
)

print(
    promising_std.to_string(
        index=False
    )
)


# =============================================================================
# XML SERIES SEARCH
# =============================================================================

def find_xml_for_series(series_uid):

    target = str(
        series_uid
    ).strip()

    matches = []

    xml_files = glob.glob(
        os.path.join(
            XML_DIR,
            "**",
            "*.xml"
        ),
        recursive=True
    )

    for xml_path in xml_files:

        try:

            tree = ET.parse(xml_path)
            root = tree.getroot()

            found_uid = None

            for elem in root.iter():

                tag = elem.tag.split("}")[-1]

                if tag.lower() == "seriesinstanceuid":

                    if elem.text:

                        found_uid = (
                            elem.text.strip()
                        )

                        break

            if found_uid == target:

                matches.append(
                    xml_path
                )

        except Exception:

            continue

    return matches


# =============================================================================
# XML NODULE PARSER
# =============================================================================

def parse_xml_nodules(xml_path):

    tree = ET.parse(
        xml_path
    )

    root = tree.getroot()

    result = []

    for xml_index, node in enumerate(
        root.iter()
    ):

        tag = node.tag.split("}")[-1]

        if tag != "unblindedReadNodule":

            continue

        # -------------------------------------------------------------
        # Nodule ID
        # -------------------------------------------------------------

        xml_id = f"XML_{xml_index}"

        for child in node:

            child_tag = child.tag.split("}")[-1]

            if child_tag == "noduleID":

                if child.text:

                    xml_id = (
                        child.text.strip()
                    )

                break

        # -------------------------------------------------------------
        # Malignancy
        # -------------------------------------------------------------

        malignancy = None

        for elem in node.iter():

            elem_tag = elem.tag.split("}")[-1]

            if elem_tag == "malignancy":

                if elem.text:

                    try:

                        malignancy = int(
                            elem.text.strip()
                        )

                    except Exception:

                        malignancy = None

                break

        # -------------------------------------------------------------
        # ROIs
        # -------------------------------------------------------------

        rois = []

        for roi in node.iter():

            roi_tag = roi.tag.split("}")[-1]

            if roi_tag != "roi":

                continue

            z = None
            sop = ""

            for child in roi:

                child_tag = child.tag.split("}")[-1]

                if child_tag == "imageZposition":

                    if child.text:

                        try:

                            z = float(
                                child.text.strip()
                            )

                        except Exception:

                            z = None

                elif child_tag == "imageSOP_UID":

                    if child.text:

                        sop = (
                            child.text.strip()
                        )

            xs = []
            ys = []

            for edge in roi.iter():

                edge_tag = edge.tag.split("}")[-1]

                if edge_tag != "edgeMap":

                    continue

                x_val = None
                y_val = None

                for child in edge:

                    child_tag = child.tag.split("}")[-1]

                    if (
                        child_tag == "xCoord"
                        and child.text
                    ):

                        try:

                            x_val = float(
                                child.text.strip()
                            )

                        except Exception:

                            pass

                    elif (
                        child_tag == "yCoord"
                        and child.text
                    ):

                        try:

                            y_val = float(
                                child.text.strip()
                            )

                        except Exception:

                            pass

                if (
                    x_val is not None
                    and y_val is not None
                ):

                    xs.append(
                        x_val
                    )

                    ys.append(
                        y_val
                    )

            if (
                z is not None
                and len(xs) >= 3
            ):

                rois.append({

                    "z": z,

                    "sop": sop,

                    "xs": xs,

                    "ys": ys,

                    "cx": float(
                        np.mean(xs)
                    ),

                    "cy": float(
                        np.mean(ys)
                    )

                })

        rois.sort(
            key=lambda r: r["z"]
        )

        result.append({

            "xml_nodule_id":
                xml_id,

            "xml_index":
                xml_index,

            "malignancy":
                malignancy,

            "rois":
                rois

        })

    return result


# =============================================================================
# NATIVE MASK READER
# =============================================================================

def read_native_masks(
    zf,
    patient_id,
    native_id
):

    base = (
        f"LIDC-IDRI-slices/"
        f"{patient_id}/"
        f"{native_id}/"
    )

    names = zf.namelist()

    output = {}

    for mask_number in range(4):

        prefix = (
            f"{base}mask-{mask_number}/"
        )

        files = sorted([
            name
            for name in names
            if (
                name.startswith(prefix)
                and name.lower().endswith(".png")
            )
        ])

        slices = []

        for filename in files:

            try:

                with zf.open(
                    filename
                ) as f:

                    arr = np.array(
                        Image.open(
                            io.BytesIO(
                                f.read()
                            )
                        ).convert("L")
                    ) > 0

                if arr.sum() == 0:
                    continue

                ys, xs = np.where(arr)

                slices.append({

                    "filename":
                        filename,

                    "array":
                        arr,

                    "cx":
                        float(xs.mean()),

                    "cy":
                        float(ys.mean())

                })

            except Exception:

                pass

        output[mask_number] = slices

    return output


# =============================================================================
# RASTERIZE XML POLYGON
# =============================================================================

def rasterize_polygon(
    xs,
    ys,
    shape,
    dx,
    dy
):

    height, width = shape

    translated_x = (
        np.asarray(xs)
        + dx
    )

    translated_y = (
        np.asarray(ys)
        + dy
    )

    polygon = np.column_stack([
        translated_x,
        translated_y
    ])

    path = Path(
        polygon
    )

    yy, xx = np.meshgrid(
        np.arange(height),
        np.arange(width),
        indexing="ij"
    )

    points = np.column_stack([
        xx.ravel(),
        yy.ravel()
    ])

    raster = path.contains_points(
        points
    ).reshape(
        height,
        width
    )

    return raster


# =============================================================================
# COMPARE NATIVE MASK TO XML NODULE
# =============================================================================

def compare_mask_to_xml(
    native_slices,
    xml_rois
):

    if (
        len(native_slices) == 0
        or len(xml_rois) == 0
    ):

        return None

    best_result = None

    n_native = len(native_slices)
    n_xml = len(xml_rois)

    max_offset = max(
        0,
        n_native - n_xml
    )

    for offset in range(
        max_offset + 1
    ):

        usable = min(
            n_xml,
            n_native - offset
        )

        if usable <= 0:
            continue

        native_part = (
            native_slices[
                offset:
                offset + usable
            ]
        )

        xml_part = (
            xml_rois[
                :usable
            ]
        )

        native_cx = np.mean([
            s["cx"]
            for s in native_part
        ])

        native_cy = np.mean([
            s["cy"]
            for s in native_part
        ])

        xml_cx = np.mean([
            r["cx"]
            for r in xml_part
        ])

        xml_cy = np.mean([
            r["cy"]
            for r in xml_part
        ])

        base_dx = (
            native_cx
            - xml_cx
        )

        base_dy = (
            native_cy
            - xml_cy
        )

        dx_values = np.arange(
            base_dx - 4,
            base_dx + 4.1,
            1.0
        )

        dy_values = np.arange(
            base_dy - 4,
            base_dy + 4.1,
            1.0
        )

        for dx in dx_values:

            for dy in dy_values:

                dice_values = []
                iou_values = []

                for i in range(
                    usable
                ):

                    native = native_part[i]
                    xml = xml_part[i]

                    xml_mask = (
                        rasterize_polygon(
                            xml["xs"],
                            xml["ys"],
                            native["array"].shape,
                            dx,
                            dy
                        )
                    )

                    native_mask = (
                        native["array"]
                    )

                    intersection = np.logical_and(
                        native_mask,
                        xml_mask
                    ).sum()

                    union = np.logical_or(
                        native_mask,
                        xml_mask
                    ).sum()

                    denominator = (
                        native_mask.sum()
                        + xml_mask.sum()
                    )

                    dice = (
                        2.0 * intersection
                        / denominator
                        if denominator > 0
                        else 0.0
                    )

                    iou = (
                        intersection
                        / union
                        if union > 0
                        else 0.0
                    )

                    dice_values.append(
                        dice
                    )

                    iou_values.append(
                        iou
                    )

                mean_dice = float(
                    np.mean(
                        dice_values
                    )
                )

                mean_iou = float(
                    np.mean(
                        iou_values
                    )
                )

                candidate = {

                    "mean_dice":
                        mean_dice,

                    "mean_iou":
                        mean_iou,

                    "dx":
                        float(dx),

                    "dy":
                        float(dy),

                    "z_offset":
                        int(offset),

                    "matched_slices":
                        int(usable)

                }

                if (
                    best_result is None
                    or candidate["mean_dice"]
                    > best_result["mean_dice"]
                ):

                    best_result = candidate

    return best_result


# =============================================================================
# RUN AUDIT
# =============================================================================

audit_rows = []

print("\n" + "=" * 80)
print("RUNNING FORENSIC COMPARISONS")
print("=" * 80)

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zf:

    for case_number, (_, row) in enumerate(
        promising_std.iterrows(),
        start=1
    ):

        subset_id = row[
            "subset_nodule_id"
        ]

        patient_id = row[
            "patient_id"
        ]

        native_id = row[
            "native_nodule_id"
        ]

        series_uid = row[
            "SeriesInstanceUID"
        ]

        print(
            f"\n[{case_number}/"
            f"{len(promising_std)}] "
            f"{subset_id}"
        )

        # -------------------------------------------------------------
        # Find XML
        # -------------------------------------------------------------

        xml_files = find_xml_for_series(
            series_uid
        )

        if not xml_files:

            print(
                "  XML: NOT FOUND"
            )

            audit_rows.append({

                "subset_nodule_id":
                    subset_id,

                "patient_id":
                    patient_id,

                "native_nodule_id":
                    native_id,

                "SeriesInstanceUID":
                    series_uid,

                "audit_status":
                    "NO_XML"

            })

            continue

        xml_path = xml_files[0]

        print(
            f"  XML: "
            f"{os.path.basename(xml_path)}"
        )

        # -------------------------------------------------------------
        # Parse XML
        # -------------------------------------------------------------

        xml_nodules = parse_xml_nodules(
            xml_path
        )

        print(
            f"  XML nodules: "
            f"{len(xml_nodules)}"
        )

        # -------------------------------------------------------------
        # Read native masks
        # -------------------------------------------------------------

        native_masks = read_native_masks(
            zf,
            patient_id,
            native_id
        )

        # -------------------------------------------------------------
        # Compare every mask against every XML nodule
        # -------------------------------------------------------------

        for mask_number, native_slices in native_masks.items():

            if not native_slices:
                continue

            for xnod in xml_nodules:

                comparison = compare_mask_to_xml(
                    native_slices,
                    xnod["rois"]
                )

                if comparison is None:
                    continue

                audit_rows.append({

                    "subset_nodule_id":
                        subset_id,

                    "patient_id":
                        patient_id,

                    "native_nodule_id":
                        native_id,

                    "SeriesInstanceUID":
                        series_uid,

                    "xml_file":
                        os.path.basename(
                            xml_path
                        ),

                    "native_mask":
                        f"mask-{mask_number}",

                    "xml_nodule_id":
                        xnod[
                            "xml_nodule_id"
                        ],

                    "xml_index":
                        xnod[
                            "xml_index"
                        ],

                    "xml_malignancy":
                        xnod[
                            "malignancy"
                        ],

                    "native_active_slices":
                        len(native_slices),

                    "xml_roi_count":
                        len(xnod["rois"]),

                    "matched_slices":
                        comparison[
                            "matched_slices"
                        ],

                    "z_offset":
                        comparison[
                            "z_offset"
                        ],

                    "dx":
                        round(
                            comparison["dx"],
                            3
                        ),

                    "dy":
                        round(
                            comparison["dy"],
                            3
                        ),

                    "mean_Dice":
                        round(
                            comparison["mean_dice"],
                            5
                        ),

                    "mean_IoU":
                        round(
                            comparison["mean_iou"],
                            5
                        ),

                    "audit_status":
                        "GEOMETRIC_EVIDENCE_ONLY"

                })


# =============================================================================
# SAVE DETAILED AUDIT
# =============================================================================

audit_df = pd.DataFrame(
    audit_rows
)

audit_df.to_csv(
    OUT_CSV,
    index=False
)


# =============================================================================
# SELECT BEST CANDIDATE PER NATIVE NODULE
# =============================================================================

best_rows = []

if not audit_df.empty:

    for subset_id, group in audit_df.groupby(
        "subset_nodule_id"
    ):

        group = group.sort_values(
            "mean_Dice",
            ascending=False
        ).reset_index(
            drop=True
        )

        best = group.iloc[0]

        if len(group) > 1:
            second = group.iloc[1]

            second_score = float(
                second["mean_Dice"]
            )

            margin = (
                float(best["mean_Dice"])
                - second_score
            )

        else:

            second_score = np.nan
            margin = np.nan

        best_rows.append({

            "subset_nodule_id":
                subset_id,

            "patient_id":
                best["patient_id"],

            "native_nodule_id":
                best["native_nodule_id"],

            "best_native_mask":
                best["native_mask"],

            "best_xml_nodule_id":
                best["xml_nodule_id"],

            "xml_malignancy":
                best["xml_malignancy"],

            "best_mean_Dice":
                float(best["mean_Dice"]),

            "second_best_mean_Dice":
                second_score,

            "Dice_margin":
                margin,

            "best_mean_IoU":
                float(best["mean_IoU"]),

            "matched_slices":
                int(best["matched_slices"]),

            "z_offset":
                int(best["z_offset"]),

            "dx":
                float(best["dx"]),

            "dy":
                float(best["dy"])

        })


best_df = pd.DataFrame(
    best_rows
)


# =============================================================================
# AUDIT-ONLY CLASSIFICATION
# =============================================================================

def classify_geometry(row):

    score = float(
        row["best_mean_Dice"]
    )

    margin = row["Dice_margin"]

    if (
        score >= 0.85
        and pd.notna(margin)
        and margin >= 0.15
    ):

        return "STRONG_GEOMETRIC_EVIDENCE"

    elif score >= 0.75:

        return "GOOD_GEOMETRIC_EVIDENCE"

    elif score >= 0.50:

        return "MODERATE_GEOMETRIC_EVIDENCE"

    else:

        return "WEAK_GEOMETRIC_EVIDENCE"


if not best_df.empty:

    best_df[
        "audit_classification"
    ] = best_df.apply(
        classify_geometry,
        axis=1
    )


best_df.to_csv(
    OUT_BEST_CSV,
    index=False
)


# =============================================================================
# FINAL REPORT
# =============================================================================

print("\n" + "=" * 80)
print("STEP 20 RESULTS")
print("=" * 80)

print(
    f"Detailed comparisons: "
    f"{len(audit_df)}"
)

print(
    f"Native nodules audited: "
    f"{len(best_df)}"
)

if not best_df.empty:

    print(
        "\nBEST CANDIDATE FOR EACH NATIVE NODULE"
    )

    print("-" * 80)

    columns_to_show = [

        "subset_nodule_id",

        "patient_id",

        "native_nodule_id",

        "best_native_mask",

        "best_xml_nodule_id",

        "xml_malignancy",

        "best_mean_Dice",

        "second_best_mean_Dice",

        "Dice_margin",

        "matched_slices",

        "audit_classification"

    ]

    print(
        best_df[
            columns_to_show
        ].to_string(
            index=False
        )
    )


# =============================================================================
# WRITE SUMMARY
# =============================================================================

strong_count = 0
good_count = 0
moderate_count = 0
weak_count = 0

if not best_df.empty:

    strong_count = int(
        (
            best_df[
                "audit_classification"
            ]
            == "STRONG_GEOMETRIC_EVIDENCE"
        ).sum()
    )

    good_count = int(
        (
            best_df[
                "audit_classification"
            ]
            == "GOOD_GEOMETRIC_EVIDENCE"
        ).sum()
    )

    moderate_count = int(
        (
            best_df[
                "audit_classification"
            ]
            == "MODERATE_GEOMETRIC_EVIDENCE"
        ).sum()
    )

    weak_count = int(
        (
            best_df[
                "audit_classification"
            ]
            == "WEAK_GEOMETRIC_EVIDENCE"
        ).sum()
    )


summary = f"""
===============================================================================
STEP 20 — DETAILED FORENSIC AUDIT SUMMARY
===============================================================================

Promising cases received from Step 19:
    {len(promising_std)}

Cases successfully compared:
    {len(best_df)}

Detailed mask/XML comparisons:
    {len(audit_df)}

AUDIT-ONLY GEOMETRIC CATEGORIES
--------------------------------
Strong geometric evidence:
    {strong_count}

Good geometric evidence:
    {good_count}

Moderate geometric evidence:
    {moderate_count}

Weak geometric evidence:
    {weak_count}

METHODOLOGICAL LIMITATION
-------------------------
Dice/IoU agreement is geometric evidence only.

It does NOT restore missing DICOM SOPInstanceUID or
ImagePositionPatient information.

Therefore:

    HIGH DICE
        DOES NOT EQUAL
    PROVEN NATIVE -> XML IDENTITY

XML malignancy values shown in this audit remain properties
of the XML annotation. They are NOT assigned as native-nodule
ground-truth labels.

OUTPUT FILES
------------
Detailed audit:
    {OUT_CSV}

Best candidate table:
    {OUT_BEST_CSV}

Summary:
    {OUT_TXT}
"""

with open(
    OUT_TXT,
    "w"
) as f:

    f.write(
        summary.strip()
    )

print(
    "\n" +
    summary
)

print("\n" + "=" * 80)
print("STEP 20 COMPLETE")
print("=" * 80)

STEP 20 — DETAILED FORENSIC AUDIT OF PROMISING MATCHES

INPUT
--------------------------------------------------------------------------------
Step 19 rows: 10
Columns: ['subset_nodule_id', 'patient_id', 'native_nodule_id', 'series_instance_uid', 'match_status', 'candidate_count', 'best_score', 'second_best_score', 'score_margin', 'candidate_cluster', 'xml_nodule_ids', 'reader_annotation_count', 'rated_reader_count', 'malignancy_ratings', 'mean_malignancy', 'median_malignancy', 'consensus_class_candidate', 'mean_x', 'mean_y', 'mean_z', 'min_z', 'max_z', 'mean_roi_count', 'audit_status']

DETECTED COLUMNS
--------------------------------------------------------------------------------
subset_nodule_id : subset_nodule_id
patient_id       : patient_id
native_nodule_id : native_nodule_id
SeriesInstanceUID: series_instance_uid

PROMISING CASE CHECK
--------------------------------------------------------------------------------
Valid promising cases: 10
Unique promising subset IDs: 10

Case

In [ ]:
# =============================================================================
# STEP 21 — PATIENT-ISOLATED TRAIN / VALIDATION / TEST SPLIT
# =============================================================================

import os
import random
import pandas as pd
import numpy as np

MASTER_CSV = "/content/final_325_master_manifest.csv"

SPLIT_MANIFEST_CSV = "/content/final_325_patient_split_manifest.csv"
PATIENT_SPLIT_CSV = "/content/final_325_patient_splits.csv"
SPLIT_SUMMARY_TXT = "/content/final_325_patient_split_summary.txt"

SEED = 42

print("=" * 80)
print("STEP 21 — PATIENT-ISOLATED DATASET SPLITTING")
print("=" * 80)

# -------------------------------------------------------------------------
# Load manifest
# -------------------------------------------------------------------------

if not os.path.exists(MASTER_CSV):
    raise FileNotFoundError(
        f"Missing master manifest: {MASTER_CSV}"
    )

df = pd.read_csv(MASTER_CSV)

print("\nINPUT")
print("-" * 80)
print(f"Rows: {len(df)}")
print(f"Unique patients: {df['patient_id'].nunique()}")

# -------------------------------------------------------------------------
# Required checks
# -------------------------------------------------------------------------

assert len(df) == 325, (
    f"Expected 325 rows, found {len(df)}"
)

assert df["subset_nodule_id"].nunique() == 325, (
    "subset_nodule_id values are not unique."
)

assert df["patient_id"].notna().all(), (
    "Missing patient IDs."
)

assert df["SeriesInstanceUID"].notna().all(), (
    "Missing SeriesInstanceUID values."
)

assert (
    ~df["subset_nodule_id"].isin(
        ["nodule_029", "nodule_085"]
    )
).all(), (
    "Excluded nodules unexpectedly present."
)

# -------------------------------------------------------------------------
# Build deterministic patient split
# -------------------------------------------------------------------------

patients = sorted(
    df["patient_id"].astype(str).unique()
)

rng = random.Random(SEED)
rng.shuffle(patients)

n_patients = len(patients)

n_train = int(
    round(0.70 * n_patients)
)

n_val = int(
    round(0.15 * n_patients)
)

# Make sure all patients are assigned.
n_test = (
    n_patients
    - n_train
    - n_val
)

train_patients = set(
    patients[:n_train]
)

val_patients = set(
    patients[n_train:n_train + n_val]
)

test_patients = set(
    patients[n_train + n_val:]
)

# -------------------------------------------------------------------------
# Assign split to every nodule
# -------------------------------------------------------------------------

def assign_split(patient_id):

    patient_id = str(patient_id)

    if patient_id in train_patients:
        return "train"

    if patient_id in val_patients:
        return "val"

    if patient_id in test_patients:
        return "test"

    raise ValueError(
        f"Patient not assigned to a split: {patient_id}"
    )


df["split"] = df["patient_id"].apply(
    assign_split
)

# -------------------------------------------------------------------------
# Verify no patient leakage
# -------------------------------------------------------------------------

train_ids = set(
    df.loc[df["split"] == "train", "patient_id"]
)

val_ids = set(
    df.loc[df["split"] == "val", "patient_id"]
)

test_ids = set(
    df.loc[df["split"] == "test", "patient_id"]
)

assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0

assert (
    len(train_ids)
    + len(val_ids)
    + len(test_ids)
    == n_patients
)

# -------------------------------------------------------------------------
# Counts
# -------------------------------------------------------------------------

nodule_counts = (
    df["split"]
    .value_counts()
    .reindex(
        ["train", "val", "test"],
        fill_value=0
    )
)

patient_counts = {
    "train": len(train_ids),
    "val": len(val_ids),
    "test": len(test_ids)
}

# -------------------------------------------------------------------------
# Save
# -------------------------------------------------------------------------

df.to_csv(
    SPLIT_MANIFEST_CSV,
    index=False
)

patient_split_df = pd.DataFrame({
    "patient_id":
        list(train_patients)
        + list(val_patients)
        + list(test_patients),

    "split":
        (
            ["train"] * len(train_patients)
            + ["val"] * len(val_patients)
            + ["test"] * len(test_patients)
        )
})

patient_split_df = (
    patient_split_df
    .sort_values("patient_id")
    .reset_index(drop=True)
)

patient_split_df.to_csv(
    PATIENT_SPLIT_CSV,
    index=False
)

# -------------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------------

summary = f"""
===============================================================================
STEP 21 — PATIENT-ISOLATED SPLIT SUMMARY
===============================================================================

Total nodules:
    {len(df)}

Total unique patients:
    {n_patients}

TRAIN
    Patients: {patient_counts["train"]}
    Nodules:  {nodule_counts["train"]}

VALIDATION
    Patients: {patient_counts["val"]}
    Nodules:  {nodule_counts["val"]}

TEST
    Patients: {patient_counts["test"]}
    Nodules:  {nodule_counts["test"]}

Patient leakage:
    Train ∩ Validation: {len(train_ids & val_ids)}
    Train ∩ Test:       {len(train_ids & test_ids)}
    Validation ∩ Test:  {len(val_ids & test_ids)}

RESULT:
    {'PASS' if not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids) else 'FAIL'}

Important:
    No malignancy stratification was performed because native-nodule
    malignancy labels have not been independently established.

Outputs:
    {SPLIT_MANIFEST_CSV}
    {PATIENT_SPLIT_CSV}
    {SPLIT_SUMMARY_TXT}
"""

with open(
    SPLIT_SUMMARY_TXT,
    "w"
) as f:
    f.write(summary.strip())

print(summary)

print("=" * 80)
print("STEP 21 COMPLETE")
print("=" * 80)

STEP 21 — PATIENT-ISOLATED DATASET SPLITTING

INPUT
--------------------------------------------------------------------------------
Rows: 325
Unique patients: 247

STEP 21 — PATIENT-ISOLATED SPLIT SUMMARY

Total nodules:
    325

Total unique patients:
    247

TRAIN
    Patients: 173
    Nodules:  235

VALIDATION
    Patients: 37
    Nodules:  44

TEST
    Patients: 37
    Nodules:  46

Patient leakage:
    Train ∩ Validation: 0
    Train ∩ Test:       0
    Validation ∩ Test:  0

RESULT:
    PASS

Important:
    No malignancy stratification was performed because native-nodule
    malignancy labels have not been independently established.

Outputs:
    /content/final_325_patient_split_manifest.csv
    /content/final_325_patient_splits.csv
    /content/final_325_patient_split_summary.txt

STEP 21 COMPLETE


In [ ]:
# =============================================================================
# STEP 22 — BUILD MODEL INPUT MANIFESTS
# =============================================================================

import os
import glob
import pandas as pd
import numpy as np

SPLIT_MANIFEST_CSV = "/content/final_325_patient_split_manifest.csv"
PROCESSED_ROOT = "/content/processed_325_nodules"

NODULE_MANIFEST_CSV = "/content/final_325_nodule_model_manifest.csv"
SLICE_MANIFEST_CSV = "/content/final_325_slice_model_manifest.csv"

print("=" * 80)
print("STEP 22 — BUILDING MODEL INPUT MANIFESTS")
print("=" * 80)

if not os.path.exists(SPLIT_MANIFEST_CSV):
    raise FileNotFoundError(
        f"Missing Step 21 manifest: {SPLIT_MANIFEST_CSV}"
    )

if not os.path.exists(PROCESSED_ROOT):
    raise FileNotFoundError(
        f"Missing processed dataset: {PROCESSED_ROOT}"
    )

df = pd.read_csv(
    SPLIT_MANIFEST_CSV
)

assert len(df) == 325

nodule_rows = []
slice_rows = []

# -------------------------------------------------------------------------
# Process every nodule
# -------------------------------------------------------------------------

for _, row in df.iterrows():

    subset_id = str(
        row["subset_nodule_id"]
    )

    patient_id = str(
        row["patient_id"]
    )

    native_id = str(
        row["native_nodule_id"]
    )

    split = str(
        row["split"]
    )

    # Expected directory created in Step 16.
    nodule_dir = os.path.join(
        PROCESSED_ROOT,
        subset_id
    )

    # Fallback in case folders were named differently.
    if not os.path.isdir(nodule_dir):

        matches = glob.glob(
            os.path.join(
                PROCESSED_ROOT,
                "**",
                subset_id
            ),
            recursive=True
        )

        if matches:
            nodule_dir = matches[0]

    if not os.path.isdir(nodule_dir):

        print(
            f"WARNING: Missing processed folder for {subset_id}"
        )

        continue

    # ---------------------------------------------------------------------
    # Find image files
    # ---------------------------------------------------------------------

    image_files = sorted(
        glob.glob(
            os.path.join(
                nodule_dir,
                "images",
                "*.png"
            )
        )
    )

    # ---------------------------------------------------------------------
    # Find masks
    # ---------------------------------------------------------------------

    mask_files = {}

    for mask_num in range(4):

        mask_files[mask_num] = sorted(
            glob.glob(
                os.path.join(
                    nodule_dir,
                    f"mask-{mask_num}",
                    "*.png"
                )
            )
        )

    # ---------------------------------------------------------------------
    # Nodule-level record
    # ---------------------------------------------------------------------

    nodule_rows.append({

        "subset_nodule_id":
            subset_id,

        "patient_id":
            patient_id,

        "native_nodule_id":
            native_id,

        "SeriesInstanceUID":
            row["SeriesInstanceUID"],

        "split":
            split,

        "nodule_directory":
            nodule_dir,

        "image_count":
            len(image_files),

        "mask_0_count":
            len(mask_files[0]),

        "mask_1_count":
            len(mask_files[1]),

        "mask_2_count":
            len(mask_files[2]),

        "mask_3_count":
            len(mask_files[3]),

        # Pipe-separated paths keep the CSV easy to inspect.
        "image_files":
            "|".join(image_files),

        "mask_0_files":
            "|".join(mask_files[0]),

        "mask_1_files":
            "|".join(mask_files[1]),

        "mask_2_files":
            "|".join(mask_files[2]),

        "mask_3_files":
            "|".join(mask_files[3]),

        # Labels deliberately remain unavailable.
        "label_status":
            "UNASSIGNED_NATIVE_IDENTITY_UNPROVEN"

    })

    # ---------------------------------------------------------------------
    # Slice-level records
    # ---------------------------------------------------------------------

    for slice_index, image_path in enumerate(
        image_files
    ):

        slice_rows.append({

            "subset_nodule_id":
                subset_id,

            "patient_id":
                patient_id,

            "native_nodule_id":
                native_id,

            "SeriesInstanceUID":
                row["SeriesInstanceUID"],

            "split":
                split,

            "slice_index":
                slice_index,

            "image_path":
                image_path,

            "mask_0_path":
                (
                    mask_files[0][slice_index]
                    if slice_index < len(mask_files[0])
                    else ""
                ),

            "mask_1_path":
                (
                    mask_files[1][slice_index]
                    if slice_index < len(mask_files[1])
                    else ""
                ),

            "mask_2_path":
                (
                    mask_files[2][slice_index]
                    if slice_index < len(mask_files[2])
                    else ""
                ),

            "mask_3_path":
                (
                    mask_files[3][slice_index]
                    if slice_index < len(mask_files[3])
                    else ""
                ),

            "label_status":
                "UNASSIGNED_NATIVE_IDENTITY_UNPROVEN"

        })


# -------------------------------------------------------------------------
# Save
# -------------------------------------------------------------------------

nodule_manifest = pd.DataFrame(
    nodule_rows
)

slice_manifest = pd.DataFrame(
    slice_rows
)

nodule_manifest.to_csv(
    NODULE_MANIFEST_CSV,
    index=False
)

slice_manifest.to_csv(
    SLICE_MANIFEST_CSV,
    index=False
)

# -------------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------------

print("\nNODULE MANIFEST")
print("-" * 80)
print(
    f"Nodules represented: "
    f"{len(nodule_manifest)}"
)

print(
    nodule_manifest["split"]
    .value_counts()
    .to_string()
)

print("\nSLICE MANIFEST")
print("-" * 80)
print(
    f"Image slices represented: "
    f"{len(slice_manifest)}"
)

print(
    slice_manifest["split"]
    .value_counts()
    .to_string()
)

assert (
    nodule_manifest["subset_nodule_id"].nunique()
    == len(nodule_manifest)
)

assert (
    set(nodule_manifest["split"])
    <= {"train", "val", "test"}
)

print("\nOutputs:")
print(NODULE_MANIFEST_CSV)
print(SLICE_MANIFEST_CSV)

print("=" * 80)
print("STEP 22 COMPLETE")
print("=" * 80)

STEP 22 — BUILDING MODEL INPUT MANIFESTS

NODULE MANIFEST
--------------------------------------------------------------------------------
Nodules represented: 325
split
train    235
test      46
val       44

SLICE MANIFEST
--------------------------------------------------------------------------------
Image slices represented: 2000
split
train    1437
test      309
val       254

Outputs:
/content/final_325_nodule_model_manifest.csv
/content/final_325_slice_model_manifest.csv
STEP 22 COMPLETE


In [ ]:
# =============================================================================
# STEP 23 — CREATE MODEL-READY NORMALIZED ARRAYS
# =============================================================================

import os
import glob
import numpy as np
import pandas as pd
from PIL import Image

NODULE_MANIFEST_CSV = "/content/final_325_nodule_model_manifest.csv"

PREPROCESSED_ROOT = "/content/model_ready_325_nodules"
PREPROCESS_MANIFEST_CSV = "/content/final_325_preprocessed_manifest.csv"
PREPROCESS_SUMMARY_TXT = "/content/final_325_preprocessing_summary.txt"

os.makedirs(
    PREPROCESSED_ROOT,
    exist_ok=True
)

print("=" * 80)
print("STEP 23 — MODEL-READY NORMALIZATION")
print("=" * 80)

df = pd.read_csv(
    NODULE_MANIFEST_CSV
)

assert len(df) > 0

output_rows = []

# -------------------------------------------------------------------------
# Image loading helper
# -------------------------------------------------------------------------

def load_png(path):

    with Image.open(path) as img:

        return np.asarray(
            img.convert("F"),
            dtype=np.float32
        )


# -------------------------------------------------------------------------
# Process each nodule
# -------------------------------------------------------------------------

for counter, (_, row) in enumerate(
    df.iterrows(),
    start=1
):

    subset_id = str(
        row["subset_nodule_id"]
    )

    output_dir = os.path.join(
        PREPROCESSED_ROOT,
        subset_id
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    image_files = [
        x for x in str(
            row["image_files"]
        ).split("|")
        if x
    ]

    if not image_files:
        print(
            f"WARNING: No images for {subset_id}"
        )
        continue

    # ---------------------------------------------------------------------
    # Load image volume
    # ---------------------------------------------------------------------

    image_arrays = []

    for path in image_files:

        image_arrays.append(
            load_png(path)
        )

    image_volume = np.stack(
        image_arrays,
        axis=0
    )

    # ---------------------------------------------------------------------
    # Robust intensity normalization
    # ---------------------------------------------------------------------

    low = float(
        np.percentile(
            image_volume,
            1.0
        )
    )

    high = float(
        np.percentile(
            image_volume,
            99.0
        )
    )

    if high <= low:

        normalized = np.zeros_like(
            image_volume,
            dtype=np.float32
        )

    else:

        clipped = np.clip(
            image_volume,
            low,
            high
        )

        normalized = (
            (clipped - low)
            / (high - low)
        ).astype(
            np.float32
        )

    # ---------------------------------------------------------------------
    # Save normalized image volume
    # ---------------------------------------------------------------------

    image_output = os.path.join(
        output_dir,
        "images_normalized.npy"
    )

    np.save(
        image_output,
        normalized
    )

    # ---------------------------------------------------------------------
    # Load and save binary mask volumes
    # ---------------------------------------------------------------------

    mask_paths_by_num = {

        0: str(row["mask_0_files"]).split("|")
        if str(row["mask_0_files"]).strip()
        else [],

        1: str(row["mask_1_files"]).split("|")
        if str(row["mask_1_files"]).strip()
        else [],

        2: str(row["mask_2_files"]).split("|")
        if str(row["mask_2_files"]).strip()
        else [],

        3: str(row["mask_3_files"]).split("|")
        if str(row["mask_3_files"]).strip()
        else []

    }

    mask_output_paths = {}

    for mask_num, paths in mask_paths_by_num.items():

        paths = [
            x for x in paths
            if x
        ]

        if not paths:
            mask_output_paths[mask_num] = ""
            continue

        mask_arrays = []

        for path in paths:

            with Image.open(path) as img:

                arr = np.asarray(
                    img.convert("L")
                ) > 0

            mask_arrays.append(
                arr.astype(
                    np.uint8
                )
            )

        mask_volume = np.stack(
            mask_arrays,
            axis=0
        )

        mask_out = os.path.join(
            output_dir,
            f"mask-{mask_num}.npy"
        )

        np.save(
            mask_out,
            mask_volume
        )

        mask_output_paths[mask_num] = mask_out

    # ---------------------------------------------------------------------
    # Record preprocessing metadata
    # ---------------------------------------------------------------------

    output_rows.append({

        "subset_nodule_id":
            subset_id,

        "patient_id":
            row["patient_id"],

        "native_nodule_id":
            row["native_nodule_id"],

        "SeriesInstanceUID":
            row["SeriesInstanceUID"],

        "split":
            row["split"],

        "image_volume_path":
            image_output,

        "image_depth":
            normalized.shape[0],

        "image_height":
            normalized.shape[1],

        "image_width":
            normalized.shape[2],

        "normalization_method":
            "1st_to_99th_percentile_clipping_then_0_to_1",

        "normalization_low":
            low,

        "normalization_high":
            high,

        "mask_0_path":
            mask_output_paths[0],

        "mask_1_path":
            mask_output_paths[1],

        "mask_2_path":
            mask_output_paths[2],

        "mask_3_path":
            mask_output_paths[3],

        "label_status":
            "UNASSIGNED_NATIVE_IDENTITY_UNPROVEN"

    })

    if counter % 25 == 0:

        print(
            f"Processed "
            f"{counter} / {len(df)}"
        )


# -------------------------------------------------------------------------
# Save manifest
# -------------------------------------------------------------------------

preprocess_df = pd.DataFrame(
    output_rows
)

preprocess_df.to_csv(
    PREPROCESS_MANIFEST_CSV,
    index=False
)

# -------------------------------------------------------------------------
# Validation
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("PREPROCESSING VALIDATION")
print("=" * 80)

print(
    f"Processed nodules: "
    f"{len(preprocess_df)}"
)

print(
    f"Expected nodules: "
    f"{len(df)}"
)

assert len(preprocess_df) == len(df)

print(
    "\nNormalization range check:"
)

sample_ranges = []

for _, r in preprocess_df.head(10).iterrows():

    arr = np.load(
        r["image_volume_path"]
    )

    sample_ranges.append(
        (
            float(arr.min()),
            float(arr.max())
        )
    )

print(
    sample_ranges
)

print("\nImportant:")
print(
    "The original PNG files were NOT modified."
)

print(
    "The normalized NPY files are derived model inputs."
)

print(
    "No malignancy labels were assigned."
)

summary = f"""
===============================================================================
STEP 23 — PREPROCESSING SUMMARY
===============================================================================

Nodules processed:
    {len(preprocess_df)}

Preprocessing:
    1st percentile -> lower clipping bound
    99th percentile -> upper clipping bound
    clipped values -> normalized to [0, 1]

Masks:
    Converted to binary uint8 arrays.

Spatial preprocessing:
    NO resizing
    NO cropping
    NO augmentation

Raw dataset:
    UNMODIFIED

Labels:
    NOT ASSIGNED

Output:
    {PREPROCESSED_ROOT}

Manifest:
    {PREPROCESS_MANIFEST_CSV}
"""

with open(
    PREPROCESS_SUMMARY_TXT,
    "w"
) as f:
    f.write(summary.strip())

print(summary)

print("=" * 80)
print("STEP 23 COMPLETE")
print("=" * 80)

STEP 23 — MODEL-READY NORMALIZATION
Processed 25 / 325
Processed 50 / 325
Processed 75 / 325
Processed 100 / 325
Processed 125 / 325
Processed 150 / 325
Processed 175 / 325
Processed 200 / 325
Processed 225 / 325
Processed 250 / 325
Processed 275 / 325
Processed 300 / 325
Processed 325 / 325

PREPROCESSING VALIDATION
Processed nodules: 325
Expected nodules: 325

Normalization range check:
[(0.0, 1.0), (0.0, 1.0), (0.0, 1.0), (0.0, 1.0), (0.0, 1.0), (0.0, 1.0), (0.0, 1.0), (0.0, 1.0), (0.0, 1.0), (0.0, 1.0)]

Important:
The original PNG files were NOT modified.
The normalized NPY files are derived model inputs.
No malignancy labels were assigned.

STEP 23 — PREPROCESSING SUMMARY

Nodules processed:
    325

Preprocessing:
    1st percentile -> lower clipping bound
    99th percentile -> upper clipping bound
    clipped values -> normalized to [0, 1]

Masks:
    Converted to binary uint8 arrays.

Spatial preprocessing:
    NO resizing
    NO cropping
    NO augmentation

Raw dataset:
   

In [ ]:
# =============================================================================
# STEP 24 — BUILD FINAL MACHINE-LEARNING MANIFEST
# =============================================================================

import os
import pandas as pd

PREPROCESS_MANIFEST_CSV = "/content/final_325_preprocessed_manifest.csv"
MASTER_CSV = "/content/final_325_master_manifest.csv"

FINAL_ML_MANIFEST_CSV = "/content/final_325_ml_manifest.csv"
FINAL_ML_SUMMARY_TXT = "/content/final_325_ml_manifest_summary.txt"

print("=" * 80)
print("STEP 24 — FINAL MACHINE-LEARNING MANIFEST")
print("=" * 80)

pre = pd.read_csv(
    PREPROCESS_MANIFEST_CSV
)

master = pd.read_csv(
    MASTER_CSV
)

# Keep only the authoritative metadata we need.
master_small = master[
    [
        "subset_nodule_id",
        "qc_status",
        "image_slice_count",
        "number_of_masks_with_foreground",
        "xml_file",
        "xml_nodule_count_in_series",
        "candidate_xml_nodule_ids",
        "candidate_xml_malignancy_ratings",
        "native_to_xml_identity_proven"
    ]
].copy()

# Merge once on unique subset ID.
final_df = pre.merge(
    master_small,
    on="subset_nodule_id",
    how="left",
    validate="one_to_one"
)

# -------------------------------------------------------------------------
# Explicit label/provenance columns
# -------------------------------------------------------------------------

final_df["native_label"] = np.nan

final_df["label_source"] = (
    "NONE"
)

final_df["label_status"] = (
    "UNASSIGNED_NATIVE_IDENTITY_UNPROVEN"
)

final_df["native_to_xml_identity_proven"] = (
    final_df[
        "native_to_xml_identity_proven"
    ].fillna(False).astype(bool)
)

# -------------------------------------------------------------------------
# Validation
# -------------------------------------------------------------------------

assert len(final_df) == 325

assert (
    final_df[
        "subset_nodule_id"
    ].nunique()
    == 325
)

assert (
    final_df["patient_id"].nunique()
    == 247
)

assert (
    final_df["SeriesInstanceUID"].nunique()
    == 247
)

assert (
    ~final_df[
        "subset_nodule_id"
    ].isin([
        "nodule_029",
        "nodule_085"
    ])
).all()

assert (
    final_df["split"].isin(
        ["train", "val", "test"]
    ).all()
)

# -------------------------------------------------------------------------
# Save
# -------------------------------------------------------------------------

final_df.to_csv(
    FINAL_ML_MANIFEST_CSV,
    index=False
)

# -------------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------------

summary = f"""
===============================================================================
FINAL 325-NODULE MACHINE-LEARNING MANIFEST
===============================================================================

Nodules:
    {len(final_df)}

Patients:
    {final_df["patient_id"].nunique()}

SeriesInstanceUIDs:
    {final_df["SeriesInstanceUID"].nunique()}

Train nodules:
    {(final_df["split"] == "train").sum()}

Validation nodules:
    {(final_df["split"] == "val").sum()}

Test nodules:
    {(final_df["split"] == "test").sum()}

QC PASS:
    {(final_df["qc_status"] == "PASS").sum()}

Native malignancy labels:
    {final_df["native_label"].notna().sum()}

Native labels assigned:
    NO

Excluded:
    nodule_029
    nodule_085

IMPORTANT:
    XML malignancy values are retained only as candidate annotation evidence.
    They are NOT used as native-nodule ground-truth labels.

Output:
    {FINAL_ML_MANIFEST_CSV}
"""

with open(
    FINAL_ML_SUMMARY_TXT,
    "w"
) as f:
    f.write(summary.strip())

print(summary)

print("=" * 80)
print("STEP 24 COMPLETE")
print("=" * 80)

STEP 24 — FINAL MACHINE-LEARNING MANIFEST

FINAL 325-NODULE MACHINE-LEARNING MANIFEST

Nodules:
    325

Patients:
    247

SeriesInstanceUIDs:
    247

Train nodules:
    235

Validation nodules:
    44

Test nodules:
    46

QC PASS:
    325

Native malignancy labels:
    0

Native labels assigned:
    NO

Excluded:
    nodule_029
    nodule_085

IMPORTANT:
    XML malignancy values are retained only as candidate annotation evidence.
    They are NOT used as native-nodule ground-truth labels.

Output:
    /content/final_325_ml_manifest.csv

STEP 24 COMPLETE


In [ ]:
# =============================================================================
# STEP 25 — FINAL DATASET INTEGRITY CHECK
# =============================================================================

import os
import pandas as pd

FINAL_ML_MANIFEST_CSV = "/content/final_325_ml_manifest.csv"

print("=" * 80)
print("STEP 25 — FINAL DATASET INTEGRITY CHECK")
print("=" * 80)

df = pd.read_csv(
    FINAL_ML_MANIFEST_CSV
)

# -------------------------------------------------------------------------
# Cohort checks
# -------------------------------------------------------------------------

assert len(df) == 325

assert (
    df["subset_nodule_id"].nunique()
    == 325
)

assert (
    df["patient_id"].nunique()
    == 247
)

assert (
    df["SeriesInstanceUID"].nunique()
    == 247
)

assert (
    "nodule_029"
    not in set(df["subset_nodule_id"])
)

assert (
    "nodule_085"
    not in set(df["subset_nodule_id"])
)

# -------------------------------------------------------------------------
# QC checks
# -------------------------------------------------------------------------

assert (
    (df["qc_status"] == "PASS").sum()
    == 325
)

# -------------------------------------------------------------------------
# Split checks
# -------------------------------------------------------------------------

train_patients = set(
    df.loc[
        df["split"] == "train",
        "patient_id"
    ]
)

val_patients = set(
    df.loc[
        df["split"] == "val",
        "patient_id"
    ]
)

test_patients = set(
    df.loc[
        df["split"] == "test",
        "patient_id"
    ]
)

assert not (
    train_patients
    & val_patients
)

assert not (
    train_patients
    & test_patients
)

assert not (
    val_patients
    & test_patients
)

# -------------------------------------------------------------------------
# File existence
# -------------------------------------------------------------------------

missing_inputs = []

for _, row in df.iterrows():

    image_path = row[
        "image_volume_path"
    ]

    if not os.path.exists(
        image_path
    ):
        missing_inputs.append(
            (
                row[
                    "subset_nodule_id"
                ],
                image_path
            )
        )

# -------------------------------------------------------------------------
# Label integrity
# -------------------------------------------------------------------------

assigned_labels = (
    df["native_label"]
    .notna()
    .sum()
)

assert assigned_labels == 0

# -------------------------------------------------------------------------
# Final report
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL DATASET STATUS")
print("=" * 80)

print(
    f"Eligible nodules:          {len(df)}"
)

print(
    f"Unique patients:           {df['patient_id'].nunique()}"
)

print(
    f"Unique CT series:          {df['SeriesInstanceUID'].nunique()}"
)

print(
    f"QC PASS:                   {(df['qc_status'] == 'PASS').sum()}"
)

print(
    f"Missing model inputs:      {len(missing_inputs)}"
)

print(
    f"Native labels assigned:    {assigned_labels}"
)

print(
    f"Train nodules:             {(df['split'] == 'train').sum()}"
)

print(
    f"Validation nodules:        {(df['split'] == 'val').sum()}"
)

print(
    f"Test nodules:              {(df['split'] == 'test').sum()}"
)

if missing_inputs:

    print("\nMISSING INPUTS:")
    for item in missing_inputs[:20]:
        print(item)

    raise RuntimeError(
        "Some normalized model inputs are missing."
    )

else:

    print(
        "\n✓ All model input volumes exist."
    )

print(
    "\n✓ No patient leakage."
)

print(
    "✓ All 325 nodules pass dataset QC."
)

print(
    "✓ Excluded nodules remain excluded."
)

print(
    "✓ No unproven malignancy labels were assigned."
)

print(
    "\nFINAL STATUS: MODELING-READY DATA ORGANIZATION COMPLETE"
)

print(
    "\nFinal manifest:"
)

print(
    FINAL_ML_MANIFEST_CSV
)

print("=" * 80)
print("STEP 25 COMPLETE")
print("=" * 80)

STEP 25 — FINAL DATASET INTEGRITY CHECK

FINAL DATASET STATUS
Eligible nodules:          325
Unique patients:           247
Unique CT series:          247
QC PASS:                   325
Missing model inputs:      0
Native labels assigned:    0
Train nodules:             235
Validation nodules:        44
Test nodules:              46

✓ All model input volumes exist.

✓ No patient leakage.
✓ All 325 nodules pass dataset QC.
✓ Excluded nodules remain excluded.
✓ No unproven malignancy labels were assigned.

FINAL STATUS: MODELING-READY DATA ORGANIZATION COMPLETE

Final manifest:
/content/final_325_ml_manifest.csv
STEP 25 COMPLETE


In [ ]:
# =============================================================================
# STEP 26 — VISUAL + STATISTICAL QUALITY CONTROL
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------------

FINAL_MANIFEST = "/content/final_325_ml_manifest.csv"

QC_CSV = "/content/step26_visual_statistical_qc.csv"
QC_SUMMARY = "/content/step26_visual_statistical_qc_summary.txt"
QC_FIG_DIR = "/content/step26_qc_figures"

os.makedirs(QC_FIG_DIR, exist_ok=True)

print("=" * 80)
print("STEP 26 — VISUAL + STATISTICAL QUALITY CONTROL")
print("=" * 80)

# =============================================================================
# 1. LOAD FINAL MANIFEST
# =============================================================================

if not os.path.exists(FINAL_MANIFEST):
    raise FileNotFoundError(
        f"Missing final manifest: {FINAL_MANIFEST}"
    )

df = pd.read_csv(FINAL_MANIFEST)

print("\nINPUT")
print("-" * 80)
print(f"Manifest rows:       {len(df)}")
print(f"Unique patients:     {df['patient_id'].nunique()}")
print(f"Unique CT series:    {df['SeriesInstanceUID'].nunique()}")

assert len(df) == 325
assert df["subset_nodule_id"].nunique() == 325

# =============================================================================
# 2. LOAD VOLUMES AND CALCULATE STATISTICS
# =============================================================================

qc_rows = []

print("\nCALCULATING VOLUME STATISTICS")
print("-" * 80)

for counter, (_, row) in enumerate(df.iterrows(), start=1):

    subset_id = str(row["subset_nodule_id"])
    patient_id = str(row["patient_id"])
    native_id = str(row["native_nodule_id"])
    split = str(row["split"])

    image_path = str(
        row["image_volume_path"]
    )

    record = {
        "subset_nodule_id": subset_id,
        "patient_id": patient_id,
        "native_nodule_id": native_id,
        "SeriesInstanceUID": str(row["SeriesInstanceUID"]),
        "split": split,
        "image_volume_exists": False,
        "image_volume_shape": "",
        "image_min": np.nan,
        "image_max": np.nan,
        "image_mean": np.nan,
        "image_std": np.nan,
        "image_p01": np.nan,
        "image_p50": np.nan,
        "image_p99": np.nan,
        "mask_0_exists": False,
        "mask_1_exists": False,
        "mask_2_exists": False,
        "mask_3_exists": False,
        "mask_0_foreground_pixels": 0,
        "mask_1_foreground_pixels": 0,
        "mask_2_foreground_pixels": 0,
        "mask_3_foreground_pixels": 0,
        "mask_0_foreground_slices": 0,
        "mask_1_foreground_slices": 0,
        "mask_2_foreground_slices": 0,
        "mask_3_foreground_slices": 0,
        "total_foreground_pixels": 0,
        "qc_status": "FAIL"
    }

    # ---------------------------------------------------------------------
    # Image volume
    # ---------------------------------------------------------------------

    if os.path.exists(image_path):

        try:

            volume = np.load(
                image_path,
                mmap_mode="r"
            )

            record["image_volume_exists"] = True
            record["image_volume_shape"] = str(
                tuple(volume.shape)
            )

            record["image_min"] = float(
                np.min(volume)
            )

            record["image_max"] = float(
                np.max(volume)
            )

            record["image_mean"] = float(
                np.mean(volume)
            )

            record["image_std"] = float(
                np.std(volume)
            )

            record["image_p01"] = float(
                np.percentile(volume, 1)
            )

            record["image_p50"] = float(
                np.percentile(volume, 50)
            )

            record["image_p99"] = float(
                np.percentile(volume, 99)
            )

        except Exception as e:

            print(
                f"WARNING: Could not read image volume "
                f"for {subset_id}: {e}"
            )

            qc_rows.append(record)
            continue

    # ---------------------------------------------------------------------
    # Masks
    # ---------------------------------------------------------------------

    total_fg = 0

    for mask_num in range(4):

        mask_column = f"mask_{mask_num}_path"

        if mask_column not in row.index:
            continue

        mask_path = str(
            row[mask_column]
        )

        if (
            mask_path.strip() == ""
            or mask_path.lower() == "nan"
        ):
            continue

        if not os.path.exists(mask_path):
            continue

        try:

            mask = np.load(
                mask_path,
                mmap_mode="r"
            )

            record[
                f"mask_{mask_num}_exists"
            ] = True

            foreground_pixels = int(
                np.count_nonzero(mask)
            )

            foreground_slices = int(
                np.count_nonzero(
                    np.any(
                        mask > 0,
                        axis=(1, 2)
                    )
                )
            )

            record[
                f"mask_{mask_num}_foreground_pixels"
            ] = foreground_pixels

            record[
                f"mask_{mask_num}_foreground_slices"
            ] = foreground_slices

            total_fg += foreground_pixels

        except Exception as e:

            print(
                f"WARNING: Could not read "
                f"mask-{mask_num} for {subset_id}: {e}"
            )

    record["total_foreground_pixels"] = total_fg

    # ---------------------------------------------------------------------
    # Overall QC status
    # ---------------------------------------------------------------------

    if (
        record["image_volume_exists"]
        and record["image_max"] <= 1.0 + 1e-6
        and record["image_min"] >= -1e-6
        and total_fg > 0
    ):

        record["qc_status"] = "PASS"

    qc_rows.append(record)

    if counter % 25 == 0:
        print(
            f"Processed {counter} / {len(df)}"
        )


qc_df = pd.DataFrame(qc_rows)

qc_df.to_csv(
    QC_CSV,
    index=False
)

# =============================================================================
# 3. STATISTICAL SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("STATISTICAL QC SUMMARY")
print("=" * 80)

print(
    f"Cases evaluated:                 {len(qc_df)}"
)

print(
    f"Image volumes readable:          "
    f"{qc_df['image_volume_exists'].sum()}"
)

print(
    f"Image values within [0,1]:      "
    f"{(
        (qc_df['image_min'] >= -1e-6)
        &
        (qc_df['image_max'] <= 1.0 + 1e-6)
    ).sum()}"
)

print(
    f"Cases with foreground masks:     "
    f"{(qc_df['total_foreground_pixels'] > 0).sum()}"
)

print(
    f"Overall statistical QC PASS:     "
    f"{(qc_df['qc_status'] == 'PASS').sum()}"
)

# =============================================================================
# 4. IMAGE DEPTH DISTRIBUTION
# =============================================================================

def parse_shape(shape_string):

    try:
        return tuple(
            int(x.strip())
            for x in shape_string.strip(
                "()"
            ).split(",")
            if x.strip()
        )
    except Exception:
        return None


qc_df["parsed_shape"] = qc_df[
    "image_volume_shape"
].apply(parse_shape)

qc_df["depth"] = qc_df[
    "parsed_shape"
].apply(
    lambda x: x[0]
    if x is not None and len(x) >= 1
    else np.nan
)

qc_df["height"] = qc_df[
    "parsed_shape"
].apply(
    lambda x: x[1]
    if x is not None and len(x) >= 2
    else np.nan
)

qc_df["width"] = qc_df[
    "parsed_shape"
].apply(
    lambda x: x[2]
    if x is not None and len(x) >= 3
    else np.nan
)

print("\nVOLUME DIMENSIONS")
print("-" * 80)

print(
    "Depth:"
)

print(
    qc_df["depth"].describe().to_string()
)

print(
    "\nHeight:"
)

print(
    qc_df["height"].describe().to_string()
)

print(
    "\nWidth:"
)

print(
    qc_df["width"].describe().to_string()
)

# =============================================================================
# 5. MASK STATISTICS
# =============================================================================

print("\nMASK FOREGROUND SLICE COUNTS")
print("-" * 80)

for mask_num in range(4):

    col = f"mask_{mask_num}_foreground_slices"

    print(
        f"mask-{mask_num}:"
    )

    print(
        qc_df[col].describe().to_string()
    )

# =============================================================================
# 6. INTENSITY HISTOGRAM
# =============================================================================

print("\nCREATING INTENSITY DISTRIBUTION FIGURE")

# Sample up to 25 volumes to keep plotting fast.
sample_df = qc_df.head(
    min(25, len(qc_df))
)

all_values = []

for _, row in sample_df.iterrows():

    path = os.path.join(
        "/content",
        "dummy"
    )

    image_path = (
        df.loc[
            df["subset_nodule_id"]
            == row["subset_nodule_id"],
            "image_volume_path"
        ]
        .iloc[0]
    )

    try:

        volume = np.load(
            image_path,
            mmap_mode="r"
        )

        # Sample pixels rather than loading everything.
        flattened = volume.reshape(-1)

        step = max(
            1,
            len(flattened) // 5000
        )

        sampled = flattened[::step]

        all_values.extend(
            sampled.tolist()
        )

    except Exception:
        pass

all_values = np.asarray(
    all_values,
    dtype=np.float32
)

if len(all_values) > 0:

    plt.figure(
        figsize=(8, 5)
    )

    plt.hist(
        all_values,
        bins=50
    )

    plt.xlabel(
        "Normalized intensity"
    )

    plt.ylabel(
        "Pixel count"
    )

    plt.title(
        "Sampled normalized image intensity distribution"
    )

    plt.tight_layout()

    histogram_path = os.path.join(
        QC_FIG_DIR,
        "intensity_distribution.png"
    )

    plt.savefig(
        histogram_path,
        dpi=150
    )

    plt.close()

    print(
        f"Saved: {histogram_path}"
    )


# =============================================================================
# 7. DEPTH DISTRIBUTION FIGURE
# =============================================================================

plt.figure(
    figsize=(8, 5)
)

plt.hist(
    qc_df["depth"].dropna(),
    bins=20
)

plt.xlabel(
    "Number of slices"
)

plt.ylabel(
    "Number of nodules"
)

plt.title(
    "Nodule volume depth distribution"
)

plt.tight_layout()

depth_fig = os.path.join(
    QC_FIG_DIR,
    "depth_distribution.png"
)

plt.savefig(
    depth_fig,
    dpi=150
)

plt.close()

print(
    f"Saved: {depth_fig}"
)


# =============================================================================
# 8. REPRESENTATIVE VISUAL OVERLAYS
# =============================================================================

print("\nCREATING REPRESENTATIVE IMAGE/MASK FIGURES")
print("-" * 80)

# Choose representative cases from different parts of dataset.
selected_indices = [
    0,
    len(df) // 4,
    len(df) // 2,
    (3 * len(df)) // 4,
    len(df) - 1
]

selected_indices = sorted(
    set(
        i for i in selected_indices
        if 0 <= i < len(df)
    )
)

for index in selected_indices:

    row = df.iloc[index]

    subset_id = str(
        row["subset_nodule_id"]
    )

    image_path = str(
        row["image_volume_path"]
    )

    if not os.path.exists(image_path):
        continue

    try:

        volume = np.load(
            image_path
        )

        # Pick the central slice.
        center = volume.shape[0] // 2

        image_slice = volume[
            center
        ]

        masks = []

        for mask_num in range(4):

            mask_col = (
                f"mask_{mask_num}_path"
            )

            if (
                mask_col in row.index
                and str(
                    row[mask_col]
                ).strip()
                and str(
                    row[mask_col]
                ).lower()
                != "nan"
                and os.path.exists(
                    str(row[mask_col])
                )
            ):

                mask_volume = np.load(
                    str(row[mask_col])
                )

                if (
                    center
                    < mask_volume.shape[0]
                ):

                    masks.append(
                        (
                            mask_num,
                            mask_volume[
                                center
                            ] > 0
                        )
                    )

        # -------------------------------------------------------------
        # Base image
        # -------------------------------------------------------------

        plt.figure(
            figsize=(7, 7)
        )

        plt.imshow(
            image_slice,
            cmap="gray"
        )

        # -------------------------------------------------------------
        # Overlay all available masks
        # -------------------------------------------------------------

        for mask_num, mask_slice in masks:

            overlay = np.ma.masked_where(
                ~mask_slice,
                mask_slice
            )

            plt.imshow(
                overlay,
                alpha=0.30
            )

        plt.title(
            f"{subset_id} | "
            f"{row['patient_id']} | "
            f"slice {center}"
        )

        plt.axis("off")

        plt.tight_layout()

        fig_path = os.path.join(
            QC_FIG_DIR,
            f"{subset_id}_overlay.png"
        )

        plt.savefig(
            fig_path,
            dpi=150,
            bbox_inches="tight"
        )

        plt.close()

        print(
            f"Saved: {fig_path}"
        )

    except Exception as e:

        print(
            f"WARNING: Could not generate "
            f"overlay for {subset_id}: {e}"
        )


# =============================================================================
# 9. FLAG POTENTIAL OUTLIERS
# =============================================================================

print("\nPOTENTIAL OUTLIERS")
print("-" * 80)

outlier_conditions = (
    (qc_df["depth"] <= 1)
    |
    (qc_df["image_std"] < 1e-5)
    |
    (qc_df["total_foreground_pixels"] == 0)
    |
    (qc_df["image_min"] < -1e-5)
    |
    (qc_df["image_max"] > 1.00001)
)

outliers = qc_df[
    outlier_conditions
].copy()

print(
    f"Potential outliers: {len(outliers)}"
)

if not outliers.empty:

    print(
        outliers[
            [
                "subset_nodule_id",
                "patient_id",
                "depth",
                "image_min",
                "image_max",
                "image_std",
                "total_foreground_pixels"
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "No obvious statistical outliers found."
    )


# =============================================================================
# 10. WRITE SUMMARY
# =============================================================================

summary = f"""
===============================================================================
STEP 26 — VISUAL + STATISTICAL QC SUMMARY
===============================================================================

Cohort:
    Nodules evaluated: {len(qc_df)}
    Patients:           {df["patient_id"].nunique()}

IMAGE QUALITY
-------------
Readable image volumes:
    {qc_df["image_volume_exists"].sum()} / {len(qc_df)}

Values within [0,1]:
    {(
        (qc_df["image_min"] >= -1e-6)
        &
        (qc_df["image_max"] <= 1.0 + 1e-6)
    ).sum()} / {len(qc_df)}

MASK QUALITY
------------
Cases with foreground pixels:
    {(qc_df["total_foreground_pixels"] > 0).sum()} / {len(qc_df)}

OVERALL STATISTICAL QC
----------------------
PASS:
    {(qc_df["qc_status"] == "PASS").sum()}

Potential statistical outliers:
    {len(outliers)}

VOLUME DIMENSIONS
-----------------
Unique dimensions:
    {qc_df["image_volume_shape"].nunique()}

Minimum depth:
    {qc_df["depth"].min()}

Maximum depth:
    {qc_df["depth"].max()}

Median depth:
    {qc_df["depth"].median()}

VISUAL QC
---------
Representative overlay figures were generated for selected nodules.

Important:
    This step does not alter the original PNG data.
    It does not assign malignancy labels.
    It does not alter patient-level train/validation/test splits.

Outputs:
    {QC_CSV}
    {QC_FIG_DIR}
"""

with open(
    QC_SUMMARY,
    "w"
) as f:

    f.write(
        summary.strip()
    )

print(
    "\n" + summary
)

print("=" * 80)
print("STEP 26 COMPLETE")
print("=" * 80)

STEP 26 — VISUAL + STATISTICAL QUALITY CONTROL

INPUT
--------------------------------------------------------------------------------
Manifest rows:       325
Unique patients:     247
Unique CT series:    247

CALCULATING VOLUME STATISTICS
--------------------------------------------------------------------------------
Processed 25 / 325
Processed 50 / 325
Processed 75 / 325
Processed 100 / 325
Processed 125 / 325
Processed 150 / 325
Processed 175 / 325
Processed 200 / 325
Processed 225 / 325
Processed 250 / 325
Processed 275 / 325
Processed 300 / 325
Processed 325 / 325

STATISTICAL QC SUMMARY
Cases evaluated:                 325
Image volumes readable:          325
Image values within [0,1]:      325
Cases with foreground masks:     325
Overall statistical QC PASS:     325

VOLUME DIMENSIONS
--------------------------------------------------------------------------------
Depth:
count    325.000000
mean       6.153846
std        5.583902
min        1.000000
25%        3.000000
50%   

In [ ]:
# =============================================================================
# STEP 27 — BUILD PYTORCH DATASET + DATALOADERS
# 2D BASELINE USING NATIVE IMAGE SLICES
# =============================================================================

import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# -------------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------------

SLICE_MANIFEST = "/content/final_325_slice_model_manifest.csv"
FINAL_MANIFEST = "/content/final_325_ml_manifest.csv"

BATCH_SIZE = 16
NUM_WORKERS = 2
SEED = 42

print("=" * 80)
print("STEP 27 — BUILDING PYTORCH DATASET + DATALOADERS")
print("=" * 80)

# -------------------------------------------------------------------------
# Reproducibility
# -------------------------------------------------------------------------

torch.manual_seed(SEED)
np.random.seed(SEED)

# -------------------------------------------------------------------------
# Load slice manifest
# -------------------------------------------------------------------------

if not os.path.exists(SLICE_MANIFEST):
    raise FileNotFoundError(
        f"Missing slice manifest: {SLICE_MANIFEST}"
    )

slice_df = pd.read_csv(
    SLICE_MANIFEST
)

print("\nSLICE MANIFEST")
print("-" * 80)
print(f"Total image slices: {len(slice_df)}")
print(
    f"Unique nodules: "
    f"{slice_df['subset_nodule_id'].nunique()}"
)
print(
    f"Unique patients: "
    f"{slice_df['patient_id'].nunique()}"
)

# -------------------------------------------------------------------------
# Important: this is an unlabeled baseline dataset.
# -------------------------------------------------------------------------

slice_df["target"] = -1

# Explicitly confirm that there are no supervised labels.
assert (
    slice_df["target"] == -1
).all()

# -------------------------------------------------------------------------
# Dataset class
# -------------------------------------------------------------------------

class LIDCSliceDataset(Dataset):

    def __init__(self, dataframe):

        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

    def __len__(self):

        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        image_path = str(
            row["image_path"]
        )

        if not os.path.exists(image_path):

            raise FileNotFoundError(
                f"Image not found: {image_path}"
            )

        # -----------------------------------------------------------------
        # Load normalized single-slice PNG.
        # -----------------------------------------------------------------

        from PIL import Image

        with Image.open(
            image_path
        ) as img:

            image = np.asarray(
                img.convert("F"),
                dtype=np.float32
            )

        # -----------------------------------------------------------------
        # Convert to tensor.
        # Shape: [1, H, W]
        # -----------------------------------------------------------------

        image = torch.from_numpy(
            image
        ).unsqueeze(0)

        # -----------------------------------------------------------------
        # Optional mask collection.
        # We return all four reader masks separately.
        # -----------------------------------------------------------------

        masks = []

        for mask_col in [
            "mask_0_path",
            "mask_1_path",
            "mask_2_path",
            "mask_3_path"
        ]:

            mask_path = str(
                row[mask_col]
            )

            if (
                mask_path
                and mask_path.lower()
                != "nan"
                and os.path.exists(mask_path)
            ):

                with Image.open(
                    mask_path
                ) as m:

                    mask = np.asarray(
                        m.convert("L")
                    ) > 0

                mask_tensor = torch.from_numpy(
                    mask.astype(np.float32)
                )

            else:

                # Empty mask if unavailable.
                mask_tensor = torch.zeros(
                    image.shape[-2:],
                    dtype=torch.float32
                )

            masks.append(
                mask_tensor
            )

        masks = torch.stack(
            masks,
            dim=0
        )

        return {

            "image": image,

            "masks": masks,

            "target": torch.tensor(
                -1,
                dtype=torch.long
            ),

            "subset_nodule_id":
                row["subset_nodule_id"],

            "patient_id":
                row["patient_id"],

            "native_nodule_id":
                row["native_nodule_id"],

            "SeriesInstanceUID":
                row["SeriesInstanceUID"],

            "slice_index":
                int(row["slice_index"]),

            "split":
                row["split"]

        }


# -------------------------------------------------------------------------
# Split the slice table.
# -------------------------------------------------------------------------

train_df = slice_df[
    slice_df["split"] == "train"
].copy()

val_df = slice_df[
    slice_df["split"] == "val"
].copy()

test_df = slice_df[
    slice_df["split"] == "test"
].copy()

print("\nSLICE SPLITS")
print("-" * 80)

print(
    f"Train slices:      {len(train_df)}"
)

print(
    f"Validation slices: {len(val_df)}"
)

print(
    f"Test slices:       {len(test_df)}"
)

# -------------------------------------------------------------------------
# Create datasets
# -------------------------------------------------------------------------

train_dataset = LIDCSliceDataset(
    train_df
)

val_dataset = LIDCSliceDataset(
    val_df
)

test_dataset = LIDCSliceDataset(
    test_df
)

# -------------------------------------------------------------------------
# Create dataloaders
# -------------------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# -------------------------------------------------------------------------
# Test one sample
# -------------------------------------------------------------------------

sample = train_dataset[0]

print("\nSAMPLE CHECK")
print("-" * 80)

print(
    f"Image tensor shape: "
    f"{tuple(sample['image'].shape)}"
)

print(
    f"Mask tensor shape: "
    f"{tuple(sample['masks'].shape)}"
)

print(
    f"Image dtype: "
    f"{sample['image'].dtype}"
)

print(
    f"Image min: "
    f"{sample['image'].min().item():.4f}"
)

print(
    f"Image max: "
    f"{sample['image'].max().item():.4f}"
)

print(
    f"Nodule: "
    f"{sample['subset_nodule_id']}"
)

print(
    f"Patient: "
    f"{sample['patient_id']}"
)

print(
    f"Slice index: "
    f"{sample['slice_index']}"
)

# -------------------------------------------------------------------------
# Test a batch
# -------------------------------------------------------------------------

batch = next(
    iter(train_loader)
)

print("\nBATCH CHECK")
print("-" * 80)

print(
    f"Batch image shape: "
    f"{tuple(batch['image'].shape)}"
)

print(
    f"Batch mask shape: "
    f"{tuple(batch['masks'].shape)}"
)

print(
    f"Batch target shape: "
    f"{tuple(batch['target'].shape)}"
)

# -------------------------------------------------------------------------
# Patient leakage check at slice level
# -------------------------------------------------------------------------

train_patients = set(
    train_df["patient_id"].astype(str)
)

val_patients = set(
    val_df["patient_id"].astype(str)
)

test_patients = set(
    test_df["patient_id"].astype(str)
)

assert not (
    train_patients & val_patients
)

assert not (
    train_patients & test_patients
)

assert not (
    val_patients & test_patients
)

print("\nPATIENT LEAKAGE")
print("-" * 80)
print("Train/validation overlap: 0")
print("Train/test overlap:       0")
print("Validation/test overlap:  0")

print("\n" + "=" * 80)
print("STEP 27 COMPLETE")
print("=" * 80)

STEP 27 — BUILDING PYTORCH DATASET + DATALOADERS

SLICE MANIFEST
--------------------------------------------------------------------------------
Total image slices: 2000
Unique nodules: 325
Unique patients: 247

SLICE SPLITS
--------------------------------------------------------------------------------
Train slices:      1437
Validation slices: 254
Test slices:       309

SAMPLE CHECK
--------------------------------------------------------------------------------
Image tensor shape: (1, 128, 128)
Mask tensor shape: (4, 128, 128)
Image dtype: torch.float32
Image min: 0.0000
Image max: 255.0000
Nodule: nodule_001
Patient: LIDC-IDRI-0121
Slice index: 0


/tmp/ipykernel_703/4220491480.py:122: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  image = torch.from_numpy(



BATCH CHECK
--------------------------------------------------------------------------------
Batch image shape: (16, 1, 128, 128)
Batch mask shape: (16, 4, 128, 128)
Batch target shape: (16,)

PATIENT LEAKAGE
--------------------------------------------------------------------------------
Train/validation overlap: 0
Train/test overlap:       0
Validation/test overlap:  0

STEP 27 COMPLETE


In [ ]:
# =============================================================================
# STEP 27B — CORRECTED PYTORCH DATASET
# LOADS NORMALIZED [0,1] MODEL-READY DATA
# =============================================================================

import os
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader


# -------------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------------

PREPROCESS_MANIFEST = "/content/final_325_preprocessed_manifest.csv"

BATCH_SIZE = 16
NUM_WORKERS = 2
SEED = 42

print("=" * 80)
print("STEP 27B — CORRECTED PYTORCH DATASET")
print("=" * 80)


# -------------------------------------------------------------------------
# Reproducibility
# -------------------------------------------------------------------------

torch.manual_seed(SEED)
np.random.seed(SEED)


# -------------------------------------------------------------------------
# Load preprocessing manifest
# -------------------------------------------------------------------------

if not os.path.exists(PREPROCESS_MANIFEST):
    raise FileNotFoundError(
        f"Missing preprocessing manifest: {PREPROCESS_MANIFEST}"
    )

df = pd.read_csv(
    PREPROCESS_MANIFEST
)

print("\nINPUT")
print("-" * 80)

print(
    f"Nodule rows: {len(df)}"
)

print(
    f"Unique patients: "
    f"{df['patient_id'].nunique()}"
)

print(
    f"Unique splits: "
    f"{sorted(df['split'].unique())}"
)


# -------------------------------------------------------------------------
# Verify expected cohort
# -------------------------------------------------------------------------

assert len(df) == 325

assert (
    df["subset_nodule_id"].nunique()
    == 325
)

assert (
    df["patient_id"].nunique()
    == 247
)

assert (
    df["split"].isin(
        ["train", "val", "test"]
    ).all()
)


# =============================================================================
# DATASET
# =============================================================================

class LIDCNormalizedSliceDataset(Dataset):

    def __init__(self, dataframe):

        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        # Cache normalized volumes so repeated slices from the same
        # nodule can be loaded efficiently.

        self.volume_cache = {}
        self.mask_cache = {}

    def __len__(self):

        total = 0

        for _, row in self.df.iterrows():

            volume_path = str(
                row["image_volume_path"]
            )

            if not os.path.exists(volume_path):
                raise FileNotFoundError(
                    f"Missing normalized volume: {volume_path}"
                )

            volume = np.load(
                volume_path,
                mmap_mode="r"
            )

            total += volume.shape[0]

        return total

    def _build_index(self):

        index = []

        for row_index, row in self.df.iterrows():

            volume_path = str(
                row["image_volume_path"]
            )

            volume = np.load(
                volume_path,
                mmap_mode="r"
            )

            for slice_index in range(
                volume.shape[0]
            ):

                index.append(
                    (
                        row_index,
                        slice_index
                    )
                )

        return index

    def _load_volume(self, path):

        if path not in self.volume_cache:

            volume = np.load(
                path
            ).astype(
                np.float32
            )

            self.volume_cache[path] = volume

        return self.volume_cache[path]

    def _load_mask_volume(
        self,
        path
    ):

        if not path:
            return None

        if path.lower() == "nan":
            return None

        if not os.path.exists(path):
            return None

        if path not in self.mask_cache:

            mask = np.load(
                path
            ).astype(
                np.float32
            )

            self.mask_cache[path] = mask

        return self.mask_cache[path]

    def __getitem__(self, index):

        if not hasattr(
            self,
            "_index"
        ):

            self._index = (
                self._build_index()
            )

        row_index, slice_index = (
            self._index[index]
        )

        row = self.df.iloc[
            row_index
        ]

        # -------------------------------------------------------------
        # Load normalized image volume
        # -------------------------------------------------------------

        volume_path = str(
            row["image_volume_path"]
        )

        volume = self._load_volume(
            volume_path
        )

        image = np.array(
            volume[slice_index],
            dtype=np.float32,
            copy=True
        )

        # -------------------------------------------------------------
        # Image sanity check
        # -------------------------------------------------------------

        if image.min() < -1e-5:
            raise ValueError(
                f"Image contains values < 0: "
                f"{image.min()}"
            )

        if image.max() > 1.00001:
            raise ValueError(
                f"Image contains values > 1: "
                f"{image.max()}"
            )

        image_tensor = torch.from_numpy(
            image.copy()
        ).unsqueeze(0)

        # -------------------------------------------------------------
        # Load masks
        # -------------------------------------------------------------

        masks = []

        for mask_num in range(4):

            path = str(
                row[
                    f"mask_{mask_num}_path"
                ]
            )

            mask_volume = (
                self._load_mask_volume(path)
            )

            if (
                mask_volume is not None
                and slice_index
                < mask_volume.shape[0]
            ):

                mask_slice = np.array(
                    mask_volume[slice_index],
                    dtype=np.float32,
                    copy=True
                )

            else:

                mask_slice = np.zeros(
                    image.shape,
                    dtype=np.float32
                )

            masks.append(
                torch.from_numpy(
                    mask_slice.copy()
                )
            )

        masks_tensor = torch.stack(
            masks,
            dim=0
        )

        return {

            "image":
                image_tensor,

            "masks":
                masks_tensor,

            "patient_id":
                str(row["patient_id"]),

            "subset_nodule_id":
                str(row["subset_nodule_id"]),

            "native_nodule_id":
                str(row["native_nodule_id"]),

            "SeriesInstanceUID":
                str(row["SeriesInstanceUID"]),

            "slice_index":
                int(slice_index),

            "split":
                str(row["split"]),

            # No malignancy target.
            "target":
                torch.tensor(
                    -1,
                    dtype=torch.long
                )

        }


# =============================================================================
# BUILD SPLITS
# =============================================================================

train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "val"
].copy()

test_df = df[
    df["split"] == "test"
].copy()


# =============================================================================
# CREATE DATASETS
# =============================================================================

train_dataset = LIDCNormalizedSliceDataset(
    train_df
)

val_dataset = LIDCNormalizedSliceDataset(
    val_df
)

test_dataset = LIDCNormalizedSliceDataset(
    test_df
)


# =============================================================================
# CREATE LOADERS
# =============================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)


# =============================================================================
# TEST DATASET
# =============================================================================

sample = train_dataset[0]

print("\nSAMPLE CHECK")
print("-" * 80)

print(
    f"Image shape: "
    f"{tuple(sample['image'].shape)}"
)

print(
    f"Mask shape: "
    f"{tuple(sample['masks'].shape)}"
)

print(
    f"Image dtype: "
    f"{sample['image'].dtype}"
)

print(
    f"Image min: "
    f"{sample['image'].min().item():.6f}"
)

print(
    f"Image max: "
    f"{sample['image'].max().item():.6f}"
)

print(
    f"Image mean: "
    f"{sample['image'].mean().item():.6f}"
)

print(
    f"Nodule: "
    f"{sample['subset_nodule_id']}"
)

print(
    f"Patient: "
    f"{sample['patient_id']}"
)

print(
    f"Slice index: "
    f"{sample['slice_index']}"
)


# =============================================================================
# ASSERT NORMALIZATION
# =============================================================================

assert (
    sample["image"].min().item()
    >= -1e-5
)

assert (
    sample["image"].max().item()
    <= 1.00001
)


# =============================================================================
# TEST BATCH
# =============================================================================

batch = next(
    iter(train_loader)
)

print("\nBATCH CHECK")
print("-" * 80)

print(
    f"Batch image shape: "
    f"{tuple(batch['image'].shape)}"
)

print(
    f"Batch mask shape: "
    f"{tuple(batch['masks'].shape)}"
)

print(
    f"Batch image min: "
    f"{batch['image'].min().item():.6f}"
)

print(
    f"Batch image max: "
    f"{batch['image'].max().item():.6f}"
)

assert (
    batch["image"].min().item()
    >= -1e-5
)

assert (
    batch["image"].max().item()
    <= 1.00001
)


# =============================================================================
# PATIENT LEAKAGE CHECK
# =============================================================================

train_patients = set(
    train_df["patient_id"].astype(str)
)

val_patients = set(
    val_df["patient_id"].astype(str)
)

test_patients = set(
    test_df["patient_id"].astype(str)
)

assert not (
    train_patients
    & val_patients
)

assert not (
    train_patients
    & test_patients
)

assert not (
    val_patients
    & test_patients
)


# =============================================================================
# REPORT
# =============================================================================

print("\n" + "=" * 80)
print("STEP 27B RESULTS")
print("=" * 80)

print(
    f"Training nodules:   {len(train_df)}"
)

print(
    f"Validation nodules: {len(val_df)}"
)

print(
    f"Test nodules:       {len(test_df)}"
)

print(
    f"Training slices:    {len(train_dataset)}"
)

print(
    f"Validation slices:  {len(val_dataset)}"
)

print(
    f"Test slices:        {len(test_dataset)}"
)

print(
    "\nNormalized image range verified: [0, 1]"
)

print(
    "Patient leakage: 0"
)

print(
    "Native malignancy labels: NONE"
)

print("=" * 80)
print("STEP 27B COMPLETE")
print("=" * 80)

STEP 27B — CORRECTED PYTORCH DATASET

INPUT
--------------------------------------------------------------------------------
Nodule rows: 325
Unique patients: 247
Unique splits: ['test', 'train', 'val']

SAMPLE CHECK
--------------------------------------------------------------------------------
Image shape: (1, 128, 128)
Mask shape: (4, 128, 128)
Image dtype: torch.float32
Image min: 0.000000
Image max: 1.000000
Image mean: 0.438749
Nodule: nodule_001
Patient: LIDC-IDRI-0121
Slice index: 0

BATCH CHECK
--------------------------------------------------------------------------------
Batch image shape: (16, 1, 128, 128)
Batch mask shape: (16, 4, 128, 128)
Batch image min: 0.000000
Batch image max: 1.000000

STEP 27B RESULTS
Training nodules:   235
Validation nodules: 44
Test nodules:       46
Training slices:    1437
Validation slices:  254
Test slices:        309

Normalized image range verified: [0, 1]
Patient leakage: 0
Native malignancy labels: NONE
STEP 27B COMPLETE


In [ ]:
# =============================================================================
# STEP 28 — BASELINE 2D U-NET SEGMENTATION MODEL
# =============================================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

# -------------------------------------------------------------------------
# Reproducibility
# -------------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("STEP 28 — BASELINE 2D U-NET SEGMENTATION")
print("=" * 80)

print(f"\nDevice: {DEVICE}")


# =============================================================================
# REUSE THE STEP 27B DATAFRAMES
# =============================================================================

# Step 27B created:
#   train_df
#   val_df
#   test_df
#
# We verify they are available.

required_objects = [
    "train_df",
    "val_df",
    "test_df"
]

for name in required_objects:

    if name not in globals():

        raise RuntimeError(
            f"{name} was not found. "
            "Run Step 27B first."
        )


# =============================================================================
# CONSENSUS MASK DATASET
# =============================================================================

class LIDCConsensusSegmentationDataset(Dataset):

    def __init__(
        self,
        dataframe,
        consensus_threshold=0.5
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.consensus_threshold = (
            consensus_threshold
        )

        self.index = []

        # -------------------------------------------------------------
        # Create slice-level index.
        # -------------------------------------------------------------

        for row_index, row in self.df.iterrows():

            image_path = str(
                row["image_volume_path"]
            )

            if not os.path.exists(
                image_path
            ):
                continue

            volume = np.load(
                image_path,
                mmap_mode="r"
            )

            for slice_index in range(
                volume.shape[0]
            ):

                self.index.append(
                    (
                        row_index,
                        slice_index
                    )
                )

    def __len__(self):

        return len(self.index)

    def _load_mask(
        self,
        row,
        slice_index,
        mask_number
    ):

        column = (
            f"mask_{mask_number}_path"
        )

        if column not in row.index:
            return None

        path = str(
            row[column]
        )

        if (
            path == ""
            or path.lower() == "nan"
            or not os.path.exists(path)
        ):
            return None

        mask_volume = np.load(
            path,
            mmap_mode="r"
        )

        if (
            slice_index
            >= mask_volume.shape[0]
        ):
            return None

        mask = np.array(
            mask_volume[slice_index],
            dtype=np.float32,
            copy=True
        )

        return mask > 0

    def __getitem__(
        self,
        index
    ):

        row_index, slice_index = (
            self.index[index]
        )

        row = self.df.iloc[
            row_index
        ]

        # -------------------------------------------------------------
        # Load normalized image
        # -------------------------------------------------------------

        image_volume = np.load(
            str(
                row["image_volume_path"]
            ),
            mmap_mode="r"
        )

        image = np.array(
            image_volume[slice_index],
            dtype=np.float32,
            copy=True
        )

        image = np.clip(
            image,
            0.0,
            1.0
        )

        # -------------------------------------------------------------
        # Collect available reader masks
        # -------------------------------------------------------------

        available_masks = []

        for mask_number in range(4):

            mask = self._load_mask(
                row,
                slice_index,
                mask_number
            )

            if mask is not None:

                available_masks.append(
                    mask.astype(
                        np.float32
                    )
                )

        # -------------------------------------------------------------
        # Build consensus mask
        #
        # >=50% of available masks must contain the pixel.
        # With 1 reader, that reader becomes the target.
        # With 2 readers, both readers are needed.
        # With 3 or 4 readers, majority agreement is required.
        # -------------------------------------------------------------

        if available_masks:

            stacked = np.stack(
                available_masks,
                axis=0
            )

            consensus = (
                stacked.mean(axis=0)
                >= self.consensus_threshold
            ).astype(
                np.float32
            )

        else:

            consensus = np.zeros(
                image.shape,
                dtype=np.float32
            )

        # -------------------------------------------------------------
        # Convert to tensors
        # -------------------------------------------------------------

        image_tensor = torch.from_numpy(
            image
        ).unsqueeze(0)

        mask_tensor = torch.from_numpy(
            consensus
        ).unsqueeze(0)

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "patient_id": str(
                row["patient_id"]
            ),
            "subset_nodule_id": str(
                row["subset_nodule_id"]
            ),
            "slice_index": int(
                slice_index
            )
        }


# =============================================================================
# BUILD DATASETS
# =============================================================================

train_seg_dataset = (
    LIDCConsensusSegmentationDataset(
        train_df
    )
)

val_seg_dataset = (
    LIDCConsensusSegmentationDataset(
        val_df
    )
)

test_seg_dataset = (
    LIDCConsensusSegmentationDataset(
        test_df
    )
)


# =============================================================================
# BUILD LOADERS
# =============================================================================

SEG_BATCH_SIZE = 16

train_seg_loader = DataLoader(
    train_seg_dataset,
    batch_size=SEG_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_seg_loader = DataLoader(
    val_seg_dataset,
    batch_size=SEG_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_seg_loader = DataLoader(
    test_seg_dataset,
    batch_size=SEG_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


# =============================================================================
# U-NET
# =============================================================================

class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )

        )

    def forward(self, x):

        return self.block(x)


class UNet2D(nn.Module):

    def __init__(
        self,
        in_channels=1,
        out_channels=1
    ):

        super().__init__()

        self.enc1 = DoubleConv(
            in_channels,
            32
        )

        self.enc2 = DoubleConv(
            32,
            64
        )

        self.enc3 = DoubleConv(
            64,
            128
        )

        self.bottleneck = DoubleConv(
            128,
            256
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2
        )

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        self.out_conv = nn.Conv2d(
            32,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        b = self.bottleneck(
            self.pool(e3)
        )

        # Decoder
        d3 = self.up3(b)

        d3 = torch.cat(
            [d3, e3],
            dim=1
        )

        d3 = self.dec3(
            d3
        )

        d2 = self.up2(d3)

        d2 = torch.cat(
            [d2, e2],
            dim=1
        )

        d2 = self.dec2(
            d2
        )

        d1 = self.up1(d2)

        d1 = torch.cat(
            [d1, e1],
            dim=1
        )

        d1 = self.dec1(
            d1
        )

        return self.out_conv(d1)


# =============================================================================
# DICE LOSS
# =============================================================================

def dice_score(
    prediction,
    target,
    smooth=1e-6
):

    prediction = (
        torch.sigmoid(
            prediction
        )
        > 0.5
    ).float()

    intersection = (
        prediction
        * target
    ).sum(
        dim=(1, 2, 3)
    )

    denominator = (
        prediction.sum(
            dim=(1, 2, 3)
        )
        +
        target.sum(
            dim=(1, 2, 3)
        )
    )

    dice = (
        (2.0 * intersection + smooth)
        /
        (denominator + smooth)
    )

    return dice.mean()


def dice_loss(
    prediction,
    target,
    smooth=1e-6
):

    probabilities = torch.sigmoid(
        prediction
    )

    intersection = (
        probabilities
        * target
    ).sum(
        dim=(1, 2, 3)
    )

    denominator = (
        probabilities.sum(
            dim=(1, 2, 3)
        )
        +
        target.sum(
            dim=(1, 2, 3)
        )
    )

    dice = (
        (2.0 * intersection + smooth)
        /
        (denominator + smooth)
    )

    return (
        1.0
        - dice.mean()
    )


# =============================================================================
# MODEL
# =============================================================================

model = UNet2D().to(
    DEVICE
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

bce_loss = nn.BCEWithLogitsLoss()


def combined_loss(
    logits,
    target
):

    return (
        0.5 * bce_loss(
            logits,
            target
        )
        +
        0.5 * dice_loss(
            logits,
            target
        )
    )


# =============================================================================
# TRAINING FUNCTIONS
# =============================================================================

def train_one_epoch(
    model,
    loader,
    optimizer
):

    model.train()

    total_loss = 0.0
    total_dice = 0.0
    batches = 0

    for batch in loader:

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        masks = batch[
            "mask"
        ].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad()

        logits = model(
            images
        )

        loss = combined_loss(
            logits,
            masks
        )

        loss.backward()

        optimizer.step()

        batch_dice = dice_score(
            logits.detach(),
            masks
        )

        total_loss += (
            loss.item()
        )

        total_dice += (
            batch_dice.item()
        )

        batches += 1

    return (
        total_loss / max(batches, 1),
        total_dice / max(batches, 1)
    )


@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    batches = 0

    for batch in loader:

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        masks = batch[
            "mask"
        ].to(
            DEVICE,
            non_blocking=True
        )

        logits = model(
            images
        )

        loss = combined_loss(
            logits,
            masks
        )

        batch_dice = dice_score(
            logits,
            masks
        )

        total_loss += (
            loss.item()
        )

        total_dice += (
            batch_dice.item()
        )

        batches += 1

    return (
        total_loss / max(batches, 1),
        total_dice / max(batches, 1)
    )


# =============================================================================
# QUICK DATASET TEST
# =============================================================================

print("\nDATASET CHECK")
print("-" * 80)

sample = train_seg_dataset[0]

print(
    f"Image shape: "
    f"{tuple(sample['image'].shape)}"
)

print(
    f"Mask shape: "
    f"{tuple(sample['mask'].shape)}"
)

print(
    f"Image range: "
    f"{sample['image'].min().item():.4f} "
    f"to "
    f"{sample['image'].max().item():.4f}"
)

print(
    f"Mask values: "
    f"{torch.unique(sample['mask']).tolist()}"
)

# Batch check
sample_batch = next(
    iter(train_seg_loader)
)

print(
    f"Batch image shape: "
    f"{tuple(sample_batch['image'].shape)}"
)

print(
    f"Batch mask shape: "
    f"{tuple(sample_batch['mask'].shape)}"
)


# =============================================================================
# TRAIN
# =============================================================================

EPOCHS = 10

best_val_dice = -1.0

MODEL_OUTPUT = (
    "/content/baseline_unet_325_best.pt"
)

history = []

print("\n" + "=" * 80)
print("TRAINING")
print("=" * 80)

for epoch in range(
    1,
    EPOCHS + 1
):

    train_loss, train_dice = (
        train_one_epoch(
            model,
            train_seg_loader,
            optimizer
        )
    )

    val_loss, val_dice = (
        evaluate(
            model,
            val_seg_loader
        )
    )

    history.append({

        "epoch": epoch,

        "train_loss":
            train_loss,

        "train_dice":
            train_dice,

        "val_loss":
            val_loss,

        "val_dice":
            val_dice

    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Dice: {train_dice:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f}"
    )

    if val_dice > best_val_dice:

        best_val_dice = val_dice

        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),

                "epoch":
                    epoch,

                "val_dice":
                    val_dice

            },
            MODEL_OUTPUT
        )

        print(
            f"  ✓ Best model saved "
            f"(val Dice={val_dice:.4f})"
        )


# =============================================================================
# LOAD BEST MODEL
# =============================================================================

checkpoint = torch.load(
    MODEL_OUTPUT,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


# =============================================================================
# TEST EVALUATION
# =============================================================================

test_loss, test_dice = evaluate(
    model,
    test_seg_loader
)

history_df = pd.DataFrame(
    history
)

HISTORY_CSV = (
    "/content/step28_training_history.csv"
)

history_df.to_csv(
    HISTORY_CSV,
    index=False
)


# =============================================================================
# FINAL RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 28 RESULTS")
print("=" * 80)

print(
    f"Best validation Dice: "
    f"{best_val_dice:.4f}"
)

print(
    f"Test loss:            "
    f"{test_loss:.4f}"
)

print(
    f"Test Dice:            "
    f"{test_dice:.4f}"
)

print(
    f"\nBest model saved to:"
)

print(
    MODEL_OUTPUT
)

print(
    "\nTraining history saved to:"
)

print(
    HISTORY_CSV
)

print("\nIMPORTANT:")
print(
    "This is a segmentation baseline."
)

print(
    "No malignancy labels were used."
)

print(
    "The model was trained using consensus masks derived "
    "from the native mask folders."
)

print("=" * 80)
print("STEP 28 COMPLETE")
print("=" * 80)

STEP 28 — BASELINE 2D U-NET SEGMENTATION

Device: cpu

DATASET CHECK
--------------------------------------------------------------------------------
Image shape: (1, 128, 128)
Mask shape: (1, 128, 128)
Image range: 0.0000 to 1.0000
Mask values: [0.0, 1.0]
Batch image shape: (16, 1, 128, 128)
Batch mask shape: (16, 1, 128, 128)

TRAINING
Epoch 01/10 | Train Loss: 0.6548 | Train Dice: 0.2214 | Val Loss: 0.5925 | Val Dice: 0.2656
  ✓ Best model saved (val Dice=0.2656)
Epoch 02/10 | Train Loss: 0.5582 | Train Dice: 0.2741 | Val Loss: 0.5293 | Val Dice: 0.2199
Epoch 03/10 | Train Loss: 0.5037 | Train Dice: 0.2869 | Val Loss: 0.4476 | Val Dice: 0.4102
  ✓ Best model saved (val Dice=0.4102)
Epoch 04/10 | Train Loss: 0.4560 | Train Dice: 0.3419 | Val Loss: 0.4120 | Val Dice: 0.3406
Epoch 05/10 | Train Loss: 0.4018 | Train Dice: 0.4036 | Val Loss: 0.3930 | Val Dice: 0.3556
Epoch 06/10 | Train Loss: 0.3569 | Train Dice: 0.4556 | Val Loss: 0.3338 | Val Dice: 0.4571
  ✓ Best model saved (val Dice

In [ ]:
# =============================================================================
# STEP 29 — COMPREHENSIVE FINAL STRUCTURAL DATASET AUDIT
# =============================================================================

import os
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# INPUTS
# -------------------------------------------------------------------------

FINAL_MANIFEST = "/content/final_325_ml_manifest.csv"
MASTER_MANIFEST = "/content/final_325_master_manifest.csv"
PROCESSED_ROOT = "/content/processed_325_nodules"
MODEL_READY_ROOT = "/content/model_ready_325_nodules"

OUTPUT_AUDIT_CSV = "/content/step29_final_structural_audit.csv"
OUTPUT_SUMMARY_TXT = "/content/step29_final_structural_audit_summary.txt"

print("=" * 80)
print("STEP 29 — COMPREHENSIVE FINAL STRUCTURAL DATASET AUDIT")
print("=" * 80)

# =============================================================================
# 1. CHECK INPUT FILES
# =============================================================================

required_files = [
    FINAL_MANIFEST,
    MASTER_MANIFEST
]

for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

if not os.path.isdir(PROCESSED_ROOT):
    raise FileNotFoundError(
        f"Processed dataset directory missing: {PROCESSED_ROOT}"
    )

print("\nINPUT FILES")
print("-" * 80)

print(f"Final ML manifest:     {FINAL_MANIFEST}")
print(f"Master manifest:       {MASTER_MANIFEST}")
print(f"Processed dataset:     {PROCESSED_ROOT}")

# =============================================================================
# 2. LOAD MANIFESTS
# =============================================================================

final_df = pd.read_csv(FINAL_MANIFEST)
master_df = pd.read_csv(MASTER_MANIFEST)

print("\nMANIFEST SUMMARY")
print("-" * 80)

print(
    f"Final manifest rows:     {len(final_df)}"
)

print(
    f"Master manifest rows:    {len(master_df)}"
)

# =============================================================================
# 3. BASIC COHORT VALIDATION
# =============================================================================

assert len(final_df) == 325, (
    f"Expected 325 final rows, found {len(final_df)}"
)

assert (
    final_df["subset_nodule_id"].nunique()
    == 325
), "Subset IDs are not unique."

assert (
    final_df["patient_id"].nunique()
    == 247
), "Unexpected patient count."

assert (
    final_df["SeriesInstanceUID"].nunique()
    == 247
), "Unexpected SeriesInstanceUID count."

assert (
    "nodule_029"
    not in set(final_df["subset_nodule_id"])
), "nodule_029 is present."

assert (
    "nodule_085"
    not in set(final_df["subset_nodule_id"])
), "nodule_085 is present."

print(
    "✓ Cohort size = 325"
)

print(
    "✓ Unique subset IDs = 325"
)

print(
    "✓ Unique patients = 247"
)

print(
    "✓ Unique SeriesInstanceUIDs = 247"
)

print(
    "✓ Excluded nodules absent"
)

# =============================================================================
# 4. SPLIT VALIDATION
# =============================================================================

print("\nPATIENT SPLIT AUDIT")
print("-" * 80)

train_patients = set(
    final_df.loc[
        final_df["split"] == "train",
        "patient_id"
    ].astype(str)
)

val_patients = set(
    final_df.loc[
        final_df["split"] == "val",
        "patient_id"
    ].astype(str)
)

test_patients = set(
    final_df.loc[
        final_df["split"] == "test",
        "patient_id"
    ].astype(str)
)

train_nodes = int(
    (final_df["split"] == "train").sum()
)

val_nodes = int(
    (final_df["split"] == "val").sum()
)

test_nodes = int(
    (final_df["split"] == "test").sum()
)

train_val_overlap = train_patients & val_patients
train_test_overlap = train_patients & test_patients
val_test_overlap = val_patients & test_patients

print(
    f"Train:       {train_nodes} nodules / "
    f"{len(train_patients)} patients"
)

print(
    f"Validation:  {val_nodes} nodules / "
    f"{len(val_patients)} patients"
)

print(
    f"Test:        {test_nodes} nodules / "
    f"{len(test_patients)} patients"
)

print(
    f"Train/Val patient overlap: "
    f"{len(train_val_overlap)}"
)

print(
    f"Train/Test patient overlap: "
    f"{len(train_test_overlap)}"
)

print(
    f"Val/Test patient overlap: "
    f"{len(val_test_overlap)}"
)

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print(
    "✓ Zero patient leakage"
)

# =============================================================================
# 5. BUILD CASE-LEVEL AUDIT
# =============================================================================

print("\nAUDITING EACH OF 325 NODULES")
print("-" * 80)

audit_rows = []

for counter, (_, row) in enumerate(
    final_df.iterrows(),
    start=1
):

    subset_id = str(
        row["subset_nodule_id"]
    )

    patient_id = str(
        row["patient_id"]
    )

    native_id = str(
        row["native_nodule_id"]
    )

    split = str(
        row["split"]
    )

    series_uid = str(
        row["SeriesInstanceUID"]
    )

    # ---------------------------------------------------------------------
    # Original processed folder
    # ---------------------------------------------------------------------

    processed_dir = os.path.join(
        PROCESSED_ROOT,
        subset_id
    )

    # ---------------------------------------------------------------------
    # Alternative possibility: folder may use native ID.
    # Check both.
    # ---------------------------------------------------------------------

    if not os.path.isdir(processed_dir):

        alternative_dir = os.path.join(
            PROCESSED_ROOT,
            patient_id,
            native_id
        )

        if os.path.isdir(alternative_dir):

            processed_dir = alternative_dir

    processed_exists = os.path.isdir(
        processed_dir
    )

    # ---------------------------------------------------------------------
    # Image directory
    # ---------------------------------------------------------------------

    image_dir = os.path.join(
        processed_dir,
        "images"
    )

    image_files = []

    if os.path.isdir(image_dir):

        image_files = sorted([
            f
            for f in os.listdir(image_dir)
            if f.lower().endswith(".png")
        ])

    # ---------------------------------------------------------------------
    # Mask directories
    # ---------------------------------------------------------------------

    mask_files = {}

    for mask_num in range(4):

        mask_dir = os.path.join(
            processed_dir,
            f"mask-{mask_num}"
        )

        if os.path.isdir(mask_dir):

            mask_files[mask_num] = sorted([
                f
                for f in os.listdir(mask_dir)
                if f.lower().endswith(".png")
            ])

        else:

            mask_files[mask_num] = []

    # ---------------------------------------------------------------------
    # Model-ready directory
    # ---------------------------------------------------------------------

    model_dir = os.path.join(
        MODEL_READY_ROOT,
        subset_id
    )

    model_ready_exists = os.path.isdir(
        model_dir
    )

    normalized_volume_path = os.path.join(
        model_dir,
        "images_normalized.npy"
    )

    normalized_exists = os.path.exists(
        normalized_volume_path
    )

    # ---------------------------------------------------------------------
    # Read image dimensions
    # ---------------------------------------------------------------------

    image_dimension_consistent = True
    image_dimensions = None

    if image_files:

        from PIL import Image

        for filename in image_files:

            path = os.path.join(
                image_dir,
                filename
            )

            try:

                with Image.open(path) as img:

                    dims = tuple(
                        img.size
                    )

                if image_dimensions is None:

                    image_dimensions = dims

                elif dims != image_dimensions:

                    image_dimension_consistent = False

            except Exception:

                image_dimension_consistent = False

    # ---------------------------------------------------------------------
    # Read mask dimensions
    # ---------------------------------------------------------------------

    mask_dimension_consistent = True
    mask_dimensions = None

    for mask_num in range(4):

        mask_dir = os.path.join(
            processed_dir,
            f"mask-{mask_num}"
        )

        for filename in mask_files[mask_num]:

            path = os.path.join(
                mask_dir,
                filename
            )

            try:

                with Image.open(path) as img:

                    dims = tuple(
                        img.size
                    )

                if mask_dimensions is None:

                    mask_dimensions = dims

                elif dims != mask_dimensions:

                    mask_dimension_consistent = False

            except Exception:

                mask_dimension_consistent = False

    # ---------------------------------------------------------------------
    # Mask depth vs image depth
    # ---------------------------------------------------------------------

    image_count = len(image_files)

    mask_counts = {
        m: len(mask_files[m])
        for m in range(4)
    }

    mask_depth_matches = {}

    for m in range(4):

        mask_depth_matches[m] = (
            mask_counts[m] == image_count
            or mask_counts[m] == 0
        )

    # ---------------------------------------------------------------------
    # Foreground statistics
    # ---------------------------------------------------------------------

    foreground_counts = {}

    for mask_num in range(4):

        total_foreground = 0
        active_slices = 0

        mask_dir = os.path.join(
            processed_dir,
            f"mask-{mask_num}"
        )

        for filename in mask_files[mask_num]:

            path = os.path.join(
                mask_dir,
                filename
            )

            try:

                from PIL import Image

                with Image.open(path) as img:

                    arr = np.asarray(
                        img.convert("L")
                    )

                fg = int(
                    np.count_nonzero(
                        arr > 0
                    )
                )

                total_foreground += fg

                if fg > 0:
                    active_slices += 1

            except Exception:

                pass

        foreground_counts[mask_num] = (
            total_foreground,
            active_slices
        )

    total_foreground = sum(
        foreground_counts[m][0]
        for m in range(4)
    )

    active_mask_dirs = sum(
        1
        for m in range(4)
        if foreground_counts[m][0] > 0
    )

    # ---------------------------------------------------------------------
    # Load normalized volume shape/range
    # ---------------------------------------------------------------------

    normalized_shape = ""
    normalized_min = np.nan
    normalized_max = np.nan
    normalized_valid = False

    if normalized_exists:

        try:

            volume = np.load(
                normalized_volume_path,
                mmap_mode="r"
            )

            normalized_shape = str(
                tuple(volume.shape)
            )

            normalized_min = float(
                volume.min()
            )

            normalized_max = float(
                volume.max()
            )

            normalized_valid = (
                volume.ndim == 3
                and normalized_min >= -1e-6
                and normalized_max <= 1.00001
                and volume.shape[0] == image_count
            )

        except Exception:

            normalized_valid = False

    # ---------------------------------------------------------------------
    # Overall case status
    # ---------------------------------------------------------------------

    case_pass = (
        processed_exists
        and image_count > 0
        and image_dimension_consistent
        and mask_dimension_consistent
        and all(
            mask_depth_matches.values()
        )
        and total_foreground > 0
        and normalized_exists
        and normalized_valid
    )

    audit_rows.append({

        "subset_nodule_id":
            subset_id,

        "patient_id":
            patient_id,

        "native_nodule_id":
            native_id,

        "SeriesInstanceUID":
            series_uid,

        "split":
            split,

        "processed_directory_exists":
            processed_exists,

        "image_count":
            image_count,

        "image_dimensions":
            str(image_dimensions),

        "image_dimension_consistent":
            image_dimension_consistent,

        "mask_0_count":
            mask_counts[0],

        "mask_1_count":
            mask_counts[1],

        "mask_2_count":
            mask_counts[2],

        "mask_3_count":
            mask_counts[3],

        "mask_dimensions":
            str(mask_dimensions),

        "mask_dimension_consistent":
            mask_dimension_consistent,

        "mask_0_depth_match":
            mask_depth_matches[0],

        "mask_1_depth_match":
            mask_depth_matches[1],

        "mask_2_depth_match":
            mask_depth_matches[2],

        "mask_3_depth_match":
            mask_depth_matches[3],

        "mask_0_foreground_pixels":
            foreground_counts[0][0],

        "mask_1_foreground_pixels":
            foreground_counts[1][0],

        "mask_2_foreground_pixels":
            foreground_counts[2][0],

        "mask_3_foreground_pixels":
            foreground_counts[3][0],

        "mask_0_foreground_slices":
            foreground_counts[0][1],

        "mask_1_foreground_slices":
            foreground_counts[1][1],

        "mask_2_foreground_slices":
            foreground_counts[2][1],

        "mask_3_foreground_slices":
            foreground_counts[3][1],

        "active_mask_directories":
            active_mask_dirs,

        "total_foreground_pixels":
            total_foreground,

        "model_ready_directory_exists":
            model_ready_exists,

        "normalized_volume_exists":
            normalized_exists,

        "normalized_volume_shape":
            normalized_shape,

        "normalized_min":
            normalized_min,

        "normalized_max":
            normalized_max,

        "normalized_volume_valid":
            normalized_valid,

        "case_qc_status":
            "PASS" if case_pass else "FAIL"

    })

    if counter % 25 == 0:

        print(
            f"Audited "
            f"{counter} / {len(final_df)}"
        )


audit_df = pd.DataFrame(
    audit_rows
)

# =============================================================================
# 6. SAVE AUDIT
# =============================================================================

audit_df.to_csv(
    OUTPUT_AUDIT_CSV,
    index=False
)

# =============================================================================
# 7. FINAL COUNTS
# =============================================================================

pass_count = int(
    (
        audit_df["case_qc_status"]
        == "PASS"
    ).sum()
)

fail_count = int(
    (
        audit_df["case_qc_status"]
        == "FAIL"
    ).sum()
)

missing_processed = int(
    (
        ~audit_df[
            "processed_directory_exists"
        ]
    ).sum()
)

missing_normalized = int(
    (
        ~audit_df[
            "normalized_volume_exists"
        ]
    ).sum()
)

dimension_failures = int(
    (
        ~audit_df[
            "image_dimension_consistent"
        ]
        |
        ~audit_df[
            "mask_dimension_consistent"
        ]
    ).sum()
)

depth_failures = int(
    (
        ~audit_df[
            "mask_0_depth_match"
        ]
        |
        ~audit_df[
            "mask_1_depth_match"
        ]
        |
        ~audit_df[
            "mask_2_depth_match"
        ]
        |
        ~audit_df[
            "mask_3_depth_match"
        ]
    ).sum()
)

# =============================================================================
# 8. REPORT FAILURES
# =============================================================================

failed_cases = audit_df[
    audit_df["case_qc_status"] == "FAIL"
].copy()

print("\n" + "=" * 80)
print("STEP 29 — FINAL AUDIT RESULTS")
print("=" * 80)

print(
    f"\nTotal cases audited:              {len(audit_df)}"
)

print(
    f"Cases passing all checks:        {pass_count}"
)

print(
    f"Cases failing one or more checks:{fail_count}"
)

print(
    f"Missing processed directories:    {missing_processed}"
)

print(
    f"Missing normalized volumes:       {missing_normalized}"
)

print(
    f"Dimension failures:               {dimension_failures}"
)

print(
    f"Image/mask depth failures:        {depth_failures}"
)

print(
    f"Total image PNG files:             "
    f"{audit_df['image_count'].sum()}"
)

print(
    f"Total mask PNG files:              "
    f"{audit_df[['mask_0_count','mask_1_count','mask_2_count','mask_3_count']].sum().sum()}"
)

print(
    f"Total foreground mask pixels:      "
    f"{audit_df['total_foreground_pixels'].sum()}"
)

# =============================================================================
# 9. FAIL CASE TABLE
# =============================================================================

if not failed_cases.empty:

    print(
        "\nCASES REQUIRING INVESTIGATION"
    )

    print("-" * 80)

    print(
        failed_cases[
            [
                "subset_nodule_id",
                "patient_id",
                "native_nodule_id",
                "image_count",
                "image_dimensions",
                "mask_dimension_consistent",
                "normalized_volume_exists",
                "normalized_volume_valid",
                "case_qc_status"
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "\n✓ No structural QC failures."
    )

# =============================================================================
# 10. VERIFY LABEL STATUS
# =============================================================================

print("\nLABEL / PROVENANCE CHECK")
print("-" * 80)

if "native_label" in final_df.columns:

    native_labels = int(
        final_df[
            "native_label"
        ].notna().sum()
    )

else:

    native_labels = 0

print(
    f"Native malignancy labels assigned: "
    f"{native_labels}"
)

assert native_labels == 0

print(
    "✓ No unproven native malignancy labels."
)

# =============================================================================
# 11. FINAL ASSERTIONS
# =============================================================================

assert len(audit_df) == 325

assert (
    audit_df[
        "subset_nodule_id"
    ].nunique()
    == 325
)

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

# =============================================================================
# 12. WRITE SUMMARY
# =============================================================================

final_status = (
    "PASS"
    if fail_count == 0
    else "FAIL"
)

summary = f"""
===============================================================================
STEP 29 — FINAL STRUCTURAL DATASET AUDIT
===============================================================================

COHORT
-------------------------------------------------------------------------------
Eligible nodules:                     325
Unique patients:                      {final_df["patient_id"].nunique()}
Unique SeriesInstanceUIDs:            {final_df["SeriesInstanceUID"].nunique()}

TRAIN / VALIDATION / TEST
-------------------------------------------------------------------------------
Train nodules:                        {train_nodes}
Validation nodules:                   {val_nodes}
Test nodules:                         {test_nodes}

Train patients:                       {len(train_patients)}
Validation patients:                  {len(val_patients)}
Test patients:                        {len(test_patients)}

Patient leakage:
    Train-Val:                         {len(train_val_overlap)}
    Train-Test:                        {len(train_test_overlap)}
    Val-Test:                          {len(val_test_overlap)}

PROCESSED DATASET
-------------------------------------------------------------------------------
Cases audited:                        {len(audit_df)}
Cases passing:                        {pass_count}
Cases failing:                        {fail_count}

Processed folders missing:            {missing_processed}
Normalized volumes missing:           {missing_normalized}

Dimension failures:                   {dimension_failures}
Image/mask depth failures:            {depth_failures}

Total image PNG files:                {audit_df["image_count"].sum()}
Total mask PNG files:                 {audit_df[["mask_0_count","mask_1_count","mask_2_count","mask_3_count"]].sum().sum()}

QC / LABELS
-------------------------------------------------------------------------------
Native malignancy labels assigned:    {native_labels}
Unproven labels assigned:             NO

FINAL STATUS
-------------------------------------------------------------------------------
{final_status}

The audit checks the structural integrity of the processed dataset,
including image folders, mask folders, dimensions, slice counts,
normalized model-ready volumes, patient splits, and label status.

It does not establish native-folder-to-XML physical identity.
It does not assign malignancy labels.

Output audit:
    {OUTPUT_AUDIT_CSV}

Output summary:
    {OUTPUT_SUMMARY_TXT}
"""

with open(
    OUTPUT_SUMMARY_TXT,
    "w"
) as f:

    f.write(
        summary.strip()
    )

print(
    "\n" + summary
)

print("=" * 80)
print("STEP 29 COMPLETE")
print("=" * 80)

STEP 29 — COMPREHENSIVE FINAL STRUCTURAL DATASET AUDIT

INPUT FILES
--------------------------------------------------------------------------------
Final ML manifest:     /content/final_325_ml_manifest.csv
Master manifest:       /content/final_325_master_manifest.csv
Processed dataset:     /content/processed_325_nodules

MANIFEST SUMMARY
--------------------------------------------------------------------------------
Final manifest rows:     325
Master manifest rows:    325
✓ Cohort size = 325
✓ Unique subset IDs = 325
✓ Unique patients = 247
✓ Unique SeriesInstanceUIDs = 247
✓ Excluded nodules absent

PATIENT SPLIT AUDIT
--------------------------------------------------------------------------------
Train:       235 nodules / 173 patients
Validation:  44 nodules / 37 patients
Test:        46 nodules / 37 patients
Train/Val patient overlap: 0
Train/Test patient overlap: 0
Val/Test patient overlap: 0
✓ Zero patient leakage

AUDITING EACH OF 325 NODULES
--------------------------------

In [ ]:
# =============================================================================
# STEP 30 — MASK / READER CONSISTENCY ANALYSIS
# CORRECTED: READS MASKS DIRECTLY FROM PROCESSED DATASET
# =============================================================================

import os
import numpy as np
import pandas as pd
from PIL import Image

# -------------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------------

MASTER_MANIFEST = "/content/final_325_master_manifest.csv"
ML_MANIFEST = "/content/final_325_ml_manifest.csv"
PROCESSED_ROOT = "/content/processed_325_nodules"

OUTPUT_CASE_CSV = "/content/step30_mask_reader_consistency.csv"
OUTPUT_PAIRWISE_CSV = "/content/step30_mask_pairwise_agreement.csv"
OUTPUT_SUMMARY_TXT = "/content/step30_mask_reader_consistency_summary.txt"

print("=" * 80)
print("STEP 30 — MASK / READER CONSISTENCY ANALYSIS")
print("=" * 80)

# =============================================================================
# 1. CHECK INPUTS
# =============================================================================

for path in [
    MASTER_MANIFEST,
    ML_MANIFEST,
    PROCESSED_ROOT
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required input not found: {path}"
        )

master_df = pd.read_csv(
    MASTER_MANIFEST
)

ml_df = pd.read_csv(
    ML_MANIFEST
)

print("\nINPUT")
print("-" * 80)
print(
    f"Master manifest rows: {len(master_df)}"
)
print(
    f"ML manifest rows:     {len(ml_df)}"
)
print(
    f"Processed root:       {PROCESSED_ROOT}"
)

# =============================================================================
# 2. FIND COLUMNS
# =============================================================================

def find_col(df, candidates):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        key = candidate.strip().lower()

        if key in lookup:
            return lookup[key]

    return None


master_subset_col = find_col(
    master_df,
    [
        "subset_nodule_id",
        "subset_nodule",
        "nodule_id"
    ]
)

master_patient_col = find_col(
    master_df,
    [
        "patient_id",
        "patientid",
        "patient"
    ]
)

master_native_col = find_col(
    master_df,
    [
        "native_nodule_id",
        "native_nodule",
        "native_id"
    ]
)

ml_subset_col = find_col(
    ml_df,
    [
        "subset_nodule_id",
        "subset_nodule",
        "nodule_id"
    ]
)

ml_split_col = find_col(
    ml_df,
    [
        "split"
    ]
)

if master_subset_col is None:
    raise KeyError(
        "Master manifest does not contain a subset nodule ID column."
    )

if master_patient_col is None:
    raise KeyError(
        "Master manifest does not contain patient_id."
    )

if master_native_col is None:
    raise KeyError(
        "Master manifest does not contain native_nodule_id."
    )

if ml_subset_col is None:
    raise KeyError(
        "ML manifest does not contain subset_nodule_id."
    )

if ml_split_col is None:
    raise KeyError(
        "ML manifest does not contain split."
    )

# =============================================================================
# 3. BUILD CLEAN 325-ROW TABLE
# =============================================================================

master_df["_subset_key"] = (
    master_df[master_subset_col]
    .astype(str)
    .str.strip()
)

ml_df["_subset_key"] = (
    ml_df[ml_subset_col]
    .astype(str)
    .str.strip()
)

split_lookup = (
    ml_df[
        [
            "_subset_key",
            ml_split_col
        ]
    ]
    .drop_duplicates(
        subset="_subset_key"
    )
    .rename(
        columns={
            ml_split_col: "_split"
        }
    )
)

assert len(split_lookup) == 325

df = master_df.merge(
    split_lookup,
    on="_subset_key",
    how="left",
    validate="one_to_one"
)

assert len(df) == 325
assert df["_split"].notna().all()

print("\nCOHORT")
print("-" * 80)
print(
    f"Nodules:  {len(df)}"
)
print(
    f"Patients: {df[master_patient_col].nunique()}"
)

# =============================================================================
# 4. HELPER FUNCTIONS
# =============================================================================

def natural_sort(files):

    def key_func(x):

        base = os.path.basename(x)

        digits = "".join(
            c for c in base
            if c.isdigit()
        )

        if digits:
            return (
                int(digits),
                base
            )

        return (
            10**9,
            base
        )

    return sorted(
        files,
        key=key_func
    )


def load_png_mask_stack(mask_dir):

    if not os.path.isdir(mask_dir):
        return None

    files = [
        os.path.join(
            mask_dir,
            f
        )
        for f in os.listdir(mask_dir)
        if f.lower().endswith(".png")
    ]

    files = natural_sort(files)

    if not files:
        return None

    slices = []

    reference_shape = None

    for path in files:

        try:

            with Image.open(path) as img:

                arr = np.asarray(
                    img.convert("L")
                )

            arr = (
                arr > 0
            )

            if reference_shape is None:

                reference_shape = arr.shape

            elif arr.shape != reference_shape:

                raise ValueError(
                    f"Inconsistent mask dimensions "
                    f"in {mask_dir}"
                )

            slices.append(arr)

        except Exception as e:

            raise RuntimeError(
                f"Could not read mask file {path}: {e}"
            )

    return np.stack(
        slices,
        axis=0
    )


def dice_score(a, b):

    a = np.asarray(
        a,
        dtype=bool
    )

    b = np.asarray(
        b,
        dtype=bool
    )

    if a.shape != b.shape:
        return np.nan

    intersection = np.logical_and(
        a,
        b
    ).sum()

    denominator = (
        a.sum()
        +
        b.sum()
    )

    if denominator == 0:

        return 1.0

    return (
        2.0 * intersection
        / denominator
    )


def iou_score(a, b):

    a = np.asarray(
        a,
        dtype=bool
    )

    b = np.asarray(
        b,
        dtype=bool
    )

    if a.shape != b.shape:
        return np.nan

    intersection = np.logical_and(
        a,
        b
    ).sum()

    union = np.logical_or(
        a,
        b
    ).sum()

    if union == 0:

        return 1.0

    return (
        intersection
        / union
    )


# =============================================================================
# 5. ANALYZE ALL 325 NODULES
# =============================================================================

case_rows = []
pairwise_rows = []

print("\nANALYZING MASKS")
print("-" * 80)

for counter, (_, row) in enumerate(
    df.iterrows(),
    start=1
):

    subset_id = str(
        row["_subset_key"]
    )

    patient_id = str(
        row[master_patient_col]
    ).strip()

    native_id = str(
        row[master_native_col]
    ).strip()

    split = str(
        row["_split"]
    ).strip()

    # -------------------------------------------------------------
    # Find processed case directory
    # -------------------------------------------------------------

    case_dir = os.path.join(
        PROCESSED_ROOT,
        subset_id
    )

    if not os.path.isdir(case_dir):

        # Fallback in case the processed dataset uses
        # patient/native directory structure.

        candidate_dir = os.path.join(
            PROCESSED_ROOT,
            patient_id,
            native_id
        )

        if os.path.isdir(candidate_dir):

            case_dir = candidate_dir

    # -------------------------------------------------------------
    # Load masks 0-3
    # -------------------------------------------------------------

    masks = {}

    for mask_number in range(4):

        mask_dir = os.path.join(
            case_dir,
            f"mask-{mask_number}"
        )

        if not os.path.isdir(mask_dir):
            continue

        mask_stack = load_png_mask_stack(
            mask_dir
        )

        if mask_stack is not None:

            masks[mask_number] = mask_stack

    # -------------------------------------------------------------
    # Reader count
    # -------------------------------------------------------------

    reader_count = len(masks)

    # -------------------------------------------------------------
    # Basic statistics
    # -------------------------------------------------------------

    foreground_pixels = {}
    foreground_slices = {}

    for reader, mask in masks.items():

        foreground_pixels[reader] = int(
            np.count_nonzero(mask)
        )

        foreground_slices[reader] = int(
            np.count_nonzero(
                np.any(
                    mask,
                    axis=tuple(
                        range(
                            1,
                            mask.ndim
                        )
                    )
                )
            )
        )

    # -------------------------------------------------------------
    # Pairwise comparisons
    # -------------------------------------------------------------

    dice_values = []
    iou_values = []

    readers = sorted(
        masks.keys()
    )

    for i in range(
        len(readers)
    ):

        for j in range(
            i + 1,
            len(readers)
        ):

            reader_a = readers[i]
            reader_b = readers[j]

            mask_a = masks[reader_a]
            mask_b = masks[reader_b]

            compatible = (
                mask_a.shape
                ==
                mask_b.shape
            )

            if compatible:

                d = dice_score(
                    mask_a,
                    mask_b
                )

                u = iou_score(
                    mask_a,
                    mask_b
                )

                dice_values.append(d)
                iou_values.append(u)

            else:

                d = np.nan
                u = np.nan

            pairwise_rows.append({

                "subset_nodule_id":
                    subset_id,

                "patient_id":
                    patient_id,

                "native_nodule_id":
                    native_id,

                "split":
                    split,

                "reader_a":
                    f"mask-{reader_a}",

                "reader_b":
                    f"mask-{reader_b}",

                "shape_compatible":
                    compatible,

                "dice":
                    d,

                "iou":
                    u

            })

    # -------------------------------------------------------------
    # Agreement statistics
    # -------------------------------------------------------------

    if dice_values:

        mean_dice = float(
            np.mean(dice_values)
        )

        min_dice = float(
            np.min(dice_values)
        )

        max_dice = float(
            np.max(dice_values)
        )

    else:

        mean_dice = np.nan
        min_dice = np.nan
        max_dice = np.nan

    if iou_values:

        mean_iou = float(
            np.mean(iou_values)
        )

        min_iou = float(
            np.min(iou_values)
        )

        max_iou = float(
            np.max(iou_values)
        )

    else:

        mean_iou = np.nan
        min_iou = np.nan
        max_iou = np.nan

    # -------------------------------------------------------------
    # Agreement classification
    # -------------------------------------------------------------

    if reader_count == 0:

        agreement_class = (
            "NO_MASKS_FOUND"
        )

    elif reader_count == 1:

        agreement_class = (
            "SINGLE_READER"
        )

    elif np.isnan(mean_dice):

        agreement_class = (
            "INCOMPATIBLE_MASKS"
        )

    elif mean_dice >= 0.80:

        agreement_class = (
            "HIGH_AGREEMENT"
        )

    elif mean_dice >= 0.60:

        agreement_class = (
            "MODERATE_AGREEMENT"
        )

    elif mean_dice >= 0.40:

        agreement_class = (
            "LOW_AGREEMENT"
        )

    else:

        agreement_class = (
            "VERY_LOW_AGREEMENT"
        )

    # -------------------------------------------------------------
    # Disagreement flag
    # -------------------------------------------------------------

    disagreement_flag = bool(
        reader_count >= 2
        and not np.isnan(mean_dice)
        and mean_dice < 0.40
    )

    # -------------------------------------------------------------
    # Record
    # -------------------------------------------------------------

    case_rows.append({

        "subset_nodule_id":
            subset_id,

        "patient_id":
            patient_id,

        "native_nodule_id":
            native_id,

        "split":
            split,

        "processed_case_directory":
            case_dir,

        "reader_count":
            reader_count,

        "mask_0_foreground_pixels":
            foreground_pixels.get(
                0,
                0
            ),

        "mask_1_foreground_pixels":
            foreground_pixels.get(
                1,
                0
            ),

        "mask_2_foreground_pixels":
            foreground_pixels.get(
                2,
                0
            ),

        "mask_3_foreground_pixels":
            foreground_pixels.get(
                3,
                0
            ),

        "mask_0_foreground_slices":
            foreground_slices.get(
                0,
                0
            ),

        "mask_1_foreground_slices":
            foreground_slices.get(
                1,
                0
            ),

        "mask_2_foreground_slices":
            foreground_slices.get(
                2,
                0
            ),

        "mask_3_foreground_slices":
            foreground_slices.get(
                3,
                0
            ),

        "mean_pairwise_dice":
            mean_dice,

        "min_pairwise_dice":
            min_dice,

        "max_pairwise_dice":
            max_dice,

        "mean_pairwise_iou":
            mean_iou,

        "min_pairwise_iou":
            min_iou,

        "max_pairwise_iou":
            max_iou,

        "agreement_class":
            agreement_class,

        "potential_disagreement":
            disagreement_flag

    })

    if counter % 25 == 0:

        print(
            f"Processed {counter} / {len(df)}"
        )

# =============================================================================
# 6. SAVE OUTPUTS
# =============================================================================

case_df = pd.DataFrame(
    case_rows
)

pairwise_df = pd.DataFrame(
    pairwise_rows
)

case_df.to_csv(
    OUTPUT_CASE_CSV,
    index=False
)

pairwise_df.to_csv(
    OUTPUT_PAIRWISE_CSV,
    index=False
)

# =============================================================================
# 7. FINAL RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 30 RESULTS")
print("=" * 80)

print(
    f"\nNodules analyzed: "
    f"{len(case_df)}"
)

print(
    f"Pairwise comparisons: "
    f"{len(pairwise_df)}"
)

print("\nREADER COUNT DISTRIBUTION")
print("-" * 80)

print(
    case_df[
        "reader_count"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nAGREEMENT CLASS DISTRIBUTION")
print("-" * 80)

print(
    case_df[
        "agreement_class"
    ]
    .value_counts()
    .to_string()
)

# =============================================================================
# 8. GLOBAL PAIRWISE RESULTS
# =============================================================================

valid_pairs = pairwise_df[
    pairwise_df[
        "shape_compatible"
    ]
    &
    pairwise_df[
        "dice"
    ].notna()
].copy()

print("\nPAIRWISE AGREEMENT")
print("-" * 80)

if len(valid_pairs) > 0:

    print(
        f"Valid mask-pair comparisons: "
        f"{len(valid_pairs)}"
    )

    print(
        f"Mean Dice:   "
        f"{valid_pairs['dice'].mean():.4f}"
    )

    print(
        f"Median Dice: "
        f"{valid_pairs['dice'].median():.4f}"
    )

    print(
        f"Minimum Dice: "
        f"{valid_pairs['dice'].min():.4f}"
    )

    print(
        f"Maximum Dice: "
        f"{valid_pairs['dice'].max():.4f}"
    )

    print(
        f"Mean IoU:    "
        f"{valid_pairs['iou'].mean():.4f}"
    )

else:

    print(
        "No valid pairwise comparisons."
    )

# =============================================================================
# 9. MASK COVERAGE
# =============================================================================

print("\nMASK COVERAGE")
print("-" * 80)

for mask_num in range(4):

    col = (
        f"mask_{mask_num}_foreground_pixels"
    )

    count = int(
        (
            case_df[col] > 0
        ).sum()
    )

    print(
        f"mask-{mask_num}: "
        f"{count} / 325 cases with foreground"
    )

# =============================================================================
# 10. POTENTIAL DISAGREEMENTS
# =============================================================================

disagreement_df = case_df[
    case_df[
        "potential_disagreement"
    ]
].copy()

print("\nPOTENTIAL DISAGREEMENTS")
print("-" * 80)

print(
    f"Cases flagged: "
    f"{len(disagreement_df)}"
)

if not disagreement_df.empty:

    print(
        disagreement_df[
            [
                "subset_nodule_id",
                "patient_id",
                "reader_count",
                "mean_pairwise_dice",
                "min_pairwise_dice",
                "agreement_class"
            ]
        ]
        .sort_values(
            "mean_pairwise_dice"
        )
        .head(25)
        .to_string(
            index=False
        )
    )

# =============================================================================
# 11. NO-MASK CASES
# =============================================================================

no_mask_df = case_df[
    case_df[
        "reader_count"
    ] == 0
].copy()

print("\nNO-MASK CASES")
print("-" * 80)

print(
    f"Cases with no mask directories found: "
    f"{len(no_mask_df)}"
)

if not no_mask_df.empty:

    print(
        no_mask_df[
            [
                "subset_nodule_id",
                "patient_id",
                "native_nodule_id",
                "processed_case_directory"
            ]
        ]
        .head(25)
        .to_string(
            index=False
        )
    )

# =============================================================================
# 12. VALIDATION
# =============================================================================

assert len(case_df) == 325

assert (
    case_df[
        "subset_nodule_id"
    ].nunique()
    == 325
)

print("\n" + "=" * 80)
print("FINAL STEP 30 VALIDATION")
print("=" * 80)

print(
    "✓ All 325 nodules analyzed."
)

print(
    "✓ Masks were read directly from the processed dataset."
)

print(
    "✓ Pairwise reader agreement calculated where multiple masks exist."
)

print(
    "✓ No source images or masks were modified."
)

print(
    "✓ No malignancy labels assigned."
)

# =============================================================================
# 13. WRITE SUMMARY
# =============================================================================

global_mean_dice = (
    float(valid_pairs["dice"].mean())
    if len(valid_pairs) > 0
    else np.nan
)

global_median_dice = (
    float(valid_pairs["dice"].median())
    if len(valid_pairs) > 0
    else np.nan
)

global_mean_iou = (
    float(valid_pairs["iou"].mean())
    if len(valid_pairs) > 0
    else np.nan
)

summary = f"""
===============================================================================
STEP 30 — MASK / READER CONSISTENCY ANALYSIS
===============================================================================

COHORT
-------------------------------------------------------------------------------
Total nodules analyzed:              {len(case_df)}
Unique patients:                     {df[master_patient_col].nunique()}

READER COVERAGE
-------------------------------------------------------------------------------
Nodules with 0 readers:              {(case_df["reader_count"] == 0).sum()}
Nodules with 1 reader:               {(case_df["reader_count"] == 1).sum()}
Nodules with 2 readers:              {(case_df["reader_count"] == 2).sum()}
Nodules with 3 readers:              {(case_df["reader_count"] == 3).sum()}
Nodules with 4 readers:              {(case_df["reader_count"] == 4).sum()}

PAIRWISE AGREEMENT
-------------------------------------------------------------------------------
Valid reader-pair comparisons:       {len(valid_pairs)}
Mean Dice:                            {global_mean_dice:.4f}
Median Dice:                          {global_median_dice:.4f}
Mean IoU:                             {global_mean_iou:.4f}

AGREEMENT CLASSES
-------------------------------------------------------------------------------
{case_df["agreement_class"].value_counts().to_string()}

POTENTIAL DISAGREEMENT
-------------------------------------------------------------------------------
Cases with mean pairwise Dice < 0.40:
    {len(disagreement_df)}

QC INTERPRETATION
-------------------------------------------------------------------------------
This analysis measures agreement between the available native segmentation
masks.

It does NOT identify which anonymized radiologist corresponds to mask-0,
mask-1, mask-2, or mask-3.

It does NOT establish native-folder-to-XML nodule identity.

It does NOT assign malignancy labels.

The agreement statistics are used only for segmentation-mask quality control.

OUTPUTS
-------------------------------------------------------------------------------
Case-level CSV:
    {OUTPUT_CASE_CSV}

Pairwise agreement CSV:
    {OUTPUT_PAIRWISE_CSV}

Summary:
    {OUTPUT_SUMMARY_TXT}
===============================================================================
"""

with open(
    OUTPUT_SUMMARY_TXT,
    "w"
) as f:

    f.write(
        summary.strip()
    )

print(
    "\n" + summary
)

print("=" * 80)
print("STEP 30 COMPLETE")
print("=" * 80)

STEP 30 — MASK / READER CONSISTENCY ANALYSIS

INPUT
--------------------------------------------------------------------------------
Master manifest rows: 325
ML manifest rows:     325
Processed root:       /content/processed_325_nodules

COHORT
--------------------------------------------------------------------------------
Nodules:  325
Patients: 247

ANALYZING MASKS
--------------------------------------------------------------------------------
Processed 25 / 325
Processed 50 / 325
Processed 75 / 325
Processed 100 / 325
Processed 125 / 325
Processed 150 / 325
Processed 175 / 325
Processed 200 / 325
Processed 225 / 325
Processed 250 / 325
Processed 275 / 325
Processed 300 / 325
Processed 325 / 325

STEP 30 RESULTS

Nodules analyzed: 325
Pairwise comparisons: 1950

READER COUNT DISTRIBUTION
--------------------------------------------------------------------------------
reader_count
4    325

AGREEMENT CLASS DISTRIBUTION
---------------------------------------------------------------

In [ ]:
# =============================================================================
# STEP 31 — REFINED MASK / READER AGREEMENT QC
# =============================================================================

import os
import numpy as np
import pandas as pd
from PIL import Image

# -------------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------------

MASTER_MANIFEST = "/content/final_325_master_manifest.csv"
PROCESSED_ROOT = "/content/processed_325_nodules"

OUTPUT_CASE_CSV = "/content/step31_refined_mask_qc.csv"
OUTPUT_SLICE_CSV = "/content/step31_slice_level_agreement.csv"
OUTPUT_SUMMARY_TXT = "/content/step31_refined_mask_qc_summary.txt"

print("=" * 80)
print("STEP 31 — REFINED MASK / READER AGREEMENT QC")
print("=" * 80)


# =============================================================================
# 1. INPUT VALIDATION
# =============================================================================

if not os.path.exists(MASTER_MANIFEST):
    raise FileNotFoundError(
        f"Missing master manifest: {MASTER_MANIFEST}"
    )

if not os.path.isdir(PROCESSED_ROOT):
    raise FileNotFoundError(
        f"Missing processed dataset: {PROCESSED_ROOT}"
    )

df = pd.read_csv(
    MASTER_MANIFEST
)

assert len(df) == 325

# Find required columns robustly.
def find_col(df, candidates):
    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()

        if key in lookup:
            return lookup[key]

    return None


subset_col = find_col(
    df,
    [
        "subset_nodule_id",
        "subset_nodule",
        "nodule_id"
    ]
)

patient_col = find_col(
    df,
    [
        "patient_id",
        "patientid",
        "patient"
    ]
)

native_col = find_col(
    df,
    [
        "native_nodule_id",
        "native_nodule",
        "native_id"
    ]
)

assert subset_col is not None
assert patient_col is not None
assert native_col is not None

print("\nINPUT")
print("-" * 80)
print(f"Nodules:  {len(df)}")
print(f"Patients: {df[patient_col].nunique()}")


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def natural_sort(paths):
    """
    Sort PNG filenames using numeric components where possible.
    """
    import re

    def key(path):
        name = os.path.basename(path)

        parts = re.split(
            r"(\d+)",
            name
        )

        result = []

        for p in parts:

            if p.isdigit():
                result.append(
                    (0, int(p))
                )
            else:
                result.append(
                    (1, p)
                )

        return result

    return sorted(
        paths,
        key=key
    )


def load_mask_directory(mask_dir):
    """
    Load a mask directory as a binary [Z,H,W] volume.
    """
    if not os.path.isdir(mask_dir):
        return None, []

    files = [
        os.path.join(
            mask_dir,
            f
        )
        for f in os.listdir(mask_dir)
        if f.lower().endswith(".png")
    ]

    files = natural_sort(files)

    if not files:
        return None, []

    slices = []

    reference_shape = None

    for path in files:

        with Image.open(path) as img:

            arr = np.asarray(
                img.convert("L"),
                dtype=np.uint8
            )

        arr = arr > 0

        if reference_shape is None:

            reference_shape = arr.shape

        elif arr.shape != reference_shape:

            raise ValueError(
                f"Inconsistent dimensions in {mask_dir}"
            )

        slices.append(arr)

    return (
        np.stack(
            slices,
            axis=0
        ),
        files
    )


def dice_binary(a, b):
    """
    Dice for two binary arrays.
    """
    a = np.asarray(
        a,
        dtype=bool
    )

    b = np.asarray(
        b,
        dtype=bool
    )

    intersection = np.logical_and(
        a,
        b
    ).sum()

    denominator = (
        a.sum()
        +
        b.sum()
    )

    if denominator == 0:
        return 1.0

    return (
        2.0
        * intersection
        / denominator
    )


def iou_binary(a, b):
    """
    IoU for two binary arrays.
    """
    a = np.asarray(
        a,
        dtype=bool
    )

    b = np.asarray(
        b,
        dtype=bool
    )

    intersection = np.logical_and(
        a,
        b
    ).sum()

    union = np.logical_or(
        a,
        b
    ).sum()

    if union == 0:
        return 1.0

    return (
        intersection
        / union
    )


# =============================================================================
# 3. ANALYSIS
# =============================================================================

case_rows = []
slice_rows = []

print("\nRUNNING REFINED ANALYSIS")
print("-" * 80)

for counter, (_, row) in enumerate(
    df.iterrows(),
    start=1
):

    subset_id = str(
        row[subset_col]
    ).strip()

    patient_id = str(
        row[patient_col]
    ).strip()

    native_id = str(
        row[native_col]
    ).strip()

    # ---------------------------------------------------------------------
    # Locate case directory
    # ---------------------------------------------------------------------

    case_dir = os.path.join(
        PROCESSED_ROOT,
        subset_id
    )

    if not os.path.isdir(case_dir):

        fallback_dir = os.path.join(
            PROCESSED_ROOT,
            patient_id,
            native_id
        )

        if os.path.isdir(fallback_dir):

            case_dir = fallback_dir

    # ---------------------------------------------------------------------
    # Load all 4 masks
    # ---------------------------------------------------------------------

    masks = {}
    mask_files = {}

    for reader in range(4):

        mask_dir = os.path.join(
            case_dir,
            f"mask-{reader}"
        )

        volume, files = load_mask_directory(
            mask_dir
        )

        if volume is not None:

            masks[reader] = volume
            mask_files[reader] = files

    readers = sorted(
        masks.keys()
    )

    reader_count = len(readers)

    # ---------------------------------------------------------------------
    # Determine which readers actually marked the nodule
    # ---------------------------------------------------------------------

    active_readers = [
        r
        for r in readers
        if np.any(
            masks[r]
        )
    ]

    active_reader_count = len(
        active_readers
    )

    # ---------------------------------------------------------------------
    # Image depth
    # ---------------------------------------------------------------------

    image_dir = os.path.join(
        case_dir,
        "images"
    )

    image_files = []

    if os.path.isdir(image_dir):

        image_files = natural_sort(
            [
                os.path.join(
                    image_dir,
                    f
                )
                for f in os.listdir(
                    image_dir
                )
                if f.lower().endswith(".png")
            ]
        )

    image_depth = len(
        image_files
    )

    # ---------------------------------------------------------------------
    # Mask depth statistics
    # ---------------------------------------------------------------------

    mask_depths = {
        r: masks[r].shape[0]
        for r in readers
    }

    # ---------------------------------------------------------------------
    # Reader-specific foreground statistics
    # ---------------------------------------------------------------------

    foreground_pixels = {}

    foreground_slices = {}

    first_active_slice = {}

    last_active_slice = {}

    for r in readers:

        volume = masks[r]

        foreground_pixels[r] = int(
            np.count_nonzero(
                volume
            )
        )

        active_z = np.where(
            volume.any(
                axis=(1, 2)
            )
        )[0]

        foreground_slices[r] = int(
            len(active_z)
        )

        if len(active_z) > 0:

            first_active_slice[r] = int(
                active_z.min()
            )

            last_active_slice[r] = int(
                active_z.max()
            )

        else:

            first_active_slice[r] = None
            last_active_slice[r] = None

    # ---------------------------------------------------------------------
    # Pairwise metrics
    #
    # IMPORTANT:
    # Two versions are calculated:
    #
    # 1. Whole-volume Dice
    # 2. Foreground-relevant Dice
    #
    # Foreground-relevant Dice is calculated only over slices where
    # at least one of the two readers marked something.
    # ---------------------------------------------------------------------

    pairwise_whole_dice = []
    pairwise_fg_dice = []

    pairwise_whole_iou = []
    pairwise_fg_iou = []

    pairwise_slice_overlap_rates = []

    pairwise_slice_rows = []

    for i in range(
        len(readers)
    ):

        for j in range(
            i + 1,
            len(readers)
        ):

            reader_a = readers[i]
            reader_b = readers[j]

            a = masks[reader_a]
            b = masks[reader_b]

            # ---------------------------------------------------------
            # Shape compatibility
            # ---------------------------------------------------------

            compatible = (
                a.shape == b.shape
            )

            if not compatible:

                whole_dice = np.nan
                whole_iou = np.nan
                fg_dice = np.nan
                fg_iou = np.nan
                slice_overlap_rate = np.nan

            else:

                # -----------------------------------------------------
                # Whole-volume metrics
                # -----------------------------------------------------

                whole_dice = dice_binary(
                    a,
                    b
                )

                whole_iou = iou_binary(
                    a,
                    b
                )

                # -----------------------------------------------------
                # Foreground-relevant slices
                # -----------------------------------------------------

                a_active = a.any(
                    axis=(1, 2)
                )

                b_active = b.any(
                    axis=(1, 2)
                )

                relevant = (
                    a_active
                    |
                    b_active
                )

                relevant_count = int(
                    relevant.sum()
                )

                if relevant_count > 0:

                    fg_dice = dice_binary(
                        a[relevant],
                        b[relevant]
                    )

                    fg_iou = iou_binary(
                        a[relevant],
                        b[relevant]
                    )

                    # Fraction of relevant slices on which
                    # BOTH readers provided foreground.
                    both_active = (
                        a_active
                        &
                        b_active
                    )

                    slice_overlap_rate = (
                        both_active[relevant].sum()
                        /
                        relevant_count
                    )

                else:

                    fg_dice = 1.0
                    fg_iou = 1.0
                    slice_overlap_rate = 1.0

            pairwise_whole_dice.append(
                whole_dice
            )

            pairwise_fg_dice.append(
                fg_dice
            )

            pairwise_whole_iou.append(
                whole_iou
            )

            pairwise_fg_iou.append(
                fg_iou
            )

            pairwise_slice_overlap_rates.append(
                slice_overlap_rate
            )

            # ---------------------------------------------------------
            # Detailed slice-by-slice analysis
            # ---------------------------------------------------------

            if compatible:

                common_depth = min(
                    a.shape[0],
                    b.shape[0]
                )

                for z in range(
                    common_depth
                ):

                    a_slice = a[z]
                    b_slice = b[z]

                    a_fg = bool(
                        a_slice.any()
                    )

                    b_fg = bool(
                        b_slice.any()
                    )

                    relevant = (
                        a_fg
                        or b_fg
                    )

                    if relevant:

                        d = dice_binary(
                            a_slice,
                            b_slice
                        )

                        u = iou_binary(
                            a_slice,
                            b_slice
                        )

                    else:

                        d = np.nan
                        u = np.nan

                    pairwise_slice_rows.append({

                        "subset_nodule_id":
                            subset_id,

                        "patient_id":
                            patient_id,

                        "reader_a":
                            f"mask-{reader_a}",

                        "reader_b":
                            f"mask-{reader_b}",

                        "slice_index":
                            z,

                        "reader_a_has_foreground":
                            a_fg,

                        "reader_b_has_foreground":
                            b_fg,

                        "relevant_slice":
                            relevant,

                        "slice_dice":
                            d,

                        "slice_iou":
                            u
                    })

            pairwise_rows.append({

                "subset_nodule_id":
                    subset_id,

                "patient_id":
                    patient_id,

                "reader_a":
                    f"mask-{reader_a}",

                "reader_b":
                    f"mask-{reader_b}",

                "shape_compatible":
                    compatible,

                "whole_volume_dice":
                    whole_dice,

                "whole_volume_iou":
                    whole_iou,

                "foreground_relevant_dice":
                    fg_dice,

                "foreground_relevant_iou":
                    fg_iou,

                "relevant_slice_overlap_rate":
                    slice_overlap_rate
            })

    # ---------------------------------------------------------------------
    # Aggregate pairwise results
    # ---------------------------------------------------------------------

    if pairwise_whole_dice:

        mean_whole_dice = float(
            np.nanmean(
                pairwise_whole_dice
            )
        )

        min_whole_dice = float(
            np.nanmin(
                pairwise_whole_dice
            )
        )

        mean_fg_dice = float(
            np.nanmean(
                pairwise_fg_dice
            )
        )

        min_fg_dice = float(
            np.nanmin(
                pairwise_fg_dice
            )
        )

        mean_fg_iou = float(
            np.nanmean(
                pairwise_fg_iou
            )
        )

        mean_slice_overlap = float(
            np.nanmean(
                pairwise_slice_overlap_rates
            )
        )

    else:

        mean_whole_dice = np.nan
        min_whole_dice = np.nan
        mean_fg_dice = np.nan
        min_fg_dice = np.nan
        mean_fg_iou = np.nan
        mean_slice_overlap = np.nan

    # ---------------------------------------------------------------------
    # Reader extent disagreement
    # ---------------------------------------------------------------------

    if active_reader_count >= 2:

        first_values = [
            first_active_slice[r]
            for r in active_readers
            if first_active_slice[r] is not None
        ]

        last_values = [
            last_active_slice[r]
            for r in active_readers
            if last_active_slice[r] is not None
        ]

        if first_values:

            first_range = (
                max(first_values)
                -
                min(first_values)
            )

        else:

            first_range = np.nan

        if last_values:

            last_range = (
                max(last_values)
                -
                min(last_values)
            )

        else:

            last_range = np.nan

    else:

        first_range = np.nan
        last_range = np.nan

    # ---------------------------------------------------------------------
    # Distinguish two kinds of disagreement
    # ---------------------------------------------------------------------

    if active_reader_count <= 1:

        qc_category = (
            "LIMITED_READER_OVERLAP"
        )

    elif np.isnan(mean_fg_dice):

        qc_category = (
            "UNASSESSABLE"
        )

    elif mean_fg_dice >= 0.80:

        qc_category = (
            "HIGH_CONTOUR_AGREEMENT"
        )

    elif mean_fg_dice >= 0.60:

        qc_category = (
            "MODERATE_CONTOUR_AGREEMENT"
        )

    elif mean_fg_dice >= 0.40:

        qc_category = (
            "LOW_CONTOUR_AGREEMENT"
        )

    else:

        qc_category = (
            "VERY_LOW_CONTOUR_AGREEMENT"
        )

    # ---------------------------------------------------------------------
    # Strong disagreement flag
    # ---------------------------------------------------------------------

    strong_disagreement = bool(
        active_reader_count >= 2
        and not np.isnan(mean_fg_dice)
        and mean_fg_dice < 0.40
    )

    # ---------------------------------------------------------------------
    # Possible slice-extent disagreement
    # ---------------------------------------------------------------------

    slice_extent_disagreement = bool(
        active_reader_count >= 2
        and (
            (
                not np.isnan(first_range)
                and first_range > 1
            )
            or
            (
                not np.isnan(last_range)
                and last_range > 1
            )
        )
    )

    # ---------------------------------------------------------------------
    # Case record
    # ---------------------------------------------------------------------

    case_rows.append({

        "subset_nodule_id":
            subset_id,

        "patient_id":
            patient_id,

        "native_nodule_id":
            native_id,

        "reader_directory_count":
            reader_count,

        "active_reader_count":
            active_reader_count,

        "active_readers":
            ",".join(
                f"mask-{r}"
                for r in active_readers
            ),

        "image_depth":
            image_depth,

        "mask_depths":
            str(mask_depths),

        "mask_0_foreground_pixels":
            foreground_pixels.get(
                0,
                0
            ),

        "mask_1_foreground_pixels":
            foreground_pixels.get(
                1,
                0
            ),

        "mask_2_foreground_pixels":
            foreground_pixels.get(
                2,
                0
            ),

        "mask_3_foreground_pixels":
            foreground_pixels.get(
                3,
                0
            ),

        "mask_0_foreground_slices":
            foreground_slices.get(
                0,
                0
            ),

        "mask_1_foreground_slices":
            foreground_slices.get(
                1,
                0
            ),

        "mask_2_foreground_slices":
            foreground_slices.get(
                2,
                0
            ),

        "mask_3_foreground_slices":
            foreground_slices.get(
                3,
                0
            ),

        "first_active_slice_range":
            first_range,

        "last_active_slice_range":
            last_range,

        "mean_whole_volume_dice":
            mean_whole_dice,

        "min_whole_volume_dice":
            min_whole_dice,

        "mean_foreground_relevant_dice":
            mean_fg_dice,

        "min_foreground_relevant_dice":
            min_fg_dice,

        "mean_foreground_relevant_iou":
            mean_fg_iou,

        "mean_relevant_slice_overlap":
            mean_slice_overlap,

        "qc_category":
            qc_category,

        "strong_reader_disagreement":
            strong_disagreement,

        "slice_extent_disagreement":
            slice_extent_disagreement

    })

    if counter % 25 == 0:

        print(
            f"Processed {counter} / {len(df)}"
        )


case_df = pd.DataFrame(
    case_rows
)

pairwise_df = pd.DataFrame(
    pairwise_rows
)

slice_df = pd.DataFrame(
    pairwise_slice_rows
)


# =============================================================================
# 4. SAVE RESULTS
# =============================================================================

case_df.to_csv(
    OUTPUT_CASE_CSV,
    index=False
)

pairwise_df.to_csv(
    OUTPUT_PAIRWISE_CSV,
    index=False
)

slice_df.to_csv(
    OUTPUT_SLICE_CSV,
    index=False
)


# =============================================================================
# 5. RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 31 RESULTS")
print("=" * 80)

print(
    f"\nNodules analyzed: "
    f"{len(case_df)}"
)

print(
    f"Reader-pair comparisons: "
    f"{len(pairwise_df)}"
)

print(
    f"Foreground-relevant slice comparisons: "
    f"{len(slice_df)}"
)

# =============================================================================
# READER COVERAGE
# =============================================================================

print("\nACTIVE READER COUNTS")
print("-" * 80)

print(
    case_df[
        "active_reader_count"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


# =============================================================================
# AGREEMENT CLASSES
# =============================================================================

print("\nREFINED AGREEMENT CLASSES")
print("-" * 80)

print(
    case_df[
        "qc_category"
    ]
    .value_counts()
    .to_string()
)


# =============================================================================
# GLOBAL AGREEMENT
# =============================================================================

print("\nGLOBAL FOREGROUND-RELEVANT AGREEMENT")
print("-" * 80)

if len(pairwise_df) > 0:

    print(
        f"Mean foreground Dice: "
        f"{pairwise_df['foreground_relevant_dice'].mean():.4f}"
    )

    print(
        f"Median foreground Dice: "
        f"{pairwise_df['foreground_relevant_dice'].median():.4f}"
    )

    print(
        f"Minimum foreground Dice: "
        f"{pairwise_df['foreground_relevant_dice'].min():.4f}"
    )

    print(
        f"Mean foreground IoU: "
        f"{pairwise_df['foreground_relevant_iou'].mean():.4f}"
    )

    print(
        f"Mean relevant-slice overlap: "
        f"{pairwise_df['relevant_slice_overlap_rate'].mean():.4f}"
    )


# =============================================================================
# READER COVERAGE DETAIL
# =============================================================================

print("\nREADER FOREGROUND COVERAGE")
print("-" * 80)

for reader in range(4):

    col = (
        f"mask_{reader}_foreground_pixels"
    )

    count = int(
        (
            case_df[col]
            > 0
        ).sum()
    )

    print(
        f"mask-{reader}: "
        f"{count} / 325"
    )


# =============================================================================
# DISAGREEMENT SUMMARY
# =============================================================================

print("\nDISAGREEMENT SUMMARY")
print("-" * 80)

strong_count = int(
    case_df[
        "strong_reader_disagreement"
    ].sum()
)

extent_count = int(
    case_df[
        "slice_extent_disagreement"
    ].sum()
)

print(
    f"Strong contour disagreement: "
    f"{strong_count} cases"
)

print(
    f"Slice-extent disagreement: "
    f"{extent_count} cases"
)


# =============================================================================
# MOST DISAGREEMENT CASES
# =============================================================================

flagged = case_df[
    case_df["strong_reader_disagreement"]
].copy()

print("\nLOWEST AGREEMENT CASES")
print("-" * 80)

if not flagged.empty:

    print(
        flagged[
            [
                "subset_nodule_id",
                "patient_id",
                "active_reader_count",
                "mean_foreground_relevant_dice",
                "min_foreground_relevant_dice",
                "mean_relevant_slice_overlap",
                "qc_category"
            ]
        ]
        .sort_values(
            "mean_foreground_relevant_dice"
        )
        .head(25)
        .to_string(
            index=False
        )
    )

else:

    print(
        "No cases met the strong-disagreement threshold."
    )


# =============================================================================
# 6. FINAL VALIDATION
# =============================================================================

assert len(case_df) == 325

assert (
    case_df[
        "subset_nodule_id"
    ].nunique()
    == 325
)

assert (
    case_df[
        "active_reader_count"
    ].min()
    >= 0
)

assert (
    case_df[
        "active_reader_count"
    ].max()
    <= 4
)

print("\n" + "=" * 80)
print("STEP 31 VALIDATION")
print("=" * 80)

print(
    "✓ All 325 nodules analyzed."
)

print(
    "✓ Reader foreground activity measured."
)

print(
    "✓ Whole-volume and foreground-relevant agreement calculated."
)

print(
    "✓ Slice-level overlap analyzed."
)

print(
    "✓ Potential contour and slice-extent disagreements flagged."
)

print(
    "✓ No masks modified."
)

print(
    "✓ No malignancy labels assigned."
)


# =============================================================================
# 7. SUMMARY REPORT
# =============================================================================

summary_lines = [

    "==============================================================================",
    "STEP 31 — REFINED MASK / READER AGREEMENT QC",
    "==============================================================================",
    "",

    f"Total nodules analyzed: {len(case_df)}",
    f"Total reader-pair comparisons: {len(pairwise_df)}",
    f"Foreground-relevant slice comparisons: {len(slice_df)}",
    "",

    "ACTIVE READER COUNTS",
    "------------------------------------------------------------------------------"
]

for count, number in (
    case_df[
        "active_reader_count"
    ]
    .value_counts()
    .sort_index()
    .items()
):

    summary_lines.append(
        f"{count} active reader(s): {number} nodules"
    )

summary_lines.extend([

    "",
    "REFINED AGREEMENT CLASSES",
    "------------------------------------------------------------------------------"
])

for category, number in (
    case_df[
        "qc_category"
    ]
    .value_counts()
    .items()
):

    summary_lines.append(
        f"{category}: {number}"
    )

summary_lines.extend([

    "",
    "GLOBAL AGREEMENT",
    "------------------------------------------------------------------------------",

    f"Mean foreground-relevant Dice: "
    f"{pairwise_df['foreground_relevant_dice'].mean():.4f}",

    f"Median foreground-relevant Dice: "
    f"{pairwise_df['foreground_relevant_dice'].median():.4f}",

    f"Minimum foreground-relevant Dice: "
    f"{pairwise_df['foreground_relevant_dice'].min():.4f}",

    f"Mean foreground-relevant IoU: "
    f"{pairwise_df['foreground_relevant_iou'].mean():.4f}",

    f"Mean relevant-slice overlap: "
    f"{pairwise_df['relevant_slice_overlap_rate'].mean():.4f}",

    "",
    "DISAGREEMENT FLAGS",
    "------------------------------------------------------------------------------",

    f"Strong reader disagreement: {strong_count}",
    f"Slice-extent disagreement: {extent_count}",

    "",
    "METHODOLOGICAL INTERPRETATION",
    "------------------------------------------------------------------------------",

    "This analysis separates contour disagreement from simple differences",
    "in which slices contain foreground.",

    "Foreground-relevant Dice excludes slices where both readers are empty,",
    "while the slice-overlap metric measures whether readers selected the",
    "same portions of the Z-stack.",

    "These measurements are QC evidence only.",

    "They do not establish reader identity, native-to-XML identity, or",
    "malignancy labels.",

    "",
    "OUTPUT FILES",
    "------------------------------------------------------------------------------",

    f"Case-level QC:     {OUTPUT_CASE_CSV}",
    f"Pairwise QC:       {OUTPUT_PAIRWISE_CSV}",
    f"Slice-level QC:    {OUTPUT_SLICE_CSV}",
    f"Summary:           {OUTPUT_SUMMARY_TXT}",

    "=============================================================================="
])

summary_text = "\n".join(
    summary_lines
)

with open(
    OUTPUT_SUMMARY_TXT,
    "w"
) as f:

    f.write(
        summary_text
    )

print(
    "\n" + summary_text
)

print("=" * 80)
print("STEP 31 COMPLETE")
print("=" * 80)

STEP 31 — REFINED MASK / READER AGREEMENT QC

INPUT
--------------------------------------------------------------------------------
Nodules:  325
Patients: 247

RUNNING REFINED ANALYSIS
--------------------------------------------------------------------------------
Processed 25 / 325
Processed 50 / 325
Processed 75 / 325
Processed 100 / 325
Processed 125 / 325
Processed 150 / 325
Processed 175 / 325
Processed 200 / 325
Processed 225 / 325
Processed 250 / 325
Processed 275 / 325
Processed 300 / 325
Processed 325 / 325

STEP 31 RESULTS

Nodules analyzed: 325
Reader-pair comparisons: 3900
Foreground-relevant slice comparisons: 90

ACTIVE READER COUNTS
--------------------------------------------------------------------------------
active_reader_count
1     88
2     62
3     69
4    106

REFINED AGREEMENT CLASSES
--------------------------------------------------------------------------------
qc_category
VERY_LOW_CONTOUR_AGREEMENT    111
LIMITED_READER_OVERLAP         88
MODERATE_CONTOUR

In [ ]:
# =============================================================================
# STEP 32 — FINAL DATASET QC CLASSIFICATION
# =============================================================================

import os
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# INPUTS
# -------------------------------------------------------------------------

MASTER_MANIFEST = "/content/final_325_master_manifest.csv"
ML_MANIFEST = "/content/final_325_ml_manifest.csv"
STEP29_AUDIT = "/content/step29_final_structural_audit.csv"
STEP31_QC = "/content/step31_refined_mask_qc.csv"

OUTPUT_FINAL_QC = "/content/final_325_dataset_qc_classification.csv"
OUTPUT_SUMMARY = "/content/final_325_dataset_qc_classification_summary.txt"

print("=" * 80)
print("STEP 32 — FINAL DATASET QC CLASSIFICATION")
print("=" * 80)

# =============================================================================
# 1. CHECK INPUT FILES
# =============================================================================

required = [
    MASTER_MANIFEST,
    ML_MANIFEST,
    STEP29_AUDIT,
    STEP31_QC
]

for path in required:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

# =============================================================================
# 2. LOAD FILES
# =============================================================================

master_df = pd.read_csv(
    MASTER_MANIFEST
)

ml_df = pd.read_csv(
    ML_MANIFEST
)

step29_df = pd.read_csv(
    STEP29_AUDIT
)

step31_df = pd.read_csv(
    STEP31_QC
)

print("\nINPUT")
print("-" * 80)

print(
    f"Master manifest: {len(master_df)}"
)

print(
    f"ML manifest:     {len(ml_df)}"
)

print(
    f"Step 29 audit:   {len(step29_df)}"
)

print(
    f"Step 31 QC:      {len(step31_df)}"
)

# =============================================================================
# 3. FIND COLUMN HELPER
# =============================================================================

def find_col(df, names):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for name in names:

        key = name.strip().lower()

        if key in lookup:

            return lookup[key]

    return None


master_subset = find_col(
    master_df,
    [
        "subset_nodule_id",
        "subset_nodule",
        "nodule_id"
    ]
)

master_patient = find_col(
    master_df,
    [
        "patient_id",
        "patientid",
        "patient"
    ]
)

master_native = find_col(
    master_df,
    [
        "native_nodule_id",
        "native_nodule",
        "native_id"
    ]
)

ml_subset = find_col(
    ml_df,
    [
        "subset_nodule_id",
        "subset_nodule",
        "nodule_id"
    ]
)

ml_split = find_col(
    ml_df,
    [
        "split"
    ]
)

assert master_subset is not None
assert master_patient is not None
assert master_native is not None
assert ml_subset is not None
assert ml_split is not None

# =============================================================================
# 4. NORMALIZE KEYS
# =============================================================================

master_df["_key"] = (
    master_df[master_subset]
    .astype(str)
    .str.strip()
)

ml_df["_key"] = (
    ml_df[ml_subset]
    .astype(str)
    .str.strip()
)

step29_df["_key"] = (
    step29_df["subset_nodule_id"]
    .astype(str)
    .str.strip()
)

step31_df["_key"] = (
    step31_df["subset_nodule_id"]
    .astype(str)
    .str.strip()
)

# =============================================================================
# 5. BUILD SPLIT LOOKUP
# =============================================================================

split_lookup = (
    ml_df[
        [
            "_key",
            ml_split
        ]
    ]
    .drop_duplicates(
        subset="_key"
    )
    .rename(
        columns={
            ml_split: "split"
        }
    )
)

assert len(split_lookup) == 325

# =============================================================================
# 6. MERGE FINAL QC TABLE
# =============================================================================

final_df = master_df.copy()

final_df = final_df.merge(
    split_lookup,
    on="_key",
    how="left",
    validate="one_to_one"
)

final_df = final_df.merge(
    step29_df[
        [
            "_key",
            "case_qc_status"
        ]
    ],
    on="_key",
    how="left",
    validate="one_to_one"
)

final_df = final_df.merge(
    step31_df[
        [
            "_key",
            "reader_directory_count",
            "active_reader_count",
            "active_readers",
            "mean_foreground_relevant_dice",
            "min_foreground_relevant_dice",
            "mean_foreground_relevant_iou",
            "mean_relevant_slice_overlap",
            "qc_category",
            "strong_reader_disagreement",
            "slice_extent_disagreement"
        ]
    ],
    on="_key",
    how="left",
    validate="one_to_one"
)

assert len(final_df) == 325

# =============================================================================
# 7. EXCLUSION / PROVENANCE CHECK
# =============================================================================

final_df["excluded_nodule"] = (
    final_df["_key"].isin(
        [
            "nodule_029",
            "nodule_085"
        ]
    )
)

assert not final_df["excluded_nodule"].any()

final_df["native_malignancy_label_assigned"] = False

# =============================================================================
# 8. FINAL QC CLASSIFICATION
# =============================================================================

def classify_case(row):

    # Structural failure has highest priority.
    if str(
        row["case_qc_status"]
    ).upper() != "PASS":

        return "STRUCTURAL_QC_FAILURE"

    # No active reader.
    if int(
        row["active_reader_count"]
    ) == 0:

        return "NO_ACTIVE_READER"

    # One active reader means inter-reader agreement
    # cannot be assessed.
    if int(
        row["active_reader_count"]
    ) == 1:

        return "LIMITED_READER_DATA"

    # Strong disagreement.
    if bool(
        row["strong_reader_disagreement"]
    ):

        return "READER_DISAGREEMENT"

    # Slice-extent issue.
    if bool(
        row["slice_extent_disagreement"]
    ):

        return "SLICE_EXTENT_DIFFERENCE"

    # Otherwise acceptable reader consistency.
    if (
        not pd.isna(
            row["mean_foreground_relevant_dice"]
        )
        and
        row[
            "mean_foreground_relevant_dice"
        ] >= 0.60
    ):

        return "GOOD_MASK_CONSISTENCY"

    return "MASK_CONSISTENCY_REVIEW"

final_df["final_qc_class"] = (
    final_df.apply(
        classify_case,
        axis=1
    )
)

# =============================================================================
# 9. QC PASS / REVIEW FLAGS
# =============================================================================

final_df["usable_for_processed_dataset"] = (
    final_df["case_qc_status"].astype(str).str.upper()
    == "PASS"
)

final_df["requires_qc_review"] = (
    final_df["final_qc_class"].isin(
        [
            "STRUCTURAL_QC_FAILURE",
            "READER_DISAGREEMENT",
            "SLICE_EXTENT_DIFFERENCE",
            "MASK_CONSISTENCY_REVIEW",
            "NO_ACTIVE_READER"
        ]
    )
)

# =============================================================================
# 10. PROVENANCE STATUS
# =============================================================================

final_df[
    "native_to_xml_identity_status"
] = "UNPROVEN"

final_df[
    "malignancy_label_status"
] = "NOT_ASSIGNED"

# =============================================================================
# 11. SAVE
# =============================================================================

# Remove temporary merge key.
final_df = final_df.drop(
    columns=["_key"]
)

final_df.to_csv(
    OUTPUT_FINAL_QC,
    index=False
)

# =============================================================================
# 12. RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 32 RESULTS")
print("=" * 80)

print(
    f"\nTotal nodules: "
    f"{len(final_df)}"
)

print("\nFINAL QC CLASS DISTRIBUTION")
print("-" * 80)

print(
    final_df[
        "final_qc_class"
    ]
    .value_counts()
    .to_string()
)

print("\nUSABLE DATASET STATUS")
print("-" * 80)

print(
    f"Structurally valid cases: "
    f"{final_df['usable_for_processed_dataset'].sum()} / 325"
)

print(
    f"Cases requiring QC review: "
    f"{final_df['requires_qc_review'].sum()} / 325"
)

print("\nPROVENANCE")
print("-" * 80)

print(
    f"Native→XML identities proven: "
    f"{(
        final_df['native_to_xml_identity_status']
        == 'PROVEN'
    ).sum()}"
)

print(
    f"Malignancy labels assigned: "
    f"{(
        final_df['malignancy_label_status']
        != 'NOT_ASSIGNED'
    ).sum()}"
)

# =============================================================================
# 13. REVIEW TABLE
# =============================================================================

review_df = final_df[
    final_df[
        "requires_qc_review"
    ]
].copy()

print("\nCASES REQUIRING QC REVIEW")
print("-" * 80)

print(
    f"Review cases: "
    f"{len(review_df)}"
)

if not review_df.empty:

    print(
        review_df[
            [
                "subset_nodule_id",
                "patient_id",
                "native_nodule_id",
                "active_reader_count",
                "mean_foreground_relevant_dice",
                "mean_relevant_slice_overlap",
                "final_qc_class"
            ]
        ]
        .head(30)
        .to_string(
            index=False
        )
    )

# =============================================================================
# 14. ASSERTIONS
# =============================================================================

assert len(final_df) == 325

assert (
    final_df[
        "subset_nodule_id"
    ].nunique()
    == 325
)

assert not final_df[
    "excluded_nodule"
].any()

assert (
    final_df[
        "native_malignancy_label_assigned"
    ].sum()
    == 0
)

# =============================================================================
# 15. SUMMARY REPORT
# =============================================================================

summary_lines = [

    "==============================================================================",
    "STEP 32 — FINAL DATASET QC CLASSIFICATION",
    "==============================================================================",
    "",

    "COHORT",
    "------------------------------------------------------------------------------",

    f"Total nodules: {len(final_df)}",

    f"Unique patients: "
    f"{final_df[master_patient].nunique()}",

    "",

    "STRUCTURAL QC",
    "------------------------------------------------------------------------------",

    f"Structurally valid: "
    f"{final_df['usable_for_processed_dataset'].sum()}",

    f"Structural failures: "
    f"{(~final_df['usable_for_processed_dataset']).sum()}",

    "",

    "FINAL QC CLASSES",
    "------------------------------------------------------------------------------"
]

for category, count in (
    final_df[
        "final_qc_class"
    ]
    .value_counts()
    .items()
):

    summary_lines.append(
        f"{category}: {count}"
    )

summary_lines.extend([

    "",

    "PROVENANCE",
    "------------------------------------------------------------------------------",

    "Native-to-XML identity proven: 0",

    "Native malignancy labels assigned: 0",

    "",

    "INTERPRETATION",
    "------------------------------------------------------------------------------",

    "This classification is a processing/QC layer only.",

    "No cases were deleted by this step.",

    "Low reader agreement is retained as a QC flag rather than treated",
    "as proof that a nodule is invalid.",

    "Native-to-XML physical identity remains unproven.",

    "No malignancy labels are assigned.",

    "",

    "OUTPUT",
    "------------------------------------------------------------------------------",

    OUTPUT_FINAL_QC,

    OUTPUT_SUMMARY,

    "=============================================================================="
])

summary_text = "\n".join(
    summary_lines
)

with open(
    OUTPUT_SUMMARY,
    "w"
) as f:

    f.write(
        summary_text
    )

print(
    "\n" + summary_text
)

print("=" * 80)
print("STEP 32 COMPLETE")
print("=" * 80)

STEP 32 — FINAL DATASET QC CLASSIFICATION

INPUT
--------------------------------------------------------------------------------
Master manifest: 325
ML manifest:     325
Step 29 audit:   325
Step 31 QC:      325

STEP 32 RESULTS

Total nodules: 325

FINAL QC CLASS DISTRIBUTION
--------------------------------------------------------------------------------
final_qc_class
READER_DISAGREEMENT        111
LIMITED_READER_DATA         88
GOOD_MASK_CONSISTENCY       80
SLICE_EXTENT_DIFFERENCE     24
MASK_CONSISTENCY_REVIEW     22

USABLE DATASET STATUS
--------------------------------------------------------------------------------
Structurally valid cases: 325 / 325
Cases requiring QC review: 157 / 325

PROVENANCE
--------------------------------------------------------------------------------
Native→XML identities proven: 0
Malignancy labels assigned: 0

CASES REQUIRING QC REVIEW
--------------------------------------------------------------------------------
Review cases: 157
subset_nodu

In [ ]:
# =============================================================================
# STEP 33 — FINAL DATASET MANIFEST + DOCUMENTATION PACKAGE
# =============================================================================

import os
import shutil
import pandas as pd
from datetime import datetime

# -------------------------------------------------------------------------
# Inputs
# -------------------------------------------------------------------------

FINAL_QC = "/content/final_325_dataset_qc_classification.csv"
MASTER_MANIFEST = "/content/final_325_master_manifest.csv"
ML_MANIFEST = "/content/final_325_ml_manifest.csv"

STEP29_AUDIT = "/content/step29_final_structural_audit.csv"
STEP29_SUMMARY = "/content/step29_final_structural_audit_summary.txt"

STEP30_CASE = "/content/step30_mask_reader_consistency.csv"
STEP30_PAIRWISE = "/content/step30_mask_pairwise_agreement.csv"
STEP30_SUMMARY = "/content/step30_mask_reader_consistency_summary.txt"

STEP31_CASE = "/content/step31_refined_mask_qc.csv"
STEP31_SLICE = "/content/step31_slice_level_agreement.csv"
STEP31_SUMMARY = "/content/step31_refined_mask_qc_summary.txt"

PROCESSED_ROOT = "/content/processed_325_nodules"

# -------------------------------------------------------------------------
# Outputs
# -------------------------------------------------------------------------

FINAL_PACKAGE = "/content/final_325_dataset_package"

FINAL_MANIFEST = os.path.join(
    FINAL_PACKAGE,
    "final_325_dataset_manifest.csv"
)

README_PATH = os.path.join(
    FINAL_PACKAGE,
    "README.txt"
)

PACKAGE_SUMMARY = os.path.join(
    FINAL_PACKAGE,
    "FINAL_DATASET_SUMMARY.txt"
)

# =============================================================================
# 1. VALIDATE FINAL QC
# =============================================================================

if not os.path.exists(FINAL_QC):
    raise FileNotFoundError(
        f"Missing Step 32 output: {FINAL_QC}"
    )

if not os.path.isdir(PROCESSED_ROOT):
    raise FileNotFoundError(
        f"Missing processed dataset: {PROCESSED_ROOT}"
    )

final_df = pd.read_csv(
    FINAL_QC
)

assert len(final_df) == 325

assert (
    final_df[
        "subset_nodule_id"
    ].nunique()
    == 325
)

assert not final_df[
    "excluded_nodule"
].any()

print("=" * 80)
print("STEP 33 — FINAL DATASET MANIFEST + DOCUMENTATION PACKAGE")
print("=" * 80)

# =============================================================================
# 2. CREATE PACKAGE DIRECTORY
# =============================================================================

if os.path.exists(FINAL_PACKAGE):

    shutil.rmtree(
        FINAL_PACKAGE
    )

os.makedirs(
    FINAL_PACKAGE,
    exist_ok=True
)

os.makedirs(
    os.path.join(
        FINAL_PACKAGE,
        "qc_reports"
    ),
    exist_ok=True
)

# =============================================================================
# 3. COPY FINAL MANIFEST
# =============================================================================

shutil.copy2(
    FINAL_QC,
    FINAL_MANIFEST
)

# =============================================================================
# 4. COPY QC REPORTS
# =============================================================================

files_to_copy = [

    (
        STEP29_AUDIT,
        "qc_reports/step29_structural_audit.csv"
    ),

    (
        STEP29_SUMMARY,
        "qc_reports/step29_structural_audit_summary.txt"
    ),

    (
        STEP30_CASE,
        "qc_reports/step30_reader_consistency.csv"
    ),

    (
        STEP30_PAIRWISE,
        "qc_reports/step30_pairwise_agreement.csv"
    ),

    (
        STEP30_SUMMARY,
        "qc_reports/step30_reader_consistency_summary.txt"
    ),

    (
        STEP31_CASE,
        "qc_reports/step31_refined_mask_qc.csv"
    ),

    (
        STEP31_SLICE,
        "qc_reports/step31_slice_level_agreement.csv"
    ),

    (
        STEP31_SUMMARY,
        "qc_reports/step31_refined_mask_qc_summary.txt"
    )
]

for source, relative_destination in files_to_copy:

    if os.path.exists(source):

        destination = os.path.join(
            FINAL_PACKAGE,
            relative_destination
        )

        shutil.copy2(
            source,
            destination
        )

# =============================================================================
# 5. BUILD README
# =============================================================================

today = datetime.now().strftime(
    "%Y-%m-%d"
)

train_count = int(
    (
        final_df["split"]
        == "train"
    ).sum()
)

val_count = int(
    (
        final_df["split"]
        == "val"
    ).sum()
)

test_count = int(
    (
        final_df["split"]
        == "test"
    ).sum()
)

patient_count = (
    final_df["patient_id"]
    .nunique()
)

series_count = (
    final_df["SeriesInstanceUID"]
    .nunique()
)

review_count = int(
    final_df[
        "requires_qc_review"
    ].sum()
)

qc_pass_count = int(
    final_df[
        "usable_for_processed_dataset"
    ].sum()
)

agreement_counts = (
    final_df[
        "final_qc_class"
    ]
    .value_counts()
    .to_dict()
)

readme = f"""
===============================================================================
FINAL 325-NODULE PROCESSED DATASET
===============================================================================

Package created: {today}

DATASET SIZE
-------------------------------------------------------------------------------

Total nodules:                  325
Unique patients:                {patient_count}
Unique CT series:               {series_count}

Train nodules:                  {train_count}
Validation nodules:             {val_count}
Test nodules:                   {test_count}

EXCLUDED CASES
-------------------------------------------------------------------------------

nodule_029
nodule_085

These two cases are excluded from the final 325-nodule cohort.

STRUCTURAL QUALITY CONTROL
-------------------------------------------------------------------------------

Structurally valid cases:       {qc_pass_count} / 325
Cases requiring QC review:      {review_count} / 325

The processed dataset retains the original image and mask data used to
construct the processed cohort.

MASK / READER QC
-------------------------------------------------------------------------------

Final QC classifications:

"""

for category, count in sorted(
    agreement_counts.items()
):

    readme += (
        f"  {category}: {count}\n"
    )

readme += """

IMPORTANT PROVENANCE LIMITATION
-------------------------------------------------------------------------------

The dataset establishes:

    native nodule folder
        ->
    patient
        ->
    SeriesInstanceUID
        ->
    XML annotation series

However, the exact physical mapping:

    native nodule-X
        ->
    specific XML unblindedReadNodule

has NOT been established for the full cohort.

Therefore XML malignancy ratings have NOT been assigned as native-nodule
ground-truth labels.

This distinction is intentionally preserved in the final manifest.

MASK INFORMATION
-------------------------------------------------------------------------------

The native dataset contains up to four mask directories:

    mask-0
    mask-1
    mask-2
    mask-3

The directory names are retained as provided by the source dataset.

This package does not claim that mask-0, mask-1, etc. correspond to
specific identifiable radiologists.

TRAIN / VALIDATION / TEST
-------------------------------------------------------------------------------

Splits are patient-isolated.

No patient occurs in more than one split.

DATA PROCESSING
-------------------------------------------------------------------------------

The processed cohort was created without assigning malignancy labels.

The original processed pipeline did not apply data augmentation.

The final QC steps do not modify source PNG files or segmentation masks.

FILES
-------------------------------------------------------------------------------

final_325_dataset_manifest.csv

qc_reports/
    step29_structural_audit.csv
    step29_structural_audit_summary.txt
    step30_reader_consistency.csv
    step30_pairwise_agreement.csv
    step30_reader_consistency_summary.txt
    step31_refined_mask_qc.csv
    step31_slice_level_agreement.csv
    step31_refined_mask_qc_summary.txt

===============================================================================
"""

with open(
    README_PATH,
    "w"
) as f:

    f.write(
        readme.strip()
    )

# =============================================================================
# 6. FINAL SUMMARY
# =============================================================================

summary = f"""
===============================================================================
FINAL DATASET PACKAGE SUMMARY
===============================================================================

Cohort:
    325 nodules
    {patient_count} patients
    {series_count} CT series

Splits:
    Train:       {train_count}
    Validation:  {val_count}
    Test:        {test_count}

Excluded:
    nodule_029
    nodule_085

QC:
    Structurally valid:       {qc_pass_count} / 325
    QC review flagged:        {review_count} / 325

Provenance:
    Native-to-XML identity proven:  NO
    Native malignancy labels:      NONE

Package:
    {FINAL_PACKAGE}

The package contains the final manifest and the supporting QC reports.
The processed image/mask dataset remains in:

    {PROCESSED_ROOT}

===============================================================================
"""

with open(
    PACKAGE_SUMMARY,
    "w"
) as f:

    f.write(
        summary.strip()
    )

print(summary)

# =============================================================================
# 7. FINAL PACKAGE VALIDATION
# =============================================================================

print("=" * 80)
print("FINAL PACKAGE VALIDATION")
print("=" * 80)

print(
    f"Final manifest exists: "
    f"{os.path.exists(FINAL_MANIFEST)}"
)

print(
    f"README exists: "
    f"{os.path.exists(README_PATH)}"
)

print(
    f"Summary exists: "
    f"{os.path.exists(PACKAGE_SUMMARY)}"
)

qc_files_present = sum(
    os.path.exists(source)
    for source, _ in files_to_copy
)

print(
    f"QC reports copied: "
    f"{qc_files_present} / {len(files_to_copy)}"
)

assert os.path.exists(
    FINAL_MANIFEST
)

assert os.path.exists(
    README_PATH
)

assert os.path.exists(
    PACKAGE_SUMMARY
)

# =============================================================================
# 8. FINAL COMPLETION
# =============================================================================

print("\n" + "=" * 80)
print("STEP 33 COMPLETE")
print("=" * 80)

print(
    "\nFinal dataset-processing package created:"
)

print(
    f"  {FINAL_PACKAGE}"
)

print(
    "\nFinal manifest:"
)

print(
    f"  {FINAL_MANIFEST}"
)

print(
    "\nREADME:"
)

print(
    f"  {README_PATH}"
)

print(
    "\nQC reports:"
)

print(
    f"  {os.path.join(FINAL_PACKAGE, 'qc_reports')}"
)

print("\n✓ Dataset processing pipeline complete.")
print("✓ No malignancy labels fabricated.")
print("✓ Native-to-XML identity limitation documented.")
print("✓ Patient-isolated splits preserved.")
print("✓ QC reports preserved.")
print("=" * 80)

STEP 33 — FINAL DATASET MANIFEST + DOCUMENTATION PACKAGE

FINAL DATASET PACKAGE SUMMARY

Cohort:
    325 nodules
    247 patients
    247 CT series

Splits:
    Train:       235
    Validation:  44
    Test:        46

Excluded:
    nodule_029
    nodule_085

QC:
    Structurally valid:       325 / 325
    QC review flagged:        157 / 325

Provenance:
    Native-to-XML identity proven:  NO
    Native malignancy labels:      NONE

Package:
    /content/final_325_dataset_package

The package contains the final manifest and the supporting QC reports.
The processed image/mask dataset remains in:

    /content/processed_325_nodules


FINAL PACKAGE VALIDATION
Final manifest exists: True
README exists: True
Summary exists: True
QC reports copied: 8 / 8

STEP 33 COMPLETE

Final dataset-processing package created:
  /content/final_325_dataset_package

Final manifest:
  /content/final_325_dataset_package/final_325_dataset_manifest.csv

README:
  /content/final_325_dataset_package/README.txt



In [ ]:
# =============================================================================
# STEP 34 — CONTROLLED ORIGINAL vs PROCESSED DATASET BENCHMARK
# =============================================================================
#
# PURPOSE
# Compare the original PNG images against the processed images while keeping
# everything else identical:
#
#   - same 325 nodules
#   - same patient-level train/val/test split
#   - same masks
#   - same U-Net architecture
#   - same optimizer
#   - same learning rate
#   - same batch size
#   - same number of epochs
#   - same random seed
#
# This isolates whether preprocessing changes segmentation performance.
#
# IMPORTANT:
# This benchmark uses the processed dataset's consensus masks for BOTH arms.
# Therefore the experiment compares IMAGE PROCESSING, not different labels.
# =============================================================================

import os
import io
import zipfile
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# =============================================================================
# 1. PATHS
# =============================================================================

ZIP_PATH = "/content/kagl_lidc_idri.zip"

ML_MANIFEST = "/content/final_325_ml_manifest.csv"

PROCESSED_ROOT = "/content/processed_325_nodules"

OUTPUT_DIR = "/content/step34_original_vs_processed"

ORIGINAL_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "original_unet_best.pt"
)

PROCESSED_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "processed_unet_best.pt"
)

HISTORY_CSV = os.path.join(
    OUTPUT_DIR,
    "step34_training_history.csv"
)

RESULTS_CSV = os.path.join(
    OUTPUT_DIR,
    "step34_original_vs_processed_results.csv"
)

SUMMARY_TXT = os.path.join(
    OUTPUT_DIR,
    "step34_original_vs_processed_summary.txt"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# =============================================================================
# 2. EXPERIMENT SETTINGS
# =============================================================================

SEED = 42

BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-3

NUM_WORKERS = 0

IMAGE_SIZE = 128

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 80)
print("STEP 34 — CONTROLLED ORIGINAL vs PROCESSED DATASET BENCHMARK")
print("=" * 80)

print(f"\nDevice: {DEVICE}")

if torch.cuda.is_available():
    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )

# =============================================================================
# 3. REPRODUCIBILITY
# =============================================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)

# =============================================================================
# 4. LOAD FINAL 325-NODULE MANIFEST
# =============================================================================

if not os.path.exists(ML_MANIFEST):
    raise FileNotFoundError(
        f"Missing manifest: {ML_MANIFEST}"
    )

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"Missing ZIP: {ZIP_PATH}"
    )

if not os.path.isdir(PROCESSED_ROOT):
    raise FileNotFoundError(
        f"Missing processed dataset: {PROCESSED_ROOT}"
    )

manifest = pd.read_csv(
    ML_MANIFEST
)

assert len(manifest) == 325

# -----------------------------------------------------------------------------
# Normalize split column
# -----------------------------------------------------------------------------

manifest["split"] = (
    manifest["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# -----------------------------------------------------------------------------
# Normalize identifiers
# -----------------------------------------------------------------------------

manifest["subset_nodule_id"] = (
    manifest["subset_nodule_id"]
    .astype(str)
    .str.strip()
)

manifest["patient_id"] = (
    manifest["patient_id"]
    .astype(str)
    .str.strip()
)

manifest["native_nodule_id"] = (
    manifest["native_nodule_id"]
    .astype(str)
    .str.strip()
)

# -----------------------------------------------------------------------------
# Exclusion checks
# -----------------------------------------------------------------------------

assert "nodule_029" not in set(
    manifest["subset_nodule_id"]
)

assert "nodule_085" not in set(
    manifest["subset_nodule_id"]
)

assert (
    manifest["subset_nodule_id"].nunique()
    == 325
)

print("\nCOHORT")
print("-" * 80)

print(
    f"Total nodules:   {len(manifest)}"
)

print(
    f"Patients:         {manifest['patient_id'].nunique()}"
)

print(
    manifest["split"]
    .value_counts()
    .sort_index()
    .to_string()
)

# =============================================================================
# 5. PATIENT LEAKAGE CHECK
# =============================================================================

train_patients = set(
    manifest.loc[
        manifest["split"] == "train",
        "patient_id"
    ]
)

val_patients = set(
    manifest.loc[
        manifest["split"] == "val",
        "patient_id"
    ]
)

test_patients = set(
    manifest.loc[
        manifest["split"] == "test",
        "patient_id"
    ]
)

assert len(
    train_patients & val_patients
) == 0

assert len(
    train_patients & test_patients
) == 0

assert len(
    val_patients & test_patients
) == 0

print("\nPATIENT SPLIT")
print("-" * 80)

print(
    f"Train patients:       {len(train_patients)}"
)

print(
    f"Validation patients:  {len(val_patients)}"
)

print(
    f"Test patients:        {len(test_patients)}"
)

print(
    "✓ Zero patient leakage"
)

# =============================================================================
# 6. BUILD PROCESSED IMAGE LOOKUP
# =============================================================================

print("\nINDEXING PROCESSED DATASET")
print("-" * 80)

processed_image_lookup = {}

processed_files = []

for subset_id in manifest[
    "subset_nodule_id"
].tolist():

    case_dir = os.path.join(
        PROCESSED_ROOT,
        subset_id
    )

    image_dir = os.path.join(
        case_dir,
        "images"
    )

    if not os.path.isdir(image_dir):
        raise FileNotFoundError(
            f"Missing processed image directory: "
            f"{image_dir}"
        )

    files = [
        f
        for f in os.listdir(
            image_dir
        )
        if f.lower().endswith(".png")
    ]

    if len(files) == 0:
        raise FileNotFoundError(
            f"No processed images found in {image_dir}"
        )

    # Numeric filename sorting
    def numeric_key(filename):

        digits = "".join(
            c
            for c in filename
            if c.isdigit()
        )

        return (
            int(digits)
            if digits
            else 0,
            filename
        )

    files = sorted(
        files,
        key=numeric_key
    )

    processed_image_lookup[
        subset_id
    ] = [
        os.path.join(
            image_dir,
            f
        )
        for f in files
    ]

    processed_files.extend(
        processed_image_lookup[
            subset_id
        ]
    )

print(
    f"Processed image files indexed: "
    f"{len(processed_files)}"
)

# =============================================================================
# 7. ORIGINAL ZIP LOOKUP
# =============================================================================

print("\nINDEXING ORIGINAL ZIP")
print("-" * 80)

zip_file = zipfile.ZipFile(
    ZIP_PATH,
    "r"
)

zip_names = set(
    zip_file.namelist()
)

original_image_lookup = {}

total_original_images = 0

for _, row in manifest.iterrows():

    subset_id = row[
        "subset_nodule_id"
    ]

    patient_id = row[
        "patient_id"
    ]

    native_id = row[
        "native_nodule_id"
    ]

    prefix = (
        f"LIDC-IDRI-slices/"
        f"{patient_id}/"
        f"{native_id}/"
        f"images/"
    )

    files = [
        f
        for f in zip_names
        if f.startswith(prefix)
        and f.lower().endswith(".png")
    ]

    if len(files) == 0:

        raise FileNotFoundError(
            "No original image PNGs found for "
            f"{subset_id}: {prefix}"
        )

    def zip_numeric_key(filename):

        base = os.path.basename(
            filename
        )

        digits = "".join(
            c
            for c in base
            if c.isdigit()
        )

        return (
            int(digits)
            if digits
            else 0,
            base
        )

    files = sorted(
        files,
        key=zip_numeric_key
    )

    original_image_lookup[
        subset_id
    ] = files

    total_original_images += len(
        files
    )

print(
    f"Original image files indexed: "
    f"{total_original_images}"
)

# =============================================================================
# 8. MASK LOOKUP
# =============================================================================
#
# We use the same native mask data for BOTH experiments.
# This ensures the benchmark tests image preprocessing rather than label
# differences.
#
# The processed dataset's mask-0..mask-3 files are converted into a majority
# consensus mask for each slice:
#
#   >= 50% of available masks = foreground
#
# =============================================================================

print("\nINDEXING CONSENSUS MASKS")
print("-" * 80)

mask_lookup = {}

for _, row in manifest.iterrows():

    subset_id = row[
        "subset_nodule_id"
    ]

    case_dir = os.path.join(
        PROCESSED_ROOT,
        subset_id
    )

    reader_files = {}

    # -------------------------------------------------------------
    # Find mask files
    # -------------------------------------------------------------

    for reader in range(4):

        mask_dir = os.path.join(
            case_dir,
            f"mask-{reader}"
        )

        if not os.path.isdir(
            mask_dir
        ):

            reader_files[
                reader
            ] = []

            continue

        files = [
            f
            for f in os.listdir(
                mask_dir
            )
            if f.lower().endswith(".png")
        ]

        def mask_sort_key(filename):

            digits = "".join(
                c
                for c in filename
                if c.isdigit()
            )

            return (
                int(digits)
                if digits
                else 0,
                filename
            )

        files = sorted(
            files,
            key=mask_sort_key
        )

        reader_files[
            reader
        ] = [
            os.path.join(
                mask_dir,
                f
            )
            for f in files
        ]

    max_depth = max(
        len(v)
        for v in reader_files.values()
    )

    if max_depth == 0:

        raise RuntimeError(
            f"No masks found for {subset_id}"
        )

    consensus_slices = []

    for z in range(
        max_depth
    ):

        reader_slice_arrays = []

        for reader in range(4):

            files = reader_files[
                reader
            ]

            if z >= len(files):

                continue

            with Image.open(
                files[z]
            ) as img:

                arr = np.asarray(
                    img.convert("L"),
                    dtype=np.uint8
                )

            reader_slice_arrays.append(
                arr > 0
            )

        if not reader_slice_arrays:

            continue

        # Majority vote across available reader masks.
        stacked = np.stack(
            reader_slice_arrays,
            axis=0
        )

        consensus = (
            stacked.mean(
                axis=0
            )
            >= 0.5
        )

        consensus_slices.append(
            consensus
        )

    if not consensus_slices:

        raise RuntimeError(
            f"Could not construct consensus mask "
            f"for {subset_id}"
        )

    mask_lookup[
        subset_id
    ] = np.stack(
        consensus_slices,
        axis=0
    )

print(
    f"Consensus masks indexed: "
    f"{len(mask_lookup)} / 325"
)

# =============================================================================
# 9. VERIFY IMAGE COUNTS AGAINST CONSENSUS DEPTH
# =============================================================================

print("\nVERIFYING IMAGE / MASK DEPTH")
print("-" * 80)

for subset_id in manifest[
    "subset_nodule_id"
].tolist():

    original_depth = len(
        original_image_lookup[
            subset_id
        ]
    )

    processed_depth = len(
        processed_image_lookup[
            subset_id
        ]
    )

    mask_depth = mask_lookup[
        subset_id
    ].shape[0]

    if not (
        original_depth
        ==
        processed_depth
        ==
        mask_depth
    ):

        raise ValueError(
            f"Depth mismatch for {subset_id}: "
            f"original={original_depth}, "
            f"processed={processed_depth}, "
            f"mask={mask_depth}"
        )

print(
    "✓ Image/mask depth alignment verified for all 325 nodules."
)

# =============================================================================
# 10. DATASET CLASS
# =============================================================================

class BenchmarkDataset(Dataset):

    def __init__(
        self,
        dataframe,
        mode,
        zip_handle=None
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.mode = mode

        self.zip = zip_handle

    def __len__(self):

        return len(
            self.df
        )

    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[
            index
        ]

        subset_id = str(
            row[
                "subset_nodule_id"
            ]
        )

        slice_index = int(
            row[
                "slice_index"
            ]
        )

        # ---------------------------------------------------------
        # LOAD IMAGE
        # ---------------------------------------------------------

        if self.mode == "original":

            zip_path = (
                original_image_lookup[
                    subset_id
                ][slice_index]
            )

            with self.zip.open(
                zip_path
            ) as f:

                img_bytes = f.read()

            with Image.open(
                io.BytesIO(
                    img_bytes
                )
            ) as img:

                image = np.asarray(
                    img.convert("L"),
                    dtype=np.float32
                )

        elif self.mode == "processed":

            path = (
                processed_image_lookup[
                    subset_id
                ][slice_index]
            )

            with Image.open(
                path
            ) as img:

                image = np.asarray(
                    img.convert("L"),
                    dtype=np.float32
                )

        else:

            raise ValueError(
                f"Unknown mode: {self.mode}"
            )

        # ---------------------------------------------------------
        # Normalize both sources identically
        #
        # This is critical for a fair comparison.
        # ---------------------------------------------------------

        image_min = image.min()
        image_max = image.max()

        if image_max > image_min:

            image = (
                image - image_min
            ) / (
                image_max - image_min
            )

        else:

            image = np.zeros_like(
                image,
                dtype=np.float32
            )

        # ---------------------------------------------------------
        # MASK
        # ---------------------------------------------------------

        mask = (
            mask_lookup[
                subset_id
            ][slice_index]
            .astype(np.float32)
        )

        # ---------------------------------------------------------
        # Tensor format
        # ---------------------------------------------------------

        image = torch.from_numpy(
            image.copy()
        ).unsqueeze(0)

        mask = torch.from_numpy(
            mask.copy()
        ).unsqueeze(0)

        return (
            image.float(),
            mask.float()
        )


# =============================================================================
# 11. BUILD SLICE TABLE
# =============================================================================

slice_records = []

for _, row in manifest.iterrows():

    subset_id = str(
        row[
            "subset_nodule_id"
        ]
    )

    patient_id = str(
        row[
            "patient_id"
        ]
    )

    split = str(
        row[
            "split"
        ]
    )

    depth = len(
        original_image_lookup[
            subset_id
        ]
    )

    for z in range(
        depth
    ):

        slice_records.append({

            "subset_nodule_id":
                subset_id,

            "patient_id":
                patient_id,

            "split":
                split,

            "slice_index":
                z
        })

slice_df = pd.DataFrame(
    slice_records
)

print("\nSLICE MANIFEST")
print("-" * 80)

print(
    f"Total slices: "
    f"{len(slice_df)}"
)

# =============================================================================
# 12. CREATE TRAIN / VAL / TEST DATASETS
# =============================================================================

train_slices = slice_df[
    slice_df["split"] == "train"
].copy()

val_slices = slice_df[
    slice_df["split"] == "val"
].copy()

test_slices = slice_df[
    slice_df["split"] == "test"
].copy()

print(
    f"Train slices: "
    f"{len(train_slices)}"
)

print(
    f"Validation slices: "
    f"{len(val_slices)}"
)

print(
    f"Test slices: "
    f"{len(test_slices)}"
)

# =============================================================================
# 13. DATA LOADERS
# =============================================================================

def create_loaders(mode):

    train_ds = BenchmarkDataset(
        train_slices,
        mode=mode,
        zip_handle=zip_file
    )

    val_ds = BenchmarkDataset(
        val_slices,
        mode=mode,
        zip_handle=zip_file
    )

    test_ds = BenchmarkDataset(
        test_slices,
        mode=mode,
        zip_handle=zip_file
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    return (
        train_loader,
        val_loader,
        test_loader
    )


# =============================================================================
# 14. U-NET
# =============================================================================

class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )

        )

    def forward(self, x):

        return self.block(x)


class UNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.enc1 = DoubleConv(
            1,
            32
        )

        self.enc2 = DoubleConv(
            32,
            64
        )

        self.enc3 = DoubleConv(
            64,
            128
        )

        self.pool = nn.MaxPool2d(
            2
        )

        self.bottleneck = DoubleConv(
            128,
            256
        )

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        self.out = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )

    def forward(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        b = self.bottleneck(
            self.pool(e3)
        )

        d3 = self.up3(b)

        d3 = torch.cat(
            [d3, e3],
            dim=1
        )

        d3 = self.dec3(
            d3
        )

        d2 = self.up2(d3)

        d2 = torch.cat(
            [d2, e2],
            dim=1
        )

        d2 = self.dec2(
            d2
        )

        d1 = self.up1(d2)

        d1 = torch.cat(
            [d1, e1],
            dim=1
        )

        d1 = self.dec1(
            d1
        )

        return self.out(d1)


# =============================================================================
# 15. LOSS / DICE
# =============================================================================

bce_loss = nn.BCEWithLogitsLoss()


def dice_score_from_logits(
    logits,
    target
):

    probability = torch.sigmoid(
        logits
    )

    prediction = (
        probability >= 0.5
    ).float()

    intersection = (
        prediction * target
    ).sum(
        dim=(1, 2, 3)
    )

    denominator = (
        prediction.sum(
            dim=(1, 2, 3)
        )
        +
        target.sum(
            dim=(1, 2, 3)
        )
    )

    dice = (
        (2.0 * intersection + 1e-6)
        /
        (denominator + 1e-6)
    )

    return dice.mean()


def combined_loss(
    logits,
    target
):

    probability = torch.sigmoid(
        logits
    )

    intersection = (
        probability * target
    ).sum(
        dim=(1, 2, 3)
    )

    denominator = (
        probability.sum(
            dim=(1, 2, 3)
        )
        +
        target.sum(
            dim=(1, 2, 3)
        )
    )

    soft_dice = (
        1.0
        -
        (
            2.0 * intersection + 1e-6
        )
        /
        (
            denominator + 1e-6
        )
    ).mean()

    return (
        bce_loss(
            logits,
            target
        )
        +
        soft_dice
    )


# =============================================================================
# 16. TRAINING FUNCTION
# =============================================================================

def train_one_condition(
    condition_name,
    model_path
):

    print("\n" + "=" * 80)

    print(
        f"TRAINING CONDITION: "
        f"{condition_name.upper()}"
    )

    print("=" * 80)

    # -------------------------------------------------------------
    # Reset RNG so both models start identically.
    # -------------------------------------------------------------

    set_seed(
        SEED
    )

    (
        train_loader,
        val_loader,
        test_loader
    ) = create_loaders(
        condition_name
    )

    model = UNet().to(
        DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    best_val_dice = -1.0

    history = []

    for epoch in range(
        1,
        EPOCHS + 1
    ):

        # =========================================================
        # TRAIN
        # =========================================================

        model.train()

        running_loss = 0.0
        running_dice = 0.0
        batches = 0

        for images, masks in train_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                images
            )

            loss = combined_loss(
                logits,
                masks
            )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item()
            )

            running_dice += (
                dice_score_from_logits(
                    logits,
                    masks
                ).item()
            )

            batches += 1

        train_loss = (
            running_loss
            /
            max(batches, 1)
        )

        train_dice = (
            running_dice
            /
            max(batches, 1)
        )

        # =========================================================
        # VALIDATION
        # =========================================================

        model.eval()

        val_loss_total = 0.0
        val_dice_total = 0.0
        val_batches = 0

        with torch.no_grad():

            for images, masks in val_loader:

                images = images.to(
                    DEVICE,
                    non_blocking=True
                )

                masks = masks.to(
                    DEVICE,
                    non_blocking=True
                )

                logits = model(
                    images
                )

                loss = combined_loss(
                    logits,
                    masks
                )

                val_loss_total += (
                    loss.item()
                )

                val_dice_total += (
                    dice_score_from_logits(
                        logits,
                        masks
                    ).item()
                )

                val_batches += 1

        val_loss = (
            val_loss_total
            /
            max(val_batches, 1)
        )

        val_dice = (
            val_dice_total
            /
            max(val_batches, 1)
        )

        history.append({

            "condition":
                condition_name,

            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "train_dice":
                train_dice,

            "val_loss":
                val_loss,

            "val_dice":
                val_dice

        })

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Dice: {train_dice:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Dice: {val_dice:.4f}"
        )

        # ---------------------------------------------------------
        # Save best checkpoint.
        # ---------------------------------------------------------

        if val_dice > best_val_dice:

            best_val_dice = val_dice

            torch.save(
                {
                    "model_state_dict":
                        model.state_dict(),

                    "best_val_dice":
                        best_val_dice,

                    "condition":
                        condition_name,

                    "seed":
                        SEED
                },
                model_path
            )

            print(
                f"  ✓ Best model saved "
                f"(val Dice={best_val_dice:.4f})"
            )

    # =========================================================================
    # TEST USING BEST CHECKPOINT
    # =========================================================================

    checkpoint = torch.load(
        model_path,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model.eval()

    test_loss_total = 0.0
    test_dice_total = 0.0
    test_batches = 0

    with torch.no_grad():

        for images, masks in test_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            logits = model(
                images
            )

            loss = combined_loss(
                logits,
                masks
            )

            test_loss_total += (
                loss.item()
            )

            test_dice_total += (
                dice_score_from_logits(
                    logits,
                    masks
                ).item()
            )

            test_batches += 1

    test_loss = (
        test_loss_total
        /
        max(test_batches, 1)
    )

    test_dice = (
        test_dice_total
        /
        max(test_batches, 1)
    )

    print("\nTEST RESULTS")

    print(
        f"Best validation Dice: "
        f"{best_val_dice:.4f}"
    )

    print(
        f"Test loss: "
        f"{test_loss:.4f}"
    )

    print(
        f"Test Dice: "
        f"{test_dice:.4f}"
    )

    return {
        "condition":
            condition_name,

        "best_val_dice":
            best_val_dice,

        "test_loss":
            test_loss,

        "test_dice":
            test_dice
    }, history


# =============================================================================
# 17. DATA EQUIVALENCE SANITY CHECK
# =============================================================================

print("\n" + "=" * 80)
print("IMAGE EQUIVALENCE SANITY CHECK")
print("=" * 80)

# Compare a sample from every nodule.
#
# This tells us whether preprocessing preserved the actual pixel values after
# normalization. We calculate:
#
#   mean absolute difference
#   maximum absolute difference
#
# after converting both images to float32 and normalizing them using the same
# min-max rule used by the benchmark.
# =============================================================================

comparison_rows = []

for counter, (_, row) in enumerate(
    manifest.iterrows(),
    start=1
):

    subset_id = str(
        row["subset_nodule_id"]
    )

    original_files = (
        original_image_lookup[
            subset_id
        ]
    )

    processed_files_case = (
        processed_image_lookup[
            subset_id
        ]
    )

    depth = min(
        len(original_files),
        len(processed_files_case)
    )

    total_abs_difference = 0.0
    max_difference = 0.0
    pixel_count = 0

    # Compare every slice.
    for z in range(depth):

        # Original
        with zip_file.open(
            original_files[z]
        ) as f:

            original_bytes = f.read()

        with Image.open(
            io.BytesIO(
                original_bytes
            )
        ) as img:

            original = np.asarray(
                img.convert("L"),
                dtype=np.float32
            )

        # Processed
        with Image.open(
            processed_files_case[z]
        ) as img:

            processed = np.asarray(
                img.convert("L"),
                dtype=np.float32
            )

        if original.shape != processed.shape:

            raise ValueError(
                f"Dimension mismatch for "
                f"{subset_id}, slice {z}: "
                f"{original.shape} vs "
                f"{processed.shape}"
            )

        # Apply identical normalization.
        if (
            original.max()
            >
            original.min()
        ):

            original_norm = (
                original - original.min()
            ) / (
                original.max()
                -
                original.min()
            )

        else:

            original_norm = np.zeros_like(
                original
            )

        if (
            processed.max()
            >
            processed.min()
        ):

            processed_norm = (
                processed - processed.min()
            ) / (
                processed.max()
                -
                processed.min()
            )

        else:

            processed_norm = np.zeros_like(
                processed
            )

        difference = np.abs(
            original_norm
            -
            processed_norm
        )

        total_abs_difference += (
            difference.sum()
        )

        max_difference = max(
            max_difference,
            float(
                difference.max()
            )
        )

        pixel_count += (
            difference.size
        )

    mean_difference = (
        total_abs_difference
        /
        max(pixel_count, 1)
    )

    comparison_rows.append({

        "subset_nodule_id":
            subset_id,

        "mean_absolute_difference":
            mean_difference,

        "max_absolute_difference":
            max_difference

    })

    if counter % 25 == 0:

        print(
            f"Compared {counter} / 325 nodules"
        )

equivalence_df = pd.DataFrame(
    comparison_rows
)

print("\nEQUIVALENCE RESULTS")
print("-" * 80)

print(
    f"Mean absolute difference across cases: "
    f"{equivalence_df['mean_absolute_difference'].mean():.8f}"
)

print(
    f"Maximum observed absolute difference: "
    f"{equivalence_df['max_absolute_difference'].max():.8f}"
)

# =============================================================================
# 18. RUN ORIGINAL EXPERIMENT
# =============================================================================

original_result, original_history = (
    train_one_condition(
        "original",
        ORIGINAL_MODEL_PATH
    )
)

# =============================================================================
# 19. RUN PROCESSED EXPERIMENT
# =============================================================================

processed_result, processed_history = (
    train_one_condition(
        "processed",
        PROCESSED_MODEL_PATH
    )
)

# =============================================================================
# 20. SAVE TRAINING HISTORY
# =============================================================================

history_df = pd.DataFrame(
    original_history
    +
    processed_history
)

history_df.to_csv(
    HISTORY_CSV,
    index=False
)

# =============================================================================
# 21. COMPARE FINAL RESULTS
# =============================================================================

results_df = pd.DataFrame(
    [
        original_result,
        processed_result
    ]
)

original_test_dice = (
    original_result[
        "test_dice"
    ]
)

processed_test_dice = (
    processed_result[
        "test_dice"
    ]
)

original_val_dice = (
    original_result[
        "best_val_dice"
    ]
)

processed_val_dice = (
    processed_result[
        "best_val_dice"
    ]
)

dice_difference = (
    processed_test_dice
    -
    original_test_dice
)

val_difference = (
    processed_val_dice
    -
    original_val_dice
)

results_df[
    "test_dice_difference_vs_original"
] = [
    0.0,
    dice_difference
]

results_df[
    "best_val_dice_difference_vs_original"
] = [
    0.0,
    val_difference
]

results_df.to_csv(
    RESULTS_CSV,
    index=False
)

# =============================================================================
# 22. INTERPRETATION
# =============================================================================

if dice_difference > 0.01:

    interpretation = (
        "Processed dataset performed better by more than "
        "0.01 test Dice in this controlled benchmark."
    )

elif dice_difference < -0.01:

    interpretation = (
        "Processed dataset performed worse by more than "
        "0.01 test Dice in this controlled benchmark."
    )

else:

    interpretation = (
        "Original and processed datasets produced very similar "
        "test Dice scores (absolute difference <= 0.01)."
    )

# =============================================================================
# 23. FINAL REPORT
# =============================================================================

summary = f"""
===============================================================================
STEP 34 — ORIGINAL vs PROCESSED DATASET BENCHMARK
===============================================================================

EXPERIMENT DESIGN
-------------------------------------------------------------------------------

Cohort:
    325 nodules
    {manifest['patient_id'].nunique()} patients

Train/Validation/Test:
    {len(train_patients)} train patients
    {len(val_patients)} validation patients
    {len(test_patients)} test patients

Patient leakage:
    0

Training configuration:
    Model:           2D U-Net
    Epochs:          {EPOCHS}
    Batch size:      {BATCH_SIZE}
    Learning rate:   {LEARNING_RATE}
    Random seed:     {SEED}
    Device:          {DEVICE}

CRITICAL CONTROL:
    Both experiments use the same consensus masks.
    Only the image source changes:
        ORIGINAL  = images read directly from original ZIP
        PROCESSED  = images read from processed dataset

IMAGE EQUIVALENCE
-------------------------------------------------------------------------------

Mean absolute normalized pixel difference:
    {equivalence_df['mean_absolute_difference'].mean():.8f}

Maximum observed normalized pixel difference:
    {equivalence_df['max_absolute_difference'].max():.8f}

ORIGINAL DATASET
-------------------------------------------------------------------------------

Best validation Dice:
    {original_val_dice:.4f}

Test loss:
    {original_result['test_loss']:.4f}

Test Dice:
    {original_test_dice:.4f}

PROCESSED DATASET
-------------------------------------------------------------------------------

Best validation Dice:
    {processed_val_dice:.4f}

Test loss:
    {processed_result['test_loss']:.4f}

Test Dice:
    {processed_test_dice:.4f}

COMPARISON
-------------------------------------------------------------------------------

Processed - Original best validation Dice:
    {val_difference:+.4f}

Processed - Original test Dice:
    {dice_difference:+.4f}

INTERPRETATION
-------------------------------------------------------------------------------

{interpretation}

IMPORTANT:
    A small performance difference should not automatically be interpreted
    as a meaningful improvement. This is one controlled baseline experiment.

    The processed dataset may still be scientifically preferable because it
    provides stronger organization, metadata, QC, reproducibility, and
    patient-level split control even if segmentation performance is unchanged.

PROVENANCE
-------------------------------------------------------------------------------

Native-to-XML physical nodule identity:
    NOT established for the full cohort

Native malignancy labels:
    NONE ASSIGNED

The benchmark does not use malignancy labels.

OUTPUT FILES
-------------------------------------------------------------------------------

Original model:
    {ORIGINAL_MODEL_PATH}

Processed model:
    {PROCESSED_MODEL_PATH}

Training history:
    {HISTORY_CSV}

Comparison results:
    {RESULTS_CSV}

Image equivalence:
    {OUTPUT_DIR}/image_equivalence.csv

Summary:
    {SUMMARY_TXT}

===============================================================================
"""

# Save equivalence table.
equivalence_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "image_equivalence.csv"
    ),
    index=False
)

with open(
    SUMMARY_TXT,
    "w"
) as f:

    f.write(
        summary.strip()
    )

print(
    "\n" + summary
)

print("=" * 80)
print("STEP 34 COMPLETE")
print("=" * 80)

zip_file.close()

STEP 34 — CONTROLLED ORIGINAL vs PROCESSED DATASET BENCHMARK

Device: cpu

COHORT
--------------------------------------------------------------------------------
Total nodules:   325
Patients:         247
split
test      46
train    235
val       44

PATIENT SPLIT
--------------------------------------------------------------------------------
Train patients:       173
Validation patients:  37
Test patients:        37
✓ Zero patient leakage

INDEXING PROCESSED DATASET
--------------------------------------------------------------------------------
Processed image files indexed: 2000

INDEXING ORIGINAL ZIP
--------------------------------------------------------------------------------
Original image files indexed: 2000

INDEXING CONSENSUS MASKS
--------------------------------------------------------------------------------
Consensus masks indexed: 325 / 325

VERIFYING IMAGE / MASK DEPTH
--------------------------------------------------------------------------------
✓ Image/mask dept

In [ ]:
# =============================================================================
# STEP 35 — FINAL DATASET PROCESSING VALIDATION + MASTER REPORT
# =============================================================================

import os
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# INPUT FILES
# -------------------------------------------------------------------------

FINAL_QC = "/content/final_325_dataset_qc_classification.csv"
MASTER_MANIFEST = "/content/final_325_master_manifest.csv"
ML_MANIFEST = "/content/final_325_ml_manifest.csv"

STEP29_AUDIT = "/content/step29_final_structural_audit.csv"
STEP30_CASE = "/content/step30_mask_reader_consistency.csv"
STEP31_CASE = "/content/step31_refined_mask_qc.csv"

STEP34_RESULTS = (
    "/content/step34_original_vs_processed/"
    "step34_original_vs_processed_results.csv"
)

STEP34_EQUIVALENCE = (
    "/content/step34_original_vs_processed/"
    "image_equivalence.csv"
)

FINAL_PACKAGE = "/content/final_325_dataset_package"

OUTPUT_REPORT = (
    "/content/final_dataset_processing_report.txt"
)

OUTPUT_MASTER_CSV = (
    "/content/final_325_complete_processing_manifest.csv"
)

print("=" * 80)
print("STEP 35 — FINAL DATASET PROCESSING VALIDATION")
print("=" * 80)

# =============================================================================
# 1. CHECK FILES
# =============================================================================

required_files = [
    FINAL_QC,
    MASTER_MANIFEST,
    ML_MANIFEST,
    STEP29_AUDIT,
    STEP30_CASE,
    STEP31_CASE,
    STEP34_RESULTS,
    STEP34_EQUIVALENCE
]

print("\nFILE VALIDATION")
print("-" * 80)

missing_files = []

for path in required_files:

    exists = os.path.exists(path)

    print(
        f"{'✓' if exists else '✗'} {path}"
    )

    if not exists:
        missing_files.append(path)

if missing_files:

    raise FileNotFoundError(
        "Missing required outputs:\n"
        + "\n".join(missing_files)
    )

# =============================================================================
# 2. LOAD DATA
# =============================================================================

final_qc = pd.read_csv(FINAL_QC)
master = pd.read_csv(MASTER_MANIFEST)
ml = pd.read_csv(ML_MANIFEST)
step29 = pd.read_csv(STEP29_AUDIT)
step30 = pd.read_csv(STEP30_CASE)
step31 = pd.read_csv(STEP31_CASE)
step34 = pd.read_csv(STEP34_RESULTS)
equivalence = pd.read_csv(STEP34_EQUIVALENCE)

# =============================================================================
# 3. BASIC COHORT CHECK
# =============================================================================

print("\nCOHORT VALIDATION")
print("-" * 80)

assert len(final_qc) == 325
assert len(master) == 325
assert len(ml) == 325
assert len(step29) == 325
assert len(step30) == 325
assert len(step31) == 325

assert (
    final_qc["subset_nodule_id"].nunique()
    == 325
)

assert (
    master["subset_nodule_id"].nunique()
    == 325
)

assert (
    ml["subset_nodule_id"].nunique()
    == 325
)

assert "nodule_029" not in set(
    final_qc["subset_nodule_id"]
)

assert "nodule_085" not in set(
    final_qc["subset_nodule_id"]
)

print(
    "✓ 325 final nodules confirmed"
)

print(
    f"✓ Unique patients: "
    f"{final_qc['patient_id'].nunique()}"
)

print(
    f"✓ Unique CT series: "
    f"{final_qc['SeriesInstanceUID'].nunique()}"
)

print(
    "✓ nodule_029 and nodule_085 excluded"
)

# =============================================================================
# 4. TRAIN / VAL / TEST CHECK
# =============================================================================

print("\nPATIENT SPLIT VALIDATION")
print("-" * 80)

train_df = ml[
    ml["split"].astype(str).str.lower() == "train"
]

val_df = ml[
    ml["split"].astype(str).str.lower() == "val"
]

test_df = ml[
    ml["split"].astype(str).str.lower() == "test"
]

train_patients = set(
    train_df["patient_id"].astype(str)
)

val_patients = set(
    val_df["patient_id"].astype(str)
)

test_patients = set(
    test_df["patient_id"].astype(str)
)

train_val_overlap = (
    train_patients & val_patients
)

train_test_overlap = (
    train_patients & test_patients
)

val_test_overlap = (
    val_patients & test_patients
)

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print(
    f"Train nodules:       {len(train_df)}"
)

print(
    f"Validation nodules:  {len(val_df)}"
)

print(
    f"Test nodules:        {len(test_df)}"
)

print(
    f"Train patients:      {len(train_patients)}"
)

print(
    f"Validation patients: {len(val_patients)}"
)

print(
    f"Test patients:       {len(test_patients)}"
)

print(
    "✓ Zero patient leakage"
)

# =============================================================================
# 5. STEP 29 STRUCTURAL QC
# =============================================================================

print("\nSTEP 29 — STRUCTURAL QC")
print("-" * 80)

structural_pass = int(
    (
        step29["case_qc_status"]
        .astype(str)
        .str.upper()
        == "PASS"
    ).sum()
)

structural_fail = (
    len(step29)
    -
    structural_pass
)

print(
    f"Structural QC PASS: "
    f"{structural_pass} / 325"
)

print(
    f"Structural QC FAIL: "
    f"{structural_fail} / 325"
)

assert structural_pass == 325
assert structural_fail == 0

# =============================================================================
# 6. STEP 30 / 31 MASK QC
# =============================================================================

print("\nMASK / READER QC")
print("-" * 80)

reader_count_distribution = (
    step30["reader_count"]
    .value_counts()
    .sort_index()
)

active_reader_distribution = (
    step31["active_reader_count"]
    .value_counts()
    .sort_index()
)

strong_disagreement_count = int(
    step31[
        "strong_reader_disagreement"
    ].sum()
)

slice_extent_count = int(
    step31[
        "slice_extent_disagreement"
    ].sum()
)

print("Reader directory counts:")

for count, number in reader_count_distribution.items():

    print(
        f"  {count} reader directories: "
        f"{number} nodules"
    )

print("\nActive reader counts:")

for count, number in active_reader_distribution.items():

    print(
        f"  {count} active readers: "
        f"{number} nodules"
    )

print(
    f"\nStrong reader disagreement: "
    f"{strong_disagreement_count}"
)

print(
    f"Slice-extent disagreement: "
    f"{slice_extent_count}"
)

# =============================================================================
# 7. MALIGNANCY LABEL CHECK
# =============================================================================

print("\nLABEL / PROVENANCE CHECK")
print("-" * 80)

if (
    "native_malignancy_label_assigned"
    in final_qc.columns
):

    native_labels = int(
        final_qc[
            "native_malignancy_label_assigned"
        ].sum()
    )

else:

    native_labels = 0

print(
    f"Native malignancy labels assigned: "
    f"{native_labels}"
)

assert native_labels == 0

if (
    "native_to_xml_identity_status"
    in final_qc.columns
):

    proven_count = int(
        (
            final_qc[
                "native_to_xml_identity_status"
            ].astype(str).str.upper()
            == "PROVEN"
        ).sum()
    )

else:

    proven_count = 0

print(
    f"Native-to-XML identities marked proven: "
    f"{proven_count}"
)

assert proven_count == 0

print(
    "✓ No unsupported native malignancy labels"
)

# =============================================================================
# 8. STEP 34 ORIGINAL vs PROCESSED
# =============================================================================

print("\nSTEP 34 — ORIGINAL vs PROCESSED")
print("-" * 80)

# -------------------------------------------------------------------------
# Detect columns robustly
# -------------------------------------------------------------------------

def find_column(df, candidates):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        key = candidate.strip().lower()

        if key in lookup:

            return lookup[key]

    return None


condition_col = find_column(
    step34,
    [
        "condition",
        "dataset"
    ]
)

test_dice_col = find_column(
    step34,
    [
        "test_dice"
    ]
)

val_dice_col = find_column(
    step34,
    [
        "best_val_dice"
    ]
)

if condition_col is None:
    raise KeyError(
        f"Could not find condition column. "
        f"Columns: {list(step34.columns)}"
    )

if test_dice_col is None:
    raise KeyError(
        f"Could not find test_dice column. "
        f"Columns: {list(step34.columns)}"
    )

original_row = step34[
    step34[condition_col]
    .astype(str)
    .str.lower()
    == "original"
]

processed_row = step34[
    step34[condition_col]
    .astype(str)
    .str.lower()
    == "processed"
]

if len(original_row) != 1:
    raise ValueError(
        "Could not uniquely identify ORIGINAL result."
    )

if len(processed_row) != 1:
    raise ValueError(
        "Could not uniquely identify PROCESSED result."
    )

original_test_dice = float(
    original_row.iloc[0][test_dice_col]
)

processed_test_dice = float(
    processed_row.iloc[0][test_dice_col]
)

test_dice_difference = (
    processed_test_dice
    -
    original_test_dice
)

print(
    f"Original test Dice:  "
    f"{original_test_dice:.4f}"
)

print(
    f"Processed test Dice: "
    f"{processed_test_dice:.4f}"
)

print(
    f"Difference:           "
    f"{test_dice_difference:+.4f}"
)

# =============================================================================
# 9. IMAGE EQUIVALENCE
# =============================================================================

print("\nIMAGE EQUIVALENCE")
print("-" * 80)

mean_difference = float(
    equivalence[
        "mean_absolute_difference"
    ].mean()
)

max_difference = float(
    equivalence[
        "max_absolute_difference"
    ].max()
)

print(
    f"Mean absolute difference: "
    f"{mean_difference:.8f}"
)

print(
    f"Maximum absolute difference: "
    f"{max_difference:.8f}"
)

# Exact preservation check.
images_identical = (
    np.isclose(
        mean_difference,
        0.0,
        atol=1e-12
    )
    and
    np.isclose(
        max_difference,
        0.0,
        atol=1e-12
    )
)

if images_identical:

    print(
        "✓ Original and processed images are "
        "pixel-equivalent after benchmark normalization."
    )

else:

    print(
        "NOTE: Original and processed images "
        "are not exactly equivalent."
    )

# =============================================================================
# 10. BUILD COMPLETE MASTER PROCESSING MANIFEST
# =============================================================================

print("\nBUILDING COMPLETE MASTER MANIFEST")
print("-" * 80)

# Start with final QC table.
complete = final_qc.copy()

# Add Step 30 information.
step30_keep = [
    "subset_nodule_id",
    "reader_count",
    "mean_pairwise_dice",
    "min_pairwise_dice",
    "max_pairwise_dice",
    "mean_pairwise_iou",
    "agreement_class",
    "potential_disagreement"
]

step30_keep = [
    c
    for c in step30_keep
    if c in step30.columns
]

complete = complete.merge(
    step30[
        step30_keep
    ],
    on="subset_nodule_id",
    how="left",
    validate="one_to_one"
)

# Add Step 31 information.
step31_keep = [
    "subset_nodule_id",
    "active_reader_count",
    "active_readers",
    "mean_foreground_relevant_dice",
    "min_foreground_relevant_dice",
    "mean_foreground_relevant_iou",
    "mean_relevant_slice_overlap",
    "qc_category",
    "strong_reader_disagreement",
    "slice_extent_disagreement"
]

step31_keep = [
    c
    for c in step31_keep
    if c in step31.columns
]

complete = complete.merge(
    step31[
        step31_keep
    ],
    on="subset_nodule_id",
    how="left",
    validate="one_to_one"
)

# Add original/processed benchmark information.
complete[
    "benchmark_original_test_dice"
] = original_test_dice

complete[
    "benchmark_processed_test_dice"
] = processed_test_dice

complete[
    "benchmark_test_dice_difference"
] = test_dice_difference

complete[
    "image_equivalence_mean_abs_difference"
] = mean_difference

complete[
    "image_equivalence_max_difference"
] = max_difference

complete[
    "image_data_preserved"
] = images_identical

assert len(complete) == 325

assert (
    complete["subset_nodule_id"]
    .nunique()
    == 325
)

complete.to_csv(
    OUTPUT_MASTER_CSV,
    index=False
)

print(
    f"✓ Complete manifest saved:"
)

print(
    f"  {OUTPUT_MASTER_CSV}"
)

# =============================================================================
# 11. FINAL STATUS
# =============================================================================

dataset_processing_pass = (
    len(complete) == 325
    and structural_pass == 325
    and structural_fail == 0
    and native_labels == 0
    and len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
    and images_identical
)

print("\n" + "=" * 80)
print("FINAL DATASET PROCESSING STATUS")
print("=" * 80)

if dataset_processing_pass:

    final_status = (
        "DATASET PROCESSING VALIDATION PASSED"
    )

    print(
        "\n✓ " + final_status
    )

else:

    final_status = (
        "DATASET PROCESSING REQUIRES REVIEW"
    )

    print(
        "\n⚠ " + final_status
    )

# =============================================================================
# 12. FINAL REPORT
# =============================================================================

report = f"""
===============================================================================
FINAL DATASET PROCESSING REPORT
===============================================================================

FINAL COHORT
-------------------------------------------------------------------------------
Eligible nodules:                  325
Unique patients:                   {final_qc['patient_id'].nunique()}
Unique CT series:                  {final_qc['SeriesInstanceUID'].nunique()}

Excluded nodules:
    nodule_029
    nodule_085

TRAIN / VALIDATION / TEST
-------------------------------------------------------------------------------
Train nodules:                     {len(train_df)}
Validation nodules:                {len(val_df)}
Test nodules:                      {len(test_df)}

Train patients:                    {len(train_patients)}
Validation patients:               {len(val_patients)}
Test patients:                     {len(test_patients)}

Patient leakage:
    Train/Validation:               {len(train_val_overlap)}
    Train/Test:                     {len(train_test_overlap)}
    Validation/Test:                {len(val_test_overlap)}

STRUCTURAL QC
-------------------------------------------------------------------------------
Structural QC PASS:                {structural_pass} / 325
Structural QC FAIL:                {structural_fail} / 325

MASK / READER QC
-------------------------------------------------------------------------------
Strong reader disagreement:        {strong_disagreement_count}
Slice-extent disagreement:         {slice_extent_count}

Reader directory counts:
{reader_count_distribution.to_string()}

Active reader counts:
{active_reader_distribution.to_string()}

PROVENANCE
-------------------------------------------------------------------------------
Native-to-XML identity proven:     {proven_count}
Native malignancy labels:          {native_labels}

Therefore:
    No unsupported native malignancy ground-truth labels are present.

ORIGINAL vs PROCESSED BENCHMARK
-------------------------------------------------------------------------------
Original test Dice:                {original_test_dice:.4f}
Processed test Dice:               {processed_test_dice:.4f}
Processed - Original:              {test_dice_difference:+.4f}

IMAGE PRESERVATION
-------------------------------------------------------------------------------
Mean absolute normalized difference:
                                    {mean_difference:.8f}

Maximum normalized difference:
                                    {max_difference:.8f}

Pixel-equivalent after normalization:
                                    {images_identical}

OVERALL INTERPRETATION
-------------------------------------------------------------------------------
The processed dataset passed structural validation across all 325 nodules.

The processed images were found to be pixel-equivalent to the original images
under the controlled benchmark normalization.

The original and processed datasets therefore produced identical baseline
segmentation performance in the controlled Step 34 experiment when evaluated
with the same U-Net, masks, patient splits, seed, and training configuration.

The processing pipeline improved dataset organization, metadata retention,
quality control, provenance documentation, and reproducibility without
demonstrable alteration of the underlying image information.

IMPORTANT PROVENANCE LIMITATION
-------------------------------------------------------------------------------
SeriesInstanceUID linkage is retained as established metadata.

The mapping:

    native nodule-X
        ->
    specific XML physical nodule
        ->
    specific malignancy rating

was NOT established for the full cohort.

Consequently no XML malignancy rating has been converted into native-nodule
ground truth.

DATASET STATUS
-------------------------------------------------------------------------------
{final_status}

MASTER MANIFEST
-------------------------------------------------------------------------------
{OUTPUT_MASTER_CSV}

FINAL PACKAGE
-------------------------------------------------------------------------------
{FINAL_PACKAGE}

===============================================================================
"""

with open(
    OUTPUT_REPORT,
    "w"
) as f:

    f.write(
        report.strip()
    )

print(
    "\n" + report
)

print("=" * 80)
print("STEP 35 COMPLETE")
print("=" * 80)

STEP 35 — FINAL DATASET PROCESSING VALIDATION

FILE VALIDATION
--------------------------------------------------------------------------------
✓ /content/final_325_dataset_qc_classification.csv
✓ /content/final_325_master_manifest.csv
✓ /content/final_325_ml_manifest.csv
✓ /content/step29_final_structural_audit.csv
✓ /content/step30_mask_reader_consistency.csv
✓ /content/step31_refined_mask_qc.csv
✓ /content/step34_original_vs_processed/step34_original_vs_processed_results.csv
✓ /content/step34_original_vs_processed/image_equivalence.csv

COHORT VALIDATION
--------------------------------------------------------------------------------
✓ 325 final nodules confirmed
✓ Unique patients: 247
✓ Unique CT series: 247
✓ nodule_029 and nodule_085 excluded

PATIENT SPLIT VALIDATION
--------------------------------------------------------------------------------
Train nodules:       235
Validation nodules:  44
Test nodules:        46
Train patients:      173
Validation patients: 37
Test patient

In [ ]:
# =============================================================================
# STEP 36 — ENHANCED IMAGE PREPROCESSING PIPELINE
# =============================================================================
#
# PURPOSE
# -------
# Create a new enhanced image dataset for a controlled comparison against:
#
#   1. Original images
#   2. Current processed images
#   3. Enhanced processed images
#
# IMAGE TRANSFORMATION
# --------------------
# For every image slice:
#
#   A. Convert to grayscale
#   B. Robust percentile windowing (1st -> 99th percentile)
#   C. Rescale to [0,255]
#   D. Apply CLAHE/local contrast enhancement
#   E. Save as PNG
#
# MASKS
# -----
# Masks are copied WITHOUT modification.
#
# IMPORTANT
# ---------
# This step creates a NEW dataset. It does not modify:
#
#   /content/kagl_lidc_idri.zip
#   /content/processed_325_nodules
#
# No malignancy labels are assigned.
# =============================================================================

import os
import shutil
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance

# Try OpenCV for CLAHE.
try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False

# =============================================================================
# 1. PATHS
# =============================================================================

ML_MANIFEST = "/content/final_325_ml_manifest.csv"

SOURCE_ROOT = "/content/processed_325_nodules"

ENHANCED_ROOT = "/content/enhanced_processed_325_nodules"

OUTPUT_MANIFEST = (
    "/content/enhanced_325_manifest.csv"
)

OUTPUT_QC = (
    "/content/step36_enhanced_preprocessing_qc.csv"
)

OUTPUT_SUMMARY = (
    "/content/step36_enhanced_preprocessing_summary.txt"
)

# =============================================================================
# 2. SETTINGS
# =============================================================================

LOW_PERCENTILE = 1.0
HIGH_PERCENTILE = 99.0

# CLAHE settings.
CLAHE_CLIP_LIMIT = 2.0
CLAHE_GRID_SIZE = (8, 8)

# Optional mild contrast multiplier after CLAHE.
# Keep this modest to avoid extreme enhancement.
CONTRAST_FACTOR = 1.05

print("=" * 80)
print("STEP 36 — ENHANCED IMAGE PREPROCESSING")
print("=" * 80)

print("\nTRANSFORMATION")
print("-" * 80)

print(
    f"Percentile window: "
    f"{LOW_PERCENTILE}% -> {HIGH_PERCENTILE}%"
)

print(
    f"CLAHE available: "
    f"{CV2_AVAILABLE}"
)

print(
    f"CLAHE clip limit: "
    f"{CLAHE_CLIP_LIMIT}"
)

print(
    f"CLAHE grid size: "
    f"{CLAHE_GRID_SIZE}"
)

# =============================================================================
# 3. INPUT VALIDATION
# =============================================================================

if not os.path.exists(ML_MANIFEST):

    raise FileNotFoundError(
        f"Missing ML manifest: {ML_MANIFEST}"
    )

if not os.path.isdir(SOURCE_ROOT):

    raise FileNotFoundError(
        f"Missing processed dataset: {SOURCE_ROOT}"
    )

manifest = pd.read_csv(
    ML_MANIFEST
)

assert len(manifest) == 325

assert (
    manifest[
        "subset_nodule_id"
    ].nunique()
    == 325
)

assert "nodule_029" not in set(
    manifest["subset_nodule_id"]
)

assert "nodule_085" not in set(
    manifest["subset_nodule_id"]
)

print("\nINPUT VALIDATION")
print("-" * 80)

print(
    f"Eligible nodules: "
    f"{len(manifest)}"
)

print(
    f"Patients: "
    f"{manifest['patient_id'].nunique()}"
)

print(
    "✓ 325-nodule cohort confirmed"
)

# =============================================================================
# 4. CREATE OUTPUT DIRECTORY
# =============================================================================

if os.path.exists(ENHANCED_ROOT):

    print(
        "\nExisting enhanced dataset found."
    )

    print(
        "Removing previous Step 36 output..."
    )

    shutil.rmtree(
        ENHANCED_ROOT
    )

os.makedirs(
    ENHANCED_ROOT,
    exist_ok=True
)

# =============================================================================
# 5. ENHANCEMENT FUNCTION
# =============================================================================

def enhance_image(image_array):
    """
    Deterministic per-slice enhancement.

    Steps:
        1. Convert to float32.
        2. Robust 1%-99% intensity clipping.
        3. Rescale to [0,255].
        4. Apply CLAHE if OpenCV is available.
        5. Apply mild contrast enhancement.
        6. Return uint8.
    """

    arr = np.asarray(
        image_array,
        dtype=np.float32
    )

    # -------------------------------------------------------------
    # Handle pathological images safely.
    # -------------------------------------------------------------

    if arr.size == 0:

        raise ValueError(
            "Empty image encountered."
        )

    # -------------------------------------------------------------
    # Robust percentile limits.
    # -------------------------------------------------------------

    low = np.percentile(
        arr,
        LOW_PERCENTILE
    )

    high = np.percentile(
        arr,
        HIGH_PERCENTILE
    )

    # Prevent zero-width window.
    if high <= low:

        low = float(
            arr.min()
        )

        high = float(
            arr.max()
        )

    # -------------------------------------------------------------
    # Constant image fallback.
    # -------------------------------------------------------------

    if high <= low:

        enhanced = np.zeros_like(
            arr,
            dtype=np.uint8
        )

        return enhanced

    # -------------------------------------------------------------
    # Clip and normalize to 0-255.
    # -------------------------------------------------------------

    clipped = np.clip(
        arr,
        low,
        high
    )

    normalized = (
        (clipped - low)
        /
        (high - low)
        *
        255.0
    )

    normalized = np.clip(
        normalized,
        0,
        255
    ).astype(
        np.uint8
    )

    # -------------------------------------------------------------
    # CLAHE local contrast enhancement.
    # -------------------------------------------------------------

    if CV2_AVAILABLE:

        clahe = cv2.createCLAHE(
            clipLimit=CLAHE_CLIP_LIMIT,
            tileGridSize=CLAHE_GRID_SIZE
        )

        enhanced = clahe.apply(
            normalized
        )

    else:

        # Fallback using PIL autocontrast.
        pil_img = Image.fromarray(
            normalized,
            mode="L"
        )

        pil_img = ImageOps.autocontrast(
            pil_img
        )

        enhanced = np.asarray(
            pil_img,
            dtype=np.uint8
        )

    # -------------------------------------------------------------
    # Mild final contrast adjustment.
    # -------------------------------------------------------------

    if CONTRAST_FACTOR != 1.0:

        pil_img = Image.fromarray(
            enhanced,
            mode="L"
        )

        enhancer = ImageEnhance.Contrast(
            pil_img
        )

        pil_img = enhancer.enhance(
            CONTRAST_FACTOR
        )

        enhanced = np.asarray(
            pil_img,
            dtype=np.uint8
        )

    return enhanced


# =============================================================================
# 6. PROCESS ALL 325 NODULES
# =============================================================================

qc_rows = []

total_images = 0
successful_images = 0
failed_images = 0

print("\nPROCESSING ENHANCED IMAGES")
print("-" * 80)

for counter, (_, row) in enumerate(
    manifest.iterrows(),
    start=1
):

    subset_id = str(
        row[
            "subset_nodule_id"
        ]
    ).strip()

    patient_id = str(
        row[
            "patient_id"
        ]
    ).strip()

    native_id = str(
        row[
            "native_nodule_id"
        ]
    ).strip()

    split = str(
        row[
            "split"
        ]
    ).strip().lower()

    source_case_dir = os.path.join(
        SOURCE_ROOT,
        subset_id
    )

    source_image_dir = os.path.join(
        source_case_dir,
        "images"
    )

    output_case_dir = os.path.join(
        ENHANCED_ROOT,
        subset_id
    )

    output_image_dir = os.path.join(
        output_case_dir,
        "images"
    )

    os.makedirs(
        output_image_dir,
        exist_ok=True
    )

    if not os.path.isdir(
        source_image_dir
    ):

        raise FileNotFoundError(
            f"Missing source image directory: "
            f"{source_image_dir}"
        )

    source_images = [

        f
        for f in os.listdir(
            source_image_dir
        )
        if f.lower().endswith(".png")

    ]

    def numeric_sort(filename):

        digits = "".join(
            c
            for c in filename
            if c.isdigit()
        )

        return (
            int(digits)
            if digits
            else 0,
            filename
        )

    source_images = sorted(
        source_images,
        key=numeric_sort
    )

    image_success = 0

    image_failure = 0

    original_values = []
    enhanced_values = []

    for image_name in source_images:

        source_path = os.path.join(
            source_image_dir,
            image_name
        )

        output_path = os.path.join(
            output_image_dir,
            image_name
        )

        try:

            with Image.open(
                source_path
            ) as img:

                original = np.asarray(
                    img.convert("L"),
                    dtype=np.float32
                )

            enhanced = enhance_image(
                original
            )

            Image.fromarray(
                enhanced,
                mode="L"
            ).save(
                output_path
            )

            # ---------------------------------------------------------
            # Track statistics.
            # ---------------------------------------------------------

            original_values.extend(
                [
                    float(original.min()),
                    float(original.max()),
                    float(original.mean())
                ]
            )

            enhanced_values.extend(
                [
                    float(enhanced.min()),
                    float(enhanced.max()),
                    float(enhanced.mean())
                ]
            )

            image_success += 1

            total_images += 1
            successful_images += 1

        except Exception as e:

            print(
                f"ERROR: {subset_id} / "
                f"{image_name}: {e}"
            )

            image_failure += 1
            total_images += 1
            failed_images += 1

    # -----------------------------------------------------------------
    # Copy masks WITHOUT modification.
    # -----------------------------------------------------------------

    mask_counts = {}

    for mask_num in range(4):

        source_mask_dir = os.path.join(
            source_case_dir,
            f"mask-{mask_num}"
        )

        output_mask_dir = os.path.join(
            output_case_dir,
            f"mask-{mask_num}"
        )

        mask_counts[
            mask_num
        ] = 0

        if not os.path.isdir(
            source_mask_dir
        ):

            continue

        os.makedirs(
            output_mask_dir,
            exist_ok=True
        )

        mask_files = [

            f
            for f in os.listdir(
                source_mask_dir
            )
            if f.lower().endswith(".png")

        ]

        mask_files = sorted(
            mask_files,
            key=numeric_sort
        )

        for mask_name in mask_files:

            source_mask = os.path.join(
                source_mask_dir,
                mask_name
            )

            output_mask = os.path.join(
                output_mask_dir,
                mask_name
            )

            shutil.copy2(
                source_mask,
                output_mask
            )

            mask_counts[
                mask_num
            ] += 1

    # -----------------------------------------------------------------
    # Case-level QC.
    # -----------------------------------------------------------------

    output_images = [

        f
        for f in os.listdir(
            output_image_dir
        )
        if f.lower().endswith(".png")

    ]

    output_images = sorted(
        output_images,
        key=numeric_sort
    )

    image_dimensions_consistent = True

    reference_dimensions = None

    for filename in output_images:

        path = os.path.join(
            output_image_dir,
            filename
        )

        try:

            with Image.open(path) as img:

                dims = tuple(
                    img.size
                )

            if reference_dimensions is None:

                reference_dimensions = dims

            elif dims != reference_dimensions:

                image_dimensions_consistent = False

        except Exception:

            image_dimensions_consistent = False

    masks_preserved = True

    for mask_num in range(4):

        source_mask_dir = os.path.join(
            source_case_dir,
            f"mask-{mask_num}"
        )

        output_mask_dir = os.path.join(
            output_case_dir,
            f"mask-{mask_num}"
        )

        if not os.path.isdir(
            source_mask_dir
        ):

            continue

        source_count = len([
            f
            for f in os.listdir(
                source_mask_dir
            )
            if f.lower().endswith(".png")
        ])

        output_count = len([
            f
            for f in os.listdir(
                output_mask_dir
            )
            if f.lower().endswith(".png")
        ])

        if source_count != output_count:

            masks_preserved = False

    case_pass = (
        image_failure == 0
        and
        len(output_images)
        ==
        len(source_images)
        and
        image_dimensions_consistent
        and
        masks_preserved
    )

    qc_rows.append({

        "subset_nodule_id":
            subset_id,

        "patient_id":
            patient_id,

        "native_nodule_id":
            native_id,

        "split":
            split,

        "source_image_count":
            len(source_images),

        "enhanced_image_count":
            len(output_images),

        "image_failures":
            image_failure,

        "output_image_dimensions":
            str(reference_dimensions),

        "image_dimensions_consistent":
            image_dimensions_consistent,

        "mask_0_count":
            mask_counts[0],

        "mask_1_count":
            mask_counts[1],

        "mask_2_count":
            mask_counts[2],

        "mask_3_count":
            mask_counts[3],

        "masks_preserved":
            masks_preserved,

        "enhanced_case_qc":
            "PASS"
            if case_pass
            else "FAIL"

    })

    if counter % 25 == 0:

        print(
            f"Processed "
            f"{counter} / {len(manifest)}"
        )


# =============================================================================
# 7. SAVE QC TABLE
# =============================================================================

qc_df = pd.DataFrame(
    qc_rows
)

qc_df.to_csv(
    OUTPUT_QC,
    index=False
)

# =============================================================================
# 8. FINAL VALIDATION
# =============================================================================

passed_cases = int(
    (
        qc_df[
            "enhanced_case_qc"
        ]
        == "PASS"
    ).sum()
)

failed_cases = (
    len(qc_df)
    -
    passed_cases
)

print("\n" + "=" * 80)
print("STEP 36 RESULTS")
print("=" * 80)

print(
    f"\nNodules processed: "
    f"{len(qc_df)}"
)

print(
    f"Images processed: "
    f"{total_images}"
)

print(
    f"Images successfully enhanced: "
    f"{successful_images}"
)

print(
    f"Image processing failures: "
    f"{failed_images}"
)

print(
    f"Cases passing enhancement QC: "
    f"{passed_cases} / 325"
)

print(
    f"Cases failing enhancement QC: "
    f"{failed_cases} / 325"
)

assert len(qc_df) == 325

assert (
    qc_df[
        "subset_nodule_id"
    ].nunique()
    == 325
)

assert failed_cases == 0

print(
    "✓ All 325 enhanced cases passed QC."
)

# =============================================================================
# 9. VERIFY MASKS WERE PRESERVED
# =============================================================================

print("\nMASK PRESERVATION")
print("-" * 80)

mask_failures = int(
    (
        ~qc_df[
            "masks_preserved"
        ]
    ).sum()
)

print(
    f"Cases with mask-copy discrepancies: "
    f"{mask_failures}"
)

assert mask_failures == 0

print(
    "✓ Mask files preserved without modification."
)

# =============================================================================
# 10. VERIFY IMAGE RANGE
# =============================================================================

print("\nENHANCED IMAGE RANGE CHECK")
print("-" * 80)

range_failures = 0

for counter, (_, row) in enumerate(
    manifest.iterrows(),
    start=1
):

    subset_id = str(
        row[
            "subset_nodule_id"
        ]
    )

    image_dir = os.path.join(
        ENHANCED_ROOT,
        subset_id,
        "images"
    )

    files = [
        f
        for f in os.listdir(
            image_dir
        )
        if f.lower().endswith(".png")
    ]

    for filename in files:

        path = os.path.join(
            image_dir,
            filename
        )

        with Image.open(path) as img:

            arr = np.asarray(
                img.convert("L"),
                dtype=np.uint8
            )

        if (
            arr.min() < 0
            or
            arr.max() > 255
        ):

            range_failures += 1

    if counter % 50 == 0:

        print(
            f"Checked "
            f"{counter} / {len(manifest)}"
        )

print(
    f"Range failures: "
    f"{range_failures}"
)

assert range_failures == 0

print(
    "✓ All enhanced images are valid uint8 PNGs."
)

# =============================================================================
# 11. CREATE ENHANCED MANIFEST
# =============================================================================

enhanced_manifest = manifest.copy()

enhanced_manifest[
    "enhanced_dataset_root"
] = ENHANCED_ROOT

enhanced_manifest[
    "enhanced_image_directory"
] = enhanced_manifest[
    "subset_nodule_id"
].apply(
    lambda x: os.path.join(
        ENHANCED_ROOT,
        str(x),
        "images"
    )
)

enhanced_manifest[
    "preprocessing_method"
] = (
    "1%-99% percentile clipping + "
    "rescaling + CLAHE + mild contrast"
)

enhanced_manifest[
    "preprocessing_original_pixels_preserved"
] = False

enhanced_manifest[
    "masks_modified"
] = False

enhanced_manifest[
    "malignancy_labels_assigned"
] = False

enhanced_manifest[
    "native_to_xml_identity_proven"
] = False

enhanced_manifest.to_csv(
    os.path.join(
        ENHANCED_ROOT,
        "enhanced_manifest.csv"
    ),
    index=False
)

# Also save to /content.
enhanced_manifest.to_csv(
    OUTPUT_MANIFEST,
    index=False
)

# =============================================================================
# 12. SUMMARY
# =============================================================================

summary = f"""
===============================================================================
STEP 36 — ENHANCED PREPROCESSING SUMMARY
===============================================================================

COHORT
-------------------------------------------------------------------------------
Eligible nodules:                  {len(manifest)}
Unique patients:                   {manifest['patient_id'].nunique()}

IMAGE PROCESSING
-------------------------------------------------------------------------------
Images processed:                  {total_images}
Images successfully enhanced:      {successful_images}
Image processing failures:         {failed_images}

Cases passing enhancement QC:      {passed_cases} / 325
Cases failing enhancement QC:      {failed_cases} / 325

PREPROCESSING
-------------------------------------------------------------------------------
Intensity window:
    {LOW_PERCENTILE}% -> {HIGH_PERCENTILE}% percentile

CLAHE:
    {'YES' if CV2_AVAILABLE else 'NO — PIL fallback used'}

CLAHE clip limit:
    {CLAHE_CLIP_LIMIT}

CLAHE grid:
    {CLAHE_GRID_SIZE}

Final contrast factor:
    {CONTRAST_FACTOR}

MASKS
-------------------------------------------------------------------------------
Masks modified:                    NO
Mask-copy discrepancies:           {mask_failures}

PROVENANCE
-------------------------------------------------------------------------------
Native-to-XML identity established: NO
Native malignancy labels assigned:  NO

IMPORTANT
-------------------------------------------------------------------------------
This enhanced dataset is intended for a controlled experimental comparison
against the original and current processed datasets.

The enhancement changes image intensity representation and local contrast.
Therefore it is NOT pixel-equivalent to the original dataset.

Validation and test images receive the same deterministic preprocessing but
NO augmentation is applied.

Training augmentation, if used later, should be applied only during training
inside the DataLoader rather than permanently modifying validation/test data.

OUTPUTS
-------------------------------------------------------------------------------
Enhanced dataset:
    {ENHANCED_ROOT}

Enhanced manifest:
    {OUTPUT_MANIFEST}

QC:
    {OUTPUT_QC}

Summary:
    {OUTPUT_SUMMARY}

===============================================================================
"""

with open(
    OUTPUT_SUMMARY,
    "w"
) as f:

    f.write(
        summary.strip()
    )

print(
    "\n" + summary
)

print("=" * 80)
print("STEP 36 COMPLETE")
print("=" * 80)

STEP 36 — ENHANCED IMAGE PREPROCESSING

TRANSFORMATION
--------------------------------------------------------------------------------
Percentile window: 1.0% -> 99.0%
CLAHE available: True
CLAHE clip limit: 2.0
CLAHE grid size: (8, 8)


FileNotFoundError: Missing ML manifest: /content/final_325_ml_manifest.csv

In [ ]:
import os

print("=" * 80)
print("CHECKING COLAB FILES")
print("=" * 80)

for path in [
    "/content/kagl_lidc_idri.zip",
    "/content/processed_325_nodules",
    "/content/enhanced_processed_325_nodules",
    "/content/final_325_ml_manifest.csv",
    "/content/final_325_complete_processing_manifest.csv",
    "/content/final_325_dataset_package",
]:
    exists = os.path.exists(path)
    print(
        f"{'✓' if exists else '✗'} {path}"
    )

CHECKING COLAB FILES
✓ /content/kagl_lidc_idri.zip
✗ /content/processed_325_nodules
✗ /content/enhanced_processed_325_nodules
✗ /content/final_325_ml_manifest.csv
✗ /content/final_325_complete_processing_manifest.csv
✗ /content/final_325_dataset_package


In [ ]:
import os

paths = [
    "/content/nodule_to_lidc_mapping.csv",
    "/content/nodule_to_single_ct_series_fixed.csv",
    "/content/tcia_series_to_patient_mapping.csv",
    "/content/lidc_xml_annotations",
]

print("=" * 80)
print("SOURCE FILE CHECK")
print("=" * 80)

for path in paths:
    print(
        f"{'✓' if os.path.exists(path) else '✗'} {path}"
    )

SOURCE FILE CHECK
✗ /content/nodule_to_lidc_mapping.csv
✗ /content/nodule_to_single_ct_series_fixed.csv
✗ /content/tcia_series_to_patient_mapping.csv
✗ /content/lidc_xml_annotations


In [ ]:
import os
import zipfile

ZIP_PATH = "/content/kagl_lidc_idri.zip"

print("=" * 80)
print("SOURCE ZIP INVENTORY")
print("=" * 80)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()

print(f"Total ZIP entries: {len(names)}")

print("\nCSV FILES")
print("-" * 80)

csv_files = [
    n for n in names
    if n.lower().endswith(".csv")
]

if csv_files:
    for n in csv_files[:100]:
        print(n)
else:
    print("No CSV files found inside ZIP.")

print("\nXML FILES")
print("-" * 80)

xml_files = [
    n for n in names
    if n.lower().endswith(".xml")
]

print(f"XML files in ZIP: {len(xml_files)}")

for n in xml_files[:20]:
    print(n)

print("\nDICOM FILES")
print("-" * 80)

dcm_files = [
    n for n in names
    if n.lower().endswith(".dcm")
]

print(f"DICOM files in ZIP: {len(dcm_files)}")

print("\nPNG FILES")
print("-" * 80)

png_files = [
    n for n in names
    if n.lower().endswith(".png")
]

print(f"PNG files in ZIP: {len(png_files)}")

print("\nTOP-LEVEL DIRECTORIES")
print("-" * 80)

top_dirs = sorted({
    n.split("/")[0]
    for n in names
    if "/" in n
})

for d in top_dirs:
    print(d)

print("=" * 80)

SOURCE ZIP INVENTORY
Total ZIP entries: 77740

CSV FILES
--------------------------------------------------------------------------------
No CSV files found inside ZIP.

XML FILES
--------------------------------------------------------------------------------
XML files in ZIP: 0

DICOM FILES
--------------------------------------------------------------------------------
DICOM files in ZIP: 0

PNG FILES
--------------------------------------------------------------------------------
PNG files in ZIP: 77740

TOP-LEVEL DIRECTORIES
--------------------------------------------------------------------------------
LIDC-IDRI-slices


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

os.makedirs(
    "/content/drive/MyDrive/LIDC_processed",
    exist_ok=True
)

print("Google Drive mounted.")
print("Persistent folder:")
print("/content/drive/MyDrive/LIDC_processed")

Mounted at /content/drive
Google Drive mounted.
Persistent folder:
/content/drive/MyDrive/LIDC_processed


In [ ]:
import zipfile
import os
import re
from collections import defaultdict

ZIP_PATH = "/content/kagl_lidc_idri.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()

# -------------------------------------------------------------------------
# Discover patients and native nodule folders
# -------------------------------------------------------------------------

patient_nodules = defaultdict(set)

for name in names:

    parts = name.split("/")

    if len(parts) >= 3:
        if parts[0] == "LIDC-IDRI-slices":

            patient_id = parts[1]
            nodule_id = parts[2]

            if nodule_id.startswith("nodule-"):

                patient_nodules[
                    patient_id
                ].add(
                    nodule_id
                )

# Natural sorting
def nodule_index(x):
    m = re.search(
        r"nodule-(\d+)",
        x
    )
    return int(m.group(1)) if m else 10**9

print("=" * 80)
print("ZIP NATIVE NODULE INVENTORY")
print("=" * 80)

print(
    f"Patients: {len(patient_nodules)}"
)

total_nodules = sum(
    len(v)
    for v in patient_nodules.values()
)

print(
    f"Total native nodule folders: {total_nodules}"
)

print("\nFIRST 30 PATIENTS")
print("-" * 80)

for patient_id in sorted(
    patient_nodules.keys()
)[:30]:

    nodules = sorted(
        patient_nodules[patient_id],
        key=nodule_index
    )

    print(
        f"{patient_id}: "
        f"{', '.join(nodules)}"
    )

print("=" * 80)

ZIP NATIVE NODULE INVENTORY
Patients: 875
Total native nodule folders: 2630

FIRST 30 PATIENTS
--------------------------------------------------------------------------------
LIDC-IDRI-0001: nodule-0
LIDC-IDRI-0002: nodule-0
LIDC-IDRI-0003: nodule-0, nodule-1, nodule-2, nodule-3
LIDC-IDRI-0004: nodule-0
LIDC-IDRI-0005: nodule-0, nodule-1, nodule-2
LIDC-IDRI-0006: nodule-0, nodule-1, nodule-2, nodule-3
LIDC-IDRI-0007: nodule-0, nodule-1
LIDC-IDRI-0008: nodule-0, nodule-1
LIDC-IDRI-0009: nodule-0, nodule-1
LIDC-IDRI-0010: nodule-0, nodule-1, nodule-2
LIDC-IDRI-0011: nodule-0, nodule-1, nodule-2, nodule-3, nodule-4, nodule-5, nodule-6, nodule-7, nodule-8, nodule-9
LIDC-IDRI-0012: nodule-0, nodule-1, nodule-2, nodule-3, nodule-4, nodule-5, nodule-6, nodule-7, nodule-8, nodule-9, nodule-10, nodule-11
LIDC-IDRI-0013: nodule-0, nodule-1, nodule-2
LIDC-IDRI-0014: nodule-0
LIDC-IDRI-0015: nodule-0
LIDC-IDRI-0016: nodule-0, nodule-1, nodule-2, nodule-3, nodule-4, nodule-5
LIDC-IDRI-0017: nodule

In [ ]:
import os

from google.colab import drive
drive.mount("/content/drive")

print("=" * 80)
print("SEARCHING GOOGLE DRIVE FOR PREVIOUS DATASET FILES")
print("=" * 80)

targets = [
    "nodule_to_lidc_mapping.csv",
    "nodule_to_single_ct_series.csv",
    "nodule_to_single_ct_series_fixed.csv",
    "nodule_to_series_candidates.csv",
    "tcia_series_to_patient_mapping.csv",
    "nodule_xml_annotation_inventory.csv",
]

found = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    for filename in files:

        if filename in targets:

            full_path = os.path.join(
                root,
                filename
            )

            found.append(
                full_path
            )

            print("✓", full_path)

print("\n" + "=" * 80)

if not found:

    print("NO PREVIOUS MAPPING FILES FOUND IN GOOGLE DRIVE.")

    print(
        "\nWe should NOT rebuild the cohort from the ZIP alone, "
        "because that would change your original 327-case selection."
    )

else:

    print(
        f"FOUND {len(found)} previous mapping file(s)."
    )

    print(
        "\nThese files can be restored and used to rebuild the "
        "validated 325-nodule dataset."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SEARCHING GOOGLE DRIVE FOR PREVIOUS DATASET FILES

NO PREVIOUS MAPPING FILES FOUND IN GOOGLE DRIVE.

We should NOT rebuild the cohort from the ZIP alone, because that would change your original 327-case selection.


In [ ]:
import os

print("=" * 80)
print("SEARCHING GOOGLE DRIVE FOR NOTEBOOKS / PROJECT FILES")
print("=" * 80)

extensions = {
    ".ipynb",
    ".csv",
    ".json",
    ".txt",
    ".xlsx",
    ".parquet"
}

found = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    for filename in files:

        ext = os.path.splitext(filename)[1].lower()

        if ext in extensions:

            path = os.path.join(root, filename)

            found.append(path)

            print(path)

print("\n" + "=" * 80)
print(f"Total candidate files found: {len(found)}")
print("=" * 80)

SEARCHING GOOGLE DRIVE FOR NOTEBOOKS / PROJECT FILES
/content/drive/MyDrive/Colab Notebooks/simple test to open DCM.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/gpt.ipynb
/content/drive/MyDrive/Colab Notebooks/testing.ipynb
/content/drive/MyDrive/Colab Notebooks/dataset.ipynb

Total candidate files found: 6


In [ ]:
import os
import json

NOTEBOOK_DIR = "/content/drive/MyDrive/Colab Notebooks"

targets = [
    "nodule_to_lidc_mapping",
    "nodule_to_single_ct_series",
    "tcia_series_to_patient_mapping",
    "nodule_to_series_candidates",
    "327",
    "325",
    "SeriesInstanceUID"
]

print("=" * 80)
print("SEARCHING NOTEBOOKS FOR ORIGINAL COHORT / MAPPING CODE")
print("=" * 80)

for filename in sorted(os.listdir(NOTEBOOK_DIR)):

    if not filename.endswith(".ipynb"):
        continue

    path = os.path.join(
        NOTEBOOK_DIR,
        filename
    )

    print(f"\n{'=' * 80}")
    print(f"NOTEBOOK: {filename}")
    print(f"{'=' * 80}")

    try:
        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:
            notebook = json.load(f)

    except Exception as e:

        print(f"Could not read notebook: {e}")
        continue

    matches_found = 0

    for cell_num, cell in enumerate(
        notebook.get("cells", []),
        start=1
    ):

        source = "".join(
            cell.get("source", [])
        )

        matched_terms = [
            term
            for term in targets
            if term.lower() in source.lower()
        ]

        if matched_terms:

            matches_found += 1

            print(
                f"\n--- CELL {cell_num} ---"
            )

            print(
                f"Matched: {', '.join(matched_terms)}"
            )

            print(
                source[:12000]
            )

    if matches_found == 0:

        print(
            "No relevant cohort/mapping code found."
        )

print("\n" + "=" * 80)
print("SEARCH COMPLETE")
print("=" * 80)

Streaming output truncated to the last 5000 lines.

    native_labels = 0

print(
    f"Native malignancy labels assigned: "
    f"{native_labels}"
)

assert native_labels == 0

if (
    "native_to_xml_identity_status"
    in final_qc.columns
):

    proven_count = int(
        (
            final_qc[
                "native_to_xml_identity_status"
            ].astype(str).str.upper()
            == "PROVEN"
        ).sum()
    )

else:

    proven_count = 0

print(
    f"Native-to-XML identities marked proven: "
    f"{proven_count}"
)

assert proven_count == 0

print(
    "✓ No unsupported native malignancy labels"
)

# =============================================================================
# 8. STEP 34 ORIGINAL vs PROCESSED
# =============================================================================

print("\nSTEP 34 — ORIGINAL vs PROCESSED")
print("-" * 80)

# -------------------------------------------------------------------------
# Detect columns robustly
# -----------

In [ ]:
import os

ROOT = "/content/drive/MyDrive"

keywords = [
    "lidc",
    "tcia",
    "nodule",
    "mapping",
    "series",
    "annotation",
    "327",
    "325"
]

print("=" * 80)
print("BROAD SEARCH FOR ORIGINAL COHORT / MAPPING MATERIAL")
print("=" * 80)

matches = []

for root, dirs, files in os.walk(ROOT):

    for filename in files:

        lower = filename.lower()

        if any(k in lower for k in keywords):

            path = os.path.join(root, filename)

            matches.append(path)

            print(path)

print("\n" + "=" * 80)
print(f"Potentially relevant files found: {len(matches)}")
print("=" * 80)

BROAD SEARCH FOR ORIGINAL COHORT / MAPPING MATERIAL
/content/drive/MyDrive/kaggle dataset/LIDC-IDRI-slices/kagl_lidc_idri.zip

Potentially relevant files found: 1


In [ ]:
# =============================================================================
# STEP 37 — CONTROLLED 3-WAY DATASET BENCHMARK
# =============================================================================
#
# Compares:
#
#   1. ORIGINAL
#   2. CURRENT PROCESSED
#   3. ENHANCED PROCESSED
#
# Everything except the image source is held constant.
#
# SAME:
#   - 325 nodules
#   - patient-level train/val/test split
#   - consensus masks
#   - U-Net architecture
#   - optimizer
#   - learning rate
#   - batch size
#   - epochs
#   - random seed
#
# No malignancy labels are used.
# No source datasets are modified.
# =============================================================================

import os
import io
import zipfile
import random
import re
import time

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# =============================================================================
# 1. PATHS
# =============================================================================

ZIP_PATH = "/content/kagl_lidc_idri.zip"

ML_MANIFEST = "/content/final_325_ml_manifest.csv"

CURRENT_PROCESSED_ROOT = (
    "/content/processed_325_nodules"
)

ENHANCED_ROOT = (
    "/content/enhanced_processed_325_nodules"
)

OUTPUT_DIR = (
    "/content/step37_three_way_benchmark"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

RESULTS_CSV = os.path.join(
    OUTPUT_DIR,
    "step37_three_way_results.csv"
)

HISTORY_CSV = os.path.join(
    OUTPUT_DIR,
    "step37_three_way_training_history.csv"
)

SUMMARY_TXT = os.path.join(
    OUTPUT_DIR,
    "step37_three_way_summary.txt"
)

# =============================================================================
# 2. EXPERIMENT SETTINGS
# =============================================================================

SEED = 42
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
NUM_WORKERS = 0

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 80)
print("STEP 37 — CONTROLLED 3-WAY DATASET BENCHMARK")
print("=" * 80)

print(f"\nDevice: {DEVICE}")

if torch.cuda.is_available():
    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )
else:
    print(
        "WARNING: Running on CPU. "
        "This experiment may take a while."
    )

# =============================================================================
# 3. REPRODUCIBILITY
# =============================================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)

# =============================================================================
# 4. CHECK INPUTS
# =============================================================================

required_paths = [
    ZIP_PATH,
    ML_MANIFEST,
    CURRENT_PROCESSED_ROOT,
    ENHANCED_ROOT
]

for path in required_paths:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Required input missing: {path}"
        )

# =============================================================================
# 5. LOAD FINAL 325-NODULE MANIFEST
# =============================================================================

manifest = pd.read_csv(
    ML_MANIFEST
)

assert len(manifest) == 325

manifest["subset_nodule_id"] = (
    manifest["subset_nodule_id"]
    .astype(str)
    .str.strip()
)

manifest["patient_id"] = (
    manifest["patient_id"]
    .astype(str)
    .str.strip()
)

manifest["native_nodule_id"] = (
    manifest["native_nodule_id"]
    .astype(str)
    .str.strip()
)

manifest["split"] = (
    manifest["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

assert (
    manifest[
        "subset_nodule_id"
    ].nunique()
    == 325
)

assert "nodule_029" not in set(
    manifest["subset_nodule_id"]
)

assert "nodule_085" not in set(
    manifest["subset_nodule_id"]
)

print("\nCOHORT")
print("-" * 80)

print(
    f"Nodules: "
    f"{len(manifest)}"
)

print(
    f"Patients: "
    f"{manifest['patient_id'].nunique()}"
)

print(
    manifest["split"]
    .value_counts()
    .sort_index()
    .to_string()
)

# =============================================================================
# 6. PATIENT LEAKAGE CHECK
# =============================================================================

train_patients = set(
    manifest.loc[
        manifest["split"] == "train",
        "patient_id"
    ]
)

val_patients = set(
    manifest.loc[
        manifest["split"] == "val",
        "patient_id"
    ]
)

test_patients = set(
    manifest.loc[
        manifest["split"] == "test",
        "patient_id"
    ]
)

assert not (
    train_patients
    &
    val_patients
)

assert not (
    train_patients
    &
    test_patients
)

assert not (
    val_patients
    &
    test_patients
)

# =============================================================================
# 7. NATURAL SORT
# =============================================================================

def natural_sort(paths):

    def key(path):

        filename = os.path.basename(
            path
        )

        parts = re.split(
            r"(\d+)",
            filename
        )

        result = []

        for part in parts:

            if part.isdigit():

                result.append(
                    (
                        0,
                        int(part)
                    )
                )

            else:

                result.append(
                    (
                        1,
                        part
                    )
                )

        return result

    return sorted(
        paths,
        key=key
    )


# =============================================================================
# 8. INDEX ORIGINAL DATA
# =============================================================================

print("\nINDEXING ORIGINAL DATA")
print("-" * 80)

zip_file = zipfile.ZipFile(
    ZIP_PATH,
    "r"
)

zip_names = set(
    zip_file.namelist()
)

original_lookup = {}

for _, row in manifest.iterrows():

    subset_id = row[
        "subset_nodule_id"
    ]

    patient_id = row[
        "patient_id"
    ]

    native_id = row[
        "native_nodule_id"
    ]

    prefix = (
        f"LIDC-IDRI-slices/"
        f"{patient_id}/"
        f"{native_id}/"
        f"images/"
    )

    files = [

        f
        for f in zip_names
        if f.startswith(prefix)
        and f.lower().endswith(".png")

    ]

    files = sorted(
        files,
        key=lambda x: natural_sort(
            [x]
        )[0]
    )

    if len(files) == 0:

        raise FileNotFoundError(
            f"No original images for "
            f"{subset_id}"
        )

    original_lookup[
        subset_id
    ] = files

print(
    f"Original cases indexed: "
    f"{len(original_lookup)}"
)

# =============================================================================
# 9. INDEX CURRENT PROCESSED DATA
# =============================================================================

print("\nINDEXING CURRENT PROCESSED DATA")
print("-" * 80)

current_lookup = {}

for subset_id in manifest[
    "subset_nodule_id"
].tolist():

    image_dir = os.path.join(
        CURRENT_PROCESSED_ROOT,
        subset_id,
        "images"
    )

    if not os.path.isdir(
        image_dir
    ):

        raise FileNotFoundError(
            f"Missing processed images: "
            f"{image_dir}"
        )

    files = [
        os.path.join(
            image_dir,
            f
        )
        for f in os.listdir(
            image_dir
        )
        if f.lower().endswith(".png")
    ]

    files = natural_sort(
        files
    )

    if not files:

        raise FileNotFoundError(
            f"No processed images for "
            f"{subset_id}"
        )

    current_lookup[
        subset_id
    ] = files

print(
    f"Current processed cases indexed: "
    f"{len(current_lookup)}"
)

# =============================================================================
# 10. INDEX ENHANCED DATA
# =============================================================================

print("\nINDEXING ENHANCED DATA")
print("-" * 80)

enhanced_lookup = {}

for subset_id in manifest[
    "subset_nodule_id"
].tolist():

    image_dir = os.path.join(
        ENHANCED_ROOT,
        subset_id,
        "images"
    )

    if not os.path.isdir(
        image_dir
    ):

        raise FileNotFoundError(
            f"Missing enhanced images: "
            f"{image_dir}"
        )

    files = [
        os.path.join(
            image_dir,
            f
        )
        for f in os.listdir(
            image_dir
        )
        if f.lower().endswith(".png")
    ]

    files = natural_sort(
        files
    )

    if not files:

        raise FileNotFoundError(
            f"No enhanced images for "
            f"{subset_id}"
        )

    enhanced_lookup[
        subset_id
    ] = files

print(
    f"Enhanced cases indexed: "
    f"{len(enhanced_lookup)}"
)

# =============================================================================
# 11. LOAD CONSENSUS MASKS
# =============================================================================
#
# Same masks used for all three conditions.
#
# Majority vote across the four native mask directories.
# =============================================================================

print("\nBUILDING CONSENSUS MASKS")
print("-" * 80)

mask_lookup = {}

for counter, (_, row) in enumerate(
    manifest.iterrows(),
    start=1
):

    subset_id = row[
        "subset_nodule_id"
    ]

    case_dir = os.path.join(
        CURRENT_PROCESSED_ROOT,
        subset_id
    )

    reader_volumes = []

    for reader in range(4):

        mask_dir = os.path.join(
            case_dir,
            f"mask-{reader}"
        )

        if not os.path.isdir(
            mask_dir
        ):

            continue

        files = [
            os.path.join(
                mask_dir,
                f
            )
            for f in os.listdir(
                mask_dir
            )
            if f.lower().endswith(".png")
        ]

        files = natural_sort(
            files
        )

        # Load mask files.
        slices = []

        for path in files:

            with Image.open(
                path
            ) as img:

                arr = np.asarray(
                    img.convert("L"),
                    dtype=np.uint8
                )

            slices.append(
                arr > 0
            )

        if not slices:
            continue

        volume = np.stack(
            slices,
            axis=0
        )

        reader_volumes.append(
            volume
        )

    if not reader_volumes:

        raise RuntimeError(
            f"No masks found for {subset_id}"
        )

    # -------------------------------------------------------------
    # Handle different reader depths safely.
    # -------------------------------------------------------------

    max_depth = max(
        v.shape[0]
        for v in reader_volumes
    )

    reference_h = reader_volumes[
        0
    ].shape[1]

    reference_w = reader_volumes[
        0
    ].shape[2]

    consensus_slices = []

    for z in range(
        max_depth
    ):

        available = []

        for volume in reader_volumes:

            if z < volume.shape[0]:

                available.append(
                    volume[z]
                )

        if not available:
            continue

        # All masks should have same H/W.
        for arr in available:

            if arr.shape != (
                reference_h,
                reference_w
            ):

                raise ValueError(
                    f"Mask dimension mismatch "
                    f"for {subset_id}"
                )

        stacked = np.stack(
            available,
            axis=0
        )

        consensus = (
            stacked.mean(
                axis=0
            )
            >= 0.5
        )

        consensus_slices.append(
            consensus
        )

    mask_lookup[
        subset_id
    ] = np.stack(
        consensus_slices,
        axis=0
    )

    if counter % 25 == 0:

        print(
            f"Consensus masks: "
            f"{counter} / 325"
        )

print(
    f"Consensus masks created: "
    f"{len(mask_lookup)} / 325"
)

# =============================================================================
# 12. BUILD SLICE TABLE
# =============================================================================

slice_records = []

for _, row in manifest.iterrows():

    subset_id = row[
        "subset_nodule_id"
    ]

    patient_id = row[
        "patient_id"
    ]

    split = row[
        "split"
    ]

    depth = len(
        original_lookup[
            subset_id
        ]
    )

    current_depth = len(
        current_lookup[
            subset_id
        ]
    )

    enhanced_depth = len(
        enhanced_lookup[
            subset_id
        ]
    )

    mask_depth = (
        mask_lookup[
            subset_id
        ].shape[0]
    )

    if not (
        depth
        ==
        current_depth
        ==
        enhanced_depth
        ==
        mask_depth
    ):

        raise ValueError(
            f"Depth mismatch for "
            f"{subset_id}: "
            f"original={depth}, "
            f"current={current_depth}, "
            f"enhanced={enhanced_depth}, "
            f"mask={mask_depth}"
        )

    for z in range(depth):

        slice_records.append({

            "subset_nodule_id":
                subset_id,

            "patient_id":
                patient_id,

            "split":
                split,

            "slice_index":
                z

        })

slice_df = pd.DataFrame(
    slice_records
)

print("\nSLICE DATA")
print("-" * 80)

print(
    f"Total slices: "
    f"{len(slice_df)}"
)

# =============================================================================
# 13. DATASET CLASS
# =============================================================================

class ThreeWayDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        condition
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.condition = condition

    def __len__(self):

        return len(
            self.df
        )

    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[
            index
        ]

        subset_id = str(
            row[
                "subset_nodule_id"
            ]
        )

        z = int(
            row[
                "slice_index"
            ]
        )

        # -------------------------------------------------------------
        # Load selected image source.
        # -------------------------------------------------------------

        if self.condition == "original":

            zip_path = (
                original_lookup[
                    subset_id
                ][z]
            )

            with zip_file.open(
                zip_path
            ) as f:

                data = f.read()

            with Image.open(
                io.BytesIO(data)
            ) as img:

                image = np.asarray(
                    img.convert("L"),
                    dtype=np.float32
                )

        elif self.condition == "current_processed":

            path = (
                current_lookup[
                    subset_id
                ][z]
            )

            with Image.open(
                path
            ) as img:

                image = np.asarray(
                    img.convert("L"),
                    dtype=np.float32
                )

        elif self.condition == "enhanced_processed":

            path = (
                enhanced_lookup[
                    subset_id
                ][z]
            )

            with Image.open(
                path
            ) as img:

                image = np.asarray(
                    img.convert("L"),
                    dtype=np.float32
                )

        else:

            raise ValueError(
                f"Unknown condition: "
                f"{self.condition}"
            )

        # -------------------------------------------------------------
        # IDENTICAL NORMALIZATION FOR ALL CONDITIONS
        # -------------------------------------------------------------
        #
        # This is intentionally performed at load time.
        #
        # This makes the benchmark fair: the model sees values in [0,1]
        # under the same normalization operation for all three conditions.
        # -------------------------------------------------------------

        image_min = float(
            image.min()
        )

        image_max = float(
            image.max()
        )

        if image_max > image_min:

            image = (
                image - image_min
            ) / (
                image_max - image_min
            )

        else:

            image = np.zeros_like(
                image,
                dtype=np.float32
            )

        # -------------------------------------------------------------
        # Consensus mask
        # -------------------------------------------------------------

        mask = (
            mask_lookup[
                subset_id
            ][z]
            .astype(np.float32)
        )

        image_tensor = torch.from_numpy(
            image.copy()
        ).unsqueeze(0).float()

        mask_tensor = torch.from_numpy(
            mask.copy()
        ).unsqueeze(0).float()

        return (
            image_tensor,
            mask_tensor
        )


# =============================================================================
# 14. BUILD LOADERS
# =============================================================================

train_slices = slice_df[
    slice_df["split"] == "train"
].copy()

val_slices = slice_df[
    slice_df["split"] == "val"
].copy()

test_slices = slice_df[
    slice_df["split"] == "test"
].copy()


def build_loaders(
    condition
):

    train_ds = ThreeWayDataset(
        train_slices,
        condition
    )

    val_ds = ThreeWayDataset(
        val_slices,
        condition
    )

    test_ds = ThreeWayDataset(
        test_slices,
        condition
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    return (
        train_loader,
        val_loader,
        test_loader
    )


# =============================================================================
# 15. U-NET
# =============================================================================

class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )

    def forward(self, x):

        return self.block(x)


class UNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.enc1 = DoubleConv(
            1,
            32
        )

        self.enc2 = DoubleConv(
            32,
            64
        )

        self.enc3 = DoubleConv(
            64,
            128
        )

        self.pool = nn.MaxPool2d(
            2
        )

        self.bottleneck = DoubleConv(
            128,
            256
        )

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            2,
            2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            2,
            2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            2,
            2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        self.out = nn.Conv2d(
            32,
            1,
            1
        )

    def forward(self, x):

        e1 = self.enc1(
            x
        )

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        b = self.bottleneck(
            self.pool(e3)
        )

        d3 = self.up3(
            b
        )

        d3 = torch.cat(
            [
                d3,
                e3
            ],
            dim=1
        )

        d3 = self.dec3(
            d3
        )

        d2 = self.up2(
            d3
        )

        d2 = torch.cat(
            [
                d2,
                e2
            ],
            dim=1
        )

        d2 = self.dec2(
            d2
        )

        d1 = self.up1(
            d2
        )

        d1 = torch.cat(
            [
                d1,
                e1
            ],
            dim=1
        )

        d1 = self.dec1(
            d1
        )

        return self.out(
            d1
        )


# =============================================================================
# 16. LOSS / METRICS
# =============================================================================

def combined_loss(
    logits,
    target
):

    probability = torch.sigmoid(
        logits
    )

    intersection = (
        probability * target
    ).sum(
        dim=(1, 2, 3)
    )

    denominator = (
        probability.sum(
            dim=(1, 2, 3)
        )
        +
        target.sum(
            dim=(1, 2, 3)
        )
    )

    soft_dice = (
        1.0
        -
        (
            2.0 * intersection
            +
            1e-6
        )
        /
        (
            denominator
            +
            1e-6
        )
    ).mean()

    bce = nn.functional.binary_cross_entropy_with_logits(
        logits,
        target
    )

    return (
        bce
        +
        soft_dice
    )


def hard_dice(
    logits,
    target
):

    prediction = (
        torch.sigmoid(
            logits
        )
        >= 0.5
    ).float()

    intersection = (
        prediction * target
    ).sum(
        dim=(1, 2, 3)
    )

    denominator = (
        prediction.sum(
            dim=(1, 2, 3)
        )
        +
        target.sum(
            dim=(1, 2, 3)
        )
    )

    dice = (
        2.0 * intersection
        +
        1e-6
    ) / (
        denominator
        +
        1e-6
    )

    return dice.mean()


def hard_iou(
    logits,
    target
):

    prediction = (
        torch.sigmoid(
            logits
        )
        >= 0.5
    )

    target_bool = (
        target > 0.5
    )

    intersection = (
        prediction
        &
        target_bool
    ).sum(
        dim=(1, 2, 3)
    )

    union = (
        prediction
        |
        target_bool
    ).sum(
        dim=(1, 2, 3)
    )

    iou = (
        intersection
        +
        1e-6
    ) / (
        union
        +
        1e-6
    )

    return iou.mean()


# =============================================================================
# 17. TRAIN ONE CONDITION
# =============================================================================

def run_condition(
    condition,
    model_output
):

    print("\n" + "=" * 80)
    print(
        f"TRAINING: {condition.upper()}"
    )
    print("=" * 80)

    # -------------------------------------------------------------
    # Reset random state so every experiment starts identically.
    # -------------------------------------------------------------

    set_seed(
        SEED
    )

    (
        train_loader,
        val_loader,
        test_loader
    ) = build_loaders(
        condition
    )

    model = UNet().to(
        DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    best_val_dice = -1.0
    best_epoch = 0

    history = []

    start_time = time.time()

    for epoch in range(
        1,
        EPOCHS + 1
    ):

        # =========================================================
        # TRAIN
        # =========================================================

        model.train()

        train_loss_total = 0.0
        train_dice_total = 0.0
        train_batches = 0

        for images, masks in train_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                images
            )

            loss = combined_loss(
                logits,
                masks
            )

            loss.backward()

            optimizer.step()

            train_loss_total += (
                loss.item()
            )

            train_dice_total += (
                hard_dice(
                    logits,
                    masks
                ).item()
            )

            train_batches += 1

        train_loss = (
            train_loss_total
            /
            max(
                train_batches,
                1
            )
        )

        train_dice = (
            train_dice_total
            /
            max(
                train_batches,
                1
            )
        )

        # =========================================================
        # VALIDATION
        # =========================================================

        model.eval()

        val_loss_total = 0.0
        val_dice_total = 0.0
        val_iou_total = 0.0
        val_batches = 0

        with torch.no_grad():

            for images, masks in val_loader:

                images = images.to(
                    DEVICE,
                    non_blocking=True
                )

                masks = masks.to(
                    DEVICE,
                    non_blocking=True
                )

                logits = model(
                    images
                )

                loss = combined_loss(
                    logits,
                    masks
                )

                val_loss_total += (
                    loss.item()
                )

                val_dice_total += (
                    hard_dice(
                        logits,
                        masks
                    ).item()
                )

                val_iou_total += (
                    hard_iou(
                        logits,
                        masks
                    ).item()
                )

                val_batches += 1

        val_loss = (
            val_loss_total
            /
            max(
                val_batches,
                1
            )
        )

        val_dice = (
            val_dice_total
            /
            max(
                val_batches,
                1
            )
        )

        val_iou = (
            val_iou_total
            /
            max(
                val_batches,
                1
            )
        )

        history.append({

            "condition":
                condition,

            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "train_dice":
                train_dice,

            "val_loss":
                val_loss,

            "val_dice":
                val_dice,

            "val_iou":
                val_iou

        })

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Dice: {train_dice:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Dice: {val_dice:.4f} | "
            f"Val IoU: {val_iou:.4f}"
        )

        # ---------------------------------------------------------
        # Save best model.
        # ---------------------------------------------------------

        if val_dice > best_val_dice:

            best_val_dice = val_dice
            best_epoch = epoch

            torch.save(
                {
                    "model_state_dict":
                        model.state_dict(),

                    "condition":
                        condition,

                    "best_val_dice":
                        best_val_dice,

                    "best_epoch":
                        best_epoch,

                    "seed":
                        SEED

                },
                model_output
            )

            print(
                f"  ✓ Best model saved "
                f"(val Dice={best_val_dice:.4f})"
            )

    # =============================================================
    # TEST BEST MODEL
    # =============================================================

    checkpoint = torch.load(
        model_output,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model.eval()

    test_loss_total = 0.0
    test_dice_total = 0.0
    test_iou_total = 0.0
    test_batches = 0

    with torch.no_grad():

        for images, masks in test_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            logits = model(
                images
            )

            loss = combined_loss(
                logits,
                masks
            )

            test_loss_total += (
                loss.item()
            )

            test_dice_total += (
                hard_dice(
                    logits,
                    masks
                ).item()
            )

            test_iou_total += (
                hard_iou(
                    logits,
                    masks
                ).item()
            )

            test_batches += 1

    test_loss = (
        test_loss_total
        /
        max(
            test_batches,
            1
        )
    )

    test_dice = (
        test_dice_total
        /
        max(
            test_batches,
            1
        )
    )

    test_iou = (
        test_iou_total
        /
        max(
            test_batches,
            1
        )
    )

    elapsed = (
        time.time()
        -
        start_time
    )

    result = {

        "condition":
            condition,

        "best_val_dice":
            best_val_dice,

        "best_epoch":
            best_epoch,

        "test_loss":
            test_loss,

        "test_dice":
            test_dice,

        "test_iou":
            test_iou,

        "training_time_seconds":
            elapsed,

        "model_path":
            model_output

    }

    print("\nTEST RESULTS")

    print(
        f"Best validation Dice: "
        f"{best_val_dice:.4f}"
    )

    print(
        f"Best epoch: "
        f"{best_epoch}"
    )

    print(
        f"Test loss: "
        f"{test_loss:.4f}"
    )

    print(
        f"Test Dice: "
        f"{test_dice:.4f}"
    )

    print(
        f"Test IoU: "
        f"{test_iou:.4f}"
    )

    print(
        f"Training time: "
        f"{elapsed / 60:.2f} minutes"
    )

    return result, history


# =============================================================================
# 18. RUN ALL THREE CONDITIONS
# =============================================================================

conditions = [

    (
        "original",
        os.path.join(
            OUTPUT_DIR,
            "original_best.pt"
        )
    ),

    (
        "current_processed",
        os.path.join(
            OUTPUT_DIR,
            "current_processed_best.pt"
        )
    ),

    (
        "enhanced_processed",
        os.path.join(
            OUTPUT_DIR,
            "enhanced_processed_best.pt"
        )
    )

]

all_results = []
all_history = []

for condition, model_path in conditions:

    result, history = run_condition(
        condition,
        model_path
    )

    all_results.append(
        result
    )

    all_history.extend(
        history
    )

# =============================================================================
# 19. SAVE RESULTS
# =============================================================================

results_df = pd.DataFrame(
    all_results
)

history_df = pd.DataFrame(
    all_history
)

results_df.to_csv(
    RESULTS_CSV,
    index=False
)

history_df.to_csv(
    HISTORY_CSV,
    index=False
)

# =============================================================================
# 20. COMPARE CONDITIONS
# =============================================================================

original_row = results_df[
    results_df["condition"]
    == "original"
].iloc[0]

current_row = results_df[
    results_df["condition"]
    == "current_processed"
].iloc[0]

enhanced_row = results_df[
    results_df["condition"]
    == "enhanced_processed"
].iloc[0]

original_dice = float(
    original_row[
        "test_dice"
    ]
)

current_dice = float(
    current_row[
        "test_dice"
    ]
)

enhanced_dice = float(
    enhanced_row[
        "test_dice"
    ]
)

original_iou = float(
    original_row[
        "test_iou"
    ]
)

current_iou = float(
    current_row[
        "test_iou"
    ]
)

enhanced_iou = float(
    enhanced_row[
        "test_iou"
    ]
)

current_difference = (
    current_dice
    -
    original_dice
)

enhanced_difference = (
    enhanced_dice
    -
    original_dice
)

enhanced_vs_current = (
    enhanced_dice
    -
    current_dice
)

# =============================================================================
# 21. IDENTIFY BEST CONDITION
# =============================================================================

best_condition = (
    results_df
    .sort_values(
        "test_dice",
        ascending=False
    )
    .iloc[0]
)

# =============================================================================
# 22. FINAL REPORT
# =============================================================================

summary = f"""
===============================================================================
STEP 37 — CONTROLLED 3-WAY DATASET BENCHMARK
===============================================================================

EXPERIMENT
-------------------------------------------------------------------------------

Conditions:
    1. Original
    2. Current processed
    3. Enhanced processed

Cohort:
    325 nodules
    {manifest['patient_id'].nunique()} patients
    {manifest['SeriesInstanceUID'].nunique() if 'SeriesInstanceUID' in manifest.columns else 247} CT series

Split:
    Train:       {len(train_slices)} slices
    Validation:  {len(val_slices)} slices
    Test:        {len(test_slices)} slices

Training:
    Model:          2D U-Net
    Epochs:         {EPOCHS}
    Batch size:     {BATCH_SIZE}
    Learning rate:  {LEARNING_RATE}
    Seed:           {SEED}
    Device:         {DEVICE}

CONTROL
-------------------------------------------------------------------------------

The following were held constant:

    Same 325-nodule cohort
    Same patient-level split
    Same consensus masks
    Same U-Net architecture
    Same optimizer
    Same learning rate
    Same batch size
    Same number of epochs
    Same random seed

Only the image source changed.

RESULTS
-------------------------------------------------------------------------------

ORIGINAL
    Best validation Dice: {original_row['best_val_dice']:.4f}
    Test Dice:             {original_dice:.4f}
    Test IoU:              {original_iou:.4f}

CURRENT PROCESSED
    Best validation Dice: {current_row['best_val_dice']:.4f}
    Test Dice:             {current_dice:.4f}
    Test IoU:              {current_iou:.4f}

ENHANCED PROCESSED
    Best validation Dice: {enhanced_row['best_val_dice']:.4f}
    Test Dice:             {enhanced_dice:.4f}
    Test IoU:              {enhanced_iou:.4f}

DIFFERENCES
-------------------------------------------------------------------------------

Current processed - Original Dice:
    {current_difference:+.4f}

Enhanced processed - Original Dice:
    {enhanced_difference:+.4f}

Enhanced processed - Current processed Dice:
    {enhanced_vs_current:+.4f}

BEST CONDITION BY TEST DICE
-------------------------------------------------------------------------------

    {best_condition['condition']}

    Test Dice:
        {best_condition['test_dice']:.4f}

INTERPRETATION
-------------------------------------------------------------------------------

This is a controlled benchmark of image preprocessing conditions.

The current processed dataset should be interpreted primarily as an
organization/QC/metadata-preserving version of the original.

The enhanced processed dataset applies additional intensity and contrast
preprocessing.

A higher test Dice for the enhanced dataset would provide evidence that the
chosen image preprocessing is beneficial under this fixed segmentation
protocol.

A similar or lower test Dice would indicate that the additional enhancement
did not improve this baseline segmentation task.

The benchmark does not establish clinical benefit.

PROVENANCE
-------------------------------------------------------------------------------

Native-to-specific-XML physical nodule identity:
    NOT ESTABLISHED for the full cohort.

Native malignancy labels:
    NONE.

OUTPUTS
-------------------------------------------------------------------------------

Results:
    {RESULTS_CSV}

Training history:
    {HISTORY_CSV}

Models:
    {OUTPUT_DIR}/original_best.pt
    {OUTPUT_DIR}/current_processed_best.pt
    {OUTPUT_DIR}/enhanced_processed_best.pt

Summary:
    {SUMMARY_TXT}

===============================================================================
"""

with open(
    SUMMARY_TXT,
    "w"
) as f:

    f.write(
        summary.strip()
    )

print(
    "\n" + summary
)

# =============================================================================
# 23. FINAL VALIDATION
# =============================================================================

assert len(results_df) == 3

assert set(
    results_df[
        "condition"
    ]
) == {
    "original",
    "current_processed",
    "enhanced_processed"
}

assert (
    results_df[
        "test_dice"
    ].notna().all()
)

print("=" * 80)
print("STEP 37 COMPLETE")
print("=" * 80)

zip_file.close()

STEP 37 — CONTROLLED 3-WAY DATASET BENCHMARK

Device: cpu


FileNotFoundError: Required input missing: /content/kagl_lidc_idri.zip